<a href="https://colab.research.google.com/github/yudhiardiyanto/TUGT-CONTEXTAWARE-CNN-BiLSTM/blob/main/Copy_of_context_skeleton_split_XYZ_midhip_shoulder3D_fisiologisdata_MODEL_18_7_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XYZ Migration Notes
Notebook ini sudah disesuaikan untuk preprocessing skeleton 3D dengan pipeline berikut:

**Raw Skeleton → Mid-Hip Centering → 3D Shoulder Width Normalization**

Perubahan utama:
1. Seluruh landmark menggunakan koordinat `x`, `y`, dan `z`.
2. Jumlah fitur skeleton menjadi `22 landmark × 3 axis = 66 fitur`.
3. Titik pusat tubuh dihitung dari rata-rata `left_hip` dan `right_hip` pada sumbu `x`, `y`, dan `z`.
4. Normalisasi skala tubuh menggunakan jarak 3D antara `left_shoulder` dan `right_shoulder`.
5. Input model skeleton otomatis mengikuti bentuk data hasil preprocessing: `(WINDOW_SIZE, 66)`.


In [ ]:
# ============================================================
# 0. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
# ============================================================
# 1. LOAD DATASET: SKELETON + CONTEXT
# ============================================================

import os
import pandas as pd

# ============================================================
# PATH DATASET
# ============================================================

DATASET_PATH = "/content/drive/MyDrive/DATASET-TUG-CONTEXT"

skeleton_list = []
context_list = []

# ============================================================
# LOAD SUBJECT01 - SUBJECT30
# ============================================================

for i in range(1, 31):

    subject_folder = f"Subject{i:02d}"   # Subject01
    subject_label  = f"subject{i:02d}"   # subject01

    # --------------------------------------------------------
    # Path Skeleton
    # --------------------------------------------------------
    skeleton_file = os.path.join(
        DATASET_PATH,
        subject_folder,
        "Skeleton",
        "all_skeleton_data.csv"
    )

    # --------------------------------------------------------
    # Path Context
    # --------------------------------------------------------
    context_file = os.path.join(
        DATASET_PATH,
        subject_folder,
        "Context",
        f"context{i:02d}.csv"
    )

    # ========================================================
    # LOAD SKELETON
    # ========================================================

    if os.path.exists(skeleton_file):

        df_skel = pd.read_csv(skeleton_file)

        df_skel.columns = (
            df_skel.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        # Pastikan subject mengacu pada folder utama
        df_skel["subject"] = subject_label

        skeleton_list.append(df_skel)

    else:
        print(f"{subject_folder}: file skeleton tidak ditemukan")

    # ========================================================
    # LOAD CONTEXT
    # ========================================================

    if os.path.exists(context_file):

        df_ctx = pd.read_csv(context_file)

        df_ctx.columns = (
            df_ctx.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        # File context belum punya subject,
        # maka subject diambil dari nama folder
        df_ctx["subject"] = subject_label

        context_list.append(df_ctx)

    else:
        print(f"{subject_folder}: file context tidak ditemukan")


# ============================================================
# GABUNGKAN SEMUA DATA
# ============================================================

skeleton_df = pd.concat(skeleton_list, ignore_index=True)
context_df  = pd.concat(context_list, ignore_index=True)

# ============================================================
# CEK DATA
# ============================================================

print("=" * 60)
print("DATASET BERHASIL DIMUAT")
print("=" * 60)

print("Skeleton shape :", skeleton_df.shape)
print("Context shape  :", context_df.shape)

print("\nJumlah subject skeleton:", skeleton_df["subject"].nunique())
print("Jumlah subject context :", context_df["subject"].nunique())

print("\nActivity:")
print(skeleton_df["activity"].unique())

print("\nSkeleton columns:")
print(skeleton_df.columns.tolist()[:15])

print("\nContext columns:")
print(context_df.columns.tolist())

display(skeleton_df.head())
display(context_df.head())

In [ ]:
# ============================================================
# 2. KONFIGURASI
# ============================================================

# Video & windowing
FPS = 30
WINDOW_SIZE = 30
STEP_SIZE = 15

# Split data berbasis subject
TEST_SIZE = 0.20
VAL_SIZE = 0.10
RANDOM_STATE = 42

# Training
EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 0.0001

# Kolom label
subject_col = "subject"
activity_col = "activity"

# Context features
context_cols = [
    "temperature",
    "humidity",
    "light_intensity",
    "pressure"
]

# Skeleton landmarks yang digunakan: 22 body landmarks
selected_landmarks = [
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "left_pinky", "right_pinky",
    "left_index", "right_index",
    "left_thumb", "right_thumb",
    "left_hip", "right_hip",
    "left_knee", "right_knee",
    "left_ankle", "right_ankle",
    "left_heel", "right_heel",
    "left_foot_index", "right_foot_index"
]

# Skeleton axis 3D
skeleton_axes = ["x", "y", "z"]

# Total fitur skeleton = 22 landmark × 3 axis = 66 fitur
skeleton_cols = [
    f"{lm}_{axis}"
    for lm in selected_landmarks
    for axis in skeleton_axes
]

print("Jumlah landmark:", len(selected_landmarks))
print("Axis skeleton:", skeleton_axes)
print("Jumlah fitur skeleton:", len(skeleton_cols))
print("Jumlah fitur context:", len(context_cols))


In [ ]:
# ============================================================
# 3. PREPROCESSING CONTEXT
#    Interpolation + Mean Sensor per Subject
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Cek kolom context
# ------------------------------------------------------------
missing_context_cols = [
    col for col in context_cols
    if col not in context_df.columns
]

if missing_context_cols:
    raise ValueError(f"Kolom context tidak ditemukan: {missing_context_cols}")

# ------------------------------------------------------------
# Pastikan data context numerik
# ------------------------------------------------------------
for col in context_cols:
    context_df[col] = pd.to_numeric(
        context_df[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# Interpolasi missing value per subject
# ------------------------------------------------------------
context_df[context_cols] = (
    context_df
    .groupby(subject_col)[context_cols]
    .transform(lambda x: x.interpolate(method="linear").ffill().bfill())
)

# ------------------------------------------------------------
# Ambil rata-rata sensor per subject
# ------------------------------------------------------------
context_mean = (
    context_df
    .groupby(subject_col)[context_cols]
    .mean()
    .reset_index()
)

context_mean[context_cols] = context_mean[context_cols].round(4)

print("Preprocessing context selesai")
print("Shape context_mean:", context_mean.shape)

display(context_mean.head())

In [ ]:
# ============================================================
# 4. PREPROCESSING SKELETON
#    Raw Skeleton -> Mid-Hip Centering -> 3D Shoulder Width Normalization
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Cek kolom skeleton 3D
# ------------------------------------------------------------
missing_skeleton_cols = [
    col for col in skeleton_cols
    if col not in skeleton_df.columns
]

if missing_skeleton_cols:
    raise ValueError(
        "Kolom skeleton 3D tidak ditemukan. "
        "Pastikan extractor MediaPipe menyimpan koordinat x, y, dan z untuk semua landmark. "
        f"Kolom yang hilang: {missing_skeleton_cols}"
    )

# ------------------------------------------------------------
# Pastikan data skeleton numerik
# ------------------------------------------------------------
for col in skeleton_cols:
    skeleton_df[col] = pd.to_numeric(
        skeleton_df[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# Interpolasi missing value per subject dan activity
# ------------------------------------------------------------
skeleton_df[skeleton_cols] = (
    skeleton_df
    .groupby([subject_col, activity_col])[skeleton_cols]
    .transform(lambda x: x.interpolate(method="linear").ffill().bfill())
)

# ------------------------------------------------------------
# Simpan salinan raw setelah interpolasi untuk audit/debugging
# ------------------------------------------------------------
skeleton_raw_interpolated = skeleton_df[[subject_col, activity_col] + skeleton_cols].copy()

# ============================================================
# STEP 1. MID-HIP CENTERING 3D
# Titik pusat tubuh = rata-rata left_hip dan right_hip pada x, y, z
# ============================================================

mid_hip = {}

for axis in skeleton_axes:
    mid_hip[axis] = (
        skeleton_df[f"left_hip_{axis}"] +
        skeleton_df[f"right_hip_{axis}"]
    ) / 2.0

for lm in selected_landmarks:
    for axis in skeleton_axes:
        skeleton_df[f"{lm}_{axis}"] = (
            skeleton_df[f"{lm}_{axis}"] - mid_hip[axis]
        )

skeleton_midhip_centered = skeleton_df[[subject_col, activity_col] + skeleton_cols].copy()

# ============================================================
# STEP 2. 3D SHOULDER WIDTH NORMALIZATION
# Shoulder width 3D = Euclidean distance antara left_shoulder dan right_shoulder
# pada sumbu x, y, dan z setelah mid-hip centering.
# ============================================================

shoulder_width_3d = np.sqrt(
    (skeleton_df["left_shoulder_x"] - skeleton_df["right_shoulder_x"]) ** 2 +
    (skeleton_df["left_shoulder_y"] - skeleton_df["right_shoulder_y"]) ** 2 +
    (skeleton_df["left_shoulder_z"] - skeleton_df["right_shoulder_z"]) ** 2
)

# Hindari pembagian dengan nol atau nilai tidak valid
shoulder_width_3d = shoulder_width_3d.replace([0, np.inf, -np.inf], np.nan)
shoulder_width_median = shoulder_width_3d.median()

if pd.isna(shoulder_width_median) or shoulder_width_median == 0:
    shoulder_width_median = 1.0

shoulder_width_3d = shoulder_width_3d.fillna(shoulder_width_median)

for lm in selected_landmarks:
    for axis in skeleton_axes:
        skeleton_df[f"{lm}_{axis}"] = (
            skeleton_df[f"{lm}_{axis}"] / shoulder_width_3d
        )

# Simpan nilai shoulder width untuk audit preprocessing
skeleton_df["shoulder_width_3d"] = shoulder_width_3d

# ------------------------------------------------------------
# Ringkasan preprocessing
# ------------------------------------------------------------
print("Preprocessing skeleton selesai")
print("Pipeline: Raw -> Mid-Hip Centering -> 3D Shoulder Width Normalization")
print("Shape skeleton_df:", skeleton_df.shape)
print("Jumlah landmark:", len(selected_landmarks))
print("Jumlah fitur skeleton:", len(skeleton_cols))
print("Shoulder width 3D median:", shoulder_width_median)

# Validasi jumlah fitur harus 66 untuk 22 landmark 3D
assert len(skeleton_cols) == len(selected_landmarks) * len(skeleton_axes)

# Tampilkan hasil preprocessing
preview_cols = [
    subject_col,
    activity_col,
    "shoulder_width_3d"
] + skeleton_cols

display(skeleton_df[preview_cols].head())


In [ ]:
# ============================================================
# 5. WINDOWING SKELETON + CONTEXT ALIGNMENT
# ============================================================

import numpy as np

X_skeleton = []
X_context = []
y = []
groups = []

# ============================================================
# Windowing berdasarkan Subject dan Activity
# ============================================================

for subject in skeleton_df[subject_col].unique():

    subject_df = skeleton_df[
        skeleton_df[subject_col] == subject
    ]

    # ------------------------------------------
    # Ambil context subject tersebut
    # ------------------------------------------
    context_vector = context_mean[
        context_mean[subject_col] == subject
    ][context_cols].values

    if len(context_vector) == 0:
        continue

    context_vector = context_vector[0]

    # ------------------------------------------
    # Windowing setiap activity
    # ------------------------------------------
    for activity in subject_df[activity_col].unique():

        activity_df = (
            subject_df[
                subject_df[activity_col] == activity
            ]
            .reset_index(drop=True)
        )

        skeleton_values = activity_df[skeleton_cols].values

        if len(skeleton_values) < WINDOW_SIZE:
            continue

        for start in range(
            0,
            len(skeleton_values) - WINDOW_SIZE + 1,
            STEP_SIZE
        ):

            end = start + WINDOW_SIZE

            window = skeleton_values[start:end]

            X_skeleton.append(window)

            # Context sama untuk seluruh window
            X_context.append(context_vector)

            y.append(activity)

            groups.append(subject)

# ============================================================
# Konversi ke NumPy
# ============================================================

X_skeleton = np.array(
    X_skeleton,
    dtype=np.float32
)

X_context = np.array(
    X_context,
    dtype=np.float32
)

y = np.array(y)

groups = np.array(groups)

# ============================================================
# Ringkasan Dataset
# ============================================================

print("="*60)
print("WINDOWING SELESAI")
print("="*60)

print("X_skeleton :", X_skeleton.shape)
print("X_context  :", X_context.shape)
print("y          :", y.shape)
print("groups     :", groups.shape)

print("\nJumlah Subject :", len(np.unique(groups)))
print("Activity :", np.unique(y))

In [ ]:
# ============================================================
# 6. GROUP SHUFFLE SPLIT + SCALING + LABEL ENCODING
# ============================================================

import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# ============================================================
# 6.1 LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Label mapping:")
for idx, label in enumerate(label_encoder.classes_):
    print(idx, "=", label)

# ============================================================
# 6.2 SPLIT TRAIN+VAL DAN TEST BERDASARKAN SUBJECT
# ============================================================

gss_test = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_val_idx, test_idx = next(
    gss_test.split(
        X_skeleton,
        y_encoded,
        groups=groups
    )
)

# ============================================================
# 6.3 SPLIT TRAIN DAN VALIDATION BERDASARKAN SUBJECT
# ============================================================

gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=RANDOM_STATE
)

train_idx_local, val_idx_local = next(
    gss_val.split(
        X_skeleton[train_val_idx],
        y_encoded[train_val_idx],
        groups=groups[train_val_idx]
    )
)

train_idx = train_val_idx[train_idx_local]
val_idx = train_val_idx[val_idx_local]

# ============================================================
# 6.4 AMBIL DATA SPLIT
# ============================================================

X_train_skeleton = X_skeleton[train_idx]
X_train_context  = X_context[train_idx]
y_train          = y_encoded[train_idx]

X_val_skeleton = X_skeleton[val_idx]
X_val_context  = X_context[val_idx]
y_val          = y_encoded[val_idx]

X_test_skeleton = X_skeleton[test_idx]
X_test_context  = X_context[test_idx]
y_test          = y_encoded[test_idx]

groups_train = groups[train_idx]
groups_val   = groups[val_idx]
groups_test  = groups[test_idx]

# ============================================================
# 6.5 SCALING SKELETON
#     Fit hanya pada train, transform val dan test
# ============================================================

n_train, n_timesteps, n_features = X_train_skeleton.shape

skeleton_scaler = StandardScaler()

X_train_skeleton = skeleton_scaler.fit_transform(
    X_train_skeleton.reshape(-1, n_features)
).reshape(n_train, n_timesteps, n_features)

X_val_skeleton = skeleton_scaler.transform(
    X_val_skeleton.reshape(-1, n_features)
).reshape(X_val_skeleton.shape[0], n_timesteps, n_features)

X_test_skeleton = skeleton_scaler.transform(
    X_test_skeleton.reshape(-1, n_features)
).reshape(X_test_skeleton.shape[0], n_timesteps, n_features)

# ============================================================
# 6.6 SCALING CONTEXT
#     Fit hanya pada train, transform val dan test
# ============================================================

context_scaler = StandardScaler()

X_train_context = context_scaler.fit_transform(X_train_context)
X_val_context   = context_scaler.transform(X_val_context)
X_test_context  = context_scaler.transform(X_test_context)


# ============================================================
# 6.7 COMPUTE CLASS WEIGHT
#     Digunakan untuk mengatasi ketidakseimbangan kelas pada training
# ============================================================

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight = dict(zip(classes, weights))

print("\nClass weight berdasarkan data training:")
for class_id, weight in class_weight.items():
    print(f"{class_id} ({label_encoder.classes_[class_id]}) = {weight:.4f}")

# ============================================================
# 6.8 CEK HASIL SPLIT
# ============================================================

print("="*60)
print("SPLIT + SCALING SELESAI")
print("="*60)

print("Train Skeleton :", X_train_skeleton.shape)
print("Train Context  :", X_train_context.shape)
print("Train Label    :", y_train.shape)

print("\nVal Skeleton   :", X_val_skeleton.shape)
print("Val Context    :", X_val_context.shape)
print("Val Label      :", y_val.shape)

print("\nTest Skeleton  :", X_test_skeleton.shape)
print("Test Context   :", X_test_context.shape)
print("Test Label     :", y_test.shape)

print("\nTrain Subjects:", sorted(np.unique(groups_train)))
print("Val Subjects  :", sorted(np.unique(groups_val)))
print("Test Subjects :", sorted(np.unique(groups_test)))


In [ ]:
# ============================================================
# 7. BUILD MODEL CNN-BiLSTM + CONTEXT FUSION
# ============================================================

import tensorflow as tf

from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    Bidirectional, LSTM, Dense, Dropout, Concatenate
)

from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam


def build_model():

    # ========================================================
    # Skeleton Branch
    # Input: WINDOW_SIZE frame × 66 fitur skeleton 3D
    # ========================================================

    skeleton_input = Input(
        shape=(X_train_skeleton.shape[1], X_train_skeleton.shape[2]),
        name="skeleton_input"
    )

    x = Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(skeleton_input)

    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(
        filters=128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(x)

    x = BatchNormalization()(x)

    x = Bidirectional(
        LSTM(
            64,
            return_sequences=False,
            kernel_regularizer=l2(0.001)
        )
    )(x)

    x = Dropout(0.3)(x)

    skeleton_feature = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(0.001),
        name="skeleton_feature"
    )(x)

    # ========================================================
    # Context Branch
    # Input: 4 fitur context
    # ========================================================

    context_input = Input(
        shape=(X_train_context.shape[1],),
        name="context_input"
    )

    c = Dense(
        16,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(context_input)

    c = Dropout(0.2)(c)

    # ========================================================
    # Fusion
    # ========================================================

    fusion = Concatenate(name="feature_fusion")(
        [skeleton_feature, c]
    )

    z = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(fusion)

    z = Dropout(0.3)(z)

    z = Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(z)

    output = Dense(
        len(label_encoder.classes_),
        activation="softmax",
        name="phase_output"
    )(z)

    model = Model(
        inputs=[skeleton_input, context_input],
        outputs=output
    )

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


model = build_model()
model.summary()


In [ ]:
# ============================================================
# 8. TRAINING MODEL
#    CNN-BiLSTM + Context Fusion
# ============================================================

import os
import tensorflow as tf

# ============================================================
# Folder penyimpanan model
# ============================================================

MODEL_DIR = "/content/drive/MyDrive/DATASET-TUG-CONTEXT/Results/Final_Context_Model_18-7-2026"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_KERAS_PATH = os.path.join(
    MODEL_DIR,
    "phase_classifier.keras"
)

MODEL_H5_PATH = os.path.join(
    MODEL_DIR,
    "phase_classifier.h5"
)

# ============================================================
# Build model
# ============================================================

tf.keras.backend.clear_session()
model = build_model()

# ============================================================
# Callback
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=MODEL_KERAS_PATH,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

# ============================================================
# Training
# ============================================================

history = model.fit(
    [X_train_skeleton, X_train_context],
    y_train,
    validation_data=(
        [X_val_skeleton, X_val_context],
        y_val
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1
)

# ============================================================
# Simpan juga format .h5
# ============================================================

model.save(MODEL_H5_PATH)

print("="*60)
print("Training selesai")
print("Model terbaik disimpan di:")
print(MODEL_KERAS_PATH)
print("Model H5 disimpan di:")
print(MODEL_H5_PATH)


In [ ]:
# ============================================================
# 9. EVALUASI MODEL PADA TEST SET
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ============================================================
# Load model terbaik dari hasil ModelCheckpoint
# ============================================================

best_model = tf.keras.models.load_model(MODEL_KERAS_PATH)

# ============================================================
# Prediksi test set
# ============================================================

y_pred_prob = best_model.predict(
    [X_test_skeleton, X_test_context],
    verbose=0
)

y_pred = np.argmax(y_pred_prob, axis=1)

# ============================================================
# Hitung metrik
# ============================================================

test_accuracy = accuracy_score(y_test, y_pred)

test_precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("="*60)
print("TEST SET EVALUATION")
print("="*60)

print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1-score  : {test_f1:.4f}")

# ============================================================
# Classification Report
# ============================================================

report = classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    digits=4,
    zero_division=0
)

print("\nClassification Report:")
print(report)

# ============================================================
# Simpan hasil evaluasi
# ============================================================

evaluation_df = pd.DataFrame({
    "metric": ["accuracy", "precision_macro", "recall_macro", "f1_macro"],
    "value": [test_accuracy, test_precision, test_recall, test_f1]
})

evaluation_df.to_csv(
    os.path.join(MODEL_DIR, "test_evaluation.csv"),
    index=False
)

print("\nHasil evaluasi disimpan di:")
print(os.path.join(MODEL_DIR, "test_evaluation.csv"))

In [ ]:
# ============================================================
# 10. SIMPAN SCALER DAN LABEL ENCODER
# ============================================================

import os
import joblib

# Folder model final
MODEL_DIR = "/content/drive/MyDrive/DATASET-TUG-CONTEXT/Results/Final_Context_Model_18-7-2026"
os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Simpan Skeleton Scaler
# ------------------------------------------------------------
joblib.dump(
    skeleton_scaler,
    os.path.join(MODEL_DIR, "skeleton_scaler.pkl")
)

# ------------------------------------------------------------
# Simpan Context Scaler
# ------------------------------------------------------------
joblib.dump(
    context_scaler,
    os.path.join(MODEL_DIR, "context_scaler.pkl")
)

# ------------------------------------------------------------
# Simpan Label Encoder
# ------------------------------------------------------------
joblib.dump(
    label_encoder,
    os.path.join(MODEL_DIR, "label_encoder.pkl")
)

print("="*60)
print("Scaler dan label encoder berhasil disimpan")
print("="*60)
print("Skeleton Scaler:", os.path.join(MODEL_DIR, "skeleton_scaler.pkl"))
print("Context Scaler :", os.path.join(MODEL_DIR, "context_scaler.pkl"))
print("Label Encoder  :", os.path.join(MODEL_DIR, "label_encoder.pkl"))

In [ ]:
# ============================================================
# 11. CONFUSION MATRIX TEST SET
# ============================================================

import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

FIGURE_DIR = os.path.join(MODEL_DIR, "Figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

class_names = label_encoder.classes_

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    square=True
)

plt.title(
    "Confusion Matrix - Test Set",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURE_DIR, "confusion_matrix_test.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(FIGURE_DIR, "confusion_matrix_test.svg"),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 12. NORMALIZED CONFUSION MATRIX (%)
# ============================================================

import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

FIGURE_DIR = os.path.join(MODEL_DIR, "Figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

class_names = label_encoder.classes_

cm_norm = confusion_matrix(
    y_test,
    y_pred,
    normalize="true"
)

cm_percent = cm_norm * 100

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm_percent,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    square=True,
    vmin=0,
    vmax=100
)

plt.title(
    "Normalized Confusion Matrix (%) - Test Set",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

cbar = plt.gca().collections[0].colorbar
cbar.set_label("Percentage (%)")

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURE_DIR, "normalized_confusion_matrix_test.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(FIGURE_DIR, "normalized_confusion_matrix_test.svg"),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 13. CLASSIFICATION REPORT
# ============================================================

import os
import pandas as pd
from sklearn.metrics import classification_report

REPORT_DIR = os.path.join(MODEL_DIR, "Reports")
os.makedirs(REPORT_DIR, exist_ok=True)

report_dict = classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    output_dict=True,
    digits=4,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()

display(report_df)

report_path = os.path.join(
    REPORT_DIR,
    "classification_report_test.csv"
)

report_df.to_csv(report_path)

print("Classification report saved to:")
print(report_path)

In [ ]:
# ============================================================
# 14. TRAINING ACCURACY AND LOSS CURVE
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

FIGURE_DIR = os.path.join(MODEL_DIR, "Figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

history_df = pd.DataFrame(history.history)

history_csv = os.path.join(
    MODEL_DIR,
    "training_history.csv"
)

history_df.to_csv(history_csv, index=False)

print("Training history saved to:")
print(history_csv)

# Accuracy Curve
plt.figure(figsize=(8, 5))

plt.plot(
    history_df["accuracy"],
    label="Training Accuracy",
    linewidth=2
)

plt.plot(
    history_df["val_accuracy"],
    label="Validation Accuracy",
    linewidth=2
)

plt.title(
    "Training and Validation Accuracy",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURE_DIR, "training_validation_accuracy.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(FIGURE_DIR, "training_validation_accuracy.svg"),
    bbox_inches="tight"
)

plt.show()

# Loss Curve
plt.figure(figsize=(8, 5))

plt.plot(
    history_df["loss"],
    label="Training Loss",
    linewidth=2
)

plt.plot(
    history_df["val_loss"],
    label="Validation Loss",
    linewidth=2
)

plt.title(
    "Training and Validation Loss",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURE_DIR, "training_validation_loss.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(FIGURE_DIR, "training_validation_loss.svg"),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 15. ROC CURVE ONE-vs-REST
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

FIGURE_DIR = os.path.join(MODEL_DIR, "Figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

class_names = label_encoder.classes_
n_classes = len(class_names)

y_test_bin = label_binarize(
    y_test,
    classes=np.arange(n_classes)
)

plt.figure(figsize=(8, 6))

roc_results = []

for i, class_name in enumerate(class_names):

    fpr, tpr, _ = roc_curve(
        y_test_bin[:, i],
        y_pred_prob[:, i]
    )

    roc_auc = auc(fpr, tpr)

    roc_results.append({
        "class": class_name,
        "auc": roc_auc
    })

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{class_name} (AUC = {roc_auc:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1
)

plt.title(
    "ROC Curve One-vs-Rest - Test Set",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.grid(alpha=0.3)
plt.legend(loc="lower right")

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURE_DIR, "roc_curve_ovr_test.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(FIGURE_DIR, "roc_curve_ovr_test.svg"),
    bbox_inches="tight"
)

plt.show()

roc_df = pd.DataFrame(roc_results)
display(roc_df)

roc_df.to_csv(
    os.path.join(MODEL_DIR, "roc_auc_test.csv"),
    index=False
)

In [ ]:
# ============================================================
# 16. PRECISION-RECALL CURVE
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve, average_precision_score

FIGURE_DIR = os.path.join(MODEL_DIR, "Figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

class_names = label_encoder.classes_
n_classes = len(class_names)

y_test_bin = label_binarize(
    y_test,
    classes=np.arange(n_classes)
)

plt.figure(figsize=(8, 6))

pr_results = []

for i, class_name in enumerate(class_names):

    precision, recall, _ = precision_recall_curve(
        y_test_bin[:, i],
        y_pred_prob[:, i]
    )

    ap = average_precision_score(
        y_test_bin[:, i],
        y_pred_prob[:, i]
    )

    pr_results.append({
        "class": class_name,
        "average_precision": ap
    })

    plt.plot(
        recall,
        precision,
        linewidth=2,
        label=f"{class_name} (AP = {ap:.3f})"
    )

plt.title(
    "Precision-Recall Curve - Test Set",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(alpha=0.3)
plt.legend(loc="lower left")

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURE_DIR, "precision_recall_curve_test.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(FIGURE_DIR, "precision_recall_curve_test.svg"),
    bbox_inches="tight"
)

plt.show()

pr_df = pd.DataFrame(pr_results)
display(pr_df)

pr_df.to_csv(
    os.path.join(MODEL_DIR, "precision_recall_auc_test.csv"),
    index=False
)

In [ ]:
# ============================================================
# 17. PER-CLASS PERFORMANCE BAR CHART
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

FIGURE_DIR = os.path.join(MODEL_DIR, "Figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

class_report_df = report_df.loc[
    label_encoder.classes_,
    ["precision", "recall", "f1-score"]
]

display(class_report_df)

ax = class_report_df.plot(
    kind="bar",
    figsize=(9, 5),
    width=0.8
)

ax.set_title(
    "Per-Class Classification Performance",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("TUG Phase")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)

ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.xticks(rotation=0)

plt.legend(
    title="Metric",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURE_DIR, "per_class_performance.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(FIGURE_DIR, "per_class_performance.svg"),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 18. SAVE FINAL EVALUATION SUMMARY
# ============================================================

import os
import pandas as pd

SUMMARY_DIR = os.path.join(MODEL_DIR, "Summary")
os.makedirs(SUMMARY_DIR, exist_ok=True)

summary_final = pd.DataFrame({
    "metric": [
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro"
    ],
    "value": [
        test_accuracy,
        test_precision,
        test_recall,
        test_f1
    ]
})

summary_final.to_csv(
    os.path.join(SUMMARY_DIR, "final_test_summary.csv"),
    index=False
)

print("Final summary saved to:")
print(os.path.join(SUMMARY_DIR, "final_test_summary.csv"))

print("\nModel directory:")
print(MODEL_DIR)

In [ ]:
# ============================================================
# SKELETON ONLY - 18.A BUILD MODEL
# ============================================================

import tensorflow as tf

from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    Bidirectional, LSTM, Dense, Dropout
)

from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam


def build_model_skeleton_only():

    skeleton_input = Input(
        shape=(X_train_skeleton.shape[1], X_train_skeleton.shape[2]),
        name="skeleton_input"
    )

    x = Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(skeleton_input)

    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(
        filters=128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(x)

    x = BatchNormalization()(x)

    x = Bidirectional(
        LSTM(
            64,
            return_sequences=False,
            kernel_regularizer=l2(0.001)
        )
    )(x)

    x = Dropout(0.3)(x)

    x = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(x)

    x = Dropout(0.3)(x)

    x = Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(x)

    output = Dense(
        len(label_encoder.classes_),
        activation="softmax",
        name="phase_output"
    )(x)

    model = Model(
        inputs=skeleton_input,
        outputs=output
    )

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


skeleton_only_model = build_model_skeleton_only()
skeleton_only_model.summary()

In [ ]:
# ============================================================
# SKELETON ONLY - 18.B TRAINING MODEL
# ============================================================

import os
import tensorflow as tf

SKELETON_ONLY_DIR = "/content/drive/MyDrive/DATASET-TUG-CONTEXT/Results/Final_Context_Model_18-7-2026"
os.makedirs(SKELETON_ONLY_DIR, exist_ok=True)

SKELETON_ONLY_KERAS_PATH = os.path.join(
    SKELETON_ONLY_DIR,
    "skeleton_only_classifier.keras"
)

SKELETON_ONLY_H5_PATH = os.path.join(
    SKELETON_ONLY_DIR,
    "skeleton_only_classifier.h5"
)

tf.keras.backend.clear_session()
skeleton_only_model = build_model_skeleton_only()

callbacks_skeleton_only = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=SKELETON_ONLY_KERAS_PATH,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

history_skeleton_only = skeleton_only_model.fit(
    X_train_skeleton,
    y_train,
    validation_data=(X_val_skeleton, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_skeleton_only,
    class_weight=class_weight,
    verbose=1
)

skeleton_only_model.save(SKELETON_ONLY_H5_PATH)

print("Skeleton-only training selesai.")
print("Model .keras:", SKELETON_ONLY_KERAS_PATH)
print("Model .h5   :", SKELETON_ONLY_H5_PATH)


In [ ]:
# ============================================================
# SKELETON ONLY - 18.C TEST EVALUATION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

best_skeleton_only_model = tf.keras.models.load_model(
    SKELETON_ONLY_KERAS_PATH
)

y_pred_prob_skeleton_only = best_skeleton_only_model.predict(
    X_test_skeleton,
    verbose=0
)

y_pred_skeleton_only = np.argmax(
    y_pred_prob_skeleton_only,
    axis=1
)

skeleton_only_accuracy = accuracy_score(
    y_test,
    y_pred_skeleton_only
)

skeleton_only_precision = precision_score(
    y_test,
    y_pred_skeleton_only,
    average="macro",
    zero_division=0
)

skeleton_only_recall = recall_score(
    y_test,
    y_pred_skeleton_only,
    average="macro",
    zero_division=0
)

skeleton_only_f1 = f1_score(
    y_test,
    y_pred_skeleton_only,
    average="macro",
    zero_division=0
)

print("="*60)
print("SKELETON ONLY TEST EVALUATION")
print("="*60)
print(f"Accuracy  : {skeleton_only_accuracy:.4f}")
print(f"Precision : {skeleton_only_precision:.4f}")
print(f"Recall    : {skeleton_only_recall:.4f}")
print(f"F1-score  : {skeleton_only_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_skeleton_only,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

skeleton_only_eval_df = pd.DataFrame({
    "metric": [
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro"
    ],
    "value": [
        skeleton_only_accuracy,
        skeleton_only_precision,
        skeleton_only_recall,
        skeleton_only_f1
    ]
})

skeleton_only_eval_df.to_csv(
    os.path.join(SKELETON_ONLY_DIR, "skeleton_only_test_evaluation.csv"),
    index=False
)

display(skeleton_only_eval_df)

In [ ]:
# ============================================================
# SKELETON ONLY - 18.D ACCURACY AND LOSS CURVE
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

FIGURE_DIR = os.path.join(
    SKELETON_ONLY_DIR,
    "Figures"
)

os.makedirs(FIGURE_DIR, exist_ok=True)

history_skeleton_only_df = pd.DataFrame(
    history_skeleton_only.history
)

history_skeleton_only_df.to_csv(
    os.path.join(
        SKELETON_ONLY_DIR,
        "skeleton_only_training_history.csv"
    ),
    index=False
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------
axes[0].plot(
    history_skeleton_only_df["accuracy"],
    label="Training Accuracy",
    linewidth=2
)

axes[0].plot(
    history_skeleton_only_df["val_accuracy"],
    label="Validation Accuracy",
    linewidth=2
)

axes[0].set_title(
    "Skeleton Only - Accuracy",
    fontsize=14,
    fontweight="bold"
)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1.05)
axes[0].grid(alpha=0.3)
axes[0].legend()

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------
axes[1].plot(
    history_skeleton_only_df["loss"],
    label="Training Loss",
    linewidth=2
)

axes[1].plot(
    history_skeleton_only_df["val_loss"],
    label="Validation Loss",
    linewidth=2
)

axes[1].set_title(
    "Skeleton Only - Loss",
    fontsize=14,
    fontweight="bold"
)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.suptitle(
    "Skeleton-Only Model Training Performance",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "skeleton_only_accuracy_loss_curve.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "skeleton_only_accuracy_loss_curve.svg"
    ),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# SKELETON ONLY - 8.E CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

class_names = label_encoder.classes_

cm_skeleton_only = confusion_matrix(
    y_test,
    y_pred_skeleton_only
)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm_skeleton_only,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    square=True
)

plt.title(
    "Skeleton Only - Confusion Matrix",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "skeleton_only_confusion_matrix.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "skeleton_only_confusion_matrix.svg"
    ),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# SKELETON ONLY - 8.F NORMALIZED CONFUSION MATRIX (%)
# ============================================================

cm_skeleton_only_norm = confusion_matrix(
    y_test,
    y_pred_skeleton_only,
    normalize="true"
) * 100

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm_skeleton_only_norm,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    square=True,
    vmin=0,
    vmax=100
)

plt.title(
    "Skeleton Only - Normalized Confusion Matrix (%)",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

cbar = plt.gca().collections[0].colorbar
cbar.set_label("Percentage (%)")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "skeleton_only_normalized_confusion_matrix.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "skeleton_only_normalized_confusion_matrix.svg"
    ),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# SKELETON ONLY - 8.G SAVE CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import classification_report

skeleton_only_report_dict = classification_report(
    y_test,
    y_pred_skeleton_only,
    target_names=label_encoder.classes_,
    output_dict=True,
    digits=4,
    zero_division=0
)

skeleton_only_report_df = pd.DataFrame(
    skeleton_only_report_dict
).transpose()

display(skeleton_only_report_df)

skeleton_only_report_df.to_csv(
    os.path.join(
        SKELETON_ONLY_DIR,
        "skeleton_only_classification_report.csv"
    )
)

In [ ]:
test_accuracy
test_precision
test_recall
test_f1

In [ ]:
# ============================================================
# SKELETON ONLY - 8.H COMPARISON WITH SKELETON + CONTEXT
# ============================================================

comparison_df = pd.DataFrame({
    "model": [
        "Skeleton Only",
        "Skeleton + Context"
    ],
    "accuracy": [
        skeleton_only_accuracy,
        test_accuracy
    ],
    "precision_macro": [
        skeleton_only_precision,
        test_precision
    ],
    "recall_macro": [
        skeleton_only_recall,
        test_recall
    ],
    "f1_macro": [
        skeleton_only_f1,
        test_f1
    ]
})

display(comparison_df)

comparison_df.to_csv(
    os.path.join(
        SKELETON_ONLY_DIR,
        "model_comparison_skeleton_vs_context.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# SKELETON ONLY - 8.I MODEL PERFORMANCE COMPARISON
# ============================================================

ax = comparison_df.set_index("model").plot(
    kind="bar",
    figsize=(9, 5),
    width=0.75
)

ax.set_title(
    "Performance Comparison: Skeleton Only vs Skeleton + Context",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.xticks(rotation=0)

plt.legend(
    title="Metric",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "model_comparison_skeleton_vs_context.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "model_comparison_skeleton_vs_context.svg"
    ),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# STEP 18.J
# TRAINING PERFORMANCE COMPARISON
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

history_context_df = pd.DataFrame(history.history)

history_skeleton_df = pd.DataFrame(
    history_skeleton_only.history
)

In [ ]:
# ============================================================
# STEP 18.K ACCURACY COMPARISON
# ============================================================

plt.figure(figsize=(9,6))

plt.plot(
    history_context_df["accuracy"],
    linewidth=2,
    label="Context+Skeleton Train"
)

plt.plot(
    history_context_df["val_accuracy"],
    linewidth=2,
    linestyle="--",
    label="Context+Skeleton Validation"
)

plt.plot(
    history_skeleton_df["accuracy"],
    linewidth=2,
    label="Skeleton Train"
)

plt.plot(
    history_skeleton_df["val_accuracy"],
    linewidth=2,
    linestyle="--",
    label="Skeleton Validation"
)

plt.title(
    "Training and Validation Accuracy Comparison",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.grid(alpha=0.3)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "accuracy_comparison.png"
    ),
    dpi=600
)

plt.show()

In [ ]:
# ============================================================
# STEP 18.L LOSS COMPARISON
# ============================================================

plt.figure(figsize=(9,6))

plt.plot(
    history_context_df["loss"],
    linewidth=2,
    label="Context+Skeleton Train"
)

plt.plot(
    history_context_df["val_loss"],
    linewidth=2,
    linestyle="--",
    label="Context+Skeleton Validation"
)

plt.plot(
    history_skeleton_df["loss"],
    linewidth=2,
    label="Skeleton Train"
)

plt.plot(
    history_skeleton_df["val_loss"],
    linewidth=2,
    linestyle="--",
    label="Skeleton Validation"
)

plt.title(
    "Training and Validation Loss Comparison",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.grid(alpha=0.3)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "loss_comparison.png"
    ),
    dpi=600
)

plt.show()

**Tabular Event**

In [ ]:
# ============================================================
# STEP 18. BUILD MODEL: CNN-BiLSTM + ATTENTION FUSION
# ============================================================

import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    MaxPooling1D,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    Concatenate,
    Multiply
)

from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam


def build_model_attention_fusion():

    # ========================================================
    # SKELETON BRANCH
    # ========================================================
    skeleton_input = Input(
        shape=(X_train_skeleton.shape[1], X_train_skeleton.shape[2]),
        name="skeleton_input"
    )

    x = Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(skeleton_input)

    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(
        filters=128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(x)

    x = BatchNormalization()(x)

    x = Bidirectional(
        LSTM(
            64,
            return_sequences=False,
            kernel_regularizer=l2(0.001)
        )
    )(x)

    x = Dropout(0.3)(x)

    skeleton_feature = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(0.001),
        name="skeleton_feature"
    )(x)

    # ========================================================
    # CONTEXT BRANCH
    # ========================================================
    context_input = Input(
        shape=(X_train_context.shape[1],),
        name="context_input"
    )

    context_feature = Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(0.001),
        name="context_feature"
    )(context_input)

    context_feature = Dropout(0.2)(context_feature)

    # ========================================================
    # ATTENTION FUSION
    # Context menghasilkan bobot perhatian untuk skeleton feature
    # ========================================================
    attention_weight = Dense(
        64,
        activation="sigmoid",
        name="attention_weight"
    )(context_feature)

    attended_skeleton = Multiply(
        name="attended_skeleton"
    )([
        skeleton_feature,
        attention_weight
    ])

    fusion = Concatenate(
        name="attention_fusion"
    )([
        attended_skeleton,
        context_feature
    ])

    # ========================================================
    # CLASSIFIER
    # ========================================================
    z = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(fusion)

    z = Dropout(0.3)(z)

    z = Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(z)

    output = Dense(
        len(label_encoder.classes_),
        activation="softmax",
        name="phase_output"
    )(z)

    model = Model(
        inputs=[skeleton_input, context_input],
        outputs=output
    )

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# Cek model
attention_model = build_model_attention_fusion()
attention_model.summary()

In [ ]:
# ============================================================
# STEP 19. TRAINING ATTENTION FUSION MODEL
# ============================================================

import os
import tensorflow as tf

ATTENTION_DIR = "/content/drive/MyDrive/DATASET-TUG-CONTEXT/Results/Final_Context_Model_18-7-2026"
os.makedirs(ATTENTION_DIR, exist_ok=True)

ATTENTION_KERAS_PATH = os.path.join(
    ATTENTION_DIR,
    "attention_fusion_classifier.keras"
)

ATTENTION_H5_PATH = os.path.join(
    ATTENTION_DIR,
    "attention_fusion_classifier.h5"
)

tf.keras.backend.clear_session()

attention_model = build_model_attention_fusion()

callbacks_attention = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=ATTENTION_KERAS_PATH,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

history_attention = attention_model.fit(
    [X_train_skeleton, X_train_context],
    y_train,
    validation_data=(
        [X_val_skeleton, X_val_context],
        y_val
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_attention,
    class_weight=class_weight,
    verbose=1
)

attention_model.save(ATTENTION_H5_PATH)

print("Attention Fusion training selesai.")
print("Model .keras:", ATTENTION_KERAS_PATH)
print("Model .h5   :", ATTENTION_H5_PATH)


In [ ]:
# ============================================================
# STEP 20. EVALUATION ATTENTION FUSION MODEL
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

best_attention_model = tf.keras.models.load_model(
    ATTENTION_KERAS_PATH
)

y_pred_prob_attention = best_attention_model.predict(
    [X_test_skeleton, X_test_context],
    verbose=0
)

y_pred_attention = np.argmax(
    y_pred_prob_attention,
    axis=1
)

attention_accuracy = accuracy_score(
    y_test,
    y_pred_attention
)

attention_precision = precision_score(
    y_test,
    y_pred_attention,
    average="macro",
    zero_division=0
)

attention_recall = recall_score(
    y_test,
    y_pred_attention,
    average="macro",
    zero_division=0
)

attention_f1 = f1_score(
    y_test,
    y_pred_attention,
    average="macro",
    zero_division=0
)

print("="*60)
print("ATTENTION FUSION TEST EVALUATION")
print("="*60)

print(f"Accuracy  : {attention_accuracy:.4f}")
print(f"Precision : {attention_precision:.4f}")
print(f"Recall    : {attention_recall:.4f}")
print(f"F1-score  : {attention_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_attention,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

attention_eval_df = pd.DataFrame({
    "metric": [
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro"
    ],
    "value": [
        attention_accuracy,
        attention_precision,
        attention_recall,
        attention_f1
    ]
})

attention_eval_df.to_csv(
    os.path.join(
        ATTENTION_DIR,
        "attention_fusion_test_evaluation.csv"
    ),
    index=False
)

display(attention_eval_df)

In [ ]:
history
history_skeleton_only
history_attention

In [ ]:
# ============================================================
# STEP 21. TRAINING PERFORMANCE COMPARISON
# Skeleton Only vs Late Fusion vs Attention Fusion
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# Folder penyimpanan
# ============================================================

COMPARE_DIR = "/content/drive/MyDrive/DATASET-TUG-CONTEXT/Results/Final_Context_Model_18-7-2026"
os.makedirs(COMPARE_DIR, exist_ok=True)

# ============================================================
# Ambil history dari masing-masing model
# ============================================================

history_late_df = pd.DataFrame(history.history)
history_skeleton_df = pd.DataFrame(history_skeleton_only.history)
history_attention_df = pd.DataFrame(history_attention.history)

# ============================================================
# Samakan panjang epoch
# Jika EarlyStopping membuat epoch berbeda
# ============================================================

min_epochs = min(
    len(history_late_df),
    len(history_skeleton_df),
    len(history_attention_df)
)

history_late_df = history_late_df.iloc[:min_epochs]
history_skeleton_df = history_skeleton_df.iloc[:min_epochs]
history_attention_df = history_attention_df.iloc[:min_epochs]

epochs = range(1, min_epochs + 1)

# ============================================================
# Plot Accuracy dan Loss dalam satu figure
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 6)
)

# ------------------------------------------------------------
# Accuracy comparison
# ------------------------------------------------------------

axes[0].plot(
    epochs,
    history_skeleton_df["accuracy"],
    linewidth=2,
    label="Skeleton Only - Train"
)

axes[0].plot(
    epochs,
    history_skeleton_df["val_accuracy"],
    linewidth=2,
    linestyle="--",
    label="Skeleton Only - Validation"
)

axes[0].plot(
    epochs,
    history_late_df["accuracy"],
    linewidth=2,
    label="Late Fusion - Train"
)

axes[0].plot(
    epochs,
    history_late_df["val_accuracy"],
    linewidth=2,
    linestyle="--",
    label="Late Fusion - Validation"
)

axes[0].plot(
    epochs,
    history_attention_df["accuracy"],
    linewidth=2,
    label="Attention Fusion - Train"
)

axes[0].plot(
    epochs,
    history_attention_df["val_accuracy"],
    linewidth=2,
    linestyle="--",
    label="Attention Fusion - Validation"
)

axes[0].set_title(
    "(a) Training and Validation Accuracy",
    fontsize=14,
    fontweight="bold"
)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1.05)
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

# ------------------------------------------------------------
# Loss comparison
# ------------------------------------------------------------

axes[1].plot(
    epochs,
    history_skeleton_df["loss"],
    linewidth=2,
    label="Skeleton Only - Train"
)

axes[1].plot(
    epochs,
    history_skeleton_df["val_loss"],
    linewidth=2,
    linestyle="--",
    label="Skeleton Only - Validation"
)

axes[1].plot(
    epochs,
    history_late_df["loss"],
    linewidth=2,
    label="Late Fusion - Train"
)

axes[1].plot(
    epochs,
    history_late_df["val_loss"],
    linewidth=2,
    linestyle="--",
    label="Late Fusion - Validation"
)

axes[1].plot(
    epochs,
    history_attention_df["loss"],
    linewidth=2,
    label="Attention Fusion - Train"
)

axes[1].plot(
    epochs,
    history_attention_df["val_loss"],
    linewidth=2,
    linestyle="--",
    label="Attention Fusion - Validation"
)

axes[1].set_title(
    "(b) Training and Validation Loss",
    fontsize=14,
    fontweight="bold"
)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

# ============================================================
# Final layout
# ============================================================

plt.suptitle(
    "Training Performance Comparison Across Models",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        COMPARE_DIR,
        "training_performance_comparison_3models.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(
        COMPARE_DIR,
        "training_performance_comparison_3models.svg"
    ),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# STEP 22. CONFUSION MATRIX
# ============================================================

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

FIGURE_DIR = os.path.join(
    ATTENTION_DIR,
    "Figures"
)

os.makedirs(FIGURE_DIR, exist_ok=True)

class_names = label_encoder.classes_

cm_attention = confusion_matrix(
    y_test,
    y_pred_attention
)

plt.figure(figsize=(7,6))

sns.heatmap(
    cm_attention,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    square=True
)

plt.title(
    "Attention Fusion Confusion Matrix",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "attention_confusion_matrix.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# STEP 23. NORMALIZED CONFUSION MATRIX
# ============================================================

cm_attention_norm = confusion_matrix(
    y_test,
    y_pred_attention,
    normalize="true"
) * 100

plt.figure(figsize=(7,6))

sns.heatmap(
    cm_attention_norm,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    square=True,
    vmin=0,
    vmax=100
)

plt.title(
    "Attention Fusion Normalized Confusion Matrix (%)",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

cbar = plt.gca().collections[0].colorbar
cbar.set_label("Percentage (%)")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "attention_normalized_confusion_matrix.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# STEP 24. MULTI-CLASS ROC CURVE
# ============================================================

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

n_classes = len(label_encoder.classes_)

y_test_bin = label_binarize(
    y_test,
    classes=np.arange(n_classes)
)

plt.figure(figsize=(8,7))

for i in range(n_classes):

    fpr, tpr, _ = roc_curve(
        y_test_bin[:, i],
        y_pred_prob_attention[:, i]
    )

    roc_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{label_encoder.classes_[i]} (AUC={roc_auc:.3f})"
    )

plt.plot(
    [0,1],
    [0,1],
    'k--'
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "Attention Fusion ROC Curve",
    fontsize=15,
    fontweight="bold"
)

plt.legend(
    loc="lower right",
    fontsize=8
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "attention_ROC_curve.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# STEP 25. PRECISION-RECALL CURVE
# ============================================================

from sklearn.metrics import precision_recall_curve

plt.figure(figsize=(8,7))

for i in range(n_classes):

    precision, recall, _ = precision_recall_curve(
        y_test_bin[:, i],
        y_pred_prob_attention[:, i]
    )

    plt.plot(
        recall,
        precision,
        linewidth=2,
        label=label_encoder.classes_[i]
    )

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title(
    "Attention Fusion Precision-Recall Curve",
    fontsize=15,
    fontweight="bold"
)

plt.legend(fontsize=8)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "attention_precision_recall_curve.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
y_test

y_pred_skeleton_only
y_pred
y_pred_attention

y_pred_prob_skeleton_only
y_pred_prob
y_pred_prob_attention

label_encoder

In [ ]:
# ============================================================
# STEP 26. CONFUSION MATRIX COMPARISON - 3 MODELS
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

COMPARE_DIR = "/content/drive/MyDrive/DATASET-TUG-CONTEXT/Results/Final_Context_Model_18-7-2026"
os.makedirs(COMPARE_DIR, exist_ok=True)

class_names = label_encoder.classes_

models_cm = {
    "Skeleton Only": y_pred_skeleton_only,
    "Late Fusion": y_pred,
    "Attention Fusion": y_pred_attention
}

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

for ax, (model_name, y_pred_model) in zip(axes, models_cm.items()):

    cm = confusion_matrix(
        y_test,
        y_pred_model
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=0.5,
        square=True,
        cbar=False,
        ax=ax
    )

    ax.set_title(
        model_name,
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.suptitle(
    "Confusion Matrix Comparison Across Models",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        COMPARE_DIR,
        "confusion_matrix_comparison_3models.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(
        COMPARE_DIR,
        "confusion_matrix_comparison_3models.svg"
    ),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# STEP 27. NORMALIZED CONFUSION MATRIX COMPARISON - 3 MODELS
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

for ax, (model_name, y_pred_model) in zip(axes, models_cm.items()):

    cm_norm = confusion_matrix(
        y_test,
        y_pred_model,
        normalize="true"
    ) * 100

    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=0.5,
        square=True,
        vmin=0,
        vmax=100,
        cbar=False,
        ax=ax
    )

    ax.set_title(
        model_name,
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.suptitle(
    "Normalized Confusion Matrix Comparison Across Models (%)",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        COMPARE_DIR,
        "normalized_confusion_matrix_comparison_3models.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(
        COMPARE_DIR,
        "normalized_confusion_matrix_comparison_3models.svg"
    ),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# STEP 28. ROC CURVE COMPARISON - 3 MODELS
# ============================================================

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

n_classes = len(label_encoder.classes_)

y_test_bin = label_binarize(
    y_test,
    classes=np.arange(n_classes)
)

models_prob = {
    "Skeleton Only": y_pred_prob_skeleton_only,
    "Late Fusion": y_pred_prob,
    "Attention Fusion": y_pred_prob_attention
}

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

roc_summary_records = []

for ax, (model_name, y_prob_model) in zip(axes, models_prob.items()):

    auc_values = []

    for i in range(n_classes):

        fpr, tpr, _ = roc_curve(
            y_test_bin[:, i],
            y_prob_model[:, i]
        )

        roc_auc = auc(fpr, tpr)
        auc_values.append(roc_auc)

        ax.plot(
            fpr,
            tpr,
            linewidth=1.8,
            label=f"{class_names[i]} AUC={roc_auc:.3f}"
        )

        roc_summary_records.append({
            "model": model_name,
            "class": class_names[i],
            "auc": roc_auc
        })

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1
    )

    ax.set_title(
        f"{model_name}\nMacro AUC={np.mean(auc_values):.3f}",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc="lower right")

plt.suptitle(
    "ROC Curve Comparison Across Models",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        COMPARE_DIR,
        "roc_curve_comparison_3models.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(
        COMPARE_DIR,
        "roc_curve_comparison_3models.svg"
    ),
    bbox_inches="tight"
)

plt.show()

roc_summary_df = pd.DataFrame(roc_summary_records)

roc_summary_df.to_csv(
    os.path.join(
        COMPARE_DIR,
        "roc_auc_comparison_3models.csv"
    ),
    index=False
)

display(roc_summary_df)

In [ ]:
# ============================================================
# STEP 29. FINAL METRIC COMPARISON TABLE
# ============================================================

comparison_3models_df = pd.DataFrame({
    "model": [
        "Skeleton Only",
        "Late Fusion",
        "Attention Fusion"
    ],
    "accuracy": [
        skeleton_only_accuracy,
        test_accuracy,
        attention_accuracy
    ],
    "precision_macro": [
        skeleton_only_precision,
        test_precision,
        attention_precision
    ],
    "recall_macro": [
        skeleton_only_recall,
        test_recall,
        attention_recall
    ],
    "f1_macro": [
        skeleton_only_f1,
        test_f1,
        attention_f1
    ]
})

display(comparison_3models_df)

comparison_3models_df.to_csv(
    os.path.join(
        COMPARE_DIR,
        "final_metric_comparison_3models.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# STEP 30. FINAL METRIC COMPARISON BAR CHART
# ============================================================

ax = comparison_3models_df.set_index("model").plot(
    kind="bar",
    figsize=(10, 5),
    width=0.75
)

ax.set_title(
    "Final Performance Comparison Across Models",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel(
    "Model",
    fontsize=12,
    fontweight="bold"
)

ax.set_ylabel(
    "Score",
    fontsize=12,
    fontweight="bold"
)

# Beri ruang di atas bar agar label tidak terpotong
ax.set_ylim(0, 1.08)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.xticks(
    rotation=0,
    fontsize=10
)

plt.yticks(fontsize=10)

# ============================================================
# Tambahkan nilai di atas setiap bar
# ============================================================
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.3f",          # tampilkan 3 digit desimal
        label_type="edge",
        padding=3,
        fontsize=8,
        fontweight="bold",
        rotation=0
    )

plt.legend(
    title="Metric",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=9,
    title_fontsize=10
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        COMPARE_DIR,
        "final_metric_comparison_3models.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(
        COMPARE_DIR,
        "final_metric_comparison_3models.svg"
    ),
    bbox_inches="tight"
)

plt.show()

In [ ]:
print(label_encoder.classes_)

In [ ]:
# ============================================================
# TABLE. AUTOMATIC PREDICTION OF TUG PHASES
# Menggunakan model dengan performa terbaik
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    mean_squared_error,
    recall_score,
    precision_score,
    f1_score
)

# ============================================================
# 1. Pilih model terbaik berdasarkan accuracy
# ============================================================

model_candidates = {
    "Skeleton Only": {
        "accuracy": skeleton_only_accuracy,
        "y_pred": y_pred_skeleton_only
    },
    "Late Fusion": {
        "accuracy": test_accuracy,
        "y_pred": y_pred
    },
    "Attention Fusion": {
        "accuracy": attention_accuracy,
        "y_pred": y_pred_attention
    }
}

best_model_name = max(
    model_candidates,
    key=lambda x: model_candidates[x]["accuracy"]
)

best_y_pred = model_candidates[best_model_name]["y_pred"]

print("Best model:", best_model_name)
print("Accuracy :", model_candidates[best_model_name]["accuracy"])

# ============================================================
# 2. Buat tabel performa per phase
# ============================================================

phase_order = ["STS", "WO", "TR", "WI", "STTS"]

table_records = []

for i, phase in enumerate(phase_order, start=1):

    phase_id = np.where(label_encoder.classes_ == phase)[0][0]

    idx_phase = np.where(y_test == phase_id)[0]

    y_true_phase = y_test[idx_phase]
    y_pred_phase = best_y_pred[idx_phase]

    n_samples = len(y_true_phase)

    class_counts = []

    for cls in range(len(label_encoder.classes_)):
        class_counts.append(
            np.sum(y_pred_phase == cls)
        )

    samples_per_class = ",".join(
        [str(x) for x in class_counts]
    )

    y_true_binary = (y_test == phase_id).astype(int)
    y_pred_binary = (best_y_pred == phase_id).astype(int)

    acc = accuracy_score(
        y_true_binary,
        y_pred_binary
    )

    mse = mean_squared_error(
        y_true_binary,
        y_pred_binary
    )

    recall = recall_score(
        y_true_binary,
        y_pred_binary,
        zero_division=0
    )

    precision = precision_score(
        y_true_binary,
        y_pred_binary,
        zero_division=0
    )

    f1 = f1_score(
        y_true_binary,
        y_pred_binary,
        zero_division=0
    )

    table_records.append({
        "TUG Phase": i,
        "Phase Description": phase,
        "N": n_samples,
        "Samples per Class <STS,WO,TR,WI,STTS>": samples_per_class,
        "Accuracy": f"{acc*100:.0f}%",
        "MSE": round(mse, 2),
        "Recall": round(recall, 2),
        "Precision": round(precision, 2),
        "F1": round(f1, 2)
    })

best_tug_phase_table = pd.DataFrame(table_records)

display(best_tug_phase_table)

# ============================================================
# 3. Simpan tabel
# ============================================================

BEST_TABLE_DIR = "/content/drive/MyDrive/DATASET-TUG-CONTEXT/Results/Final_Context_Model_18-7-2026/Best_Model_Tables"
os.makedirs(BEST_TABLE_DIR, exist_ok=True)

best_tug_phase_table.to_csv(
    os.path.join(
        BEST_TABLE_DIR,
        f"tug_phase_prediction_table_{best_model_name.replace(' ', '_')}.csv"
    ),
    index=False
)

print("Tabel disimpan di:")
print(BEST_TABLE_DIR)


# Multimodal Fall Risk Prediction Pipeline

Bagian ini merapikan tahap akhir notebook sehingga alurnya menjadi:

**Skeleton Preprocessing → Windowing → CNN-BiLSTM Phase Classification → Phase Duration Extraction → Skeleton + Context + Duration + Physiological Data Fusion → Fall Risk Prediction → Evaluation → Model Export (.keras, .tflite, .onnx)**

Catatan metodologis:
- Model CNN-BiLSTM sebelumnya digunakan untuk menghasilkan **skeleton embedding** pada level window.
- Embedding window dirata-ratakan per subject agar bisa digabung dengan **duration**, **environmental context**, dan **physiological/anthropometric features**.
- Label risiko jatuh pada template ini masih dibuat dari ambang durasi TUG (`LOW`, `MEDIUM`, `HIGH`). Jika tersedia label klinis sebenarnya, ganti kolom target dengan label klinis tersebut.


In [ ]:
# ============================================================
# STEP 21. CONFIGURATION & LOAD PHYSIOLOGICAL DATA
# ============================================================

import os
import glob
import re
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight

# ============================================================
# DATASET DIRECTORY
# ============================================================

DATASET_DIR = "/content/drive/MyDrive/DATASET-TUG-CONTEXT"

# ============================================================
# OUTPUT DIRECTORY
# ============================================================

RISK_OUTPUT_DIR = os.path.join(
    DATASET_DIR,
    "Results",
    "Final_Context_Model_18-7-2026_Multimodal_Fall_Risk"
)

RISK_MODEL_DIR = os.path.join(RISK_OUTPUT_DIR, "Models")
RISK_TABLE_DIR = os.path.join(RISK_OUTPUT_DIR, "Tables")
RISK_FIGURE_DIR = os.path.join(RISK_OUTPUT_DIR, "Figures")

for folder in [
    RISK_OUTPUT_DIR,
    RISK_MODEL_DIR,
    RISK_TABLE_DIR,
    RISK_FIGURE_DIR
]:
    os.makedirs(folder, exist_ok=True)

# ============================================================
# COLUMN CONFIGURATION
# ============================================================

PHASE_ORDER = [
    "STS",
    "WO",
    "TR",
    "WI",
    "STTS"
]

DURATION_COLS = [
    "duration_STS",
    "duration_WO",
    "duration_TR",
    "duration_WI",
    "duration_STTS",
    "total_tug_time"
]

CONTEXT_COLS = [
    "temperature",
    "humidity",
    "light_intensity",
    "pressure"
]

# Semua kolom pada dataset fisiologis
PHYSIO_ALL_COLS = [
    "Subject",
    "Age",
    "Height_cm",
    "Weight_kg",
    "BMI",
    "Body_Temperature",
    "Mean_SpO2",
    "Mean_Heart_Rate"
]

# Kolom yang digunakan untuk model
PHYSIO_FEATURE_COLS = [
    "Age",
    "BMI",
    "Body_Temperature",
    "Mean_SpO2",
    "Mean_Heart_Rate"
]

# ============================================================
# LOAD PHYSIOLOGICAL DATA
# ============================================================

physio_pattern = os.path.join(
    DATASET_DIR,
    "Subject*",
    "Physiological_Data",
    "Physiological_Data_subject*.csv"
)

physio_files = sorted(glob.glob(physio_pattern))

print("=" * 60)
print("Loading Physiological Data...")
print("=" * 60)
print(f"Found {len(physio_files)} physiological files.\n")

physio_list = []

for file in physio_files:

    df = pd.read_csv(file)

    # Jika kolom Subject tidak ada
    if "Subject" not in df.columns:

        match = re.search(
            r"subject(\d+)",
            file.lower()
        )

        if match:
            df["Subject"] = f"subject{int(match.group(1)):02d}"

    # Standarisasi format subject
    df["Subject"] = (
        df["Subject"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    physio_list.append(df)

physio_df = pd.concat(
    physio_list,
    ignore_index=True
)

physio_df = (
    physio_df
    .sort_values("Subject")
    .reset_index(drop=True)
)

print("=" * 60)
print("Physiological Dataset")
print("=" * 60)
print("Shape :", physio_df.shape)
print()

display(physio_df.head())

print()
print("Physiological Features used for Fall Risk Prediction:")
print(PHYSIO_FEATURE_COLS)

print()
print("Risk Output Directory:")
print(RISK_OUTPUT_DIR)


In [ ]:
display(physio_df)

In [ ]:

# ============================================================
# 22. UTILITY FUNCTIONS
# ============================================================

def normalize_subject_id(value):
    """Normalize subject id into format subject01, subject02, ..."""
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    s = s.replace(" ", "")
    match = re.search(r"(\d+)", s)
    if match:
        return f"subject{int(match.group(1)):02d}"
    return s


def standardize_columns(df):
    """Lowercase column names and replace spaces/symbols with underscore."""
    df = df.copy()
    df.columns = (
        pd.Index(df.columns)
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df


def classify_tug_risk(total_time):
    """
    Template label berbasis clinical cutoff TUG.
    LOW    : < 10 s
    MEDIUM : 10 - 13.5 s
    HIGH   : > 13.5 s
    """
    if pd.isna(total_time):
        return np.nan
    if total_time <= 13.5:
        return "LOW/NO"
    #elif total_time <= 13.5:
    #    return "MEDIU"
    else:
        return "HIGH"


def safe_numeric(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


In [ ]:

# ============================================================
# 23. PHASE DURATION EXTRACTION FROM BORIS TABULAR FILES
# ============================================================

subject_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "Subject*")))
print("Jumlah folder subject ditemukan:", len(subject_folders))

duration_records = []

for subject_path in subject_folders:
    subject_name = os.path.basename(subject_path)
    subject_id = normalize_subject_id(subject_name)

    # Mendukung beberapa kemungkinan nama folder/file BORIS
    boris_files = []
    boris_files.extend(glob.glob(os.path.join(subject_path, "Boris_Tabular", "*.xlsx")))
    boris_files.extend(glob.glob(os.path.join(subject_path, "BORIS", "*.xlsx")))
    boris_files.extend(glob.glob(os.path.join(subject_path, "*BORIS*", "*.xlsx")))
    boris_files.extend(glob.glob(os.path.join(subject_path, "*.xlsx")))
    boris_files = sorted(list(set(boris_files)))

    if len(boris_files) == 0:
        print(f"{subject_name}: file BORIS tidak ditemukan")
        continue

    boris_file = boris_files[0]
    df = pd.read_excel(boris_file)
    df = standardize_columns(df)

    # BORIS umumnya punya kolom behavior dan time.
    behavior_col = None
    time_col = None

    for candidate in ["behavior", "behaviour", "event_type"]:
        if candidate in df.columns:
            behavior_col = candidate
            break

    for candidate in ["time", "time_s", "start_s", "start_time", "media_time"]:
        if candidate in df.columns:
            time_col = candidate
            break

    if behavior_col is None or time_col is None:
        print(f"{subject_name}: kolom Behavior/Time tidak ditemukan. Kolom tersedia: {df.columns.tolist()}")
        continue

    df[behavior_col] = df[behavior_col].astype(str).str.strip().str.upper()
    df[time_col] = pd.to_numeric(df[time_col], errors="coerce")

    row = {"subject": subject_id}
    start_times, end_times = [], []

    for phase in PHASE_ORDER:
        phase_df = (
            df[df[behavior_col] == phase]
            .dropna(subset=[time_col])
            .sort_values(time_col)
        )

        # Format point-event: titik pertama = start, titik kedua = end.
        if len(phase_df) < 2:
            row[f"duration_{phase}"] = np.nan
            continue

        start_time = phase_df.iloc[0][time_col]
        end_time = phase_df.iloc[1][time_col]
        duration = end_time - start_time

        row[f"duration_{phase}"] = duration
        start_times.append(start_time)
        end_times.append(end_time)

    row["total_tug_time"] = (max(end_times) - min(start_times)) if start_times and end_times else np.nan
    duration_records.append(row)

duration_dataset = pd.DataFrame(duration_records)
duration_dataset = duration_dataset.drop_duplicates(subset=["subject"], keep="first")

display(duration_dataset.head())
print("Shape duration_dataset:", duration_dataset.shape)
print("Missing values:")
print(duration_dataset.isna().sum())

duration_dataset.to_csv(os.path.join(RISK_TABLE_DIR, "phase_duration_dataset.csv"), index=False)


In [ ]:

# ============================================================
# 24. ENVIRONMENTAL CONTEXT MEAN PER SUBJECT
# ============================================================

context_records = []

for subject_path in subject_folders:
    subject_id = normalize_subject_id(os.path.basename(subject_path))

    context_files = sorted(glob.glob(os.path.join(subject_path, "Context", "*.csv")))
    if len(context_files) == 0:
        print(f"{subject_id}: context file tidak ditemukan")
        continue

    df = pd.read_csv(context_files[0])
    df = standardize_columns(df)

    # Normalisasi alias kolom yang mungkin berbeda
    alias_map = {
        "light_intensity_lux": "light_intensity",
        "light": "light_intensity",
        "temp": "temperature",
        "humid": "humidity",
        "barometric_pressure": "pressure"
    }
    df = df.rename(columns={k: v for k, v in alias_map.items() if k in df.columns})

    missing = [col for col in CONTEXT_COLS if col not in df.columns]
    if missing:
        print(f"{subject_id}: kolom context hilang {missing}. Kolom tersedia: {df.columns.tolist()}")
        continue

    df = safe_numeric(df, CONTEXT_COLS)
    df[CONTEXT_COLS] = df[CONTEXT_COLS].interpolate(method="linear").ffill().bfill()

    row = {"subject": subject_id}
    for col in CONTEXT_COLS:
        row[col] = df[col].mean()
    context_records.append(row)

context_mean = pd.DataFrame(context_records).drop_duplicates(subset=["subject"], keep="first")

display(context_mean.head())
print("Shape context_mean:", context_mean.shape)
print("Missing values:")
print(context_mean.isna().sum())

context_mean.to_csv(os.path.join(RISK_TABLE_DIR, "context_mean_per_subject.csv"), index=False)


In [ ]:

# ============================================================
# 25. LOAD PHYSIOLOGICAL AND ANTHROPOMETRIC DATA
# ============================================================

# Data fisiologis berada di:
# DATASET-TUG-CONTEXT/
# ├── Subject01/
# │   └── Physiological_Data/
# │       └── Physiological_Data_subject01.csv
# ├── Subject02/
# │   └── Physiological_Data/
# │       └── Physiological_Data_subject02.csv
# ...
# └── Subject30/
#     └── Physiological_Data/
#         └── Physiological_Data_subject30.csv

physio_pattern = os.path.join(
    DATASET_DIR,
    "Subject*",
    "Physiological_Data",
    "Physiological_Data_subject*.csv"
)

physio_files = sorted(glob.glob(physio_pattern))

if len(physio_files) == 0:
    raise FileNotFoundError(
        "File fisiologis tidak ditemukan. Pastikan struktur folder adalah: "
        "SubjectXX/Physiological_Data/Physiological_Data_subjectXX.csv"
    )

physio_list = []

for file_path in physio_files:
    temp_df = pd.read_csv(file_path)

    # Standarisasi nama kolom agar konsisten dengan pipeline lama
    temp_df = temp_df.rename(columns={
        "Subject": "subject",
        "Age": "age",
        "Height_cm": "height_cm",
        "Weight_kg": "weight_kg",
        "BMI": "bmi",
        "Body_Temperature": "body_temperature",
        "Mean_SpO2": "mean_spo2",
        "Mean_Heart_Rate": "mean_heart_rate"
    })

    # Jika kolom subject tidak ada, ambil dari nama/path file
    if "subject" not in temp_df.columns:
        match = re.search(r"subject(\d+)", file_path.lower())
        if match:
            temp_df["subject"] = f"subject{int(match.group(1)):02d}"

    temp_df["subject"] = temp_df["subject"].apply(normalize_subject_id)

    physio_list.append(temp_df)

physio_df = pd.concat(
    physio_list,
    ignore_index=True
)

physio_df = physio_df.dropna(subset=["subject"])

# Kolom dasar yang tetap dimunculkan di tabel
PHYSIO_BASE_COLS = [
    "age",
    "height_cm",
    "weight_kg",
    "bmi",
    "body_temperature",
    "mean_spo2",
    "mean_heart_rate"
]

physio_df = safe_numeric(
    physio_df,
    PHYSIO_BASE_COLS
)

# Jika BMI belum tersedia di beberapa file, hitung ulang dari tinggi dan berat
if "bmi" not in physio_df.columns or physio_df["bmi"].isna().any():
    physio_df["height_m"] = physio_df["height_cm"] / 100.0
    physio_df["bmi"] = physio_df["weight_kg"] / (physio_df["height_m"] ** 2)

# ============================================================
# Kolom yang digunakan untuk model fall-risk prediction
# Height dan Weight tetap tampil, tetapi tidak dipakai sebagai fitur model
# ============================================================

PHYSIO_FEATURE_COLS = [
    "age",
    "bmi",
    "body_temperature",
    "mean_spo2",
    "mean_heart_rate"
]

# Tabel lengkap untuk dokumentasi
physio_features = physio_df[
    [
        "subject",
        "age",
        "height_cm",
        "weight_kg",
        "bmi",
        "body_temperature",
        "mean_spo2",
        "mean_heart_rate"
    ]
].drop_duplicates(
    subset=["subject"],
    keep="first"
)

physio_features = physio_features.sort_values("subject").reset_index(drop=True)

display(physio_features.head())

print("Total physiological files found:", len(physio_files))
print("Shape physio_features:", physio_features.shape)

print("\nColumns displayed in table:")
print(physio_features.columns.tolist())

print("\nPhysiological columns used for model:")
print(PHYSIO_FEATURE_COLS)

print("\nMissing values:")
print(physio_features.isna().sum())

physio_features.to_csv(
    os.path.join(
        RISK_TABLE_DIR,
        "physiological_features.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# 26. SUBJECT-LEVEL SKELETON EMBEDDING FROM CNN-BiLSTM
# ============================================================

# Cell ini mengambil representasi skeleton dari model phase classification.
# Karena notebook ini tidak memiliki variabel X_skeleton_scaled,
# embedding dibuat dari data split yang benar-benar dipakai saat training:
#
# X_train_skeleton, X_val_skeleton, X_test_skeleton
# X_train_context,  X_val_context,  X_test_context
# groups_train,     groups_val,     groups_test
#
# Output:
# window-level embedding -> rata-rata per subject -> subject_skeleton_embedding

if "model" not in globals():
    raise NameError(
        "Variabel 'model' belum tersedia. "
        "Jalankan cell training CNN-BiLSTM Phase Classification terlebih dahulu."
    )

# ============================================================
# Cari layer embedding skeleton
# ============================================================

candidate_embedding_layers = [
    "skeleton_feature",    # paling direkomendasikan: representasi skeleton sebelum fusion
    "skeleton_embedding",
    "dense_skeleton",
    "feature_fusion",
    "fusion_feature"
]

embedding_layer = None
selected_embedding_layer_name = None

for layer_name in candidate_embedding_layers:
    try:
        embedding_layer = model.get_layer(layer_name)
        selected_embedding_layer_name = layer_name
        print(f"Embedding layer ditemukan: {layer_name}")
        break
    except ValueError:
        pass

if embedding_layer is None:
    print("Layer embedding spesifik tidak ditemukan.")
    print("Daftar layer model:")
    for layer in model.layers:
        try:
            output_shape = layer.output_shape
        except Exception:
            output_shape = "unknown"
        print("-", layer.name, output_shape)

    raise ValueError(
        "Tidak ditemukan layer embedding. "
        "Pastikan model phase classifier memiliki layer Dense bernama 'skeleton_feature'."
    )

# ============================================================
# Buat embedding extractor model
# ============================================================

embedding_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=embedding_layer.output,
    name="skeleton_embedding_extractor"
)

# ============================================================
# Validasi variabel split yang tersedia
# ============================================================

required_split_vars = [
    "X_train_skeleton", "X_val_skeleton", "X_test_skeleton",
    "groups_train", "groups_val", "groups_test"
]

missing_split_vars = [
    var for var in required_split_vars
    if var not in globals()
]

if missing_split_vars:
    raise NameError(
        "Variabel split skeleton/group belum lengkap: "
        f"{missing_split_vars}. Jalankan cell GROUP SHUFFLE SPLIT + SCALING terlebih dahulu."
    )

# Gabungkan train + val + test agar embedding tersedia untuk semua subject.
X_embedding_skeleton = np.concatenate(
    [
        X_train_skeleton,
        X_val_skeleton,
        X_test_skeleton
    ],
    axis=0
)

groups_embedding = np.concatenate(
    [
        groups_train,
        groups_val,
        groups_test
    ],
    axis=0
)

# ============================================================
# Siapkan input sesuai jumlah input model
# ============================================================

n_model_inputs = len(model.inputs)

if n_model_inputs == 1:
    embedding_input = X_embedding_skeleton

elif n_model_inputs == 2:
    required_context_vars = [
        "X_train_context",
        "X_val_context",
        "X_test_context"
    ]

    missing_context_vars = [
        var for var in required_context_vars
        if var not in globals()
    ]

    if missing_context_vars:
        raise NameError(
            "Model membutuhkan input context, tetapi variabel context belum lengkap: "
            f"{missing_context_vars}"
        )

    X_embedding_context = np.concatenate(
        [
            X_train_context,
            X_val_context,
            X_test_context
        ],
        axis=0
    )

    embedding_input = [
        X_embedding_skeleton,
        X_embedding_context
    ]

else:
    raise ValueError(
        f"Model memiliki {n_model_inputs} input. "
        "Cell ini hanya mendukung model dengan 1 input skeleton "
        "atau 2 input skeleton + context."
    )

print("=" * 70)
print("SKELETON EMBEDDING EXTRACTION")
print("=" * 70)
print("Selected embedding layer :", selected_embedding_layer_name)
print("X_embedding_skeleton    :", X_embedding_skeleton.shape)

if n_model_inputs == 2:
    print("X_embedding_context     :", X_embedding_context.shape)

print("groups_embedding        :", groups_embedding.shape)

# ============================================================
# Extract window-level skeleton embedding
# ============================================================

window_embeddings = embedding_model.predict(
    embedding_input,
    batch_size=64,
    verbose=1
)

# Jika output embedding masih berdimensi > 2, flatten dulu.
if len(window_embeddings.shape) > 2:
    window_embeddings = window_embeddings.reshape(
        window_embeddings.shape[0],
        -1
    )

embedding_cols = [
    f"skel_emb_{i:03d}"
    for i in range(window_embeddings.shape[1])
]

embedding_df = pd.DataFrame(
    window_embeddings,
    columns=embedding_cols
)

embedding_df["subject"] = [
    normalize_subject_id(g)
    for g in groups_embedding
]

# ============================================================
# Aggregate window-level embedding menjadi subject-level embedding
# ============================================================

subject_skeleton_embedding = (
    embedding_df
    .groupby("subject", as_index=False)[embedding_cols]
    .mean()
)

display(subject_skeleton_embedding.head())

print("Window embedding shape:", window_embeddings.shape)
print("Subject skeleton embedding shape:", subject_skeleton_embedding.shape)
print("Total subject with embedding:", subject_skeleton_embedding["subject"].nunique())

subject_skeleton_embedding.to_csv(
    os.path.join(
        RISK_TABLE_DIR,
        "subject_skeleton_embedding.csv"
    ),
    index=False
)

In [ ]:

# ============================================================
# 27. MULTIMODAL FEATURE FUSION DATASET
# Skeleton + Context + Duration + Physiological Data
# ============================================================

required_fusion_objects = [
    "duration_dataset",
    "context_mean",
    "physio_features",
    "subject_skeleton_embedding",
    "embedding_cols"
]

missing_fusion_objects = [
    obj for obj in required_fusion_objects
    if obj not in globals()
]

if missing_fusion_objects:
    raise NameError(
        "Objek untuk feature fusion belum lengkap: "
        f"{missing_fusion_objects}. Jalankan STEP 23-26 terlebih dahulu."
    )

risk_dataset = (
    duration_dataset
    .merge(context_mean, on="subject", how="left")
    .merge(physio_features, on="subject", how="left")
    .merge(subject_skeleton_embedding, on="subject", how="left")
)

# Label risiko berbasis total TUG time.
# Ganti bagian ini jika Anda punya label klinis/manual sebenarnya.
risk_dataset["fall_risk"] = risk_dataset["total_tug_time"].apply(classify_tug_risk)

# Hapus subject tanpa label.
risk_dataset = risk_dataset.dropna(subset=["fall_risk"])

# ============================================================
# Feature columns untuk model
# ============================================================

feature_cols = (
    embedding_cols +
    CONTEXT_COLS +
    DURATION_COLS +
    PHYSIO_FEATURE_COLS
)

feature_cols = [
    col for col in feature_cols
    if col in risk_dataset.columns
]

missing_from_dataset = [
    col for col in (CONTEXT_COLS + DURATION_COLS + PHYSIO_FEATURE_COLS)
    if col not in risk_dataset.columns
]

if missing_from_dataset:
    print("PERINGATAN: Kolom berikut tidak ditemukan dalam risk_dataset:")
    print(missing_from_dataset)

risk_dataset[feature_cols] = risk_dataset[feature_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

# Imputasi missing numeric features dengan median.
for col in feature_cols:
    median_value = risk_dataset[col].median()

    if pd.isna(median_value):
        median_value = 0.0

    risk_dataset[col] = risk_dataset[col].fillna(median_value)

risk_dataset = (
    risk_dataset
    .drop_duplicates(subset=["subject"], keep="first")
    .sort_values("subject")
    .reset_index(drop=True)
)

print("=" * 70)
print("MULTIMODAL FALL RISK DATASET")
print("=" * 70)
print("Risk dataset shape:", risk_dataset.shape)
print("Jumlah subject:", risk_dataset["subject"].nunique())
print("Jumlah feature untuk model:", len(feature_cols))

print("\nDistribusi label risiko:")
print(risk_dataset["fall_risk"].value_counts())

selected_preview_cols = (
    ["subject", "fall_risk", "total_tug_time"] +
    CONTEXT_COLS +
    [col for col in ["age", "height_cm", "weight_kg", "bmi", "body_temperature", "mean_spo2", "mean_heart_rate"] if col in risk_dataset.columns]
)

display(risk_dataset[selected_preview_cols].head())

risk_dataset.to_csv(
    os.path.join(
        RISK_TABLE_DIR,
        "multimodal_fall_risk_dataset.csv"
    ),
    index=False
)

pd.DataFrame({"feature": feature_cols}).to_csv(
    os.path.join(
        RISK_TABLE_DIR,
        "fall_risk_feature_columns.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# STEP 1. INITIALIZE PCA–MACHINE LEARNING EXPERIMENT
# Imports, Configuration, Cross-Validation, and Dataset Check
# ============================================================

# Pipeline penelitian:
#
# Multimodal features
#       ↓
# Median imputation
#       ↓
# StandardScaler
#       ↓
# PCA
#       ↓
# Machine-learning classifier
#       ↓
# Inner GridSearchCV
#       ↓
# Outer Stratified K-Fold evaluation
#
# Evaluasi utama dilakukan menggunakan nested cross-validation:
# - Outer CV: mengukur performa generalisasi.
# - Inner CV: memilih hyperparameter dan jumlah komponen PCA.


# ============================================================
# 1. IMPORT LIBRARY
# ============================================================

import os
import json
import time
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)


# ============================================================
# 2. RANDOM SEED
# ============================================================

RANDOM_STATE = 42

np.random.seed(
    RANDOM_STATE
)


# ============================================================
# 3. NAMA KOLOM UTAMA
# ============================================================

SUBJECT_COLUMN = "subject"
TARGET_COLUMN = "fall_risk"


# ============================================================
# 4. EXPECTED FEATURE COMPOSITION
# ============================================================

# Input model:
# 64 skeleton embedding
# 4 environmental context
# 5 physiological
#
# Duration tidak digunakan karena total_tug_time digunakan
# untuk membentuk label fall_risk.
#
# Height dan weight tidak digunakan karena telah
# direpresentasikan oleh BMI.

EXPECTED_FEATURE_COUNT = 73

EXCLUDED_MODEL_FEATURES = [
    "duration_STS",
    "duration_WO",
    "duration_TR",
    "duration_WI",
    "duration_STTS",
    "total_tug_time",
    "height_cm",
    "weight_kg"
]


# ============================================================
# 5. CONFIGURATION FOR NESTED CROSS-VALIDATION
# ============================================================

# Outer CV digunakan untuk evaluasi performa akhir.
#
# Dengan 30 subject dan kelas minoritas sebanyak 10 subject,
# 5-fold menghasilkan sekitar:
# - 24 subject training
# - 6 subject testing
# pada setiap outer fold.

OUTER_N_SPLITS = 5

outer_cv = StratifiedKFold(
    n_splits=OUTER_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


# Inner CV digunakan oleh GridSearchCV untuk memilih:
# - jumlah komponen PCA;
# - hyperparameter classifier.

INNER_N_SPLITS = 3

inner_cv = StratifiedKFold(
    n_splits=INNER_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 6. GRID SEARCH CONFIGURATION
# ============================================================

GRID_SCORING = "roc_auc"

GRID_N_JOBS = -1

GRID_VERBOSE = 0

REFIT_METRIC = True


# ============================================================
# 7. CLASSIFICATION THRESHOLD
# ============================================================

# Threshold digunakan untuk classifier yang menghasilkan
# probability_HIGH.

CLASSIFICATION_THRESHOLD = 0.50

CLASS_LABEL_MAPPING = {
    0: "LOW/NO",
    1: "HIGH"
}


# ============================================================
# 8. PCA CONFIGURATION
# ============================================================

# Daftar komponen PCA akan disesuaikan lagi dengan ukuran
# training set di dalam nested cross-validation.
#
# PCA juga dapat menggunakan proporsi variance, misalnya 0.90
# atau 0.95.

PCA_COMPONENT_CANDIDATES = [
    2,
    3,
    4,
    5,
    6,
    8,
    10,
    12,
    15,
    0.90,
    0.95
]


# ============================================================
# 9. OUTPUT DIRECTORIES
# ============================================================

if "RISK_TABLE_DIR" not in globals():
    RISK_TABLE_DIR = os.path.join(
        os.getcwd(),
        "pca_ml_fall_risk_results"
    )

PCA_ML_OUTPUT_DIR = os.path.join(
    RISK_TABLE_DIR,
    "pca_machine_learning"
)

PCA_ML_TABLE_DIR = os.path.join(
    PCA_ML_OUTPUT_DIR,
    "tables"
)

PCA_ML_FIGURE_DIR = os.path.join(
    PCA_ML_OUTPUT_DIR,
    "figures"
)

PCA_ML_MODEL_DIR = os.path.join(
    PCA_ML_OUTPUT_DIR,
    "models"
)

for output_directory in [
    PCA_ML_OUTPUT_DIR,
    PCA_ML_TABLE_DIR,
    PCA_ML_FIGURE_DIR,
    PCA_ML_MODEL_DIR
]:
    os.makedirs(
        output_directory,
        exist_ok=True
    )


# ============================================================
# 10. VALIDASI OBJEK DATASET
# ============================================================

required_dataset_objects = [
    "risk_dataset",
    "feature_cols"
]

missing_dataset_objects = [
    object_name
    for object_name in required_dataset_objects
    if object_name not in globals()
]

if missing_dataset_objects:
    raise NameError(
        "Objek dataset belum tersedia: "
        f"{missing_dataset_objects}. "
        "Jalankan tahap multimodal feature fusion terlebih dahulu."
    )


# ============================================================
# 11. MEMBUAT SALINAN DATASET
# ============================================================

ml_dataset = (
    risk_dataset
    .copy()
    .reset_index(drop=True)
)

model_feature_columns = list(
    dict.fromkeys(
        feature_cols
    )
)


# ============================================================
# 12. VALIDASI KOLOM UTAMA
# ============================================================

required_main_columns = [
    SUBJECT_COLUMN,
    TARGET_COLUMN
]

missing_main_columns = [
    column
    for column in required_main_columns
    if column not in ml_dataset.columns
]

if missing_main_columns:
    raise ValueError(
        "Kolom utama tidak ditemukan dalam risk_dataset: "
        f"{missing_main_columns}"
    )


# ============================================================
# 13. VALIDASI FEATURE COLUMNS
# ============================================================

missing_model_features = [
    feature
    for feature in model_feature_columns
    if feature not in ml_dataset.columns
]

if missing_model_features:
    raise ValueError(
        "Fitur berikut tidak ditemukan dalam risk_dataset: "
        f"{missing_model_features}"
    )


# ============================================================
# 14. MEMASTIKAN FITUR YANG DIKECUALIKAN TIDAK MASUK MODEL
# ============================================================

incorrectly_included_features = [
    feature
    for feature in EXCLUDED_MODEL_FEATURES
    if feature in model_feature_columns
]

if incorrectly_included_features:
    raise ValueError(
        "Fitur yang seharusnya dikeluarkan masih terdapat "
        "dalam model_feature_columns: "
        f"{incorrectly_included_features}"
    )


# ============================================================
# 15. VALIDASI JUMLAH FITUR
# ============================================================

if len(model_feature_columns) != EXPECTED_FEATURE_COUNT:
    raise ValueError(
        f"Jumlah fitur model seharusnya "
        f"{EXPECTED_FEATURE_COUNT}, tetapi diperoleh "
        f"{len(model_feature_columns)}."
    )


# ============================================================
# 16. PEMBERSIHAN SUBJECT DAN TARGET
# ============================================================

ml_dataset[SUBJECT_COLUMN] = (
    ml_dataset[SUBJECT_COLUMN]
    .astype(str)
    .str.strip()
)

ml_dataset[TARGET_COLUMN] = (
    ml_dataset[TARGET_COLUMN]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ============================================================
# 17. MEMERIKSA SUBJECT DUPLIKAT
# ============================================================

duplicated_subjects = (
    ml_dataset.loc[
        ml_dataset[
            SUBJECT_COLUMN
        ].duplicated(
            keep=False
        ),
        SUBJECT_COLUMN
    ]
    .unique()
    .tolist()
)

if duplicated_subjects:
    raise ValueError(
        "Setiap subject harus memiliki tepat satu baris. "
        "Ditemukan subject duplikat: "
        f"{duplicated_subjects}"
    )


# ============================================================
# 18. MEMERIKSA LABEL YANG TERSEDIA
# ============================================================

valid_text_labels = {
    "LOW/NO",
    "HIGH"
}

dataset_text_labels = set(
    ml_dataset[
        TARGET_COLUMN
    ].dropna().unique()
)

invalid_text_labels = (
    dataset_text_labels
    - valid_text_labels
)

if invalid_text_labels:
    raise ValueError(
        "Dataset mengandung label yang tidak dikenali: "
        f"{invalid_text_labels}"
    )


# ============================================================
# 19. KONVERSI FITUR MENJADI NUMERIK
# ============================================================

ml_dataset[
    model_feature_columns
] = (
    ml_dataset[
        model_feature_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )
)


# ============================================================
# 20. PEMERIKSAAN MISSING VALUE
# ============================================================

missing_value_summary = (
    ml_dataset[
        model_feature_columns
    ]
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

missing_value_summary = (
    missing_value_summary[
        missing_value_summary > 0
    ]
)

# Missing value tidak diisi pada STEP 1.
# Imputasi dilakukan di dalam Pipeline agar hanya belajar
# dari training fold dan tidak menyebabkan data leakage.


# ============================================================
# 21. PEMERIKSAAN CONSTANT FEATURES
# ============================================================

constant_features = [
    feature
    for feature in model_feature_columns
    if ml_dataset[
        feature
    ].nunique(
        dropna=True
    ) <= 1
]


# ============================================================
# 22. PEMERIKSAAN DISTRIBUSI KELAS
# ============================================================

class_distribution_df = (
    ml_dataset[
        TARGET_COLUMN
    ]
    .value_counts()
    .rename_axis(
        "fall_risk"
    )
    .reset_index(
        name="number_of_subjects"
    )
)

class_distribution_df[
    "percentage"
] = (
    class_distribution_df[
        "number_of_subjects"
    ]
    / len(ml_dataset)
    * 100
)


minimum_class_count = int(
    class_distribution_df[
        "number_of_subjects"
    ].min()
)

if minimum_class_count < OUTER_N_SPLITS:
    raise ValueError(
        "Jumlah anggota kelas terkecil tidak mencukupi "
        f"untuk {OUTER_N_SPLITS}-fold stratified CV. "
        f"Jumlah kelas terkecil: {minimum_class_count}."
    )


# ============================================================
# 23. MENAMPILKAN RINGKASAN KONFIGURASI
# ============================================================

print("=" * 72)
print("PCA–MACHINE LEARNING EXPERIMENT INITIALIZATION")
print("=" * 72)

print(
    "Dataset shape                 :",
    ml_dataset.shape
)

print(
    "Number of subjects            :",
    ml_dataset[
        SUBJECT_COLUMN
    ].nunique()
)

print(
    "Number of model features      :",
    len(
        model_feature_columns
    )
)

print(
    "Expected feature count        :",
    EXPECTED_FEATURE_COUNT
)

print(
    "Outer cross-validation        :",
    f"Stratified {OUTER_N_SPLITS}-Fold"
)

print(
    "Inner GridSearchCV            :",
    f"Stratified {INNER_N_SPLITS}-Fold"
)

print(
    "Primary GridSearch metric     :",
    GRID_SCORING
)

print(
    "PCA candidates                :",
    PCA_COMPONENT_CANDIDATES
)

print(
    "Missing values                :",
    int(
        ml_dataset[
            model_feature_columns
        ]
        .isna()
        .sum()
        .sum()
    )
)

print(
    "Constant features             :",
    len(
        constant_features
    )
)

print(
    "Output directory              :",
    PCA_ML_OUTPUT_DIR
)


# ============================================================
# 24. MENAMPILKAN DISTRIBUSI KELAS
# ============================================================

print("\nClass distribution:")

display(
    class_distribution_df
)


# ============================================================
# 25. MENAMPILKAN MISSING VALUE
# ============================================================

if missing_value_summary.empty:

    print(
        "\nTidak terdapat missing value pada fitur model."
    )

else:

    print(
        "\nMissing values akan ditangani oleh median imputation "
        "di dalam setiap training fold:"
    )

    display(
        missing_value_summary.to_frame(
            "missing_count"
        )
    )


# ============================================================
# 26. MENAMPILKAN CONSTANT FEATURES
# ============================================================

if constant_features:

    print(
        "\nPERINGATAN: Fitur berikut memiliki nilai konstan:"
    )

    print(
        constant_features
    )

else:

    print(
        "\nTidak terdapat fitur konstan."
    )


# ============================================================
# 27. MENYIMPAN KONFIGURASI AWAL
# ============================================================

experiment_configuration = {
    "random_state": RANDOM_STATE,
    "number_of_subjects": int(
        ml_dataset[
            SUBJECT_COLUMN
        ].nunique()
    ),
    "number_of_features": int(
        len(
            model_feature_columns
        )
    ),
    "outer_cv": (
        f"StratifiedKFold("
        f"n_splits={OUTER_N_SPLITS}, "
        f"shuffle=True)"
    ),
    "inner_cv": (
        f"StratifiedKFold("
        f"n_splits={INNER_N_SPLITS}, "
        f"shuffle=True)"
    ),
    "grid_scoring": GRID_SCORING,
    "classification_threshold": (
        CLASSIFICATION_THRESHOLD
    ),
    "pca_candidates": (
        PCA_COMPONENT_CANDIDATES
    ),
    "excluded_features": (
        EXCLUDED_MODEL_FEATURES
    ),
    "positive_class": "HIGH",
    "negative_class": "LOW/NO"
}

configuration_path = os.path.join(
    PCA_ML_OUTPUT_DIR,
    "experiment_configuration.json"
)

with open(
    configuration_path,
    "w",
    encoding="utf-8"
) as configuration_file:

    json.dump(
        experiment_configuration,
        configuration_file,
        indent=2,
        ensure_ascii=False
    )


feature_list_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "model_feature_columns.csv"
)

pd.DataFrame({
    "feature_order": np.arange(
        1,
        len(model_feature_columns) + 1
    ),
    "feature": model_feature_columns
}).to_csv(
    feature_list_path,
    index=False
)


class_distribution_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "class_distribution.csv"
)

class_distribution_df.to_csv(
    class_distribution_path,
    index=False
)


# ============================================================
# 28. KONFIRMASI STEP 1
# ============================================================

print("\n" + "=" * 72)
print("STEP 1 COMPLETED")
print("=" * 72)

print(
    "Dataset telah divalidasi dan siap untuk "
    "pembentukan X, y, serta nested cross-validation."
)

print(
    "\nConfiguration saved to:"
)

print(
    configuration_path
)

In [ ]:
# ============================================================
# STEP 2. PREPARE FEATURE MATRIX, TARGET, AND SUBJECT IDENTIFIERS
# Encode Fall-Risk Labels and Validate Model Inputs
# ============================================================

# Output utama STEP 2:
# X                : dataframe fitur multimodal
# y                : target numerik
# subjects         : identitas subject
# feature_columns  : urutan fitur model
#
# Target encoding:
# 0 = LOW/NO
# 1 = HIGH


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP 1
# ============================================================

required_step_2_objects = [
    "ml_dataset",
    "model_feature_columns",
    "SUBJECT_COLUMN",
    "TARGET_COLUMN",
    "EXPECTED_FEATURE_COUNT",
    "CLASS_LABEL_MAPPING",
    "PCA_ML_TABLE_DIR"
]

missing_step_2_objects = [
    object_name
    for object_name in required_step_2_objects
    if object_name not in globals()
]

if missing_step_2_objects:
    raise NameError(
        "Objek untuk STEP 2 belum lengkap: "
        f"{missing_step_2_objects}. "
        "Jalankan STEP 1 terlebih dahulu."
    )


# ============================================================
# 2. MEMBUAT DATAFRAME KERJA
# ============================================================

model_dataset = (
    ml_dataset
    .copy()
    .reset_index(drop=True)
)

feature_columns = (
    model_feature_columns
    .copy()
)


# ============================================================
# 3. MEMBENTUK MATRIKS FITUR X
# ============================================================

X = (
    model_dataset[
        feature_columns
    ]
    .copy()
)

X = (
    X
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )
)


# ============================================================
# 4. MENYIAPKAN IDENTITAS SUBJECT
# ============================================================

subjects = (
    model_dataset[
        SUBJECT_COLUMN
    ]
    .astype(str)
    .str.strip()
    .to_numpy()
)


# ============================================================
# 5. MEMBERSIHKAN LABEL FALL RISK
# ============================================================

raw_labels = (
    model_dataset[
        TARGET_COLUMN
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ============================================================
# 6. MENDEFINISIKAN PEMETAAN LABEL
# ============================================================

label_mapping = {
    "LOW/NO": 0,
    "LOW / NO": 0,
    "LOW-NO": 0,
    "LOW": 0,
    "NO": 0,
    "HIGH": 1
}

y_series = (
    raw_labels
    .map(
        label_mapping
    )
)


# ============================================================
# 7. MEMERIKSA LABEL YANG TIDAK DIKENALI
# ============================================================

unknown_labels = (
    raw_labels[
        y_series.isna()
    ]
    .unique()
    .tolist()
)

if unknown_labels:
    raise ValueError(
        "Terdapat label fall risk yang tidak dikenali: "
        f"{unknown_labels}. "
        "Periksa kolom fall_risk pada dataset."
    )


# ============================================================
# 8. MEMBENTUK TARGET NUMERIK y
# ============================================================

y = (
    y_series
    .astype(int)
    .to_numpy()
)


# ============================================================
# 9. VALIDASI DIMENSI INPUT
# ============================================================

if X.shape[0] != len(y):
    raise ValueError(
        "Jumlah baris X tidak sama dengan jumlah label y. "
        f"X={X.shape[0]}, y={len(y)}."
    )

if X.shape[0] != len(subjects):
    raise ValueError(
        "Jumlah baris X tidak sama dengan jumlah subject. "
        f"X={X.shape[0]}, subjects={len(subjects)}."
    )

if X.shape[1] != EXPECTED_FEATURE_COUNT:
    raise ValueError(
        f"Jumlah fitur seharusnya "
        f"{EXPECTED_FEATURE_COUNT}, tetapi diperoleh "
        f"{X.shape[1]}."
    )

if X.shape[1] != len(
    feature_columns
):
    raise ValueError(
        "Jumlah kolom X tidak sama dengan jumlah "
        "feature_columns."
    )


# ============================================================
# 10. VALIDASI SUBJECT UNIK
# ============================================================

duplicated_subject_count = int(
    pd.Series(
        subjects
    )
    .duplicated()
    .sum()
)

if duplicated_subject_count > 0:
    raise ValueError(
        f"Ditemukan {duplicated_subject_count} "
        "subject duplikat."
    )


# ============================================================
# 11. VALIDASI KELAS TARGET
# ============================================================

unique_target_values = np.unique(
    y
)

if not np.array_equal(
    unique_target_values,
    np.array([
        0,
        1
    ])
):
    raise ValueError(
        "Target harus memiliki tepat dua kelas "
        "[0, 1]. "
        f"Diperoleh: {unique_target_values}."
    )


# ============================================================
# 12. MEMERIKSA KESELARASAN LABEL TEKS DAN NUMERIK
# ============================================================

label_alignment_df = pd.DataFrame({
    SUBJECT_COLUMN: subjects,
    "fall_risk_text": raw_labels.to_numpy(),
    "fall_risk_encoded": y
})

label_alignment_table = pd.crosstab(
    label_alignment_df[
        "fall_risk_text"
    ],
    label_alignment_df[
        "fall_risk_encoded"
    ],
    margins=True
)

expected_low_count = int(
    (
        raw_labels == "LOW/NO"
    ).sum()
)

expected_high_count = int(
    (
        raw_labels == "HIGH"
    ).sum()
)

actual_low_encoded_count = int(
    (
        (
            raw_labels == "LOW/NO"
        )
        & (
            y_series == 0
        )
    ).sum()
)

actual_high_encoded_count = int(
    (
        (
            raw_labels == "HIGH"
        )
        & (
            y_series == 1
        )
    ).sum()
)

if (
    actual_low_encoded_count
    != expected_low_count
):
    raise ValueError(
        "Mapping LOW/NO → 0 tidak konsisten."
    )

if (
    actual_high_encoded_count
    != expected_high_count
):
    raise ValueError(
        "Mapping HIGH → 1 tidak konsisten."
    )


# ============================================================
# 13. MEMERIKSA MISSING VALUE
# ============================================================

missing_value_summary = (
    X
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

missing_value_summary = (
    missing_value_summary[
        missing_value_summary > 0
    ]
)

total_missing_values = int(
    X
    .isna()
    .sum()
    .sum()
)


# ============================================================
# 14. MEMERIKSA NILAI KONSTAN
# ============================================================

constant_features = [
    column
    for column in feature_columns
    if X[
        column
    ].nunique(
        dropna=True
    ) <= 1
]


# ============================================================
# 15. MEMERIKSA FITUR DENGAN VARIANSI SANGAT RENDAH
# ============================================================

numeric_variance = (
    X
    .var(
        axis=0,
        skipna=True
    )
)

LOW_VARIANCE_THRESHOLD = 1e-12

near_constant_features = (
    numeric_variance[
        numeric_variance
        <= LOW_VARIANCE_THRESHOLD
    ]
    .index
    .tolist()
)


# ============================================================
# 16. MEMERIKSA DISTRIBUSI KELAS
# ============================================================

class_distribution_df = pd.DataFrame({
    "class_id": [
        0,
        1
    ],
    "class_name": [
        CLASS_LABEL_MAPPING[0],
        CLASS_LABEL_MAPPING[1]
    ],
    "number_of_subjects": [
        int(
            (
                y == 0
            ).sum()
        ),
        int(
            (
                y == 1
            ).sum()
        )
    ]
})

class_distribution_df[
    "percentage"
] = (
    class_distribution_df[
        "number_of_subjects"
    ]
    / len(y)
    * 100
)


# ============================================================
# 17. MEMERIKSA KELAYAKAN STRATIFIED CROSS-VALIDATION
# ============================================================

minimum_class_count = int(
    class_distribution_df[
        "number_of_subjects"
    ].min()
)

if minimum_class_count < OUTER_N_SPLITS:
    raise ValueError(
        "Jumlah anggota kelas terkecil tidak mencukupi "
        f"untuk outer {OUTER_N_SPLITS}-fold CV. "
        f"Jumlah kelas terkecil: {minimum_class_count}."
    )

# Pada outer 5-fold, setiap training fold berisi sekitar 80%
# data. Inner 3-fold harus tetap memiliki minimal satu sampel
# kelas minoritas pada setiap fold.
estimated_outer_training_minority_count = int(
    np.floor(
        minimum_class_count
        * (
            OUTER_N_SPLITS - 1
        )
        / OUTER_N_SPLITS
    )
)

if (
    estimated_outer_training_minority_count
    < INNER_N_SPLITS
):
    raise ValueError(
        "Perkiraan jumlah kelas minoritas pada outer training "
        "fold tidak mencukupi untuk inner cross-validation. "
        f"Perkiraan: {estimated_outer_training_minority_count}, "
        f"inner folds: {INNER_N_SPLITS}."
    )


# ============================================================
# 18. MEMBENTUK ARRAY UNTUK SCIKIT-LEARN
# ============================================================

X_array = (
    X
    .to_numpy(
        dtype=float
    )
)

y_array = (
    np.asarray(
        y,
        dtype=int
    )
)

subject_array = (
    np.asarray(
        subjects
    )
)


# ============================================================
# 19. MENAMPILKAN RINGKASAN INPUT MODEL
# ============================================================

print("=" * 72)
print("PCA–MACHINE LEARNING MODEL INPUTS")
print("=" * 72)

print(
    "Feature matrix shape         :",
    X.shape
)

print(
    "Target vector shape          :",
    y.shape
)

print(
    "Subject vector shape         :",
    subjects.shape
)

print(
    "Number of subjects           :",
    len(
        subjects
    )
)

print(
    "Number of features           :",
    len(
        feature_columns
    )
)

print(
    "Number of classes            :",
    len(
        unique_target_values
    )
)

print(
    "Missing values               :",
    total_missing_values
)

print(
    "Constant features            :",
    len(
        constant_features
    )
)

print(
    "Near-constant features       :",
    len(
        near_constant_features
    )
)

print(
    "Duplicated subjects          :",
    duplicated_subject_count
)


# ============================================================
# 20. MENAMPILKAN DISTRIBUSI KELAS
# ============================================================

print("\nClass distribution:")

display(
    class_distribution_df
)


# ============================================================
# 21. MENAMPILKAN KESELARASAN LABEL
# ============================================================

print("\nText and numeric label alignment:")

display(
    label_alignment_table
)


# ============================================================
# 22. MENAMPILKAN MISSING VALUES
# ============================================================

if missing_value_summary.empty:

    print(
        "\nTidak terdapat missing value pada fitur model."
    )

else:

    print(
        "\nMissing value akan diimputasi menggunakan median "
        "di dalam training fold:"
    )

    display(
        missing_value_summary.to_frame(
            "missing_count"
        )
    )


# ============================================================
# 23. MENAMPILKAN FITUR KONSTAN
# ============================================================

if constant_features:

    print(
        "\nPERINGATAN: Fitur konstan:"
    )

    print(
        constant_features
    )

else:

    print(
        "\nTidak terdapat fitur konstan."
    )


if near_constant_features:

    print(
        "\nPERINGATAN: Fitur dengan variansi sangat rendah:"
    )

    print(
        near_constant_features
    )


# ============================================================
# 24. PREVIEW DATASET MODEL
# ============================================================

preview_feature_count = min(
    10,
    len(
        feature_columns
    )
)

input_preview_df = pd.concat(
    [
        label_alignment_df.reset_index(
            drop=True
        ),
        X[
            feature_columns[
                :preview_feature_count
            ]
        ].reset_index(
            drop=True
        )
    ],
    axis=1
)

print(
    f"\nPreview input model dengan "
    f"{preview_feature_count} fitur pertama:"
)

display(
    input_preview_df.head()
)


# ============================================================
# 25. MENYIMPAN INPUT MODEL
# ============================================================

model_input_dataset = pd.concat(
    [
        label_alignment_df.reset_index(
            drop=True
        ),
        X.reset_index(
            drop=True
        )
    ],
    axis=1
)

model_input_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "pca_ml_model_input_dataset.csv"
)

model_input_dataset.to_csv(
    model_input_path,
    index=False
)


label_alignment_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "label_alignment.csv"
)

label_alignment_df.to_csv(
    label_alignment_path,
    index=False
)


class_distribution_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "class_distribution_encoded.csv"
)

class_distribution_df.to_csv(
    class_distribution_path,
    index=False
)


missing_value_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "missing_value_summary.csv"
)

missing_value_summary.to_frame(
    "missing_count"
).to_csv(
    missing_value_path
)


# ============================================================
# 26. KONFIRMASI STEP 2
# ============================================================

print("\n" + "=" * 72)
print("STEP 2 COMPLETED")
print("=" * 72)

print(
    "Target encoding:"
)

print(
    "0 = LOW/NO"
)

print(
    "1 = HIGH"
)

print(
    "\nX, y, subjects, feature_columns, X_array, "
    "y_array, dan subject_array siap digunakan."
)

print(
    "\nModel input saved to:"
)

print(
    model_input_path
)

In [ ]:
# ============================================================
# STEP 4. DEFINE HYPERPARAMETER GRIDS
# PCA Components and Classifier-Specific Parameters
# ============================================================

# GridSearchCV akan memilih:
# 1. Jumlah komponen PCA atau proporsi explained variance.
# 2. Hyperparameter classifier.
#
# Seluruh proses tuning dilakukan hanya pada inner training
# fold dalam nested cross-validation.


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP 3
# ============================================================

required_step_4_objects = [
    "ml_pipelines",
    "valid_pca_candidates",
    "RANDOM_STATE",
    "PCA_ML_TABLE_DIR"
]

missing_step_4_objects = [
    object_name
    for object_name in required_step_4_objects
    if object_name not in globals()
]

if missing_step_4_objects:
    raise NameError(
        "Objek untuk STEP 4 belum lengkap: "
        f"{missing_step_4_objects}. "
        "Jalankan STEP 3 terlebih dahulu."
    )


# ============================================================
# 2. PCA GRID DASAR
# ============================================================

pca_grid = {
    "pca__n_components": (
        valid_pca_candidates
    )
}


# ============================================================
# 3. LOGISTIC REGRESSION GRID
# ============================================================

# Solver liblinear mendukung:
# - L1 penalty
# - L2 penalty
#
# C kecil menghasilkan regularisasi lebih kuat.

logistic_regression_grid = {
    **pca_grid,

    "classifier__C": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],

    "classifier__penalty": [
        "l1",
        "l2"
    ],

    "classifier__solver": [
        "liblinear"
    ]
}


# ============================================================
# 4. SVM-RBF GRID
# ============================================================

# C:
# - kecil  = margin lebih lunak dan regularisasi lebih kuat
# - besar  = penalti kesalahan klasifikasi lebih kuat
#
# gamma:
# - kecil  = decision boundary lebih halus
# - besar  = decision boundary lebih kompleks

svm_rbf_grid = {
    **pca_grid,

    "classifier__C": [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],

    "classifier__gamma": [
        "scale",
        "auto",
        0.001,
        0.01,
        0.1,
        1.0
    ],

    "classifier__kernel": [
        "rbf"
    ]
}


# ============================================================
# 5. RANDOM FOREST GRID
# ============================================================

# Random Forest tetap menerima data hasil PCA agar struktur
# eksperimen sama untuk seluruh classifier.
#
# max_depth dan min_samples_leaf dibatasi untuk mengurangi
# risiko overfitting pada dataset 30 subject.

random_forest_grid = {
    **pca_grid,

    "classifier__n_estimators": [
        100,
        300,
        500
    ],

    "classifier__max_depth": [
        2,
        3,
        5,
        None
    ],

    "classifier__min_samples_split": [
        2,
        4,
        6
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        3
    ],

    "classifier__max_features": [
        "sqrt",
        "log2",
        None
    ],

    "classifier__criterion": [
        "gini",
        "entropy"
    ]
}


# ============================================================
# 6. DECISION TREE GRID
# ============================================================

decision_tree_grid = {
    **pca_grid,

    "classifier__criterion": [
        "gini",
        "entropy",
        "log_loss"
    ],

    "classifier__max_depth": [
        1,
        2,
        3,
        4,
        5,
        None
    ],

    "classifier__min_samples_split": [
        2,
        4,
        6,
        8
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        3,
        4
    ],

    "classifier__max_features": [
        None,
        "sqrt",
        "log2"
    ]
}


# ============================================================
# 7. K-NEAREST NEIGHBORS GRID
# ============================================================

# Jumlah tetangga tidak boleh terlalu besar terhadap ukuran
# inner training fold.
#
# Nilai ganjil diprioritaskan untuk mengurangi kemungkinan tie.

knn_grid = {
    **pca_grid,

    "classifier__n_neighbors": [
        3,
        5,
        7,
        9,
        11
    ],

    "classifier__weights": [
        "uniform",
        "distance"
    ],

    "classifier__metric": [
        "euclidean",
        "manhattan",
        "minkowski"
    ],

    "classifier__p": [
        1,
        2
    ]
}


# ============================================================
# 8. GAUSSIAN NAIVE BAYES GRID
# ============================================================

# var_smoothing menambahkan sebagian variansi terbesar
# untuk menjaga kestabilan numerik.

gaussian_nb_grid = {
    **pca_grid,

    "classifier__var_smoothing": np.logspace(
        -12,
        -6,
        7
    ).tolist()
}


# ============================================================
# 9. MENYIMPAN SELURUH GRID DALAM DICTIONARY
# ============================================================

ml_param_grids = {
    "Logistic Regression": (
        logistic_regression_grid
    ),
    "SVM-RBF": (
        svm_rbf_grid
    ),
    "Random Forest": (
        random_forest_grid
    ),
    "Decision Tree": (
        decision_tree_grid
    ),
    "k-Nearest Neighbors": (
        knn_grid
    ),
    "Gaussian Naive Bayes": (
        gaussian_nb_grid
    )
}


# ============================================================
# 10. VALIDASI KESESUAIAN NAMA MODEL
# ============================================================

pipeline_model_names = set(
    ml_pipelines.keys()
)

grid_model_names = set(
    ml_param_grids.keys()
)

if pipeline_model_names != grid_model_names:
    raise ValueError(
        "Nama model pada ml_pipelines dan ml_param_grids "
        "tidak sama.\n"
        f"Pipeline: {sorted(pipeline_model_names)}\n"
        f"Grid: {sorted(grid_model_names)}"
    )


# ============================================================
# 11. VALIDASI SETIAP PARAMETER GRID
# ============================================================

grid_validation_records = []

for model_name, parameter_grid in (
    ml_param_grids.items()
):

    model_pipeline = (
        ml_pipelines[
            model_name
        ]
    )

    valid_pipeline_parameters = (
        model_pipeline
        .get_params()
        .keys()
    )

    invalid_parameters = [
        parameter_name
        for parameter_name
        in parameter_grid.keys()
        if parameter_name
        not in valid_pipeline_parameters
    ]

    if invalid_parameters:
        raise ValueError(
            f"Parameter grid {model_name} mengandung "
            f"parameter tidak valid: {invalid_parameters}"
        )

    parameter_value_counts = {
        parameter_name: len(
            parameter_values
        )
        for parameter_name, parameter_values
        in parameter_grid.items()
    }

    number_of_combinations = int(
        np.prod(
            list(
                parameter_value_counts.values()
            )
        )
    )

    grid_validation_records.append({
        "model": model_name,
        "number_of_parameters": int(
            len(
                parameter_grid
            )
        ),
        "number_of_combinations": (
            number_of_combinations
        ),
        "parameter_names": ", ".join(
            parameter_grid.keys()
        )
    })


grid_summary_df = pd.DataFrame(
    grid_validation_records
)


# ============================================================
# 12. MENGHITUNG ESTIMASI JUMLAH FIT
# ============================================================

# Estimasi:
# jumlah kombinasi × inner folds × outer folds
#
# Nilai ini belum memasukkan refit model terbaik pada
# setiap outer fold.

grid_summary_df[
    "estimated_inner_fits_per_outer_fold"
] = (
    grid_summary_df[
        "number_of_combinations"
    ]
    * INNER_N_SPLITS
)

grid_summary_df[
    "estimated_total_nested_fits"
] = (
    grid_summary_df[
        "estimated_inner_fits_per_outer_fold"
    ]
    * OUTER_N_SPLITS
)


# ============================================================
# 13. PERINGATAN KOMPUTASI
# ============================================================

TOTAL_ESTIMATED_NESTED_FITS = int(
    grid_summary_df[
        "estimated_total_nested_fits"
    ].sum()
)

LARGE_GRID_WARNING_THRESHOLD = 100000

if (
    TOTAL_ESTIMATED_NESTED_FITS
    > LARGE_GRID_WARNING_THRESHOLD
):
    print(
        "PERINGATAN: Total estimasi model fit sangat besar:",
        TOTAL_ESTIMATED_NESTED_FITS
    )

    print(
        "Grid mungkin membutuhkan waktu komputasi yang panjang."
    )


# ============================================================
# 14. MENAMPILKAN RINGKASAN GRID
# ============================================================

print("=" * 72)
print("PCA–MACHINE LEARNING HYPERPARAMETER GRIDS")
print("=" * 72)

print(
    "Number of models            :",
    len(
        ml_param_grids
    )
)

print(
    "PCA candidates              :",
    valid_pca_candidates
)

print(
    "Inner CV folds              :",
    INNER_N_SPLITS
)

print(
    "Outer CV folds              :",
    OUTER_N_SPLITS
)

print(
    "Total estimated nested fits :",
    TOTAL_ESTIMATED_NESTED_FITS
)

display(
    grid_summary_df
)


# ============================================================
# 15. MENAMPILKAN DETAIL GRID SETIAP MODEL
# ============================================================

for model_name, parameter_grid in (
    ml_param_grids.items()
):

    print("\n" + "=" * 72)
    print(model_name.upper())
    print("=" * 72)

    for parameter_name, parameter_values in (
        parameter_grid.items()
    ):

        print(
            f"{parameter_name}:"
        )

        print(
            parameter_values
        )


# ============================================================
# 16. MEMBENTUK VERSI GRID RINGKAS
# ============================================================

# Grid ringkas dapat dipakai untuk uji awal apabila grid lengkap
# terlalu berat secara komputasi.

compact_param_grids = {
    "Logistic Regression": {
        "pca__n_components": [
            3,
            5,
            8,
            10,
            0.90,
            0.95
        ],
        "classifier__C": [
            0.01,
            0.1,
            1.0,
            10.0
        ],
        "classifier__penalty": [
            "l2"
        ],
        "classifier__solver": [
            "liblinear"
        ]
    },

    "SVM-RBF": {
        "pca__n_components": [
            3,
            5,
            8,
            10,
            0.90,
            0.95
        ],
        "classifier__C": [
            0.1,
            1.0,
            10.0
        ],
        "classifier__gamma": [
            "scale",
            0.01,
            0.1
        ],
        "classifier__kernel": [
            "rbf"
        ]
    },

    "Random Forest": {
        "pca__n_components": [
            3,
            5,
            8,
            10,
            0.90,
            0.95
        ],
        "classifier__n_estimators": [
            100,
            300
        ],
        "classifier__max_depth": [
            2,
            3,
            5
        ],
        "classifier__min_samples_leaf": [
            1,
            2
        ],
        "classifier__max_features": [
            "sqrt"
        ]
    },

    "Decision Tree": {
        "pca__n_components": [
            3,
            5,
            8,
            10,
            0.90,
            0.95
        ],
        "classifier__criterion": [
            "gini",
            "entropy"
        ],
        "classifier__max_depth": [
            1,
            2,
            3,
            5
        ],
        "classifier__min_samples_leaf": [
            1,
            2,
            3
        ]
    },

    "k-Nearest Neighbors": {
        "pca__n_components": [
            3,
            5,
            8,
            10,
            0.90,
            0.95
        ],
        "classifier__n_neighbors": [
            3,
            5,
            7,
            9
        ],
        "classifier__weights": [
            "uniform",
            "distance"
        ],
        "classifier__metric": [
            "euclidean",
            "manhattan"
        ],
        "classifier__p": [
            1,
            2
        ]
    },

    "Gaussian Naive Bayes": {
        "pca__n_components": [
            3,
            5,
            8,
            10,
            0.90,
            0.95
        ],
        "classifier__var_smoothing": [
            1e-11,
            1e-10,
            1e-9,
            1e-8,
            1e-7
        ]
    }
}


# ============================================================
# 17. MEMILIH MODE GRID
# ============================================================

# Gunakan:
# "compact" untuk uji awal yang lebih cepat.
# "full" untuk eksperimen lengkap.
#
# Dengan 30 subject, saya menyarankan memulai dari compact.

GRID_MODE = "compact"

if GRID_MODE == "compact":

    active_param_grids = (
        compact_param_grids
    )

elif GRID_MODE == "full":

    active_param_grids = (
        ml_param_grids
    )

else:
    raise ValueError(
        "GRID_MODE harus bernilai 'compact' atau 'full'."
    )


# ============================================================
# 18. RINGKASAN ACTIVE GRID
# ============================================================

active_grid_records = []

for model_name, parameter_grid in (
    active_param_grids.items()
):

    number_of_combinations = int(
        np.prod([
            len(parameter_values)
            for parameter_values
            in parameter_grid.values()
        ])
    )

    active_grid_records.append({
        "model": model_name,
        "grid_mode": GRID_MODE,
        "number_of_combinations": (
            number_of_combinations
        ),
        "estimated_nested_fits": (
            number_of_combinations
            * INNER_N_SPLITS
            * OUTER_N_SPLITS
        )
    })


active_grid_summary_df = pd.DataFrame(
    active_grid_records
)

print("\n" + "=" * 72)
print("ACTIVE GRID CONFIGURATION")
print("=" * 72)

print(
    "Grid mode:",
    GRID_MODE
)

display(
    active_grid_summary_df
)

print(
    "Total active estimated fits:",
    int(
        active_grid_summary_df[
            "estimated_nested_fits"
        ].sum()
    )
)


# ============================================================
# 19. MENYIMPAN RINGKASAN GRID
# ============================================================

full_grid_summary_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "full_hyperparameter_grid_summary.csv"
)

grid_summary_df.to_csv(
    full_grid_summary_path,
    index=False
)


active_grid_summary_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "active_hyperparameter_grid_summary.csv"
)

active_grid_summary_df.to_csv(
    active_grid_summary_path,
    index=False
)


active_grid_configuration_path = os.path.join(
    PCA_ML_OUTPUT_DIR,
    "active_hyperparameter_grids.json"
)

serializable_active_grids = {}

for model_name, parameter_grid in (
    active_param_grids.items()
):

    serializable_active_grids[
        model_name
    ] = {
        parameter_name: [
            (
                value.item()
                if isinstance(
                    value,
                    np.generic
                )
                else value
            )
            for value in parameter_values
        ]
        for parameter_name, parameter_values
        in parameter_grid.items()
    }


with open(
    active_grid_configuration_path,
    "w",
    encoding="utf-8"
) as grid_file:

    json.dump(
        serializable_active_grids,
        grid_file,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 20. KONFIRMASI STEP 4
# ============================================================

print("\n" + "=" * 72)
print("STEP 4 COMPLETED")
print("=" * 72)

print(
    "Hyperparameter grids berhasil disiapkan."
)

print(
    "Active grid mode:",
    GRID_MODE
)

print(
    "\nGridSearchCV akan menggunakan:"
)

for model_name in active_param_grids:
    print(
        f"- {model_name}"
    )

print(
    "\nPada STEP 5, GridSearchCV akan dijalankan "
    "di dalam outer stratified cross-validation."
)

In [ ]:
# ============================================================
# STEP 5. RUN NESTED STRATIFIED CROSS-VALIDATION
# Inner GridSearchCV + Outer Model Evaluation
# ============================================================

# Proses pada setiap outer fold:
#
# Outer training data
#       ↓
# Inner GridSearchCV
#       ↓
# Imputation + Scaling + PCA + Hyperparameter tuning
#       ↓
# Best estimator
#       ↓
# Prediction on untouched outer test fold
#
# Hasil utama STEP 5:
# 1. Prediksi out-of-fold setiap subject.
# 2. Metrik setiap outer fold.
# 3. Hyperparameter terbaik setiap fold.
# 4. Jumlah komponen PCA terbaik setiap fold.
# 5. Ringkasan performa nested cross-validation.


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP SEBELUMNYA
# ============================================================

required_step_5_objects = [
    "X",
    "y",
    "subjects",
    "feature_columns",
    "ml_pipelines",
    "active_param_grids",
    "model_score_methods",
    "model_abbreviations",
    "outer_cv",
    "INNER_N_SPLITS",
    "GRID_SCORING",
    "GRID_N_JOBS",
    "GRID_VERBOSE",
    "RANDOM_STATE",
    "CLASS_LABEL_MAPPING",
    "PCA_ML_TABLE_DIR"
]

missing_step_5_objects = [
    object_name
    for object_name in required_step_5_objects
    if object_name not in globals()
]

if missing_step_5_objects:
    raise NameError(
        "Objek untuk STEP 5 belum lengkap: "
        f"{missing_step_5_objects}. "
        "Jalankan STEP 1–4 terlebih dahulu."
    )


# ============================================================
# 2. KONVERSI INPUT MENJADI ARRAY
# ============================================================

X_nested = (
    X
    .to_numpy(
        dtype=float
    )
)

y_nested = np.asarray(
    y,
    dtype=int
)

subjects_nested = np.asarray(
    subjects
)


if X_nested.shape[0] != len(y_nested):
    raise ValueError(
        "Jumlah baris X tidak sama dengan jumlah target."
    )

if X_nested.shape[0] != len(
    subjects_nested
):
    raise ValueError(
        "Jumlah baris X tidak sama dengan jumlah subject."
    )

if X_nested.shape[1] != len(
    feature_columns
):
    raise ValueError(
        "Jumlah kolom X tidak sama dengan feature_columns."
    )


# ============================================================
# 3. FUNGSI MENGAMBIL SKOR KELAS HIGH
# ============================================================

def extract_positive_class_score(
    fitted_estimator,
    X_input,
    score_method
):
    """
    Mengambil skor kontinu untuk kelas HIGH = 1.

    Parameters
    ----------
    fitted_estimator : sklearn Pipeline
        Pipeline yang telah dilatih.

    X_input : array-like
        Data yang akan diprediksi.

    score_method : str
        predict_proba atau decision_function.

    Returns
    -------
    numpy.ndarray
        Skor kontinu kelas HIGH.
    """

    classifier = (
        fitted_estimator
        .named_steps["classifier"]
    )

    classifier_classes = np.asarray(
        classifier.classes_
    )

    if not np.array_equal(
        classifier_classes,
        np.array([
            0,
            1
        ])
    ):
        raise ValueError(
            "Urutan kelas classifier harus [0, 1]. "
            f"Diperoleh: {classifier_classes}."
        )

    if score_method == "predict_proba":

        probabilities = (
            fitted_estimator
            .predict_proba(
                X_input
            )
        )

        positive_class_index = int(
            np.where(
                classifier_classes == 1
            )[0][0]
        )

        positive_scores = (
            probabilities[
                :,
                positive_class_index
            ]
        )

    elif score_method == "decision_function":

        positive_scores = np.ravel(
            fitted_estimator
            .decision_function(
                X_input
            )
        )

    else:
        raise ValueError(
            "score_method harus 'predict_proba' "
            "atau 'decision_function'."
        )

    positive_scores = np.asarray(
        positive_scores,
        dtype=float
    )

    if np.isnan(
        positive_scores
    ).any():
        raise ValueError(
            "Skor prediksi mengandung NaN."
        )

    if np.isinf(
        positive_scores
    ).any():
        raise ValueError(
            "Skor prediksi mengandung infinite."
        )

    return positive_scores


# ============================================================
# 4. FUNGSI MENGHITUNG METRIK BINARY CLASSIFICATION
# ============================================================

def calculate_nested_binary_metrics(
    y_true,
    y_pred,
    y_score
):
    """
    Menghitung metrik klasifikasi dengan HIGH = 1.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int
    )

    y_score = np.asarray(
        y_score,
        dtype=float
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1
        ]
    )

    tn, fp, fn, tp = cm.ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    negative_predictive_value = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else 0.0
    )

    roc_auc = (
        roc_auc_score(
            y_true,
            y_score
        )
        if len(
            np.unique(
                y_true
            )
        ) == 2
        else np.nan
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                y_pred
            )
        ),

        "precision": precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),

        "recall": sensitivity,

        "sensitivity": sensitivity,

        "specificity": specificity,

        "negative_predictive_value": (
            negative_predictive_value
        ),

        "f1_score": f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),

        "roc_auc": roc_auc,

        "true_negative": int(tn),

        "false_positive": int(fp),

        "false_negative": int(fn),

        "true_positive": int(tp)
    }


# ============================================================
# 5. INISIALISASI PENYIMPAN HASIL
# ============================================================

nested_cv_results = {}

all_model_prediction_records = []

all_model_fold_metric_records = []

all_model_best_parameter_records = []

all_model_grid_search_records = []

all_model_summary_records = []


# ============================================================
# 6. EVALUASI SETIAP MODEL
# ============================================================

for model_index, (
    model_name,
    base_pipeline
) in enumerate(
    ml_pipelines.items(),
    start=1
):

    print("\n" + "=" * 72)
    print(
        f"MODEL {model_index}/{len(ml_pipelines)}: "
        f"{model_name.upper()}"
    )
    print("=" * 72)

    parameter_grid = (
        active_param_grids[
            model_name
        ]
    )

    score_method = (
        model_score_methods[
            model_name
        ]
    )

    model_start_time = time.time()

    model_prediction_records = []

    model_fold_metric_records = []

    model_best_parameter_records = []

    model_grid_search_records = []

    model_y_true = []

    model_y_pred = []

    model_y_score = []

    model_subjects = []


    # ========================================================
    # 7. OUTER STRATIFIED CROSS-VALIDATION
    # ========================================================

    for outer_fold, (
        outer_train_indices,
        outer_test_indices
    ) in enumerate(
        outer_cv.split(
            X_nested,
            y_nested
        ),
        start=1
    ):

        fold_start_time = time.time()

        print(
            f"\nOuter fold "
            f"{outer_fold}/{OUTER_N_SPLITS}"
        )

        X_outer_train = X_nested[
            outer_train_indices
        ]

        X_outer_test = X_nested[
            outer_test_indices
        ]

        y_outer_train = y_nested[
            outer_train_indices
        ]

        y_outer_test = y_nested[
            outer_test_indices
        ]

        subjects_outer_test = (
            subjects_nested[
                outer_test_indices
            ]
        )


        # ====================================================
        # 8. VALIDASI DISTRIBUSI KELAS OUTER FOLD
        # ====================================================

        train_class_counts = (
            pd.Series(
                y_outer_train
            )
            .value_counts()
            .sort_index()
        )

        test_class_counts = (
            pd.Series(
                y_outer_test
            )
            .value_counts()
            .sort_index()
        )

        if len(
            np.unique(
                y_outer_train
            )
        ) != 2:
            raise ValueError(
                f"Outer training fold {outer_fold} "
                "tidak memiliki dua kelas."
            )

        if len(
            np.unique(
                y_outer_test
            )
        ) != 2:
            raise ValueError(
                f"Outer test fold {outer_fold} "
                "tidak memiliki dua kelas."
            )


        # ====================================================
        # 9. INNER CROSS-VALIDATION KHUSUS OUTER FOLD
        # ====================================================

        inner_cv_fold = StratifiedKFold(
            n_splits=INNER_N_SPLITS,
            shuffle=True,
            random_state=(
                RANDOM_STATE
                + outer_fold
            )
        )


        # ====================================================
        # 10. GRID SEARCH
        # ====================================================

        grid_search = GridSearchCV(
            estimator=clone(
                base_pipeline
            ),
            param_grid=parameter_grid,
            scoring=GRID_SCORING,
            cv=inner_cv_fold,
            n_jobs=GRID_N_JOBS,
            verbose=GRID_VERBOSE,
            refit=True,
            return_train_score=False,
            error_score="raise"
        )

        grid_search.fit(
            X_outer_train,
            y_outer_train
        )


        # ====================================================
        # 11. BEST ESTIMATOR DAN HYPERPARAMETER
        # ====================================================

        best_estimator = (
            grid_search
            .best_estimator_
        )

        best_parameters = (
            grid_search
            .best_params_
        )

        best_inner_auc = float(
            grid_search
            .best_score_
        )

        best_pca = (
            best_estimator
            .named_steps["pca"]
        )

        selected_pca_parameter = (
            best_parameters.get(
                "pca__n_components"
            )
        )

        actual_pca_components = int(
            best_pca.n_components_
        )

        pca_explained_variance = float(
            best_pca
            .explained_variance_ratio_
            .sum()
        )


        # ====================================================
        # 12. PREDIKSI OUTER TEST FOLD
        # ====================================================

        outer_predicted_labels = (
            best_estimator
            .predict(
                X_outer_test
            )
            .astype(int)
        )

        outer_prediction_scores = (
            extract_positive_class_score(
                fitted_estimator=(
                    best_estimator
                ),
                X_input=(
                    X_outer_test
                ),
                score_method=(
                    score_method
                )
            )
        )


        # ====================================================
        # 13. VALIDASI PREDIKSI OUTER TEST
        # ====================================================

        if len(
            outer_predicted_labels
        ) != len(
            y_outer_test
        ):
            raise ValueError(
                f"Jumlah prediksi pada fold "
                f"{outer_fold} tidak sesuai."
            )

        if len(
            outer_prediction_scores
        ) != len(
            y_outer_test
        ):
            raise ValueError(
                f"Jumlah skor pada fold "
                f"{outer_fold} tidak sesuai."
            )


        # ====================================================
        # 14. METRIK OUTER FOLD
        # ====================================================

        fold_metrics = (
            calculate_nested_binary_metrics(
                y_true=(
                    y_outer_test
                ),
                y_pred=(
                    outer_predicted_labels
                ),
                y_score=(
                    outer_prediction_scores
                )
            )
        )


        # ====================================================
        # 15. MENYIMPAN PREDIKSI PER SUBJECT
        # ====================================================

        for local_index, subject_name in enumerate(
            subjects_outer_test
        ):

            actual_label = int(
                y_outer_test[
                    local_index
                ]
            )

            predicted_label = int(
                outer_predicted_labels[
                    local_index
                ]
            )

            prediction_score = float(
                outer_prediction_scores[
                    local_index
                ]
            )

            model_prediction_records.append({
                "model": model_name,
                "model_abbreviation": (
                    model_abbreviations[
                        model_name
                    ]
                ),
                "outer_fold": int(
                    outer_fold
                ),
                "subject": subject_name,
                "actual_label": actual_label,
                "actual_class": (
                    CLASS_LABEL_MAPPING[
                        actual_label
                    ]
                ),
                "predicted_label": predicted_label,
                "predicted_class": (
                    CLASS_LABEL_MAPPING[
                        predicted_label
                    ]
                ),
                "prediction_score": (
                    prediction_score
                ),
                "score_method": (
                    score_method
                ),
                "correct_prediction": bool(
                    actual_label
                    == predicted_label
                )
            })


        # ====================================================
        # 16. MENYIMPAN METRIK PER OUTER FOLD
        # ====================================================

        fold_elapsed_seconds = float(
            time.time()
            - fold_start_time
        )

        model_fold_metric_records.append({
            "model": model_name,
            "model_abbreviation": (
                model_abbreviations[
                    model_name
                ]
            ),
            "outer_fold": int(
                outer_fold
            ),
            "outer_train_size": int(
                len(
                    outer_train_indices
                )
            ),
            "outer_test_size": int(
                len(
                    outer_test_indices
                )
            ),
            "training_class_0": int(
                train_class_counts.get(
                    0,
                    0
                )
            ),
            "training_class_1": int(
                train_class_counts.get(
                    1,
                    0
                )
            ),
            "test_class_0": int(
                test_class_counts.get(
                    0,
                    0
                )
            ),
            "test_class_1": int(
                test_class_counts.get(
                    1,
                    0
                )
            ),
            "best_inner_auc": (
                best_inner_auc
            ),
            "selected_pca_parameter": (
                selected_pca_parameter
            ),
            "actual_pca_components": (
                actual_pca_components
            ),
            "pca_explained_variance": (
                pca_explained_variance
            ),
            "elapsed_seconds": (
                fold_elapsed_seconds
            ),
            **fold_metrics
        })


        # ====================================================
        # 17. MENYIMPAN HYPERPARAMETER TERBAIK
        # ====================================================

        parameter_record = {
            "model": model_name,
            "outer_fold": int(
                outer_fold
            ),
            "best_inner_auc": (
                best_inner_auc
            ),
            "actual_pca_components": (
                actual_pca_components
            ),
            "pca_explained_variance": (
                pca_explained_variance
            )
        }

        for parameter_name, parameter_value in (
            best_parameters.items()
        ):

            parameter_record[
                parameter_name
            ] = parameter_value

        model_best_parameter_records.append(
            parameter_record
        )


        # ====================================================
        # 18. MENYIMPAN SELURUH HASIL GRID SEARCH
        # ====================================================

        grid_result_df = pd.DataFrame(
            grid_search.cv_results_
        )

        selected_grid_columns = [
            column
            for column in [
                "rank_test_score",
                "mean_test_score",
                "std_test_score",
                "mean_fit_time",
                "std_fit_time",
                "mean_score_time",
                "params"
            ]
            if column in grid_result_df.columns
        ]

        fold_grid_result_df = (
            grid_result_df[
                selected_grid_columns
            ]
            .copy()
        )

        fold_grid_result_df.insert(
            0,
            "outer_fold",
            outer_fold
        )

        fold_grid_result_df.insert(
            0,
            "model",
            model_name
        )

        model_grid_search_records.append(
            fold_grid_result_df
        )


        # ====================================================
        # 19. MENAMBAHKAN KE ARRAY HASIL MODEL
        # ====================================================

        model_y_true.extend(
            y_outer_test.tolist()
        )

        model_y_pred.extend(
            outer_predicted_labels.tolist()
        )

        model_y_score.extend(
            outer_prediction_scores.tolist()
        )

        model_subjects.extend(
            subjects_outer_test.tolist()
        )


        # ====================================================
        # 20. MENAMPILKAN HASIL OUTER FOLD
        # ====================================================

        print(
            "  Train/Test size       :",
            f"{len(outer_train_indices)}/"
            f"{len(outer_test_indices)}"
        )

        print(
            "  Best inner ROC-AUC    :",
            f"{best_inner_auc:.4f}"
        )

        print(
            "  PCA parameter         :",
            selected_pca_parameter
        )

        print(
            "  Actual PCA components :",
            actual_pca_components
        )

        print(
            "  Explained variance    :",
            f"{pca_explained_variance:.4f}"
        )

        print(
            "  Outer accuracy        :",
            f"{fold_metrics['accuracy']:.4f}"
        )

        print(
            "  Outer ROC-AUC         :",
            f"{fold_metrics['roc_auc']:.4f}"
        )

        print(
            "  Best parameters       :",
            best_parameters
        )


    # ========================================================
    # 21. VALIDASI OUT-OF-FOLD PREDICTIONS
    # ========================================================

    model_y_true = np.asarray(
        model_y_true,
        dtype=int
    )

    model_y_pred = np.asarray(
        model_y_pred,
        dtype=int
    )

    model_y_score = np.asarray(
        model_y_score,
        dtype=float
    )

    model_subjects = np.asarray(
        model_subjects
    )

    if len(
        model_y_true
    ) != len(
        y_nested
    ):
        raise ValueError(
            f"Jumlah out-of-fold prediction "
            f"{model_name} tidak sama dengan jumlah subject."
        )

    if len(
        np.unique(
            model_subjects
        )
    ) != len(
        subjects_nested
    ):
        raise ValueError(
            f"Tidak semua subject memperoleh tepat satu "
            f"prediksi untuk model {model_name}."
        )


    # ========================================================
    # 22. METRIK OUT-OF-FOLD KESELURUHAN
    # ========================================================

    overall_metrics = (
        calculate_nested_binary_metrics(
            y_true=model_y_true,
            y_pred=model_y_pred,
            y_score=model_y_score
        )
    )

    model_elapsed_seconds = float(
        time.time()
        - model_start_time
    )

    all_model_summary_records.append({
        "model": model_name,
        "model_abbreviation": (
            model_abbreviations[
                model_name
            ]
        ),
        "score_method": (
            score_method
        ),
        "number_of_subjects": int(
            len(
                model_y_true
            )
        ),
        "number_of_outer_folds": int(
            OUTER_N_SPLITS
        ),
        "mean_inner_best_auc": float(
            np.mean([
                record[
                    "best_inner_auc"
                ]
                for record in (
                    model_fold_metric_records
                )
            ])
        ),
        "std_inner_best_auc": float(
            np.std([
                record[
                    "best_inner_auc"
                ]
                for record in (
                    model_fold_metric_records
                )
            ])
        ),
        "mean_selected_pca_components": float(
            np.mean([
                record[
                    "actual_pca_components"
                ]
                for record in (
                    model_fold_metric_records
                )
            ])
        ),
        "mean_explained_variance": float(
            np.mean([
                record[
                    "pca_explained_variance"
                ]
                for record in (
                    model_fold_metric_records
                )
            ])
        ),
        "total_elapsed_seconds": (
            model_elapsed_seconds
        ),
        **overall_metrics
    })


    # ========================================================
    # 23. MEMBENTUK DATAFRAME HASIL MODEL
    # ========================================================

    model_predictions_df = pd.DataFrame(
        model_prediction_records
    )

    model_fold_metrics_df = pd.DataFrame(
        model_fold_metric_records
    )

    model_best_parameters_df = pd.DataFrame(
        model_best_parameter_records
    )

    model_grid_search_df = pd.concat(
        model_grid_search_records,
        ignore_index=True
    )


    # ========================================================
    # 24. MENYIMPAN HASIL MODEL KE DICTIONARY
    # ========================================================

    nested_cv_results[
        model_name
    ] = {
        "predictions": (
            model_predictions_df
        ),
        "fold_metrics": (
            model_fold_metrics_df
        ),
        "best_parameters": (
            model_best_parameters_df
        ),
        "grid_search_results": (
            model_grid_search_df
        ),
        "y_true": (
            model_y_true
        ),
        "y_pred": (
            model_y_pred
        ),
        "y_score": (
            model_y_score
        ),
        "overall_metrics": (
            overall_metrics
        )
    }


    # ========================================================
    # 25. MENAMBAHKAN KE HASIL GABUNGAN
    # ========================================================

    all_model_prediction_records.append(
        model_predictions_df
    )

    all_model_fold_metric_records.append(
        model_fold_metrics_df
    )

    all_model_best_parameter_records.append(
        model_best_parameters_df
    )

    all_model_grid_search_records.append(
        model_grid_search_df
    )


    # ========================================================
    # 26. MENAMPILKAN HASIL KESELURUHAN MODEL
    # ========================================================

    print("\n" + "-" * 72)
    print(
        f"OVERALL OUT-OF-FOLD RESULTS: "
        f"{model_name}"
    )
    print("-" * 72)

    print(
        "Accuracy             :",
        f"{overall_metrics['accuracy']:.4f}"
    )

    print(
        "Balanced accuracy    :",
        f"{overall_metrics['balanced_accuracy']:.4f}"
    )

    print(
        "Precision            :",
        f"{overall_metrics['precision']:.4f}"
    )

    print(
        "Recall/Sensitivity   :",
        f"{overall_metrics['recall']:.4f}"
    )

    print(
        "Specificity          :",
        f"{overall_metrics['specificity']:.4f}"
    )

    print(
        "F1-score             :",
        f"{overall_metrics['f1_score']:.4f}"
    )

    print(
        "ROC-AUC              :",
        f"{overall_metrics['roc_auc']:.4f}"
    )

    print(
        "Elapsed time (second):",
        f"{model_elapsed_seconds:.2f}"
    )


# ============================================================
# 27. MENGGABUNGKAN SELURUH HASIL MODEL
# ============================================================

nested_predictions_df = pd.concat(
    all_model_prediction_records,
    ignore_index=True
)

nested_fold_metrics_df = pd.concat(
    all_model_fold_metric_records,
    ignore_index=True
)

nested_best_parameters_df = pd.concat(
    all_model_best_parameter_records,
    ignore_index=True
)

nested_grid_search_results_df = pd.concat(
    all_model_grid_search_records,
    ignore_index=True
)

nested_model_summary_df = (
    pd.DataFrame(
        all_model_summary_records
    )
    .sort_values(
        by=[
            "roc_auc",
            "balanced_accuracy",
            "f1_score"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

nested_model_summary_df.insert(
    0,
    "performance_rank",
    np.arange(
        1,
        len(
            nested_model_summary_df
        ) + 1
    )
)


# ============================================================
# 28. MENAMPILKAN RINGKASAN PERFORMA MODEL
# ============================================================

summary_display_columns = [
    "performance_rank",
    "model",
    "mean_selected_pca_components",
    "mean_explained_variance",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1_score",
    "roc_auc"
]

print("\n" + "=" * 72)
print("NESTED CROSS-VALIDATION MODEL SUMMARY")
print("=" * 72)

display(
    nested_model_summary_df[
        summary_display_columns
    ]
)


# ============================================================
# 29. MENENTUKAN MODEL TERBAIK SEMENTARA
# ============================================================

best_nested_model_row = (
    nested_model_summary_df
    .iloc[0]
)

best_nested_model_name = str(
    best_nested_model_row[
        "model"
    ]
)

best_nested_model_auc = float(
    best_nested_model_row[
        "roc_auc"
    ]
)

print("\n" + "=" * 72)
print("BEST NESTED CV MODEL")
print("=" * 72)

print(
    "Model             :",
    best_nested_model_name
)

print(
    "ROC-AUC           :",
    f"{best_nested_model_auc:.4f}"
)

print(
    "Accuracy          :",
    f"{best_nested_model_row['accuracy']:.4f}"
)

print(
    "Balanced accuracy :",
    f"{best_nested_model_row['balanced_accuracy']:.4f}"
)

print(
    "Recall            :",
    f"{best_nested_model_row['recall']:.4f}"
)

print(
    "Specificity       :",
    f"{best_nested_model_row['specificity']:.4f}"
)

print(
    "F1-score          :",
    f"{best_nested_model_row['f1_score']:.4f}"
)


# ============================================================
# 30. MENYIMPAN SELURUH HASIL
# ============================================================

nested_model_summary_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_model_summary.csv"
)

nested_model_summary_df.to_csv(
    nested_model_summary_path,
    index=False
)


nested_predictions_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_subject_predictions.csv"
)

nested_predictions_df.to_csv(
    nested_predictions_path,
    index=False
)


nested_fold_metrics_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_fold_metrics.csv"
)

nested_fold_metrics_df.to_csv(
    nested_fold_metrics_path,
    index=False
)


nested_best_parameters_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_best_parameters.csv"
)

nested_best_parameters_df.to_csv(
    nested_best_parameters_path,
    index=False
)


nested_grid_results_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_grid_search_results.csv"
)

nested_grid_search_results_df.to_csv(
    nested_grid_results_path,
    index=False
)


# ============================================================
# 31. KONFIRMASI STEP 5
# ============================================================

print("\n" + "=" * 72)
print("STEP 5 COMPLETED")
print("=" * 72)

print(
    "Nested Stratified Cross-Validation dan GridSearchCV "
    "berhasil dijalankan untuk seluruh model."
)

print(
    "Best model:",
    best_nested_model_name
)

print(
    "Best ROC-AUC:",
    f"{best_nested_model_auc:.4f}"
)

print(
    "\nModel summary saved to:"
)

print(
    nested_model_summary_path
)

In [ ]:
# ============================================================
# STEP 7. VISUALIZE NESTED CROSS-VALIDATION PERFORMANCE
# ROC Curves, Confusion Matrices, Precision–Recall Curves,
# and Model Performance Comparison
# ============================================================

# Visualisasi menggunakan seluruh out-of-fold prediction dari
# outer Stratified K-Fold.
#
# Dengan demikian, setiap subject hanya diprediksi ketika
# berada pada outer test fold dan tidak digunakan untuk
# melatih model yang menghasilkan prediksi tersebut.


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP 5 DAN STEP 6
# ============================================================

required_step_7_objects = [
    "nested_cv_results",
    "nested_model_summary_df",
    "nested_predictions_df",
    "ml_pipelines",
    "model_abbreviations",
    "CLASS_LABEL_MAPPING",
    "PCA_ML_FIGURE_DIR",
    "PCA_ML_TABLE_DIR"
]

missing_step_7_objects = [
    object_name
    for object_name in required_step_7_objects
    if object_name not in globals()
]

if missing_step_7_objects:
    raise NameError(
        "Objek untuk STEP 7 belum lengkap: "
        f"{missing_step_7_objects}. "
        "Jalankan STEP 5 dan STEP 6 terlebih dahulu."
    )


# ============================================================
# 2. IMPORT METRIK VISUALISASI
# ============================================================

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    ConfusionMatrixDisplay
)


# ============================================================
# 3. KONFIGURASI GAMBAR
# ============================================================

FIGURE_DPI = 300

os.makedirs(
    PCA_ML_FIGURE_DIR,
    exist_ok=True
)


# ============================================================
# 4. MEMBENTUK TABEL ROC CURVE SELURUH MODEL
# ============================================================

all_roc_curve_records = []

plt.figure(
    figsize=(9, 7)
)

for model_name in ml_pipelines.keys():

    model_result = (
        nested_cv_results[
            model_name
        ]
    )

    y_true_model = np.asarray(
        model_result[
            "y_true"
        ],
        dtype=int
    )

    y_score_model = np.asarray(
        model_result[
            "y_score"
        ],
        dtype=float
    )

    false_positive_rate, true_positive_rate, (
        roc_thresholds
    ) = roc_curve(
        y_true_model,
        y_score_model,
        pos_label=1
    )

    model_auc = roc_auc_score(
        y_true_model,
        y_score_model
    )

    plt.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        marker="o",
        markersize=3,
        label=(
            f"{model_abbreviations[model_name]} "
            f"(AUC = {model_auc:.3f})"
        )
    )

    for point_index in range(
        len(false_positive_rate)
    ):

        all_roc_curve_records.append({
            "model": model_name,
            "model_abbreviation": (
                model_abbreviations[
                    model_name
                ]
            ),
            "point_index": int(
                point_index
            ),
            "false_positive_rate": float(
                false_positive_rate[
                    point_index
                ]
            ),
            "true_positive_rate": float(
                true_positive_rate[
                    point_index
                ]
            ),
            "threshold": float(
                roc_thresholds[
                    point_index
                ]
            ),
            "roc_auc": float(
                model_auc
            )
        })


plt.plot(
    [
        0,
        1
    ],
    [
        0,
        1
    ],
    linestyle="--",
    linewidth=1.5,
    label="Chance level"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curves of PCA–Machine Learning Models"
)

plt.xlim(
    0.0,
    1.0
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    alpha=0.25
)

plt.legend(
    loc="lower right"
)

plt.tight_layout()


combined_roc_figure_path = os.path.join(
    PCA_ML_FIGURE_DIR,
    "combined_nested_cv_roc_curves.png"
)

plt.savefig(
    combined_roc_figure_path,
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


all_roc_curve_df = pd.DataFrame(
    all_roc_curve_records
)


# ============================================================
# 5. PRECISION–RECALL CURVES SELURUH MODEL
# ============================================================

all_pr_curve_records = []

plt.figure(
    figsize=(9, 7)
)

positive_class_prevalence = float(
    np.mean(
        y == 1
    )
)

for model_name in ml_pipelines.keys():

    model_result = (
        nested_cv_results[
            model_name
        ]
    )

    y_true_model = np.asarray(
        model_result[
            "y_true"
        ],
        dtype=int
    )

    y_score_model = np.asarray(
        model_result[
            "y_score"
        ],
        dtype=float
    )

    precision_values, recall_values, (
        pr_thresholds
    ) = precision_recall_curve(
        y_true_model,
        y_score_model,
        pos_label=1
    )

    average_precision = (
        average_precision_score(
            y_true_model,
            y_score_model
        )
    )

    plt.plot(
        recall_values,
        precision_values,
        linewidth=2,
        marker="o",
        markersize=3,
        label=(
            f"{model_abbreviations[model_name]} "
            f"(AP = {average_precision:.3f})"
        )
    )

    for point_index in range(
        len(precision_values)
    ):

        threshold_value = (
            float(
                pr_thresholds[
                    point_index
                ]
            )
            if point_index
            < len(pr_thresholds)
            else np.nan
        )

        all_pr_curve_records.append({
            "model": model_name,
            "model_abbreviation": (
                model_abbreviations[
                    model_name
                ]
            ),
            "point_index": int(
                point_index
            ),
            "recall": float(
                recall_values[
                    point_index
                ]
            ),
            "precision": float(
                precision_values[
                    point_index
                ]
            ),
            "threshold": (
                threshold_value
            ),
            "average_precision": float(
                average_precision
            )
        })


plt.axhline(
    y=positive_class_prevalence,
    linestyle="--",
    linewidth=1.5,
    label=(
        "Positive-class prevalence "
        f"({positive_class_prevalence:.3f})"
    )
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Precision–Recall Curves of PCA–Machine Learning Models"
)

plt.xlim(
    0.0,
    1.0
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    alpha=0.25
)

plt.legend(
    loc="best"
)

plt.tight_layout()


combined_pr_figure_path = os.path.join(
    PCA_ML_FIGURE_DIR,
    "combined_nested_cv_precision_recall_curves.png"
)

plt.savefig(
    combined_pr_figure_path,
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


all_pr_curve_df = pd.DataFrame(
    all_pr_curve_records
)


# ============================================================
# 6. CONFUSION MATRIX SETIAP MODEL
# ============================================================

confusion_matrix_records = []

for model_name in ml_pipelines.keys():

    model_result = (
        nested_cv_results[
            model_name
        ]
    )

    y_true_model = np.asarray(
        model_result[
            "y_true"
        ],
        dtype=int
    )

    y_pred_model = np.asarray(
        model_result[
            "y_pred"
        ],
        dtype=int
    )

    model_confusion_matrix = confusion_matrix(
        y_true_model,
        y_pred_model,
        labels=[
            0,
            1
        ]
    )

    tn, fp, fn, tp = (
        model_confusion_matrix
        .ravel()
    )

    confusion_matrix_records.append({
        "model": model_name,
        "true_negative": int(
            tn
        ),
        "false_positive": int(
            fp
        ),
        "false_negative": int(
            fn
        ),
        "true_positive": int(
            tp
        )
    })


    plt.figure(
        figsize=(5.5, 5)
    )

    display_object = ConfusionMatrixDisplay(
        confusion_matrix=(
            model_confusion_matrix
        ),
        display_labels=[
            "LOW/NO",
            "HIGH"
        ]
    )

    display_object.plot(
        values_format="d",
        ax=plt.gca(),
        colorbar=True
    )

    plt.title(
        f"Confusion Matrix—{model_name}"
    )

    plt.xlabel(
        "Predicted Class"
    )

    plt.ylabel(
        "Actual Class"
    )

    plt.tight_layout()


    safe_model_name = (
        model_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )

    confusion_figure_path = os.path.join(
        PCA_ML_FIGURE_DIR,
        (
            f"confusion_matrix_"
            f"{safe_model_name}.png"
        )
    )

    plt.savefig(
        confusion_figure_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()


confusion_matrix_summary_df = pd.DataFrame(
    confusion_matrix_records
)


# ============================================================
# 7. PERBANDINGAN METRIK UTAMA
# ============================================================

performance_metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1_score",
    "roc_auc"
]

performance_comparison_long_df = (
    nested_model_summary_df[
        [
            "model"
        ]
        + performance_metric_columns
    ]
    .melt(
        id_vars=[
            "model"
        ],
        value_vars=(
            performance_metric_columns
        ),
        var_name="metric",
        value_name="score"
    )
)


for metric_name in performance_metric_columns:

    metric_plot_df = (
        nested_model_summary_df[
            [
                "model",
                metric_name
            ]
        ]
        .sort_values(
            by=metric_name,
            ascending=False
        )
        .copy()
    )

    plt.figure(
        figsize=(10, 5.5)
    )

    plt.bar(
        metric_plot_df[
            "model"
        ],
        metric_plot_df[
            metric_name
        ]
    )

    plt.xlabel(
        "Machine-Learning Model"
    )

    plt.ylabel(
        metric_name.replace(
            "_",
            " "
        ).title()
    )

    plt.title(
        (
            f"Nested Cross-Validation "
            f"{metric_name.replace('_', ' ').title()}"
        )
    )

    plt.xticks(
        rotation=25,
        ha="right"
    )

    plt.ylim(
        0.0,
        1.05
    )

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.tight_layout()


    metric_figure_path = os.path.join(
        PCA_ML_FIGURE_DIR,
        (
            f"model_comparison_"
            f"{metric_name}.png"
        )
    )

    plt.savefig(
        metric_figure_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# 8. GRAFIK GABUNGAN METRIK UTAMA
# ============================================================

selected_combined_metrics = [
    "accuracy",
    "recall",
    "specificity",
    "f1_score",
    "roc_auc"
]

combined_metric_plot_df = (
    nested_model_summary_df[
        [
            "model"
        ]
        + selected_combined_metrics
    ]
    .set_index(
        "model"
    )
)

combined_metric_plot_df.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.xlabel(
    "Machine-Learning Model"
)

plt.ylabel(
    "Performance Score"
)

plt.title(
    "Comparison of Nested Cross-Validation Performance"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.legend(
    title="Metric",
    loc="best"
)

plt.tight_layout()


combined_metrics_figure_path = os.path.join(
    PCA_ML_FIGURE_DIR,
    "combined_model_performance_metrics.png"
)

plt.savefig(
    combined_metrics_figure_path,
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. VISUALISASI PERFORMA PER OUTER FOLD
# ============================================================

fold_auc_pivot_df = (
    nested_fold_metrics_df
    .pivot(
        index="outer_fold",
        columns="model",
        values="roc_auc"
    )
    .sort_index()
)

plt.figure(
    figsize=(11, 6)
)

for model_name in fold_auc_pivot_df.columns:

    plt.plot(
        fold_auc_pivot_df.index,
        fold_auc_pivot_df[
            model_name
        ],
        marker="o",
        linewidth=2,
        label=(
            model_abbreviations[
                model_name
            ]
        )
    )

plt.xlabel(
    "Outer Fold"
)

plt.ylabel(
    "ROC-AUC"
)

plt.title(
    "ROC-AUC Variability Across Outer Folds"
)

plt.xticks(
    fold_auc_pivot_df.index
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    alpha=0.25
)

plt.legend(
    title="Model",
    loc="best"
)

plt.tight_layout()


fold_auc_figure_path = os.path.join(
    PCA_ML_FIGURE_DIR,
    "outer_fold_roc_auc_variability.png"
)

plt.savefig(
    fold_auc_figure_path,
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 10. DISTRIBUSI ROC-AUC OUTER FOLD
# ============================================================

boxplot_data = []

boxplot_labels = []

for model_name in ml_pipelines.keys():

    model_fold_auc = (
        nested_fold_metrics_df.loc[
            nested_fold_metrics_df[
                "model"
            ]
            == model_name,
            "roc_auc"
        ]
        .dropna()
        .to_numpy()
    )

    boxplot_data.append(
        model_fold_auc
    )

    boxplot_labels.append(
        model_abbreviations[
            model_name
        ]
    )


plt.figure(
    figsize=(10, 6)
)

plt.boxplot(
    boxplot_data,
    tick_labels=boxplot_labels
)

plt.xlabel(
    "Machine-Learning Model"
)

plt.ylabel(
    "Outer-Fold ROC-AUC"
)

plt.title(
    "Distribution of ROC-AUC Across Outer Folds"
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()


fold_auc_boxplot_path = os.path.join(
    PCA_ML_FIGURE_DIR,
    "outer_fold_roc_auc_boxplot.png"
)

plt.savefig(
    fold_auc_boxplot_path,
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 11. MENAMPILKAN RINGKASAN CONFUSION MATRIX
# ============================================================

print("=" * 72)
print("CONFUSION MATRIX SUMMARY")
print("=" * 72)

display(
    confusion_matrix_summary_df
)


# ============================================================
# 12. MENAMPILKAN RINGKASAN PERFORMA
# ============================================================

print("\n" + "=" * 72)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 72)

display(
    nested_model_summary_df[
        [
            "performance_rank",
            "model",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "f1_score",
            "roc_auc"
        ]
    ]
)


# ============================================================
# 13. MENYIMPAN DATA VISUALISASI
# ============================================================

roc_curve_table_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_roc_curve_points.csv"
)

all_roc_curve_df.to_csv(
    roc_curve_table_path,
    index=False
)


pr_curve_table_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_precision_recall_curve_points.csv"
)

all_pr_curve_df.to_csv(
    pr_curve_table_path,
    index=False
)


confusion_summary_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_confusion_matrix_summary.csv"
)

confusion_matrix_summary_df.to_csv(
    confusion_summary_path,
    index=False
)


performance_long_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_performance_long_format.csv"
)

performance_comparison_long_df.to_csv(
    performance_long_path,
    index=False
)


fold_auc_pivot_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "outer_fold_roc_auc_pivot.csv"
)

fold_auc_pivot_df.to_csv(
    fold_auc_pivot_path
)


# ============================================================
# 14. RINGKASAN PATH GAMBAR
# ============================================================

visualization_summary_df = pd.DataFrame([
    {
        "visualization": "Combined ROC curves",
        "path": combined_roc_figure_path
    },
    {
        "visualization": (
            "Combined precision–recall curves"
        ),
        "path": combined_pr_figure_path
    },
    {
        "visualization": (
            "Combined performance metrics"
        ),
        "path": combined_metrics_figure_path
    },
    {
        "visualization": (
            "Outer-fold ROC-AUC variability"
        ),
        "path": fold_auc_figure_path
    },
    {
        "visualization": (
            "Outer-fold ROC-AUC boxplot"
        ),
        "path": fold_auc_boxplot_path
    }
])

visualization_summary_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "visualization_output_summary.csv"
)

visualization_summary_df.to_csv(
    visualization_summary_path,
    index=False
)


# ============================================================
# 15. KONFIRMASI STEP 7
# ============================================================

print("\n" + "=" * 72)
print("STEP 7 COMPLETED")
print("=" * 72)

print(
    "Visualisasi nested cross-validation berhasil dibuat."
)

print(
    "\nCombined ROC curve:"
)

print(
    combined_roc_figure_path
)

print(
    "\nCombined Precision–Recall curve:"
)

print(
    combined_pr_figure_path
)

print(
    "\nBest-performing model:"
)

print(
    best_nested_model_name
)

print(
    "Best ROC-AUC:"
)

print(
    f"{best_nested_model_auc:.4f}"
)

In [ ]:
# ============================================================
# STEP 8. STATISTICAL COMPARISON AND UNCERTAINTY ANALYSIS
# Confidence Intervals, Bootstrap Analysis, and Paired Tests
# ============================================================

# Analisis pada STEP ini mencakup:
#
# 1. Mean ± standard deviation metrik pada outer folds.
# 2. 95% confidence interval berbasis distribusi t.
# 3. Bootstrap 95% confidence interval menggunakan seluruh
#    out-of-fold prediction pada tingkat subject.
# 4. Friedman test untuk membandingkan seluruh model.
# 5. Pairwise Wilcoxon signed-rank test sebagai analisis
#    eksploratif antarmodel.
# 6. Koreksi Holm untuk multiple comparisons.
#
# Catatan:
# Outer cross-validation hanya memiliki 5 folds. Oleh karena
# itu, uji statistik berbasis fold memiliki statistical power
# yang terbatas dan harus diinterpretasikan secara hati-hati.


# ============================================================
# 1. IMPORT LIBRARY STATISTIK
# ============================================================

from itertools import combinations

from scipy.stats import (
    t,
    friedmanchisquare,
    wilcoxon
)


# ============================================================
# 2. VALIDASI OBJEK HASIL STEP SEBELUMNYA
# ============================================================

required_step_8_objects = [
    "nested_cv_results",
    "nested_model_summary_df",
    "nested_fold_metrics_df",
    "nested_predictions_df",
    "ml_pipelines",
    "model_abbreviations",
    "PCA_ML_TABLE_DIR",
    "RANDOM_STATE"
]

missing_step_8_objects = [
    object_name
    for object_name in required_step_8_objects
    if object_name not in globals()
]

if missing_step_8_objects:
    raise NameError(
        "Objek untuk STEP 8 belum lengkap: "
        f"{missing_step_8_objects}. "
        "Jalankan STEP 5–7 terlebih dahulu."
    )


# ============================================================
# 3. KONFIGURASI ANALISIS STATISTIK
# ============================================================

CONFIDENCE_LEVEL = 0.95

ALPHA = (
    1.0
    - CONFIDENCE_LEVEL
)

BOOTSTRAP_ITERATIONS = 5000

BOOTSTRAP_RANDOM_STATE = (
    RANDOM_STATE + 100
)

STATISTICAL_PRIMARY_METRIC = (
    "roc_auc"
)

FOLD_METRICS_FOR_ANALYSIS = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1_score",
    "roc_auc"
]


# ============================================================
# 4. FUNGSI CONFIDENCE INTERVAL BERBASIS t
# ============================================================

def calculate_t_confidence_interval(
    values,
    confidence_level=0.95
):
    """
    Menghitung mean, standard deviation, standard error,
    dan confidence interval berbasis distribusi t.

    Parameters
    ----------
    values : array-like
        Nilai metrik dari outer folds.

    confidence_level : float
        Tingkat confidence interval.

    Returns
    -------
    dict
        Ringkasan statistik.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(
            values
        )
    ]

    sample_size = len(
        values
    )

    if sample_size == 0:
        return {
            "n_folds": 0,
            "mean": np.nan,
            "std": np.nan,
            "standard_error": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "minimum": np.nan,
            "maximum": np.nan
        }

    mean_value = float(
        np.mean(
            values
        )
    )

    minimum_value = float(
        np.min(
            values
        )
    )

    maximum_value = float(
        np.max(
            values
        )
    )

    if sample_size == 1:
        return {
            "n_folds": 1,
            "mean": mean_value,
            "std": np.nan,
            "standard_error": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "minimum": minimum_value,
            "maximum": maximum_value
        }

    standard_deviation = float(
        np.std(
            values,
            ddof=1
        )
    )

    standard_error = float(
        standard_deviation
        / np.sqrt(
            sample_size
        )
    )

    degrees_of_freedom = (
        sample_size - 1
    )

    critical_value = float(
        t.ppf(
            (
                1.0
                + confidence_level
            )
            / 2.0,
            degrees_of_freedom
        )
    )

    margin_of_error = (
        critical_value
        * standard_error
    )

    confidence_interval_lower = float(
        mean_value
        - margin_of_error
    )

    confidence_interval_upper = float(
        mean_value
        + margin_of_error
    )

    # Metrik klasifikasi umumnya berada pada rentang 0–1.
    confidence_interval_lower = max(
        0.0,
        confidence_interval_lower
    )

    confidence_interval_upper = min(
        1.0,
        confidence_interval_upper
    )

    return {
        "n_folds": int(
            sample_size
        ),
        "mean": mean_value,
        "std": standard_deviation,
        "standard_error": standard_error,
        "ci_lower": confidence_interval_lower,
        "ci_upper": confidence_interval_upper,
        "minimum": minimum_value,
        "maximum": maximum_value
    }


# ============================================================
# 5. RINGKASAN MEAN ± SD DAN 95% CI OUTER FOLDS
# ============================================================

fold_uncertainty_records = []

for model_name in ml_pipelines.keys():

    model_fold_df = (
        nested_fold_metrics_df[
            nested_fold_metrics_df[
                "model"
            ]
            == model_name
        ]
        .sort_values(
            by="outer_fold"
        )
        .copy()
    )

    for metric_name in (
        FOLD_METRICS_FOR_ANALYSIS
    ):

        metric_summary = (
            calculate_t_confidence_interval(
                model_fold_df[
                    metric_name
                ].to_numpy(),
                confidence_level=(
                    CONFIDENCE_LEVEL
                )
            )
        )

        fold_uncertainty_records.append({
            "model": model_name,
            "model_abbreviation": (
                model_abbreviations[
                    model_name
                ]
            ),
            "metric": metric_name,
            "confidence_level": (
                CONFIDENCE_LEVEL
            ),
            **metric_summary
        })


fold_uncertainty_df = pd.DataFrame(
    fold_uncertainty_records
)

fold_uncertainty_df[
    "mean_plus_minus_std"
] = (
    fold_uncertainty_df.apply(
        lambda row: (
            f"{row['mean']:.4f} ± "
            f"{row['std']:.4f}"
        ),
        axis=1
    )
)

fold_uncertainty_df[
    "confidence_interval_text"
] = (
    fold_uncertainty_df.apply(
        lambda row: (
            f"[{row['ci_lower']:.4f}, "
            f"{row['ci_upper']:.4f}]"
        ),
        axis=1
    )
)


print("=" * 72)
print("OUTER-FOLD PERFORMANCE UNCERTAINTY")
print("=" * 72)

display(
    fold_uncertainty_df[
        [
            "model",
            "metric",
            "n_folds",
            "mean_plus_minus_std",
            "standard_error",
            "confidence_interval_text",
            "minimum",
            "maximum"
        ]
    ]
)


# ============================================================
# 6. MEMBENTUK TABEL WIDE MEAN ± SD
# ============================================================

mean_sd_performance_table = (
    fold_uncertainty_df
    .pivot(
        index="model",
        columns="metric",
        values="mean_plus_minus_std"
    )
    .reset_index()
)

mean_sd_performance_table.columns.name = None

print("\n" + "=" * 72)
print("MEAN ± STANDARD DEVIATION TABLE")
print("=" * 72)

display(
    mean_sd_performance_table
)


# ============================================================
# 7. FUNGSI BOOTSTRAP METRIK OOF
# ============================================================

def calculate_metrics_for_bootstrap(
    y_true,
    y_pred,
    y_score
):
    """
    Menghitung metrik utama untuk satu bootstrap sample.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int
    )

    y_score = np.asarray(
        y_score,
        dtype=float
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1
        ]
    )

    tn, fp, fn, tp = (
        cm.ravel()
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    if len(
        np.unique(
            y_true
        )
    ) == 2:

        auc_value = roc_auc_score(
            y_true,
            y_score
        )

    else:

        auc_value = np.nan

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                y_pred
            )
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),
        "recall": sensitivity,
        "specificity": specificity,
        "f1_score": f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),
        "roc_auc": auc_value
    }


# ============================================================
# 8. FUNGSI STRATIFIED SUBJECT-LEVEL BOOTSTRAP
# ============================================================

def stratified_bootstrap_confidence_intervals(
    y_true,
    y_pred,
    y_score,
    n_iterations=5000,
    confidence_level=0.95,
    random_state=42
):
    """
    Menghitung bootstrap confidence intervals dengan sampling
    terpisah pada kelas LOW/NO dan HIGH.

    Stratified bootstrap menjaga keberadaan kedua kelas pada
    setiap bootstrap sample.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int
    )

    y_score = np.asarray(
        y_score,
        dtype=float
    )

    if not (
        len(y_true)
        == len(y_pred)
        == len(y_score)
    ):
        raise ValueError(
            "Panjang y_true, y_pred, dan y_score harus sama."
        )

    class_zero_indices = np.where(
        y_true == 0
    )[0]

    class_one_indices = np.where(
        y_true == 1
    )[0]

    if (
        len(class_zero_indices) == 0
        or len(class_one_indices) == 0
    ):
        raise ValueError(
            "Bootstrap membutuhkan kedua kelas."
        )

    random_generator = (
        np.random.default_rng(
            random_state
        )
    )

    bootstrap_metric_records = []

    for bootstrap_iteration in range(
        n_iterations
    ):

        sampled_class_zero = (
            random_generator.choice(
                class_zero_indices,
                size=len(
                    class_zero_indices
                ),
                replace=True
            )
        )

        sampled_class_one = (
            random_generator.choice(
                class_one_indices,
                size=len(
                    class_one_indices
                ),
                replace=True
            )
        )

        bootstrap_indices = np.concatenate(
            [
                sampled_class_zero,
                sampled_class_one
            ]
        )

        random_generator.shuffle(
            bootstrap_indices
        )

        bootstrap_metrics = (
            calculate_metrics_for_bootstrap(
                y_true[
                    bootstrap_indices
                ],
                y_pred[
                    bootstrap_indices
                ],
                y_score[
                    bootstrap_indices
                ]
            )
        )

        bootstrap_metric_records.append(
            bootstrap_metrics
        )


    bootstrap_metrics_df = pd.DataFrame(
        bootstrap_metric_records
    )

    lower_percentile = (
        (
            1.0
            - confidence_level
        )
        / 2.0
        * 100
    )

    upper_percentile = (
        (
            1.0
            + confidence_level
        )
        / 2.0
        * 100
    )

    observed_metrics = (
        calculate_metrics_for_bootstrap(
            y_true,
            y_pred,
            y_score
        )
    )

    summary_records = []

    for metric_name in (
        FOLD_METRICS_FOR_ANALYSIS
    ):

        valid_bootstrap_values = (
            bootstrap_metrics_df[
                metric_name
            ]
            .dropna()
            .to_numpy()
        )

        summary_records.append({
            "metric": metric_name,
            "observed_value": float(
                observed_metrics[
                    metric_name
                ]
            ),
            "bootstrap_mean": float(
                np.mean(
                    valid_bootstrap_values
                )
            ),
            "bootstrap_std": float(
                np.std(
                    valid_bootstrap_values,
                    ddof=1
                )
            ),
            "ci_lower": float(
                np.percentile(
                    valid_bootstrap_values,
                    lower_percentile
                )
            ),
            "ci_upper": float(
                np.percentile(
                    valid_bootstrap_values,
                    upper_percentile
                )
            ),
            "valid_bootstrap_samples": int(
                len(
                    valid_bootstrap_values
                )
            )
        })

    return (
        pd.DataFrame(
            summary_records
        ),
        bootstrap_metrics_df
    )


# ============================================================
# 9. BOOTSTRAP CI UNTUK SELURUH MODEL
# ============================================================

bootstrap_summary_records = []

bootstrap_distribution_frames = []

for model_index, model_name in enumerate(
    ml_pipelines.keys(),
    start=1
):

    model_result = (
        nested_cv_results[
            model_name
        ]
    )

    model_bootstrap_summary, (
        model_bootstrap_distribution
    ) = (
        stratified_bootstrap_confidence_intervals(
            y_true=model_result[
                "y_true"
            ],
            y_pred=model_result[
                "y_pred"
            ],
            y_score=model_result[
                "y_score"
            ],
            n_iterations=(
                BOOTSTRAP_ITERATIONS
            ),
            confidence_level=(
                CONFIDENCE_LEVEL
            ),
            random_state=(
                BOOTSTRAP_RANDOM_STATE
                + model_index
            )
        )
    )

    model_bootstrap_summary.insert(
        0,
        "model_abbreviation",
        model_abbreviations[
            model_name
        ]
    )

    model_bootstrap_summary.insert(
        0,
        "model",
        model_name
    )

    model_bootstrap_distribution.insert(
        0,
        "bootstrap_iteration",
        np.arange(
            1,
            len(
                model_bootstrap_distribution
            ) + 1
        )
    )

    model_bootstrap_distribution.insert(
        0,
        "model",
        model_name
    )

    bootstrap_summary_records.append(
        model_bootstrap_summary
    )

    bootstrap_distribution_frames.append(
        model_bootstrap_distribution
    )


bootstrap_confidence_interval_df = (
    pd.concat(
        bootstrap_summary_records,
        ignore_index=True
    )
)

bootstrap_distribution_df = (
    pd.concat(
        bootstrap_distribution_frames,
        ignore_index=True
    )
)

bootstrap_confidence_interval_df[
    "confidence_interval_text"
] = (
    bootstrap_confidence_interval_df.apply(
        lambda row: (
            f"[{row['ci_lower']:.4f}, "
            f"{row['ci_upper']:.4f}]"
        ),
        axis=1
    )
)


print("\n" + "=" * 72)
print("STRATIFIED BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 72)

display(
    bootstrap_confidence_interval_df[
        [
            "model",
            "metric",
            "observed_value",
            "bootstrap_mean",
            "bootstrap_std",
            "confidence_interval_text",
            "valid_bootstrap_samples"
        ]
    ]
)


# ============================================================
# 10. RINGKASAN BOOTSTRAP ROC-AUC
# ============================================================

bootstrap_auc_summary_df = (
    bootstrap_confidence_interval_df[
        bootstrap_confidence_interval_df[
            "metric"
        ]
        == "roc_auc"
    ]
    .sort_values(
        by="observed_value",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 72)
print("BOOTSTRAP ROC-AUC SUMMARY")
print("=" * 72)

display(
    bootstrap_auc_summary_df[
        [
            "model",
            "observed_value",
            "bootstrap_mean",
            "bootstrap_std",
            "ci_lower",
            "ci_upper",
            "confidence_interval_text"
        ]
    ]
)


# ============================================================
# 11. MENYIAPKAN DATA FOLD UNTUK UJI STATISTIK
# ============================================================

primary_metric_pivot_df = (
    nested_fold_metrics_df
    .pivot(
        index="outer_fold",
        columns="model",
        values=STATISTICAL_PRIMARY_METRIC
    )
    .sort_index()
)

primary_metric_pivot_df = (
    primary_metric_pivot_df[
        list(
            ml_pipelines.keys()
        )
    ]
)


print("\n" + "=" * 72)
print("OUTER-FOLD ROC-AUC MATRIX")
print("=" * 72)

display(
    primary_metric_pivot_df
)


# ============================================================
# 12. FRIEDMAN TEST SELURUH MODEL
# ============================================================

friedman_statistic = np.nan

friedman_p_value = np.nan

friedman_interpretation = (
    "Not available"
)

if (
    primary_metric_pivot_df.shape[1]
    >= 3
    and not primary_metric_pivot_df
    .isna()
    .any()
    .any()
):

    friedman_result = friedmanchisquare(
        *[
            primary_metric_pivot_df[
                model_name
            ].to_numpy()
            for model_name in (
                primary_metric_pivot_df.columns
            )
        ]
    )

    friedman_statistic = float(
        friedman_result.statistic
    )

    friedman_p_value = float(
        friedman_result.pvalue
    )

    friedman_interpretation = (
        "Significant difference among models"
        if friedman_p_value < ALPHA
        else "No statistically significant difference"
    )


friedman_result_df = pd.DataFrame([
    {
        "test": "Friedman test",
        "metric": (
            STATISTICAL_PRIMARY_METRIC
        ),
        "number_of_models": int(
            primary_metric_pivot_df.shape[1]
        ),
        "number_of_outer_folds": int(
            primary_metric_pivot_df.shape[0]
        ),
        "statistic": (
            friedman_statistic
        ),
        "p_value": (
            friedman_p_value
        ),
        "alpha": ALPHA,
        "interpretation": (
            friedman_interpretation
        )
    }
])


print("\n" + "=" * 72)
print("FRIEDMAN TEST")
print("=" * 72)

display(
    friedman_result_df
)


# ============================================================
# 13. PAIRWISE WILCOXON SIGNED-RANK TESTS
# ============================================================

pairwise_wilcoxon_records = []

model_pairs = list(
    combinations(
        primary_metric_pivot_df.columns,
        2
    )
)

for model_a, model_b in model_pairs:

    model_a_values = (
        primary_metric_pivot_df[
            model_a
        ]
        .to_numpy(
            dtype=float
        )
    )

    model_b_values = (
        primary_metric_pivot_df[
            model_b
        ]
        .to_numpy(
            dtype=float
        )
    )

    valid_pair_mask = (
        np.isfinite(
            model_a_values
        )
        & np.isfinite(
            model_b_values
        )
    )

    model_a_valid = (
        model_a_values[
            valid_pair_mask
        ]
    )

    model_b_valid = (
        model_b_values[
            valid_pair_mask
        ]
    )

    paired_differences = (
        model_a_valid
        - model_b_valid
    )

    wilcoxon_statistic = np.nan

    wilcoxon_p_value = np.nan

    test_note = ""

    if len(
        model_a_valid
    ) < 2:

        test_note = (
            "Insufficient paired observations"
        )

    elif np.allclose(
        paired_differences,
        0.0
    ):

        wilcoxon_statistic = 0.0

        wilcoxon_p_value = 1.0

        test_note = (
            "All paired differences are zero"
        )

    else:

        try:

            wilcoxon_result = wilcoxon(
                model_a_valid,
                model_b_valid,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto"
            )

            wilcoxon_statistic = float(
                wilcoxon_result.statistic
            )

            wilcoxon_p_value = float(
                wilcoxon_result.pvalue
            )

            test_note = "Completed"

        except ValueError as error:

            test_note = str(
                error
            )


    pairwise_wilcoxon_records.append({
        "model_a": model_a,
        "model_b": model_b,
        "metric": (
            STATISTICAL_PRIMARY_METRIC
        ),
        "number_of_pairs": int(
            len(
                model_a_valid
            )
        ),
        "mean_model_a": float(
            np.mean(
                model_a_valid
            )
        ),
        "mean_model_b": float(
            np.mean(
                model_b_valid
            )
        ),
        "mean_difference_a_minus_b": float(
            np.mean(
                paired_differences
            )
        ),
        "median_difference_a_minus_b": float(
            np.median(
                paired_differences
            )
        ),
        "wilcoxon_statistic": (
            wilcoxon_statistic
        ),
        "raw_p_value": (
            wilcoxon_p_value
        ),
        "test_note": test_note
    })


pairwise_wilcoxon_df = pd.DataFrame(
    pairwise_wilcoxon_records
)


# ============================================================
# 14. HOLM CORRECTION
# ============================================================

def holm_adjust_p_values(
    p_values
):
    """
    Melakukan Holm step-down correction.
    """

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    adjusted_p_values = np.full(
        shape=len(
            p_values
        ),
        fill_value=np.nan,
        dtype=float
    )

    valid_indices = np.where(
        np.isfinite(
            p_values
        )
    )[0]

    if len(
        valid_indices
    ) == 0:

        return (
            adjusted_p_values
        )

    valid_p_values = (
        p_values[
            valid_indices
        ]
    )

    sorted_local_indices = np.argsort(
        valid_p_values
    )

    number_of_tests = len(
        valid_p_values
    )

    running_maximum = 0.0

    for rank_position, local_index in enumerate(
        sorted_local_indices
    ):

        multiplier = (
            number_of_tests
            - rank_position
        )

        adjusted_value = min(
            1.0,
            valid_p_values[
                local_index
            ]
            * multiplier
        )

        running_maximum = max(
            running_maximum,
            adjusted_value
        )

        original_index = (
            valid_indices[
                local_index
            ]
        )

        adjusted_p_values[
            original_index
        ] = running_maximum

    return adjusted_p_values


pairwise_wilcoxon_df[
    "holm_adjusted_p_value"
] = (
    holm_adjust_p_values(
        pairwise_wilcoxon_df[
            "raw_p_value"
        ].to_numpy()
    )
)

pairwise_wilcoxon_df[
    "significant_after_holm"
] = (
    pairwise_wilcoxon_df[
        "holm_adjusted_p_value"
    ]
    < ALPHA
)

pairwise_wilcoxon_df[
    "interpretation"
] = np.where(
    pairwise_wilcoxon_df[
        "significant_after_holm"
    ],
    "Statistically significant",
    "Not statistically significant"
)


print("\n" + "=" * 72)
print("PAIRWISE WILCOXON TESTS WITH HOLM CORRECTION")
print("=" * 72)

display(
    pairwise_wilcoxon_df
)


# ============================================================
# 15. MODEL RANKING PER OUTER FOLD
# ============================================================

fold_rank_df = (
    primary_metric_pivot_df
    .rank(
        axis=1,
        method="average",
        ascending=False
    )
)

mean_model_rank_df = (
    fold_rank_df
    .mean(
        axis=0
    )
    .rename(
        "mean_rank"
    )
    .reset_index()
    .rename(
        columns={
            "model": "model"
        }
    )
)

mean_model_rank_df.columns = [
    "model",
    "mean_rank"
]

mean_model_rank_df = (
    mean_model_rank_df
    .sort_values(
        by="mean_rank",
        ascending=True
    )
    .reset_index(drop=True)
)

mean_model_rank_df.insert(
    0,
    "rank_position",
    np.arange(
        1,
        len(
            mean_model_rank_df
        ) + 1
    )
)


print("\n" + "=" * 72)
print("MEAN MODEL RANK ACROSS OUTER FOLDS")
print("=" * 72)

display(
    mean_model_rank_df
)


# ============================================================
# 16. MENGGABUNGKAN HASIL UTAMA DAN BOOTSTRAP CI
# ============================================================

final_uncertainty_summary_df = (
    nested_model_summary_df[
        [
            "performance_rank",
            "model",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "f1_score",
            "roc_auc"
        ]
    ]
    .merge(
        bootstrap_auc_summary_df[
            [
                "model",
                "ci_lower",
                "ci_upper",
                "bootstrap_std"
            ]
        ].rename(
            columns={
                "ci_lower": (
                    "roc_auc_bootstrap_ci_lower"
                ),
                "ci_upper": (
                    "roc_auc_bootstrap_ci_upper"
                ),
                "bootstrap_std": (
                    "roc_auc_bootstrap_std"
                )
            }
        ),
        on="model",
        how="left"
    )
    .merge(
        mean_model_rank_df[
            [
                "model",
                "mean_rank"
            ]
        ],
        on="model",
        how="left"
    )
)


final_uncertainty_summary_df[
    "roc_auc_95_ci"
] = (
    final_uncertainty_summary_df.apply(
        lambda row: (
            f"[{row['roc_auc_bootstrap_ci_lower']:.4f}, "
            f"{row['roc_auc_bootstrap_ci_upper']:.4f}]"
        ),
        axis=1
    )
)


print("\n" + "=" * 72)
print("FINAL MODEL PERFORMANCE WITH UNCERTAINTY")
print("=" * 72)

display(
    final_uncertainty_summary_df[
        [
            "performance_rank",
            "model",
            "accuracy",
            "balanced_accuracy",
            "recall",
            "specificity",
            "f1_score",
            "roc_auc",
            "roc_auc_95_ci",
            "mean_rank"
        ]
    ]
)


# ============================================================
# 17. MENYIMPAN HASIL ANALISIS STATISTIK
# ============================================================

fold_uncertainty_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "outer_fold_performance_uncertainty.csv"
)

fold_uncertainty_df.to_csv(
    fold_uncertainty_path,
    index=False
)


mean_sd_table_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "outer_fold_mean_std_table.csv"
)

mean_sd_performance_table.to_csv(
    mean_sd_table_path,
    index=False
)


bootstrap_ci_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "bootstrap_confidence_intervals.csv"
)

bootstrap_confidence_interval_df.to_csv(
    bootstrap_ci_path,
    index=False
)


bootstrap_distribution_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "bootstrap_metric_distributions.csv"
)

bootstrap_distribution_df.to_csv(
    bootstrap_distribution_path,
    index=False
)


friedman_result_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "friedman_test_results.csv"
)

friedman_result_df.to_csv(
    friedman_result_path,
    index=False
)


pairwise_wilcoxon_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "pairwise_wilcoxon_holm_results.csv"
)

pairwise_wilcoxon_df.to_csv(
    pairwise_wilcoxon_path,
    index=False
)


mean_model_rank_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "mean_model_rank_outer_folds.csv"
)

mean_model_rank_df.to_csv(
    mean_model_rank_path,
    index=False
)


final_uncertainty_summary_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "final_model_performance_with_uncertainty.csv"
)

final_uncertainty_summary_df.to_csv(
    final_uncertainty_summary_path,
    index=False
)


# ============================================================
# 18. MENAMPILKAN PERINGATAN INTERPRETASI
# ============================================================

print("\n" + "=" * 72)
print("STATISTICAL INTERPRETATION NOTES")
print("=" * 72)

print(
    "1. Outer-fold confidence intervals menggunakan hanya "
    f"{OUTER_N_SPLITS} paired folds."
)

print(
    "2. Bootstrap confidence intervals menggunakan "
    f"{BOOTSTRAP_ITERATIONS} stratified subject-level samples."
)

print(
    "3. Friedman dan Wilcoxon tests bersifat eksploratif "
    "karena jumlah outer folds sangat kecil."
)

print(
    "4. Prediksi antar-fold tidak sepenuhnya independen karena "
    "training sets saling tumpang tindih."
)

print(
    "5. Model terbaik sebaiknya ditentukan berdasarkan "
    "kombinasi ROC-AUC, sensitivity, specificity, stability, "
    "dan konteks klinis; bukan hanya p-value."
)


# ============================================================
# 19. KONFIRMASI STEP 8
# ============================================================

print("\n" + "=" * 72)
print("STEP 8 COMPLETED")
print("=" * 72)

print(
    "Analisis uncertainty dan perbandingan statistik "
    "seluruh model berhasil dilakukan."
)

print(
    "\nBest model from nested CV:"
)

print(
    best_nested_model_name
)

print(
    "Best ROC-AUC:"
)

print(
    f"{best_nested_model_auc:.4f}"
)

print(
    "\nFinal uncertainty summary saved to:"
)

print(
    final_uncertainty_summary_path
)

In [ ]:
# ============================================================
# STEP 9. SELECT AND TRAIN THE FINAL PCA–ML MODEL
# Full-Data GridSearchCV for Final Deployment Configuration
# ============================================================

# Tujuan STEP 9:
# 1. Memilih classifier terbaik berdasarkan nested CV.
# 2. Menjalankan GridSearchCV kembali pada seluruh dataset.
# 3. Menentukan konfigurasi PCA dan hyperparameter final.
# 4. Melatih final pipeline menggunakan seluruh subject.
#
# PENTING:
# Performa ilmiah tetap menggunakan hasil nested CV pada
# STEP 5–8. Hasil prediksi terhadap seluruh training dataset
# pada STEP ini hanya merupakan sanity check, bukan estimasi
# performa generalisasi.


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP SEBELUMNYA
# ============================================================

required_step_9_objects = [
    "X",
    "y",
    "subjects",
    "feature_columns",
    "ml_pipelines",
    "active_param_grids",
    "model_score_methods",
    "nested_model_summary_df",
    "nested_cv_results",
    "best_nested_model_name",
    "best_nested_model_auc",
    "RANDOM_STATE",
    "GRID_SCORING",
    "GRID_N_JOBS",
    "GRID_VERBOSE",
    "CLASS_LABEL_MAPPING",
    "PCA_ML_TABLE_DIR",
    "PCA_ML_MODEL_DIR"
]

missing_step_9_objects = [
    object_name
    for object_name in required_step_9_objects
    if object_name not in globals()
]

if missing_step_9_objects:
    raise NameError(
        "Objek untuk STEP 9 belum lengkap: "
        f"{missing_step_9_objects}. "
        "Jalankan STEP 1–8 terlebih dahulu."
    )


# ============================================================
# 2. MENENTUKAN MODEL FINAL
# ============================================================

final_model_name = str(
    best_nested_model_name
)

if final_model_name not in ml_pipelines:
    raise ValueError(
        "Model terbaik tidak ditemukan dalam ml_pipelines: "
        f"{final_model_name}"
    )

if final_model_name not in active_param_grids:
    raise ValueError(
        "Parameter grid model terbaik tidak ditemukan: "
        f"{final_model_name}"
    )

final_base_pipeline = clone(
    ml_pipelines[
        final_model_name
    ]
)

final_parameter_grid = (
    active_param_grids[
        final_model_name
    ]
)

final_score_method = (
    model_score_methods[
        final_model_name
    ]
)


# ============================================================
# 3. MENYIAPKAN INPUT FINAL
# ============================================================

X_final = (
    X
    .to_numpy(
        dtype=float
    )
)

y_final = np.asarray(
    y,
    dtype=int
)

subjects_final = np.asarray(
    subjects
)


if X_final.shape[0] != len(y_final):
    raise ValueError(
        "Jumlah baris X_final tidak sama dengan y_final."
    )

if X_final.shape[0] != len(subjects_final):
    raise ValueError(
        "Jumlah baris X_final tidak sama dengan subject."
    )

if X_final.shape[1] != len(feature_columns):
    raise ValueError(
        "Jumlah fitur X_final tidak sama dengan "
        "feature_columns."
    )

if not np.array_equal(
    np.unique(y_final),
    np.array([0, 1])
):
    raise ValueError(
        "Target final harus memiliki kelas [0, 1]."
    )


# ============================================================
# 4. CROSS-VALIDATION UNTUK FINAL GRID SEARCH
# ============================================================

# GridSearchCV ini digunakan untuk memilih konfigurasi final
# sebelum deployment.
#
# Karena kelas minoritas berjumlah 10 subject, Stratified
# 5-Fold masih dapat digunakan dengan sekitar dua subject
# LOW/NO pada setiap validation fold.

FINAL_GRID_N_SPLITS = 5

final_grid_cv = StratifiedKFold(
    n_splits=FINAL_GRID_N_SPLITS,
    shuffle=True,
    random_state=(
        RANDOM_STATE + 500
    )
)


minimum_final_class_count = int(
    pd.Series(
        y_final
    )
    .value_counts()
    .min()
)

if minimum_final_class_count < FINAL_GRID_N_SPLITS:
    raise ValueError(
        "Jumlah kelas terkecil tidak mencukupi untuk "
        f"{FINAL_GRID_N_SPLITS}-fold final GridSearchCV. "
        f"Jumlah minimum: {minimum_final_class_count}."
    )


# ============================================================
# 5. MENAMPILKAN KONFIGURASI FINAL GRID SEARCH
# ============================================================

number_of_final_combinations = int(
    np.prod([
        len(parameter_values)
        for parameter_values
        in final_parameter_grid.values()
    ])
)

estimated_final_fits = (
    number_of_final_combinations
    * FINAL_GRID_N_SPLITS
)

print("=" * 72)
print("FINAL PCA–MACHINE LEARNING MODEL SELECTION")
print("=" * 72)

print(
    "Selected classifier       :",
    final_model_name
)

print(
    "Nested CV ROC-AUC         :",
    f"{best_nested_model_auc:.4f}"
)

print(
    "Final score method        :",
    final_score_method
)

print(
    "Number of subjects        :",
    len(y_final)
)

print(
    "Number of input features  :",
    X_final.shape[1]
)

print(
    "Final GridSearchCV        :",
    f"Stratified {FINAL_GRID_N_SPLITS}-Fold"
)

print(
    "Grid scoring              :",
    GRID_SCORING
)

print(
    "Parameter combinations    :",
    number_of_final_combinations
)

print(
    "Estimated CV fits         :",
    estimated_final_fits
)


# ============================================================
# 6. MENJALANKAN FINAL GRID SEARCH
# ============================================================

final_grid_start_time = time.time()

final_grid_search = GridSearchCV(
    estimator=final_base_pipeline,
    param_grid=final_parameter_grid,
    scoring=GRID_SCORING,
    cv=final_grid_cv,
    n_jobs=GRID_N_JOBS,
    verbose=GRID_VERBOSE,
    refit=True,
    return_train_score=True,
    error_score="raise"
)

final_grid_search.fit(
    X_final,
    y_final
)

final_grid_elapsed_seconds = float(
    time.time()
    - final_grid_start_time
)


# ============================================================
# 7. MENGAMBIL FINAL TRAINED PIPELINE
# ============================================================

final_trained_pipeline = (
    final_grid_search
    .best_estimator_
)

final_best_parameters = (
    final_grid_search
    .best_params_
)

final_best_cv_auc = float(
    final_grid_search
    .best_score_
)


# ============================================================
# 8. MEMERIKSA CLASSIFIER DAN URUTAN KELAS
# ============================================================

final_classifier = (
    final_trained_pipeline
    .named_steps["classifier"]
)

final_classifier_classes = np.asarray(
    final_classifier.classes_
)

if not np.array_equal(
    final_classifier_classes,
    np.array([0, 1])
):
    raise ValueError(
        "Urutan kelas final classifier harus [0, 1]. "
        f"Diperoleh: {final_classifier_classes}."
    )


# ============================================================
# 9. MENGAMBIL INFORMASI PCA FINAL
# ============================================================

final_pca = (
    final_trained_pipeline
    .named_steps["pca"]
)

final_selected_pca_parameter = (
    final_best_parameters.get(
        "pca__n_components"
    )
)

final_actual_pca_components = int(
    final_pca.n_components_
)

final_pca_explained_variance = float(
    final_pca
    .explained_variance_ratio_
    .sum()
)

final_pca_component_variance_df = pd.DataFrame({
    "principal_component": [
        f"PC{component_index}"
        for component_index in range(
            1,
            final_actual_pca_components + 1
        )
    ],
    "explained_variance_ratio": (
        final_pca
        .explained_variance_ratio_
    )
})

final_pca_component_variance_df[
    "cumulative_explained_variance"
] = (
    final_pca_component_variance_df[
        "explained_variance_ratio"
    ]
    .cumsum()
)


# ============================================================
# 10. MENGAMBIL SKOR FINAL PADA DATA TRAINING
# SANITY CHECK SAJA
# ============================================================

final_training_predictions = (
    final_trained_pipeline
    .predict(
        X_final
    )
    .astype(int)
)

final_training_scores = (
    extract_positive_class_score(
        fitted_estimator=(
            final_trained_pipeline
        ),
        X_input=X_final,
        score_method=(
            final_score_method
        )
    )
)


if len(final_training_predictions) != len(y_final):
    raise ValueError(
        "Jumlah prediksi sanity check tidak sesuai."
    )

if len(final_training_scores) != len(y_final):
    raise ValueError(
        "Jumlah skor sanity check tidak sesuai."
    )


# ============================================================
# 11. MENGHITUNG TRAINING SANITY-CHECK METRICS
# ============================================================

final_training_sanity_metrics = (
    calculate_nested_binary_metrics(
        y_true=y_final,
        y_pred=final_training_predictions,
        y_score=final_training_scores
    )
)

# Nilai ini tidak digunakan untuk klaim performa model.
# Nilai performa utama tetap berasal dari nested outer CV.


# ============================================================
# 12. MEMBENTUK TABEL PREDIKSI SANITY CHECK
# ============================================================

final_training_sanity_predictions_df = pd.DataFrame({
    SUBJECT_COLUMN: subjects_final,
    "actual_label": y_final,
    "actual_class": [
        CLASS_LABEL_MAPPING[
            int(label)
        ]
        for label in y_final
    ],
    "predicted_label": (
        final_training_predictions
    ),
    "predicted_class": [
        CLASS_LABEL_MAPPING[
            int(label)
        ]
        for label in (
            final_training_predictions
        )
    ],
    "prediction_score": (
        final_training_scores
    ),
    "score_method": (
        final_score_method
    ),
    "correct_prediction": (
        y_final
        == final_training_predictions
    )
})


# ============================================================
# 13. MEMBENTUK RINGKASAN MODEL FINAL
# ============================================================

nested_best_row = (
    nested_model_summary_df[
        nested_model_summary_df[
            "model"
        ]
        == final_model_name
    ]
    .iloc[0]
)


final_model_summary_df = pd.DataFrame([
    {
        "selected_model": (
            final_model_name
        ),
        "number_of_subjects": int(
            len(y_final)
        ),
        "number_of_original_features": int(
            X_final.shape[1]
        ),
        "final_grid_cv_folds": int(
            FINAL_GRID_N_SPLITS
        ),
        "grid_scoring": (
            GRID_SCORING
        ),
        "number_of_grid_combinations": int(
            number_of_final_combinations
        ),
        "selected_pca_parameter": (
            final_selected_pca_parameter
        ),
        "actual_pca_components": int(
            final_actual_pca_components
        ),
        "pca_explained_variance": float(
            final_pca_explained_variance
        ),
        "final_grid_best_auc": float(
            final_best_cv_auc
        ),
        "nested_cv_accuracy": float(
            nested_best_row[
                "accuracy"
            ]
        ),
        "nested_cv_balanced_accuracy": float(
            nested_best_row[
                "balanced_accuracy"
            ]
        ),
        "nested_cv_precision": float(
            nested_best_row[
                "precision"
            ]
        ),
        "nested_cv_recall": float(
            nested_best_row[
                "recall"
            ]
        ),
        "nested_cv_specificity": float(
            nested_best_row[
                "specificity"
            ]
        ),
        "nested_cv_f1_score": float(
            nested_best_row[
                "f1_score"
            ]
        ),
        "nested_cv_roc_auc": float(
            nested_best_row[
                "roc_auc"
            ]
        ),
        "training_sanity_accuracy": float(
            final_training_sanity_metrics[
                "accuracy"
            ]
        ),
        "training_sanity_roc_auc": float(
            final_training_sanity_metrics[
                "roc_auc"
            ]
        ),
        "score_method": (
            final_score_method
        ),
        "elapsed_seconds": (
            final_grid_elapsed_seconds
        )
    }
])


# ============================================================
# 14. MEMBENTUK TABEL HYPERPARAMETER FINAL
# ============================================================

final_best_parameters_df = pd.DataFrame([
    {
        "parameter": parameter_name,
        "selected_value": parameter_value
    }
    for parameter_name, parameter_value
    in final_best_parameters.items()
])


# ============================================================
# 15. MENYIMPAN SELURUH HASIL GRID SEARCH FINAL
# ============================================================

final_grid_results_df = pd.DataFrame(
    final_grid_search.cv_results_
)

final_grid_result_columns = [
    column_name
    for column_name in [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "std_train_score",
        "mean_fit_time",
        "std_fit_time",
        "mean_score_time",
        "params"
    ]
    if column_name
    in final_grid_results_df.columns
]

final_grid_results_summary_df = (
    final_grid_results_df[
        final_grid_result_columns
    ]
    .sort_values(
        by="rank_test_score",
        ascending=True
    )
    .reset_index(drop=True)
)


# ============================================================
# 16. MEMERIKSA GAP INNER-CV TRAIN DAN VALIDATION
# ============================================================

best_grid_row = (
    final_grid_results_df.loc[
        final_grid_search.best_index_
    ]
)

final_grid_mean_train_auc = float(
    best_grid_row[
        "mean_train_score"
    ]
)

final_grid_mean_validation_auc = float(
    best_grid_row[
        "mean_test_score"
    ]
)

final_grid_train_validation_gap = float(
    final_grid_mean_train_auc
    - final_grid_mean_validation_auc
)

final_model_summary_df[
    "final_grid_mean_train_auc"
] = (
    final_grid_mean_train_auc
)

final_model_summary_df[
    "final_grid_mean_validation_auc"
] = (
    final_grid_mean_validation_auc
)

final_model_summary_df[
    "final_grid_train_validation_gap"
] = (
    final_grid_train_validation_gap
)


# ============================================================
# 17. MENAMPILKAN MODEL FINAL
# ============================================================

print("\n" + "=" * 72)
print("FINAL PCA–ML MODEL CONFIGURATION")
print("=" * 72)

print(
    "Selected classifier           :",
    final_model_name
)

print(
    "Selected PCA parameter        :",
    final_selected_pca_parameter
)

print(
    "Actual PCA components         :",
    final_actual_pca_components
)

print(
    "PCA explained variance        :",
    f"{final_pca_explained_variance:.4f}"
)

print(
    "Final GridSearchCV best AUC   :",
    f"{final_best_cv_auc:.4f}"
)

print(
    "Final grid mean train AUC     :",
    f"{final_grid_mean_train_auc:.4f}"
)

print(
    "Final grid mean validation AUC:",
    f"{final_grid_mean_validation_auc:.4f}"
)

print(
    "Train–validation AUC gap      :",
    f"{final_grid_train_validation_gap:.4f}"
)

print(
    "Nested CV ROC-AUC             :",
    f"{nested_best_row['roc_auc']:.4f}"
)

print(
    "Nested CV accuracy            :",
    f"{nested_best_row['accuracy']:.4f}"
)

print(
    "Nested CV sensitivity         :",
    f"{nested_best_row['recall']:.4f}"
)

print(
    "Nested CV specificity         :",
    f"{nested_best_row['specificity']:.4f}"
)

print(
    "Nested CV F1-score            :",
    f"{nested_best_row['f1_score']:.4f}"
)

print(
    "Elapsed time                  :",
    f"{final_grid_elapsed_seconds:.2f} seconds"
)


# ============================================================
# 18. MENAMPILKAN HYPERPARAMETER TERBAIK
# ============================================================

print("\n" + "=" * 72)
print("FINAL SELECTED HYPERPARAMETERS")
print("=" * 72)

display(
    final_best_parameters_df
)


# ============================================================
# 19. MENAMPILKAN EXPLAINED VARIANCE PCA
# ============================================================

print("\n" + "=" * 72)
print("FINAL PCA EXPLAINED VARIANCE")
print("=" * 72)

display(
    final_pca_component_variance_df
)


# ============================================================
# 20. MENAMPILKAN RINGKASAN FINAL MODEL
# ============================================================

print("\n" + "=" * 72)
print("FINAL MODEL SUMMARY")
print("=" * 72)

display(
    final_model_summary_df
)


# ============================================================
# 21. MENAMPILKAN SANITY CHECK
# ============================================================

print("\n" + "=" * 72)
print("TRAINING SANITY CHECK")
print("=" * 72)

print(
    "Training accuracy:",
    f"{final_training_sanity_metrics['accuracy']:.4f}"
)

print(
    "Training ROC-AUC :",
    f"{final_training_sanity_metrics['roc_auc']:.4f}"
)

print(
    "\nCatatan: hasil training sanity check bukan estimasi "
    "performa generalisasi dan tidak digunakan sebagai "
    "hasil utama penelitian."
)


# ============================================================
# 22. PERINGATAN OVERFITTING
# ============================================================

if final_grid_train_validation_gap > 0.20:

    print(
        "\nPERINGATAN: Ditemukan train–validation AUC gap "
        "lebih besar dari 0.20."
    )

    print(
        "Model masih menunjukkan indikasi overfitting. "
        "Gunakan hasil nested CV sebagai estimasi performa "
        "utama dan interpretasikan deployment secara hati-hati."
    )

elif final_grid_train_validation_gap > 0.10:

    print(
        "\nCatatan: Terdapat train–validation AUC gap "
        "moderat."
    )

else:

    print(
        "\nTrain–validation AUC gap relatif kecil."
    )


# ============================================================
# 23. MENYIMPAN TABEL HASIL STEP 9
# ============================================================

final_model_summary_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "final_model_training_summary.csv"
)

final_model_summary_df.to_csv(
    final_model_summary_path,
    index=False
)


final_best_parameters_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "final_selected_hyperparameters.csv"
)

final_best_parameters_df.to_csv(
    final_best_parameters_path,
    index=False
)


final_pca_variance_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "final_pca_explained_variance.csv"
)

final_pca_component_variance_df.to_csv(
    final_pca_variance_path,
    index=False
)


final_grid_results_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "final_full_data_grid_search_results.csv"
)

final_grid_results_summary_df.to_csv(
    final_grid_results_path,
    index=False
)


final_training_sanity_predictions_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "final_training_sanity_predictions.csv"
)

final_training_sanity_predictions_df.to_csv(
    final_training_sanity_predictions_path,
    index=False
)


# ============================================================
# 24. KONFIRMASI STEP 9
# ============================================================

print("\n" + "=" * 72)
print("STEP 9 COMPLETED")
print("=" * 72)

print(
    "Final PCA–machine learning pipeline berhasil dipilih "
    "dan dilatih menggunakan seluruh dataset."
)

print(
    "\nSelected model:"
)

print(
    final_model_name
)

print(
    "\nNested CV performance remains the primary result:"
)

print(
    f"ROC-AUC = {nested_best_row['roc_auc']:.4f}"
)

print(
    "\nFinal trained pipeline tersedia dalam variabel:"
)

print(
    "final_trained_pipeline"
)

print(
    "\nPada STEP 10, pipeline, feature order, metadata, "
    "dan fungsi inference akan disimpan untuk deployment."
)

In [ ]:
# ============================================================
# STEP 10. SAVE FINAL MODEL AND BUILD INFERENCE FUNCTION
# Deployment Pipeline, Feature Order, Metadata, and Validation
# ============================================================

# Tujuan STEP 10:
# 1. Menyimpan final PCA–ML pipeline.
# 2. Menyimpan urutan fitur input.
# 3. Menyimpan metadata eksperimen.
# 4. Membuat fungsi prediksi untuk subject baru.
# 5. Memastikan model yang dimuat ulang menghasilkan prediksi
#    yang identik.
#
# Catatan:
# Pipeline sudah mencakup:
# - median imputation;
# - StandardScaler;
# - PCA;
# - classifier.
#
# Karena seluruh preprocessing berada di dalam pipeline,
# inference hanya perlu menyediakan fitur mentah dalam urutan
# dan nama kolom yang sesuai.


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP 9
# ============================================================

required_step_10_objects = [
    "final_trained_pipeline",
    "final_model_name",
    "final_best_parameters",
    "final_score_method",
    "final_selected_pca_parameter",
    "final_actual_pca_components",
    "final_pca_explained_variance",
    "final_model_summary_df",
    "feature_columns",
    "X",
    "y",
    "subjects",
    "CLASS_LABEL_MAPPING",
    "PCA_ML_MODEL_DIR",
    "PCA_ML_TABLE_DIR",
    "RANDOM_STATE"
]

missing_step_10_objects = [
    object_name
    for object_name in required_step_10_objects
    if object_name not in globals()
]

if missing_step_10_objects:
    raise NameError(
        "Objek untuk STEP 10 belum lengkap: "
        f"{missing_step_10_objects}. "
        "Jalankan STEP 9 terlebih dahulu."
    )


# ============================================================
# 2. IMPORT LIBRARY DEPLOYMENT
# ============================================================

import platform
import sklearn


# ============================================================
# 3. MEMBUAT NAMA FILE MODEL YANG AMAN
# ============================================================

safe_final_model_name = (
    final_model_name
    .lower()
    .replace(
        " ",
        "_"
    )
    .replace(
        "-",
        "_"
    )
)

final_pipeline_filename = (
    f"final_pca_{safe_final_model_name}_pipeline.joblib"
)

final_pipeline_path = os.path.join(
    PCA_ML_MODEL_DIR,
    final_pipeline_filename
)


# ============================================================
# 4. VALIDASI URUTAN FITUR
# ============================================================

deployment_feature_columns = list(
    feature_columns
)

if len(
    deployment_feature_columns
) != len(
    set(
        deployment_feature_columns
    )
):
    raise ValueError(
        "feature_columns mengandung nama fitur duplikat."
    )

missing_features_in_X = [
    feature
    for feature in deployment_feature_columns
    if feature not in X.columns
]

if missing_features_in_X:
    raise ValueError(
        "Fitur deployment berikut tidak ditemukan dalam X: "
        f"{missing_features_in_X}"
    )

if len(
    deployment_feature_columns
) != X.shape[1]:
    raise ValueError(
        "Jumlah deployment_feature_columns tidak sama "
        "dengan jumlah kolom X."
    )


# ============================================================
# 5. VALIDASI STRUKTUR FINAL PIPELINE
# ============================================================

required_final_pipeline_steps = [
    "imputer",
    "scaler",
    "pca",
    "classifier"
]

available_final_pipeline_steps = list(
    final_trained_pipeline
    .named_steps
    .keys()
)

missing_final_pipeline_steps = [
    step_name
    for step_name in required_final_pipeline_steps
    if step_name not in available_final_pipeline_steps
]

if missing_final_pipeline_steps:
    raise ValueError(
        "Final pipeline belum memiliki step: "
        f"{missing_final_pipeline_steps}"
    )


final_classifier = (
    final_trained_pipeline
    .named_steps["classifier"]
)

final_classifier_classes = np.asarray(
    final_classifier.classes_
)

if not np.array_equal(
    final_classifier_classes,
    np.array([
        0,
        1
    ])
):
    raise ValueError(
        "Urutan kelas final classifier harus [0, 1]. "
        f"Diperoleh: {final_classifier_classes}."
    )


# ============================================================
# 6. MENYIMPAN FINAL PIPELINE
# ============================================================

joblib.dump(
    final_trained_pipeline,
    final_pipeline_path
)


# ============================================================
# 7. MENYIMPAN URUTAN FITUR
# ============================================================

deployment_feature_order_df = pd.DataFrame({
    "feature_order": np.arange(
        1,
        len(
            deployment_feature_columns
        ) + 1
    ),
    "feature": deployment_feature_columns
})

deployment_feature_order_csv_path = os.path.join(
    PCA_ML_MODEL_DIR,
    "deployment_feature_order.csv"
)

deployment_feature_order_df.to_csv(
    deployment_feature_order_csv_path,
    index=False
)


deployment_feature_order_json_path = os.path.join(
    PCA_ML_MODEL_DIR,
    "deployment_feature_order.json"
)

with open(
    deployment_feature_order_json_path,
    "w",
    encoding="utf-8"
) as feature_json_file:

    json.dump(
        deployment_feature_columns,
        feature_json_file,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 8. MENYIAPKAN HYPERPARAMETER SERIALIZABLE
# ============================================================

serializable_final_best_parameters = {}

for parameter_name, parameter_value in (
    final_best_parameters.items()
):

    if isinstance(
        parameter_value,
        np.generic
    ):
        parameter_value = (
            parameter_value.item()
        )

    serializable_final_best_parameters[
        parameter_name
    ] = parameter_value


# ============================================================
# 9. MENGAMBIL HASIL NESTED CV MODEL FINAL
# ============================================================

final_nested_summary_row = (
    nested_model_summary_df[
        nested_model_summary_df[
            "model"
        ]
        == final_model_name
    ]
    .iloc[0]
)


# ============================================================
# 10. MEMBENTUK METADATA DEPLOYMENT
# ============================================================

deployment_metadata = {
    "model_name": final_model_name,

    "model_file": (
        final_pipeline_filename
    ),

    "pipeline_steps": (
        available_final_pipeline_steps
    ),

    "number_of_training_subjects": int(
        len(
            y
        )
    ),

    "number_of_input_features": int(
        len(
            deployment_feature_columns
        )
    ),

    "selected_pca_parameter": (
        final_selected_pca_parameter
    ),

    "actual_pca_components": int(
        final_actual_pca_components
    ),

    "pca_explained_variance": float(
        final_pca_explained_variance
    ),

    "selected_hyperparameters": (
        serializable_final_best_parameters
    ),

    "score_method": (
        final_score_method
    ),

    "target_encoding": {
        "0": "LOW/NO",
        "1": "HIGH"
    },

    "positive_class": {
        "id": 1,
        "name": "HIGH"
    },

    "negative_class": {
        "id": 0,
        "name": "LOW/NO"
    },

    "validation_method": (
        "Nested Stratified Cross-Validation"
    ),

    "outer_cross_validation": (
        f"StratifiedKFold(n_splits={OUTER_N_SPLITS}, "
        f"shuffle=True)"
    ),

    "inner_grid_search_cross_validation": (
        f"StratifiedKFold(n_splits={INNER_N_SPLITS}, "
        f"shuffle=True)"
    ),

    "grid_search_scoring": (
        GRID_SCORING
    ),

    "nested_cv_performance": {
        "accuracy": float(
            final_nested_summary_row[
                "accuracy"
            ]
        ),
        "balanced_accuracy": float(
            final_nested_summary_row[
                "balanced_accuracy"
            ]
        ),
        "precision": float(
            final_nested_summary_row[
                "precision"
            ]
        ),
        "recall_sensitivity": float(
            final_nested_summary_row[
                "recall"
            ]
        ),
        "specificity": float(
            final_nested_summary_row[
                "specificity"
            ]
        ),
        "f1_score": float(
            final_nested_summary_row[
                "f1_score"
            ]
        ),
        "roc_auc": float(
            final_nested_summary_row[
                "roc_auc"
            ]
        )
    },

    "random_state": int(
        RANDOM_STATE
    ),

    "software_versions": {
        "python": (
            platform.python_version()
        ),
        "numpy": (
            np.__version__
        ),
        "pandas": (
            pd.__version__
        ),
        "scikit_learn": (
            sklearn.__version__
        ),
        "joblib": (
            joblib.__version__
        )
    },

    "important_note": (
        "The deployment model was trained using all available "
        "subjects. Reported scientific performance must use "
        "nested cross-validation results, not training-set "
        "predictions."
    )
}


deployment_metadata_path = os.path.join(
    PCA_ML_MODEL_DIR,
    "deployment_metadata.json"
)

with open(
    deployment_metadata_path,
    "w",
    encoding="utf-8"
) as metadata_file:

    json.dump(
        deployment_metadata,
        metadata_file,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 11. MEMBUAT TEMPLATE INPUT SUBJECT BARU
# ============================================================

deployment_input_template_df = pd.DataFrame(
    columns=[
        SUBJECT_COLUMN
    ]
    + deployment_feature_columns
)

deployment_input_template_path = os.path.join(
    PCA_ML_MODEL_DIR,
    "new_subject_input_template.csv"
)

deployment_input_template_df.to_csv(
    deployment_input_template_path,
    index=False
)


# ============================================================
# 12. FUNGSI MEMUAT MODEL DAN FITUR DEPLOYMENT
# ============================================================

def load_fall_risk_deployment_artifacts(
    model_path=final_pipeline_path,
    feature_path=deployment_feature_order_json_path
):
    """
    Memuat final pipeline dan urutan fitur deployment.

    Returns
    -------
    tuple
        loaded_pipeline, required_feature_columns
    """

    if not os.path.exists(
        model_path
    ):
        raise FileNotFoundError(
            f"File model tidak ditemukan: {model_path}"
        )

    if not os.path.exists(
        feature_path
    ):
        raise FileNotFoundError(
            f"File feature order tidak ditemukan: "
            f"{feature_path}"
        )

    loaded_pipeline = joblib.load(
        model_path
    )

    with open(
        feature_path,
        "r",
        encoding="utf-8"
    ) as feature_file:

        required_feature_columns = json.load(
            feature_file
        )

    return (
        loaded_pipeline,
        required_feature_columns
    )


# ============================================================
# 13. FUNGSI MENGAMBIL SKOR INFERENCE
# ============================================================

def extract_deployment_score(
    fitted_pipeline,
    X_input,
    score_method
):
    """
    Mengambil skor kontinu kelas HIGH dari pipeline.
    """

    classifier = (
        fitted_pipeline
        .named_steps["classifier"]
    )

    classifier_classes = np.asarray(
        classifier.classes_
    )

    positive_class_indices = np.where(
        classifier_classes == 1
    )[0]

    if len(
        positive_class_indices
    ) != 1:
        raise ValueError(
            "Kelas positif HIGH=1 tidak ditemukan secara unik."
        )

    positive_class_index = int(
        positive_class_indices[0]
    )

    if score_method == "predict_proba":

        predicted_probabilities = (
            fitted_pipeline
            .predict_proba(
                X_input
            )
        )

        prediction_scores = (
            predicted_probabilities[
                :,
                positive_class_index
            ]
        )

    elif score_method == "decision_function":

        prediction_scores = np.ravel(
            fitted_pipeline
            .decision_function(
                X_input
            )
        )

    else:
        raise ValueError(
            "score_method tidak dikenali: "
            f"{score_method}"
        )

    prediction_scores = np.asarray(
        prediction_scores,
        dtype=float
    )

    if not np.isfinite(
        prediction_scores
    ).all():
        raise ValueError(
            "Prediction score mengandung NaN atau infinite."
        )

    return prediction_scores


# ============================================================
# 14. FUNGSI INFERENCE SUBJECT BARU
# ============================================================

def predict_fall_risk(
    new_data,
    model_pipeline=final_trained_pipeline,
    required_features=deployment_feature_columns,
    score_method=final_score_method,
    subject_column=SUBJECT_COLUMN
):
    """
    Memprediksi fall risk satu atau lebih subject baru.

    Parameters
    ----------
    new_data : pandas.DataFrame
        DataFrame yang memuat seluruh fitur model.
        Setiap baris merepresentasikan satu subject.

    model_pipeline : sklearn Pipeline
        Pipeline deployment yang telah dilatih.

    required_features : list
        Nama dan urutan fitur yang dibutuhkan model.

    score_method : str
        predict_proba atau decision_function.

    subject_column : str
        Nama kolom identitas subject.

    Returns
    -------
    pandas.DataFrame
        Hasil prediksi per subject.
    """

    if not isinstance(
        new_data,
        pd.DataFrame
    ):
        raise TypeError(
            "new_data harus berupa pandas DataFrame."
        )

    if len(
        new_data
    ) == 0:
        raise ValueError(
            "new_data tidak boleh kosong."
        )

    duplicated_required_features = (
        pd.Series(
            required_features
        )
        .duplicated()
        .sum()
    )

    if duplicated_required_features > 0:
        raise ValueError(
            "required_features mengandung nama duplikat."
        )

    missing_input_features = [
        feature
        for feature in required_features
        if feature not in new_data.columns
    ]

    if missing_input_features:
        raise ValueError(
            "Fitur berikut belum tersedia pada new_data: "
            f"{missing_input_features}"
        )

    inference_X = (
        new_data[
            required_features
        ]
        .copy()
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
    )

    # Missing value dipertahankan sebagai NaN karena median
    # imputation sudah berada di dalam pipeline.

    predicted_labels = (
        model_pipeline
        .predict(
            inference_X
        )
        .astype(int)
    )

    prediction_scores = (
        extract_deployment_score(
            fitted_pipeline=(
                model_pipeline
            ),
            X_input=inference_X,
            score_method=(
                score_method
            )
        )
    )

    prediction_result_df = pd.DataFrame({
        "predicted_label": (
            predicted_labels
        ),
        "predicted_class": [
            CLASS_LABEL_MAPPING[
                int(label)
            ]
            for label in predicted_labels
        ],
        "prediction_score_HIGH": (
            prediction_scores
        ),
        "score_method": (
            score_method
        )
    })

    if (
        subject_column
        in new_data.columns
    ):

        prediction_result_df.insert(
            0,
            subject_column,
            new_data[
                subject_column
            ]
            .astype(str)
            .str.strip()
            .reset_index(
                drop=True
            )
        )

    return prediction_result_df


# ============================================================
# 15. SANITY CHECK FUNGSI INFERENCE
# ============================================================

sanity_check_size = min(
    3,
    len(
        X
    )
)

sanity_check_input_df = pd.concat(
    [
        pd.Series(
            subjects[
                :sanity_check_size
            ],
            name=SUBJECT_COLUMN
        ),
        X.iloc[
            :sanity_check_size
        ].reset_index(
            drop=True
        )
    ],
    axis=1
)

sanity_check_output_df = predict_fall_risk(
    sanity_check_input_df
)

print("=" * 72)
print("DEPLOYMENT INFERENCE SANITY CHECK")
print("=" * 72)

display(
    sanity_check_output_df
)


# ============================================================
# 16. MEMUAT ULANG MODEL DAN MEMVERIFIKASI PREDIKSI
# ============================================================

(
    reloaded_pipeline,
    reloaded_feature_columns
) = load_fall_risk_deployment_artifacts()


if reloaded_feature_columns != deployment_feature_columns:
    raise ValueError(
        "Urutan fitur setelah dimuat ulang tidak identik."
    )


original_reload_test_predictions = (
    final_trained_pipeline
    .predict(
        X[
            deployment_feature_columns
        ]
    )
)

reloaded_test_predictions = (
    reloaded_pipeline
    .predict(
        X[
            reloaded_feature_columns
        ]
    )
)


if not np.array_equal(
    original_reload_test_predictions,
    reloaded_test_predictions
):
    raise ValueError(
        "Prediksi model sebelum dan setelah penyimpanan "
        "tidak identik."
    )


original_reload_test_scores = (
    extract_deployment_score(
        fitted_pipeline=(
            final_trained_pipeline
        ),
        X_input=X[
            deployment_feature_columns
        ],
        score_method=(
            final_score_method
        )
    )
)

reloaded_test_scores = (
    extract_deployment_score(
        fitted_pipeline=(
            reloaded_pipeline
        ),
        X_input=X[
            reloaded_feature_columns
        ],
        score_method=(
            final_score_method
        )
    )
)


if not np.allclose(
    original_reload_test_scores,
    reloaded_test_scores
):
    raise ValueError(
        "Skor model sebelum dan setelah penyimpanan "
        "tidak identik."
    )


print(
    "\nVerifikasi penyimpanan model berhasil."
)

print(
    "Prediksi dan skor sebelum serta setelah reload identik."
)


# ============================================================
# 17. MEMBENTUK RINGKASAN ARTEFAK DEPLOYMENT
# ============================================================

deployment_artifacts_df = pd.DataFrame([
    {
        "artifact": "Final PCA–ML pipeline",
        "path": final_pipeline_path
    },
    {
        "artifact": "Feature order CSV",
        "path": (
            deployment_feature_order_csv_path
        )
    },
    {
        "artifact": "Feature order JSON",
        "path": (
            deployment_feature_order_json_path
        )
    },
    {
        "artifact": "Deployment metadata",
        "path": (
            deployment_metadata_path
        )
    },
    {
        "artifact": "New-subject input template",
        "path": (
            deployment_input_template_path
        )
    }
])


deployment_artifacts_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "deployment_artifacts.csv"
)

deployment_artifacts_df.to_csv(
    deployment_artifacts_path,
    index=False
)


# ============================================================
# 18. MENAMPILKAN ARTEFAK DEPLOYMENT
# ============================================================

print("\n" + "=" * 72)
print("DEPLOYMENT ARTIFACTS")
print("=" * 72)

display(
    deployment_artifacts_df
)


# ============================================================
# 19. RINGKASAN MODEL DEPLOYMENT
# ============================================================

print("\n" + "=" * 72)
print("FINAL DEPLOYMENT MODEL")
print("=" * 72)

print(
    "Model                  :",
    final_model_name
)

print(
    "Original features      :",
    len(
        deployment_feature_columns
    )
)

print(
    "PCA components         :",
    final_actual_pca_components
)

print(
    "Explained variance     :",
    f"{final_pca_explained_variance:.4f}"
)

print(
    "Score method           :",
    final_score_method
)

print(
    "Nested CV ROC-AUC      :",
    f"{final_nested_summary_row['roc_auc']:.4f}"
)

print(
    "Nested CV sensitivity  :",
    f"{final_nested_summary_row['recall']:.4f}"
)

print(
    "Nested CV specificity  :",
    f"{final_nested_summary_row['specificity']:.4f}"
)

print(
    "\nFinal pipeline saved to:"
)

print(
    final_pipeline_path
)


# ============================================================
# 20. KONFIRMASI STEP 10
# ============================================================

print("\n" + "=" * 72)
print("STEP 10 COMPLETED")
print("=" * 72)

print(
    "Final PCA–machine learning pipeline, feature order, "
    "metadata, input template, dan fungsi inference "
    "berhasil disiapkan."
)

print(
    "\nGunakan fungsi berikut untuk subject baru:"
)

print(
    "predict_fall_risk(new_data)"
)

print(
    "\nPerforma penelitian tetap dilaporkan menggunakan "
    "hasil nested cross-validation."
)

In [ ]:
# ============================================================
# STEP 11. GENERATE FINAL CONFUSION MATRICES
# Nested Cross-Validation Confusion Matrix for All Models
# ============================================================

# Confusion matrix utama menggunakan out-of-fold predictions
# dari outer nested cross-validation.
#
# Setiap subject diprediksi hanya saat berada pada outer
# test fold, sehingga hasil ini valid untuk evaluasi model.


# ============================================================
# 1. VALIDASI OBJEK
# ============================================================

required_step_11_objects = [
    "nested_cv_results",
    "nested_model_summary_df",
    "best_nested_model_name",
    "ml_pipelines",
    "CLASS_LABEL_MAPPING",
    "PCA_ML_FIGURE_DIR",
    "PCA_ML_TABLE_DIR"
]

missing_step_11_objects = [
    object_name
    for object_name in required_step_11_objects
    if object_name not in globals()
]

if missing_step_11_objects:
    raise NameError(
        "Objek untuk STEP 11 belum lengkap: "
        f"{missing_step_11_objects}. "
        "Jalankan STEP 5 terlebih dahulu."
    )


# ============================================================
# 2. IMPORT VISUALISASI CONFUSION MATRIX
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)


# ============================================================
# 3. KONFIGURASI
# ============================================================

FIGURE_DPI = 300

class_ids = [
    0,
    1
]

class_names = [
    CLASS_LABEL_MAPPING[0],
    CLASS_LABEL_MAPPING[1]
]

os.makedirs(
    PCA_ML_FIGURE_DIR,
    exist_ok=True
)


# ============================================================
# 4. CONFUSION MATRIX SELURUH MODEL
# ============================================================

all_confusion_matrix_records = []

all_classification_report_frames = []

confusion_matrix_paths = []


for model_name in ml_pipelines.keys():

    model_result = (
        nested_cv_results[
            model_name
        ]
    )

    model_y_true = np.asarray(
        model_result[
            "y_true"
        ],
        dtype=int
    )

    model_y_pred = np.asarray(
        model_result[
            "y_pred"
        ],
        dtype=int
    )


    # ========================================================
    # Confusion matrix
    # ========================================================

    model_cm = confusion_matrix(
        model_y_true,
        model_y_pred,
        labels=class_ids
    )

    tn, fp, fn, tp = (
        model_cm.ravel()
    )


    # ========================================================
    # Metrik dari confusion matrix
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    positive_predictive_value = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    negative_predictive_value = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else 0.0
    )


    all_confusion_matrix_records.append({
        "model": model_name,
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "sensitivity": float(
            sensitivity
        ),
        "specificity": float(
            specificity
        ),
        "positive_predictive_value": float(
            positive_predictive_value
        ),
        "negative_predictive_value": float(
            negative_predictive_value
        ),
        "correct_predictions": int(
            tn + tp
        ),
        "incorrect_predictions": int(
            fp + fn
        ),
        "total_subjects": int(
            model_cm.sum()
        )
    })


    # ========================================================
    # Classification report
    # ========================================================

    report_dictionary = classification_report(
        model_y_true,
        model_y_pred,
        labels=class_ids,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    model_report_df = (
        pd.DataFrame(
            report_dictionary
        )
        .transpose()
        .reset_index()
        .rename(
            columns={
                "index": "class_or_metric"
            }
        )
    )

    model_report_df.insert(
        0,
        "model",
        model_name
    )

    all_classification_report_frames.append(
        model_report_df
    )


    # ========================================================
    # Visualisasi confusion matrix
    # ========================================================

    figure, axis = plt.subplots(
        figsize=(6, 5)
    )

    confusion_display = ConfusionMatrixDisplay(
        confusion_matrix=model_cm,
        display_labels=class_names
    )

    confusion_display.plot(
        ax=axis,
        values_format="d",
        colorbar=False
    )

    axis.set_title(
        f"Nested CV Confusion Matrix—{model_name}"
    )

    axis.set_xlabel(
        "Predicted Fall-Risk Class"
    )

    axis.set_ylabel(
        "Actual Fall-Risk Class"
    )

    plt.tight_layout()


    safe_model_name = (
        model_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )

    model_confusion_path = os.path.join(
        PCA_ML_FIGURE_DIR,
        (
            "nested_cv_confusion_matrix_"
            f"{safe_model_name}.png"
        )
    )

    plt.savefig(
        model_confusion_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()

    confusion_matrix_paths.append({
        "model": model_name,
        "figure_path": model_confusion_path
    })


# ============================================================
# 5. MEMBENTUK DATAFRAME RINGKASAN
# ============================================================

confusion_matrix_summary_df = (
    pd.DataFrame(
        all_confusion_matrix_records
    )
    .merge(
        nested_model_summary_df[
            [
                "model",
                "performance_rank",
                "accuracy",
                "balanced_accuracy",
                "precision",
                "recall",
                "specificity",
                "f1_score",
                "roc_auc"
            ]
        ],
        on="model",
        how="left",
        suffixes=(
            "_from_cm",
            "_from_summary"
        )
    )
    .sort_values(
        by="performance_rank"
    )
    .reset_index(drop=True)
)


classification_reports_all_models_df = (
    pd.concat(
        all_classification_report_frames,
        ignore_index=True
    )
)


confusion_matrix_figure_paths_df = (
    pd.DataFrame(
        confusion_matrix_paths
    )
)


# ============================================================
# 6. MENAMPILKAN RINGKASAN SELURUH MODEL
# ============================================================

print("=" * 72)
print("NESTED CROSS-VALIDATION CONFUSION MATRIX SUMMARY")
print("=" * 72)

display(
    confusion_matrix_summary_df[
        [
            "performance_rank",
            "model",
            "true_negative",
            "false_positive",
            "false_negative",
            "true_positive",
            "accuracy",
            "sensitivity",
            "specificity_from_cm",
            "positive_predictive_value",
            "negative_predictive_value",
            "f1_score",
            "roc_auc"
        ]
    ]
)


# ============================================================
# 7. CONFUSION MATRIX MODEL TERBAIK
# ============================================================

best_model_result = (
    nested_cv_results[
        best_nested_model_name
    ]
)

best_model_y_true = np.asarray(
    best_model_result[
        "y_true"
    ],
    dtype=int
)

best_model_y_pred = np.asarray(
    best_model_result[
        "y_pred"
    ],
    dtype=int
)

best_model_cm = confusion_matrix(
    best_model_y_true,
    best_model_y_pred,
    labels=class_ids
)

best_tn, best_fp, best_fn, best_tp = (
    best_model_cm.ravel()
)


best_model_confusion_df = pd.DataFrame(
    best_model_cm,
    index=[
        "Actual LOW/NO",
        "Actual HIGH"
    ],
    columns=[
        "Predicted LOW/NO",
        "Predicted HIGH"
    ]
)


print("\n" + "=" * 72)
print("BEST MODEL CONFUSION MATRIX")
print("=" * 72)

print(
    "Best model:",
    best_nested_model_name
)

display(
    best_model_confusion_df
)


# ============================================================
# 8. VISUALISASI MODEL TERBAIK
# ============================================================

figure, axis = plt.subplots(
    figsize=(6.5, 5.5)
)

best_confusion_display = ConfusionMatrixDisplay(
    confusion_matrix=best_model_cm,
    display_labels=class_names
)

best_confusion_display.plot(
    ax=axis,
    values_format="d",
    colorbar=False
)

axis.set_title(
    (
        "Best Nested CV Model Confusion Matrix\n"
        f"{best_nested_model_name}"
    )
)

axis.set_xlabel(
    "Predicted Fall-Risk Class"
)

axis.set_ylabel(
    "Actual Fall-Risk Class"
)

plt.tight_layout()


best_model_confusion_figure_path = os.path.join(
    PCA_ML_FIGURE_DIR,
    "best_nested_cv_model_confusion_matrix.png"
)

plt.savefig(
    best_model_confusion_figure_path,
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. CLASSIFICATION REPORT MODEL TERBAIK
# ============================================================

best_model_classification_report_dict = (
    classification_report(
        best_model_y_true,
        best_model_y_pred,
        labels=class_ids,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
)

best_model_classification_report_df = (
    pd.DataFrame(
        best_model_classification_report_dict
    )
    .transpose()
)


print("\n" + "=" * 72)
print("BEST MODEL CLASSIFICATION REPORT")
print("=" * 72)

display(
    best_model_classification_report_df
)

print(
    classification_report(
        best_model_y_true,
        best_model_y_pred,
        labels=class_ids,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 10. INTERPRETASI CONFUSION MATRIX MODEL TERBAIK
# ============================================================

best_sensitivity = (
    best_tp / (
        best_tp + best_fn
    )
    if (
        best_tp + best_fn
    ) > 0
    else 0.0
)

best_specificity = (
    best_tn / (
        best_tn + best_fp
    )
    if (
        best_tn + best_fp
    ) > 0
    else 0.0
)

best_accuracy_from_cm = (
    (
        best_tn + best_tp
    )
    / best_model_cm.sum()
)

print("\n" + "=" * 72)
print("BEST MODEL CONFUSION MATRIX INTERPRETATION")
print("=" * 72)

print(
    "True Negative  :",
    best_tn,
    "subject LOW/NO diprediksi LOW/NO"
)

print(
    "False Positive :",
    best_fp,
    "subject LOW/NO diprediksi HIGH"
)

print(
    "False Negative :",
    best_fn,
    "subject HIGH diprediksi LOW/NO"
)

print(
    "True Positive  :",
    best_tp,
    "subject HIGH diprediksi HIGH"
)

print(
    "\nAccuracy       :",
    f"{best_accuracy_from_cm:.4f}"
)

print(
    "Sensitivity    :",
    f"{best_sensitivity:.4f}"
)

print(
    "Specificity    :",
    f"{best_specificity:.4f}"
)


# ============================================================
# 11. MENYIMPAN HASIL CONFUSION MATRIX
# ============================================================

confusion_summary_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_confusion_matrix_summary_complete.csv"
)

confusion_matrix_summary_df.to_csv(
    confusion_summary_path,
    index=False
)


classification_reports_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "nested_cv_classification_reports_all_models.csv"
)

classification_reports_all_models_df.to_csv(
    classification_reports_path,
    index=False
)


best_model_confusion_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "best_nested_cv_model_confusion_matrix.csv"
)

best_model_confusion_df.to_csv(
    best_model_confusion_path
)


best_model_report_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "best_nested_cv_model_classification_report.csv"
)

best_model_classification_report_df.to_csv(
    best_model_report_path
)


confusion_figure_paths_path = os.path.join(
    PCA_ML_TABLE_DIR,
    "confusion_matrix_figure_paths.csv"
)

confusion_matrix_figure_paths_df.to_csv(
    confusion_figure_paths_path,
    index=False
)


# ============================================================
# 12. KONFIRMASI
# ============================================================

print("\n" + "=" * 72)
print("STEP 11 COMPLETED")
print("=" * 72)

print(
    "Confusion matrix seluruh model dan model terbaik "
    "berhasil dibuat."
)

print(
    "\nBest model confusion matrix saved to:"
)

print(
    best_model_confusion_figure_path
)

ENSEMBLE LEARNING

In [ ]:

# ============================================================
# STEP 1. INITIALIZE PURE ENSEMBLE MACHINE-LEARNING EXPERIMENT
# Imports, Configuration, Cross-Validation, and Output Directory
# ============================================================

# Pipeline penelitian:
#
# Multimodal features
#       ↓
# Median imputation
#       ↓
# Ensemble classifier
#       ↓
# Inner GridSearchCV
#       ↓
# Outer Stratified K-Fold evaluation
#
# Tidak digunakan:
# - StandardScaler
# - PCA
#
# Model ensemble:
# 1. Random Forest
# 2. Extra Trees
# 3. Gradient Boosting
# 4. AdaBoost
# 5. XGBoost


# ============================================================
# 1. IMPORT LIBRARY DASAR
# ============================================================

import os
import json
import time
import warnings
import platform
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import __version__ as sklearn_version
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)


# ============================================================
# 2. IMPORT XGBOOST
# ============================================================

try:

    import xgboost
    from xgboost import XGBClassifier

    XGBOOST_AVAILABLE = True

except ImportError as error:

    XGBOOST_AVAILABLE = False

    raise ImportError(
        "Library xgboost belum tersedia. "
        "Instal terlebih dahulu dengan perintah:\n"
        "pip install xgboost"
    ) from error


# ============================================================
# 3. WARNING CONFIGURATION
# ============================================================

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 4. RANDOM SEED
# ============================================================

RANDOM_STATE = 42

np.random.seed(
    RANDOM_STATE
)


# ============================================================
# 5. NAMA KOLOM UTAMA
# ============================================================

SUBJECT_COLUMN = "subject"

TARGET_COLUMN = "fall_risk"


# ============================================================
# 6. KOMPOSISI FITUR MODEL
# ============================================================

# Fitur input:
# - 64 skeleton embedding
# - 4 environmental context
# - 5 physiological
#
# Total:
# 64 + 4 + 5 = 73 fitur
#
# Duration tidak digunakan sebagai fitur karena total TUG time
# digunakan untuk membentuk label fall_risk.
#
# Height dan weight tidak digunakan karena sudah
# direpresentasikan melalui BMI.

EXPECTED_FEATURE_COUNT = 73


EXCLUDED_MODEL_FEATURES = [
    "duration_STS",
    "duration_WO",
    "duration_TR",
    "duration_WI",
    "duration_STTS",
    "total_tug_time",
    "height_cm",
    "weight_kg"
]


# ============================================================
# 7. DAFTAR MODEL ENSEMBLE
# ============================================================

ENSEMBLE_MODEL_NAMES = [
    "Random Forest",
    "Extra Trees",
    "Gradient Boosting",
    "AdaBoost",
    "XGBoost"
]


MODEL_ABBREVIATIONS = {
    "Random Forest": "RF",
    "Extra Trees": "ET",
    "Gradient Boosting": "GB",
    "AdaBoost": "ADA",
    "XGBoost": "XGB"
}


# ============================================================
# 8. KONFIGURASI NESTED CROSS-VALIDATION
# ============================================================

# Distribusi dataset:
# LOW/NO = 10 subject
# HIGH   = 20 subject
#
# Outer 5-fold menghasilkan kira-kira:
# - 24 training subject
# - 6 testing subject
#
# Setiap outer test fold akan memuat kira-kira:
# - 2 LOW/NO
# - 4 HIGH

OUTER_N_SPLITS = 5

outer_cv = StratifiedKFold(
    n_splits=OUTER_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


# Inner CV digunakan GridSearchCV untuk memilih
# hyperparameter hanya dari outer training fold.

INNER_N_SPLITS = 3

inner_cv = StratifiedKFold(
    n_splits=INNER_N_SPLITS,
    shuffle=True,
    random_state=(
        RANDOM_STATE + 1
    )
)


# ============================================================
# 9. GRID SEARCH CONFIGURATION
# ============================================================

GRID_SCORING = "roc_auc"

GRID_N_JOBS = -1

GRID_VERBOSE = 0

GRID_REFIT = True

GRID_ERROR_SCORE = "raise"


# ============================================================
# 10. KELAS TARGET
# ============================================================

CLASS_LABEL_MAPPING = {
    0: "LOW/NO",
    1: "HIGH"
}

NEGATIVE_CLASS_ID = 0
POSITIVE_CLASS_ID = 1

NEGATIVE_CLASS_NAME = "LOW/NO"
POSITIVE_CLASS_NAME = "HIGH"


# ============================================================
# 11. REFERENSI CLASS IMBALANCE
# ============================================================

# Untuk XGBoost:
#
# scale_pos_weight =
# jumlah kelas negatif / jumlah kelas positif
#
# LOW/NO = kelas 0 = 10
# HIGH   = kelas 1 = 20
#
# Nilai referensi = 10 / 20 = 0.5
#
# Nilai aktual akan tetap dipilih melalui GridSearchCV.

REFERENCE_SCALE_POS_WEIGHT = 0.5


# ============================================================
# 12. OUTPUT DIRECTORIES
# ============================================================

# Semua hasil eksperimen baru disimpan pada direktori
# yang memiliki tambahan nama "ensemble".

if "RISK_TABLE_DIR" not in globals():

    RISK_TABLE_DIR = os.path.join(
        os.getcwd(),
        "fall_risk_results"
    )


ENSEMBLE_OUTPUT_DIR = os.path.join(
    RISK_TABLE_DIR,
    "ensemble_machine_learning"
)

ENSEMBLE_TABLE_DIR = os.path.join(
    ENSEMBLE_OUTPUT_DIR,
    "tables"
)

ENSEMBLE_FIGURE_DIR = os.path.join(
    ENSEMBLE_OUTPUT_DIR,
    "figures"
)

ENSEMBLE_MODEL_DIR = os.path.join(
    ENSEMBLE_OUTPUT_DIR,
    "models"
)

ENSEMBLE_LOG_DIR = os.path.join(
    ENSEMBLE_OUTPUT_DIR,
    "logs"
)


for output_directory in [
    ENSEMBLE_OUTPUT_DIR,
    ENSEMBLE_TABLE_DIR,
    ENSEMBLE_FIGURE_DIR,
    ENSEMBLE_MODEL_DIR,
    ENSEMBLE_LOG_DIR
]:

    os.makedirs(
        output_directory,
        exist_ok=True
    )


# ============================================================
# 13. VALIDASI OBJEK DATASET
# ============================================================

required_dataset_objects = [
    "risk_dataset",
    "feature_cols"
]


missing_dataset_objects = [
    object_name
    for object_name in required_dataset_objects
    if object_name not in globals()
]


if missing_dataset_objects:

    raise NameError(
        "Objek dataset belum tersedia: "
        f"{missing_dataset_objects}. "
        "Jalankan tahap multimodal feature fusion terlebih dahulu."
    )


# ============================================================
# 14. MEMBUAT SALINAN DATASET
# ============================================================

ensemble_dataset = (
    risk_dataset
    .copy()
    .reset_index(drop=True)
)


ensemble_feature_columns = list(
    dict.fromkeys(
        feature_cols
    )
)


# ============================================================
# 15. VALIDASI KOLOM UTAMA
# ============================================================

required_main_columns = [
    SUBJECT_COLUMN,
    TARGET_COLUMN
]


missing_main_columns = [
    column_name
    for column_name in required_main_columns
    if column_name not in ensemble_dataset.columns
]


if missing_main_columns:

    raise ValueError(
        "Kolom utama tidak ditemukan dalam risk_dataset: "
        f"{missing_main_columns}"
    )


# ============================================================
# 16. VALIDASI FEATURE COLUMNS
# ============================================================

missing_model_features = [
    feature_name
    for feature_name in ensemble_feature_columns
    if feature_name not in ensemble_dataset.columns
]


if missing_model_features:

    raise ValueError(
        "Fitur berikut tidak ditemukan dalam risk_dataset: "
        f"{missing_model_features}"
    )


# ============================================================
# 17. MEMASTIKAN FITUR YANG DIKECUALIKAN TIDAK MASUK
# ============================================================

incorrectly_included_features = [
    feature_name
    for feature_name in EXCLUDED_MODEL_FEATURES
    if feature_name in ensemble_feature_columns
]


if incorrectly_included_features:

    raise ValueError(
        "Fitur yang seharusnya dikeluarkan masih terdapat "
        "dalam ensemble_feature_columns: "
        f"{incorrectly_included_features}"
    )


# ============================================================
# 18. VALIDASI JUMLAH FITUR
# ============================================================

if len(
    ensemble_feature_columns
) != EXPECTED_FEATURE_COUNT:

    raise ValueError(
        f"Jumlah fitur ensemble seharusnya "
        f"{EXPECTED_FEATURE_COUNT}, tetapi diperoleh "
        f"{len(ensemble_feature_columns)}."
    )


# ============================================================
# 19. PEMBERSIHAN SUBJECT DAN TARGET
# ============================================================

ensemble_dataset[
    SUBJECT_COLUMN
] = (
    ensemble_dataset[
        SUBJECT_COLUMN
    ]
    .astype(str)
    .str.strip()
)


ensemble_dataset[
    TARGET_COLUMN
] = (
    ensemble_dataset[
        TARGET_COLUMN
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ============================================================
# 20. VALIDASI SUBJECT DUPLIKAT
# ============================================================

duplicated_subjects = (
    ensemble_dataset.loc[
        ensemble_dataset[
            SUBJECT_COLUMN
        ].duplicated(
            keep=False
        ),
        SUBJECT_COLUMN
    ]
    .unique()
    .tolist()
)


if duplicated_subjects:

    raise ValueError(
        "Setiap subject harus memiliki tepat satu baris. "
        "Ditemukan subject duplikat: "
        f"{duplicated_subjects}"
    )


# ============================================================
# 21. VALIDASI LABEL TARGET
# ============================================================

valid_text_labels = {
    "LOW/NO",
    "HIGH"
}


available_text_labels = set(
    ensemble_dataset[
        TARGET_COLUMN
    ]
    .dropna()
    .unique()
)


invalid_text_labels = (
    available_text_labels
    - valid_text_labels
)


if invalid_text_labels:

    raise ValueError(
        "Dataset mengandung label tidak dikenali: "
        f"{invalid_text_labels}"
    )


# ============================================================
# 22. KONVERSI FITUR MENJADI NUMERIK
# ============================================================

ensemble_dataset[
    ensemble_feature_columns
] = (
    ensemble_dataset[
        ensemble_feature_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )
)


# ============================================================
# 23. MISSING VALUE SUMMARY
# ============================================================

ensemble_missing_value_summary = (
    ensemble_dataset[
        ensemble_feature_columns
    ]
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)


ensemble_missing_value_summary = (
    ensemble_missing_value_summary[
        ensemble_missing_value_summary > 0
    ]
)


ensemble_total_missing_values = int(
    ensemble_dataset[
        ensemble_feature_columns
    ]
    .isna()
    .sum()
    .sum()
)


# Missing value belum diimputasi pada STEP 1.
# Median imputation dilakukan di dalam Pipeline supaya hanya
# belajar dari training fold.


# ============================================================
# 24. CONSTANT FEATURE SUMMARY
# ============================================================

ensemble_constant_features = [
    feature_name
    for feature_name in ensemble_feature_columns
    if ensemble_dataset[
        feature_name
    ].nunique(
        dropna=True
    ) <= 1
]


# ============================================================
# 25. DISTRIBUSI KELAS
# ============================================================

ensemble_class_distribution_df = (
    ensemble_dataset[
        TARGET_COLUMN
    ]
    .value_counts()
    .rename_axis(
        "fall_risk"
    )
    .reset_index(
        name="number_of_subjects"
    )
)


ensemble_class_distribution_df[
    "percentage"
] = (
    ensemble_class_distribution_df[
        "number_of_subjects"
    ]
    / len(
        ensemble_dataset
    )
    * 100
)


minimum_class_count = int(
    ensemble_class_distribution_df[
        "number_of_subjects"
    ].min()
)


if minimum_class_count < OUTER_N_SPLITS:

    raise ValueError(
        "Jumlah anggota kelas terkecil tidak mencukupi "
        f"untuk outer {OUTER_N_SPLITS}-fold CV. "
        f"Jumlah kelas terkecil: {minimum_class_count}."
    )


# ============================================================
# 26. VALIDASI INNER CROSS-VALIDATION
# ============================================================

estimated_outer_training_minority_count = int(
    np.floor(
        minimum_class_count
        * (
            OUTER_N_SPLITS - 1
        )
        / OUTER_N_SPLITS
    )
)


if (
    estimated_outer_training_minority_count
    < INNER_N_SPLITS
):

    raise ValueError(
        "Jumlah kelas minoritas pada outer training fold "
        "diperkirakan tidak mencukupi untuk inner CV. "
        f"Perkiraan minority training count: "
        f"{estimated_outer_training_minority_count}; "
        f"inner folds: {INNER_N_SPLITS}."
    )


# ============================================================
# 27. EXPERIMENT CONFIGURATION
# ============================================================

ensemble_experiment_configuration = {
    "experiment_name": (
        "Pure Ensemble Fall-Risk Classification"
    ),

    "preprocessing": [
        "Median imputation"
    ],

    "standard_scaler_used": False,

    "pca_used": False,

    "models": (
        ENSEMBLE_MODEL_NAMES
    ),

    "random_state": int(
        RANDOM_STATE
    ),

    "number_of_subjects": int(
        ensemble_dataset[
            SUBJECT_COLUMN
        ].nunique()
    ),

    "number_of_input_features": int(
        len(
            ensemble_feature_columns
        )
    ),

    "expected_feature_count": int(
        EXPECTED_FEATURE_COUNT
    ),

    "outer_cross_validation": {
        "method": "StratifiedKFold",
        "n_splits": int(
            OUTER_N_SPLITS
        ),
        "shuffle": True
    },

    "inner_cross_validation": {
        "method": "StratifiedKFold",
        "n_splits": int(
            INNER_N_SPLITS
        ),
        "shuffle": True
    },

    "grid_search_scoring": (
        GRID_SCORING
    ),

    "class_mapping": {
        "0": "LOW/NO",
        "1": "HIGH"
    },

    "reference_scale_pos_weight": float(
        REFERENCE_SCALE_POS_WEIGHT
    ),

    "excluded_features": (
        EXCLUDED_MODEL_FEATURES
    ),

    "output_directory": (
        ENSEMBLE_OUTPUT_DIR
    ),

    "software_versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn_version,
        "xgboost": xgboost.__version__,
        "joblib": joblib.__version__
    }
}


# ============================================================
# 28. MENAMPILKAN RINGKASAN KONFIGURASI
# ============================================================

print("=" * 76)
print("PURE ENSEMBLE MACHINE-LEARNING EXPERIMENT")
print("=" * 76)

print(
    "Dataset shape                 :",
    ensemble_dataset.shape
)

print(
    "Number of subjects            :",
    ensemble_dataset[
        SUBJECT_COLUMN
    ].nunique()
)

print(
    "Number of model features      :",
    len(
        ensemble_feature_columns
    )
)

print(
    "Expected feature count        :",
    EXPECTED_FEATURE_COUNT
)

print(
    "PCA used                      :",
    False
)

print(
    "StandardScaler used           :",
    False
)

print(
    "Preprocessing                 :",
    "Median imputation only"
)

print(
    "Ensemble classifiers          :",
    len(
        ENSEMBLE_MODEL_NAMES
    )
)

print(
    "Models                        :",
    ENSEMBLE_MODEL_NAMES
)

print(
    "Outer cross-validation        :",
    f"Stratified {OUTER_N_SPLITS}-Fold"
)

print(
    "Inner GridSearchCV            :",
    f"Stratified {INNER_N_SPLITS}-Fold"
)

print(
    "Primary metric                :",
    GRID_SCORING
)

print(
    "Missing values                :",
    ensemble_total_missing_values
)

print(
    "Constant features             :",
    len(
        ensemble_constant_features
    )
)

print(
    "Output directory              :",
    ENSEMBLE_OUTPUT_DIR
)


# ============================================================
# 29. MENAMPILKAN DISTRIBUSI KELAS
# ============================================================

print("\nClass distribution:")

display(
    ensemble_class_distribution_df
)


# ============================================================
# 30. MENAMPILKAN MISSING VALUES
# ============================================================

if ensemble_missing_value_summary.empty:

    print(
        "\nTidak terdapat missing value pada fitur ensemble."
    )

else:

    print(
        "\nMissing values akan ditangani dengan median "
        "imputation di dalam setiap training fold:"
    )

    display(
        ensemble_missing_value_summary.to_frame(
            "missing_count"
        )
    )


# ============================================================
# 31. MENAMPILKAN CONSTANT FEATURES
# ============================================================

if ensemble_constant_features:

    print(
        "\nPERINGATAN: Fitur berikut memiliki nilai konstan:"
    )

    print(
        ensemble_constant_features
    )

else:

    print(
        "\nTidak terdapat fitur konstan."
    )


# ============================================================
# 32. MENYIMPAN KONFIGURASI AWAL
# ============================================================

ensemble_configuration_path = os.path.join(
    ENSEMBLE_OUTPUT_DIR,
    "ensemble_experiment_configuration.json"
)


with open(
    ensemble_configuration_path,
    "w",
    encoding="utf-8"
) as configuration_file:

    json.dump(
        ensemble_experiment_configuration,
        configuration_file,
        indent=2,
        ensure_ascii=False
    )


ensemble_feature_list_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_feature_columns.csv"
)


pd.DataFrame({
    "feature_order": np.arange(
        1,
        len(
            ensemble_feature_columns
        ) + 1
    ),

    "feature": (
        ensemble_feature_columns
    )
}).to_csv(
    ensemble_feature_list_path,
    index=False
)


ensemble_class_distribution_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_class_distribution.csv"
)


ensemble_class_distribution_df.to_csv(
    ensemble_class_distribution_path,
    index=False
)


ensemble_missing_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_missing_value_summary.csv"
)


ensemble_missing_value_summary.to_frame(
    "missing_count"
).to_csv(
    ensemble_missing_summary_path
)


# ============================================================
# 33. KONFIRMASI STEP 1
# ============================================================

print("\n" + "=" * 76)
print("STEP 1 COMPLETED")
print("=" * 76)

print(
    "Eksperimen ensemble tanpa PCA berhasil diinisialisasi."
)

print(
    "Seluruh hasil akan disimpan dalam direktori:"
)

print(
    ENSEMBLE_OUTPUT_DIR
)

print(
    "\nConfiguration saved to:"
)

print(
    ensemble_configuration_path
)

print(
    "\nSTEP berikutnya:"
)

print(
    "STEP 2 — Prepare ensemble feature matrix, target vector, "
    "and subject identifiers."
)

In [ ]:
# ============================================================
# STEP 2. PREPARE ENSEMBLE MODEL INPUTS
# Feature Matrix X, Target Vector y, and Subject Identifiers
# ============================================================

# Output utama STEP 2:
#
# ensemble_X
# ensemble_y
# ensemble_subjects
# ensemble_feature_columns
#
# Target encoding:
# 0 = LOW/NO
# 1 = HIGH
#
# Tidak dilakukan:
# - imputasi global;
# - standardisasi;
# - PCA;
# - oversampling.
#
# Median imputation akan dilakukan di dalam Pipeline pada
# setiap training fold agar tidak terjadi data leakage.


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP 1
# ============================================================

required_step_2_objects = [
    "ensemble_dataset",
    "ensemble_feature_columns",
    "SUBJECT_COLUMN",
    "TARGET_COLUMN",
    "EXPECTED_FEATURE_COUNT",
    "CLASS_LABEL_MAPPING",
    "OUTER_N_SPLITS",
    "INNER_N_SPLITS",
    "ENSEMBLE_TABLE_DIR"
]

missing_step_2_objects = [
    object_name
    for object_name in required_step_2_objects
    if object_name not in globals()
]

if missing_step_2_objects:
    raise NameError(
        "Objek untuk STEP 2 belum lengkap: "
        f"{missing_step_2_objects}. "
        "Jalankan STEP 1 terlebih dahulu."
    )


# ============================================================
# 2. MEMBUAT DATASET KERJA
# ============================================================

ensemble_model_dataset = (
    ensemble_dataset
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 3. MEMBENTUK MATRIKS FITUR
# ============================================================

ensemble_X = (
    ensemble_model_dataset[
        ensemble_feature_columns
    ]
    .copy()
)

ensemble_X = (
    ensemble_X
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )
)


# ============================================================
# 4. MENYIAPKAN IDENTITAS SUBJECT
# ============================================================

ensemble_subjects = (
    ensemble_model_dataset[
        SUBJECT_COLUMN
    ]
    .astype(str)
    .str.strip()
    .to_numpy()
)


# ============================================================
# 5. MEMBERSIHKAN LABEL FALL RISK
# ============================================================

ensemble_raw_labels = (
    ensemble_model_dataset[
        TARGET_COLUMN
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ============================================================
# 6. PEMETAAN LABEL
# ============================================================

ensemble_label_mapping = {
    "LOW/NO": 0,
    "LOW / NO": 0,
    "LOW-NO": 0,
    "LOW": 0,
    "NO": 0,
    "HIGH": 1
}

ensemble_y_series = (
    ensemble_raw_labels
    .map(
        ensemble_label_mapping
    )
)


# ============================================================
# 7. MEMERIKSA LABEL YANG TIDAK DIKENALI
# ============================================================

unknown_ensemble_labels = (
    ensemble_raw_labels[
        ensemble_y_series.isna()
    ]
    .unique()
    .tolist()
)

if unknown_ensemble_labels:
    raise ValueError(
        "Terdapat label fall risk yang tidak dikenali: "
        f"{unknown_ensemble_labels}."
    )


# ============================================================
# 8. MEMBENTUK TARGET VECTOR
# ============================================================

ensemble_y = (
    ensemble_y_series
    .astype(int)
    .to_numpy()
)


# ============================================================
# 9. VALIDASI DIMENSI
# ============================================================

if ensemble_X.shape[0] != len(
    ensemble_y
):
    raise ValueError(
        "Jumlah baris ensemble_X tidak sama dengan "
        "jumlah label ensemble_y."
    )

if ensemble_X.shape[0] != len(
    ensemble_subjects
):
    raise ValueError(
        "Jumlah baris ensemble_X tidak sama dengan "
        "jumlah ensemble_subjects."
    )

if ensemble_X.shape[1] != EXPECTED_FEATURE_COUNT:
    raise ValueError(
        f"Jumlah fitur seharusnya "
        f"{EXPECTED_FEATURE_COUNT}, tetapi diperoleh "
        f"{ensemble_X.shape[1]}."
    )

if ensemble_X.shape[1] != len(
    ensemble_feature_columns
):
    raise ValueError(
        "Jumlah kolom ensemble_X tidak sama dengan "
        "ensemble_feature_columns."
    )


# ============================================================
# 10. VALIDASI SUBJECT UNIK
# ============================================================

ensemble_duplicated_subject_count = int(
    pd.Series(
        ensemble_subjects
    )
    .duplicated()
    .sum()
)

if ensemble_duplicated_subject_count > 0:
    raise ValueError(
        f"Ditemukan {ensemble_duplicated_subject_count} "
        "subject duplikat."
    )


# ============================================================
# 11. VALIDASI KELAS TARGET
# ============================================================

ensemble_unique_target_values = np.unique(
    ensemble_y
)

if not np.array_equal(
    ensemble_unique_target_values,
    np.array([
        0,
        1
    ])
):
    raise ValueError(
        "Target ensemble harus memiliki tepat dua kelas "
        "[0, 1]. "
        f"Diperoleh: {ensemble_unique_target_values}."
    )


# ============================================================
# 12. MEMERIKSA KESELARASAN LABEL
# ============================================================

ensemble_label_alignment_df = pd.DataFrame({
    SUBJECT_COLUMN: ensemble_subjects,
    "fall_risk_text": (
        ensemble_raw_labels
        .to_numpy()
    ),
    "fall_risk_encoded": (
        ensemble_y
    )
})

ensemble_label_alignment_table = pd.crosstab(
    ensemble_label_alignment_df[
        "fall_risk_text"
    ],
    ensemble_label_alignment_df[
        "fall_risk_encoded"
    ],
    margins=True
)


expected_low_count = int(
    (
        ensemble_raw_labels
        == "LOW/NO"
    ).sum()
)

expected_high_count = int(
    (
        ensemble_raw_labels
        == "HIGH"
    ).sum()
)

actual_low_encoded_count = int(
    (
        (
            ensemble_raw_labels
            == "LOW/NO"
        )
        & (
            ensemble_y_series == 0
        )
    ).sum()
)

actual_high_encoded_count = int(
    (
        (
            ensemble_raw_labels
            == "HIGH"
        )
        & (
            ensemble_y_series == 1
        )
    ).sum()
)

if (
    expected_low_count
    != actual_low_encoded_count
):
    raise ValueError(
        "Mapping LOW/NO → 0 tidak konsisten."
    )

if (
    expected_high_count
    != actual_high_encoded_count
):
    raise ValueError(
        "Mapping HIGH → 1 tidak konsisten."
    )


# ============================================================
# 13. MISSING VALUE SUMMARY
# ============================================================

ensemble_input_missing_summary = (
    ensemble_X
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

ensemble_input_missing_summary = (
    ensemble_input_missing_summary[
        ensemble_input_missing_summary > 0
    ]
)

ensemble_input_total_missing = int(
    ensemble_X
    .isna()
    .sum()
    .sum()
)


# ============================================================
# 14. CONSTANT FEATURES
# ============================================================

ensemble_input_constant_features = [
    feature_name
    for feature_name in ensemble_feature_columns
    if ensemble_X[
        feature_name
    ].nunique(
        dropna=True
    ) <= 1
]


# ============================================================
# 15. NEAR-CONSTANT FEATURES
# ============================================================

ensemble_feature_variance = (
    ensemble_X
    .var(
        axis=0,
        skipna=True
    )
)

ENSEMBLE_LOW_VARIANCE_THRESHOLD = 1e-12

ensemble_near_constant_features = (
    ensemble_feature_variance[
        ensemble_feature_variance
        <= ENSEMBLE_LOW_VARIANCE_THRESHOLD
    ]
    .index
    .tolist()
)


# ============================================================
# 16. DISTRIBUSI KELAS
# ============================================================

ensemble_encoded_class_distribution_df = pd.DataFrame({
    "class_id": [
        0,
        1
    ],
    "class_name": [
        CLASS_LABEL_MAPPING[0],
        CLASS_LABEL_MAPPING[1]
    ],
    "number_of_subjects": [
        int(
            (
                ensemble_y == 0
            ).sum()
        ),
        int(
            (
                ensemble_y == 1
            ).sum()
        )
    ]
})

ensemble_encoded_class_distribution_df[
    "percentage"
] = (
    ensemble_encoded_class_distribution_df[
        "number_of_subjects"
    ]
    / len(
        ensemble_y
    )
    * 100
)


# ============================================================
# 17. MEMERIKSA RASIO KELAS
# ============================================================

ensemble_negative_count = int(
    (
        ensemble_y == 0
    ).sum()
)

ensemble_positive_count = int(
    (
        ensemble_y == 1
    ).sum()
)

ensemble_class_ratio = float(
    ensemble_positive_count
    / ensemble_negative_count
)

ensemble_scale_pos_weight_reference = float(
    ensemble_negative_count
    / ensemble_positive_count
)


# ============================================================
# 18. VALIDASI STRATIFIED CROSS-VALIDATION
# ============================================================

ensemble_minimum_class_count = int(
    ensemble_encoded_class_distribution_df[
        "number_of_subjects"
    ].min()
)

if (
    ensemble_minimum_class_count
    < OUTER_N_SPLITS
):
    raise ValueError(
        "Jumlah anggota kelas terkecil tidak mencukupi "
        f"untuk outer {OUTER_N_SPLITS}-fold CV."
    )


estimated_outer_training_minority_count = int(
    np.floor(
        ensemble_minimum_class_count
        * (
            OUTER_N_SPLITS - 1
        )
        / OUTER_N_SPLITS
    )
)

if (
    estimated_outer_training_minority_count
    < INNER_N_SPLITS
):
    raise ValueError(
        "Jumlah anggota kelas minoritas pada outer training "
        "fold tidak mencukupi untuk inner CV."
    )


# ============================================================
# 19. MEMBENTUK ARRAY UNTUK SCIKIT-LEARN
# ============================================================

ensemble_X_array = (
    ensemble_X
    .to_numpy(
        dtype=float
    )
)

ensemble_y_array = np.asarray(
    ensemble_y,
    dtype=int
)

ensemble_subject_array = np.asarray(
    ensemble_subjects
)


# ============================================================
# 20. MENYIAPKAN PREVIEW DATASET
# ============================================================

ensemble_preview_feature_count = min(
    10,
    len(
        ensemble_feature_columns
    )
)

ensemble_input_preview_df = pd.concat(
    [
        ensemble_label_alignment_df.reset_index(
            drop=True
        ),
        ensemble_X[
            ensemble_feature_columns[
                :ensemble_preview_feature_count
            ]
        ].reset_index(
            drop=True
        )
    ],
    axis=1
)


# ============================================================
# 21. MENYIAPKAN DATASET INPUT LENGKAP
# ============================================================

ensemble_model_input_dataset = pd.concat(
    [
        ensemble_label_alignment_df.reset_index(
            drop=True
        ),
        ensemble_X.reset_index(
            drop=True
        )
    ],
    axis=1
)


# ============================================================
# 22. MENAMPILKAN RINGKASAN INPUT
# ============================================================

print("=" * 76)
print("PURE ENSEMBLE MODEL INPUTS")
print("=" * 76)

print(
    "Feature matrix shape         :",
    ensemble_X.shape
)

print(
    "Target vector shape          :",
    ensemble_y.shape
)

print(
    "Subject vector shape         :",
    ensemble_subjects.shape
)

print(
    "Number of subjects           :",
    len(
        ensemble_subjects
    )
)

print(
    "Number of features           :",
    len(
        ensemble_feature_columns
    )
)

print(
    "Number of classes            :",
    len(
        ensemble_unique_target_values
    )
)

print(
    "LOW/NO subjects              :",
    ensemble_negative_count
)

print(
    "HIGH subjects                :",
    ensemble_positive_count
)

print(
    "HIGH-to-LOW/NO ratio         :",
    f"{ensemble_class_ratio:.2f}:1"
)

print(
    "Reference scale_pos_weight   :",
    f"{ensemble_scale_pos_weight_reference:.4f}"
)

print(
    "Missing values               :",
    ensemble_input_total_missing
)

print(
    "Constant features            :",
    len(
        ensemble_input_constant_features
    )
)

print(
    "Near-constant features       :",
    len(
        ensemble_near_constant_features
    )
)

print(
    "Duplicated subjects          :",
    ensemble_duplicated_subject_count
)


# ============================================================
# 23. MENAMPILKAN DISTRIBUSI KELAS
# ============================================================

print("\nClass distribution:")

display(
    ensemble_encoded_class_distribution_df
)


# ============================================================
# 24. MENAMPILKAN KESELARASAN LABEL
# ============================================================

print("\nText and numeric label alignment:")

display(
    ensemble_label_alignment_table
)


# ============================================================
# 25. MENAMPILKAN MISSING VALUES
# ============================================================

if ensemble_input_missing_summary.empty:

    print(
        "\nTidak terdapat missing value pada fitur ensemble."
    )

else:

    print(
        "\nMissing values akan diimputasi menggunakan median "
        "di dalam setiap training fold:"
    )

    display(
        ensemble_input_missing_summary.to_frame(
            "missing_count"
        )
    )


# ============================================================
# 26. MENAMPILKAN FITUR KONSTAN
# ============================================================

if ensemble_input_constant_features:

    print(
        "\nPERINGATAN: Fitur konstan:"
    )

    print(
        ensemble_input_constant_features
    )

else:

    print(
        "\nTidak terdapat fitur konstan."
    )


if ensemble_near_constant_features:

    print(
        "\nPERINGATAN: Fitur dengan variansi sangat rendah:"
    )

    print(
        ensemble_near_constant_features
    )


# ============================================================
# 27. MENAMPILKAN PREVIEW
# ============================================================

print(
    f"\nPreview input model dengan "
    f"{ensemble_preview_feature_count} fitur pertama:"
)

display(
    ensemble_input_preview_df.head()
)


# ============================================================
# 28. MENYIMPAN INPUT MODEL
# ============================================================

ensemble_model_input_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_model_input_dataset.csv"
)

ensemble_model_input_dataset.to_csv(
    ensemble_model_input_path,
    index=False
)


ensemble_label_alignment_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_label_alignment.csv"
)

ensemble_label_alignment_df.to_csv(
    ensemble_label_alignment_path,
    index=False
)


ensemble_encoded_distribution_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_encoded_class_distribution.csv"
)

ensemble_encoded_class_distribution_df.to_csv(
    ensemble_encoded_distribution_path,
    index=False
)


ensemble_input_missing_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_input_missing_value_summary.csv"
)

ensemble_input_missing_summary.to_frame(
    "missing_count"
).to_csv(
    ensemble_input_missing_path
)


ensemble_input_validation_summary_df = pd.DataFrame([
    {
        "number_of_subjects": int(
            len(
                ensemble_subjects
            )
        ),
        "number_of_features": int(
            len(
                ensemble_feature_columns
            )
        ),
        "low_no_subjects": (
            ensemble_negative_count
        ),
        "high_subjects": (
            ensemble_positive_count
        ),
        "high_to_low_ratio": (
            ensemble_class_ratio
        ),
        "reference_scale_pos_weight": (
            ensemble_scale_pos_weight_reference
        ),
        "missing_values": (
            ensemble_input_total_missing
        ),
        "constant_features": int(
            len(
                ensemble_input_constant_features
            )
        ),
        "near_constant_features": int(
            len(
                ensemble_near_constant_features
            )
        ),
        "duplicated_subjects": (
            ensemble_duplicated_subject_count
        )
    }
])

ensemble_input_validation_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_input_validation_summary.csv"
)

ensemble_input_validation_summary_df.to_csv(
    ensemble_input_validation_path,
    index=False
)


# ============================================================
# 29. KONFIRMASI STEP 2
# ============================================================

print("\n" + "=" * 76)
print("STEP 2 COMPLETED")
print("=" * 76)

print(
    "Input model ensemble tanpa PCA berhasil disiapkan."
)

print(
    "\nTarget encoding:"
)

print(
    "0 = LOW/NO"
)

print(
    "1 = HIGH"
)

print(
    "\nObjek yang siap digunakan:"
)

print(
    "- ensemble_X"
)

print(
    "- ensemble_y"
)

print(
    "- ensemble_subjects"
)

print(
    "- ensemble_feature_columns"
)

print(
    "- ensemble_X_array"
)

print(
    "- ensemble_y_array"
)

print(
    "\nModel input saved to:"
)

print(
    ensemble_model_input_path
)

In [ ]:
# ============================================================
# STEP 3. DEFINE PURE ENSEMBLE MACHINE-LEARNING PIPELINES
# Random Forest, Extra Trees, Gradient Boosting,
# AdaBoost, and XGBoost
# ============================================================

# Struktur pipeline:
#
# Median Imputation
#       ↓
# Ensemble Classifier
#
# Tidak digunakan:
# - StandardScaler
# - PCA
#
# Seluruh hyperparameter, termasuk class imbalance handling,
# akan dioptimasi menggunakan GridSearchCV pada STEP 4.


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP 1 DAN STEP 2
# ============================================================

required_step_3_objects = [
    "ensemble_X",
    "ensemble_y",
    "ensemble_subjects",
    "ensemble_feature_columns",
    "RANDOM_STATE",
    "ENSEMBLE_MODEL_NAMES",
    "MODEL_ABBREVIATIONS",
    "ENSEMBLE_TABLE_DIR"
]

missing_step_3_objects = [
    object_name
    for object_name in required_step_3_objects
    if object_name not in globals()
]

if missing_step_3_objects:
    raise NameError(
        "Objek untuk STEP 3 belum lengkap: "
        f"{missing_step_3_objects}. "
        "Jalankan STEP 1 dan STEP 2 terlebih dahulu."
    )


# ============================================================
# 2. VALIDASI IMPORT CLASSIFIER
# ============================================================

required_classifier_objects = [
    "RandomForestClassifier",
    "ExtraTreesClassifier",
    "GradientBoostingClassifier",
    "AdaBoostClassifier",
    "XGBClassifier"
]

missing_classifier_objects = [
    object_name
    for object_name in required_classifier_objects
    if object_name not in globals()
]

if missing_classifier_objects:
    raise NameError(
        "Classifier berikut belum diimpor: "
        f"{missing_classifier_objects}. "
        "Periksa bagian import pada STEP 1."
    )


# ============================================================
# 3. FUNGSI PEMBENTUKAN PIPELINE
# ============================================================

def build_ensemble_pipeline(
    classifier
):
    """
    Membentuk pipeline ensemble tanpa StandardScaler dan PCA.

    Parameters
    ----------
    classifier : sklearn-compatible estimator
        Classifier ensemble yang akan digunakan.

    Returns
    -------
    sklearn.pipeline.Pipeline
        Pipeline berisi median imputation dan classifier.
    """

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "classifier",
            classifier
        )
    ])


# ============================================================
# 4. RANDOM FOREST CLASSIFIER
# ============================================================

# class_weight tidak ditetapkan permanen.
# GridSearchCV pada STEP 4 akan memilih:
# - None
# - balanced

random_forest_classifier = (
    RandomForestClassifier(
        n_estimators=300,
        criterion="gini",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=True,
        class_weight=None,
        random_state=RANDOM_STATE,
        n_jobs=1
    )
)

random_forest_pipeline = (
    build_ensemble_pipeline(
        random_forest_classifier
    )
)


# ============================================================
# 5. EXTRA TREES CLASSIFIER
# ============================================================

# Extra Trees menggunakan split yang lebih acak dibanding
# Random Forest.
#
# class_weight juga akan dipilih oleh GridSearchCV.

extra_trees_classifier = (
    ExtraTreesClassifier(
        n_estimators=300,
        criterion="gini",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=False,
        class_weight=None,
        random_state=RANDOM_STATE,
        n_jobs=1
    )
)

extra_trees_pipeline = (
    build_ensemble_pipeline(
        extra_trees_classifier
    )
)


# ============================================================
# 6. GRADIENT BOOSTING CLASSIFIER
# ============================================================

# GradientBoostingClassifier tidak mendukung class_weight
# secara langsung.
#
# Karena dataset relatif kecil, base tree dibuat dangkal
# sebagai konfigurasi awal untuk mengurangi overfitting.

gradient_boosting_classifier = (
    GradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.05,
        n_estimators=100,
        subsample=1.0,
        criterion="friedman_mse",
        min_samples_split=2,
        min_samples_leaf=1,
        max_depth=2,
        max_features=None,
        random_state=RANDOM_STATE
    )
)

gradient_boosting_pipeline = (
    build_ensemble_pipeline(
        gradient_boosting_classifier
    )
)


# ============================================================
# 7. ADABOOST CLASSIFIER
# ============================================================

# AdaBoost tidak mendukung class_weight secara langsung.
#
# estimator=None menggunakan decision stump bawaan.
# Hyperparameter akan dioptimasi pada STEP 4.

adaboost_classifier = (
    AdaBoostClassifier(
        estimator=None,
        n_estimators=100,
        learning_rate=0.5,
        algorithm="SAMME",
        random_state=RANDOM_STATE
    )
)

adaboost_pipeline = (
    build_ensemble_pipeline(
        adaboost_classifier
    )
)


# ============================================================
# 8. XGBOOST CLASSIFIER
# ============================================================

# XGBoost menggunakan scale_pos_weight untuk menangani
# ketidakseimbangan kelas.
#
# Nilainya tidak dikunci di sini. GridSearchCV pada STEP 4
# akan menguji beberapa kandidat, termasuk nilai referensi 0.5.

xgboost_classifier = (
    XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        booster="gbtree",
        tree_method="hist",
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        min_child_weight=1,
        subsample=1.0,
        colsample_bytree=1.0,
        reg_alpha=0.0,
        reg_lambda=1.0,
        gamma=0.0,
        scale_pos_weight=1.0,
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbosity=0
    )
)

xgboost_pipeline = (
    build_ensemble_pipeline(
        xgboost_classifier
    )
)


# ============================================================
# 9. MENYIMPAN SELURUH PIPELINE DALAM DICTIONARY
# ============================================================

ensemble_pipelines = {
    "Random Forest": (
        random_forest_pipeline
    ),

    "Extra Trees": (
        extra_trees_pipeline
    ),

    "Gradient Boosting": (
        gradient_boosting_pipeline
    ),

    "AdaBoost": (
        adaboost_pipeline
    ),

    "XGBoost": (
        xgboost_pipeline
    )
}


# ============================================================
# 10. MENDEFINISIKAN SINGKATAN MODEL
# ============================================================

ensemble_model_abbreviations = {
    "Random Forest": "RF",
    "Extra Trees": "ET",
    "Gradient Boosting": "GB",
    "AdaBoost": "ADA",
    "XGBoost": "XGB"
}


# ============================================================
# 11. METODE SKOR UNTUK ROC-AUC
# ============================================================

# Seluruh ensemble classifier menggunakan predict_proba().
#
# Skor ROC-AUC akan diambil dari probabilitas kelas:
# HIGH = 1

ensemble_model_score_methods = {
    "Random Forest": "predict_proba",
    "Extra Trees": "predict_proba",
    "Gradient Boosting": "predict_proba",
    "AdaBoost": "predict_proba",
    "XGBoost": "predict_proba"
}


# ============================================================
# 12. VALIDASI KESESUAIAN NAMA MODEL
# ============================================================

expected_ensemble_models = set(
    ENSEMBLE_MODEL_NAMES
)

actual_ensemble_models = set(
    ensemble_pipelines.keys()
)

if actual_ensemble_models != expected_ensemble_models:

    missing_models = (
        expected_ensemble_models
        - actual_ensemble_models
    )

    unexpected_models = (
        actual_ensemble_models
        - expected_ensemble_models
    )

    raise ValueError(
        "Daftar model pada ensemble_pipelines tidak sesuai.\n"
        f"Model belum tersedia: {sorted(missing_models)}\n"
        f"Model tidak diharapkan: {sorted(unexpected_models)}"
    )


if set(
    ensemble_model_abbreviations.keys()
) != actual_ensemble_models:
    raise ValueError(
        "Nama model pada ensemble_model_abbreviations "
        "tidak sama dengan ensemble_pipelines."
    )


if set(
    ensemble_model_score_methods.keys()
) != actual_ensemble_models:
    raise ValueError(
        "Nama model pada ensemble_model_score_methods "
        "tidak sama dengan ensemble_pipelines."
    )


# ============================================================
# 13. VALIDASI STRUKTUR PIPELINE
# ============================================================

required_pipeline_steps = [
    "imputer",
    "classifier"
]

pipeline_validation_records = []

for model_name, model_pipeline in (
    ensemble_pipelines.items()
):

    available_steps = list(
        model_pipeline
        .named_steps
        .keys()
    )

    missing_steps = [
        step_name
        for step_name in required_pipeline_steps
        if step_name not in available_steps
    ]

    if missing_steps:
        raise ValueError(
            f"Pipeline {model_name} belum memiliki step: "
            f"{missing_steps}"
        )


    if "scaler" in available_steps:
        raise ValueError(
            f"Pipeline {model_name} masih memiliki scaler."
        )


    if "pca" in available_steps:
        raise ValueError(
            f"Pipeline {model_name} masih memiliki PCA."
        )


    classifier_object = (
        model_pipeline
        .named_steps["classifier"]
    )

    if not hasattr(
        classifier_object,
        "predict_proba"
    ):
        raise AttributeError(
            f"{model_name} tidak menyediakan predict_proba()."
        )


    classifier_class_weight = getattr(
        classifier_object,
        "class_weight",
        "not supported"
    )

    classifier_scale_pos_weight = getattr(
        classifier_object,
        "scale_pos_weight",
        "not supported"
    )


    pipeline_validation_records.append({
        "model": model_name,
        "abbreviation": (
            ensemble_model_abbreviations[
                model_name
            ]
        ),
        "classifier_class": (
            classifier_object
            .__class__
            .__name__
        ),
        "pipeline_steps": (
            " → ".join(
                available_steps
            )
        ),
        "standard_scaler_used": False,
        "pca_used": False,
        "score_method": (
            ensemble_model_score_methods[
                model_name
            ]
        ),
        "base_class_weight": str(
            classifier_class_weight
        ),
        "base_scale_pos_weight": str(
            classifier_scale_pos_weight
        )
    })


ensemble_pipeline_summary_df = pd.DataFrame(
    pipeline_validation_records
)


# ============================================================
# 14. VALIDASI OBJEK PIPELINE UNIK
# ============================================================

ensemble_pipeline_object_ids = [
    id(
        model_pipeline
    )
    for model_pipeline in (
        ensemble_pipelines.values()
    )
]

if len(
    ensemble_pipeline_object_ids
) != len(
    set(
        ensemble_pipeline_object_ids
    )
):
    raise ValueError(
        "Terdapat pipeline yang memakai objek sama "
        "secara tidak sengaja."
    )


# ============================================================
# 15. KARAKTERISTIK MODEL ENSEMBLE
# ============================================================

ensemble_model_characteristics_df = pd.DataFrame([
    {
        "model": "Random Forest",
        "abbreviation": "RF",
        "ensemble_strategy": "Bagging",
        "base_learner": "Decision trees",
        "class_imbalance_handling": (
            "class_weight optimized by GridSearchCV"
        ),
        "roc_auc_source": (
            "Probability of HIGH class"
        )
    },
    {
        "model": "Extra Trees",
        "abbreviation": "ET",
        "ensemble_strategy": (
            "Extremely randomized tree ensemble"
        ),
        "base_learner": "Randomized decision trees",
        "class_imbalance_handling": (
            "class_weight optimized by GridSearchCV"
        ),
        "roc_auc_source": (
            "Probability of HIGH class"
        )
    },
    {
        "model": "Gradient Boosting",
        "abbreviation": "GB",
        "ensemble_strategy": (
            "Sequential gradient boosting"
        ),
        "base_learner": "Shallow regression trees",
        "class_imbalance_handling": (
            "No direct class_weight parameter"
        ),
        "roc_auc_source": (
            "Probability of HIGH class"
        )
    },
    {
        "model": "AdaBoost",
        "abbreviation": "ADA",
        "ensemble_strategy": (
            "Adaptive boosting"
        ),
        "base_learner": "Decision stumps or shallow trees",
        "class_imbalance_handling": (
            "No direct class_weight parameter"
        ),
        "roc_auc_source": (
            "Probability of HIGH class"
        )
    },
    {
        "model": "XGBoost",
        "abbreviation": "XGB",
        "ensemble_strategy": (
            "Regularized gradient boosting"
        ),
        "base_learner": "Decision trees",
        "class_imbalance_handling": (
            "scale_pos_weight optimized by GridSearchCV"
        ),
        "roc_auc_source": (
            "Probability of HIGH class"
        )
    }
])


# ============================================================
# 16. MENAMPILKAN RINGKASAN PIPELINE
# ============================================================

print("=" * 76)
print("PURE ENSEMBLE MACHINE-LEARNING PIPELINES")
print("=" * 76)

print(
    "Number of classifiers :",
    len(
        ensemble_pipelines
    )
)

print(
    "Pipeline structure    :",
    "Median Imputer → Ensemble Classifier"
)

print(
    "StandardScaler used   :",
    False
)

print(
    "PCA used              :",
    False
)

print(
    "Positive class        :",
    "HIGH = 1"
)

print(
    "Score method          :",
    "predict_proba()"
)

print(
    "Models                :",
    list(
        ensemble_pipelines.keys()
    )
)


print("\nPipeline summary:")

display(
    ensemble_pipeline_summary_df
)


print("\nModel characteristics:")

display(
    ensemble_model_characteristics_df
)


# ============================================================
# 17. MENAMPILKAN BASE CONFIGURATION
# ============================================================

for model_name, model_pipeline in (
    ensemble_pipelines.items()
):

    classifier_object = (
        model_pipeline
        .named_steps["classifier"]
    )

    print("\n" + "-" * 76)

    print(
        model_name
    )

    print(
        "-" * 76
    )

    print(
        classifier_object
    )

    print(
        "ROC-AUC source:",
        ensemble_model_score_methods[
            model_name
        ]
    )


# ============================================================
# 18. MENYIMPAN RINGKASAN PIPELINE
# ============================================================

ensemble_pipeline_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_pipeline_summary.csv"
)

ensemble_pipeline_summary_df.to_csv(
    ensemble_pipeline_summary_path,
    index=False
)


ensemble_model_characteristics_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_model_characteristics.csv"
)

ensemble_model_characteristics_df.to_csv(
    ensemble_model_characteristics_path,
    index=False
)


# ============================================================
# 19. KONFIRMASI STEP 3
# ============================================================

print("\n" + "=" * 76)
print("STEP 3 COMPLETED")
print("=" * 76)

print(
    "Lima pipeline ensemble tanpa PCA dan StandardScaler "
    "berhasil dibuat:"
)

for model_name in ensemble_pipelines:
    print(
        f"- {model_name}"
    )

print(
    "\nSeluruh pipeline siap menerima hyperparameter grid "
    "pada STEP 4."
)

print(
    "\nPipeline summary saved to:"
)

print(
    ensemble_pipeline_summary_path
)

In [ ]:
# ============================================================
# STEP 4. DEFINE PURE ENSEMBLE HYPERPARAMETER GRIDS
# Random Forest, Extra Trees, Gradient Boosting,
# AdaBoost, and XGBoost
# ============================================================

# Tujuan:
# 1. Menentukan hyperparameter setiap ensemble classifier.
# 2. Menentukan strategi class-imbalance handling.
# 3. Menyiapkan parameter grid untuk inner GridSearchCV.
#
# Tidak digunakan:
# - StandardScaler
# - PCA
#
# Class-imbalance handling:
# - Random Forest : class_weight
# - Extra Trees   : class_weight
# - Gradient Boosting : tidak memiliki class_weight langsung
# - AdaBoost      : tidak memiliki class_weight langsung
# - XGBoost       : scale_pos_weight
#
# Dataset:
# - LOW/NO = 10 subject, kelas 0
# - HIGH   = 20 subject, kelas 1
#
# Referensi scale_pos_weight:
# negative / positive = 10 / 20 = 0.5


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP 1–3
# ============================================================

required_step_4_objects = [
    "ensemble_pipelines",
    "ensemble_X",
    "ensemble_y",
    "ensemble_feature_columns",
    "ensemble_scale_pos_weight_reference",
    "INNER_N_SPLITS",
    "OUTER_N_SPLITS",
    "ENSEMBLE_TABLE_DIR",
    "ENSEMBLE_OUTPUT_DIR"
]

missing_step_4_objects = [
    object_name
    for object_name in required_step_4_objects
    if object_name not in globals()
]

if missing_step_4_objects:
    raise NameError(
        "Objek untuk STEP 4 belum lengkap: "
        f"{missing_step_4_objects}. "
        "Jalankan STEP 1–3 terlebih dahulu."
    )


# ============================================================
# 2. VALIDASI DAFTAR MODEL
# ============================================================

expected_ensemble_model_names = {
    "Random Forest",
    "Extra Trees",
    "Gradient Boosting",
    "AdaBoost",
    "XGBoost"
}

actual_ensemble_model_names = set(
    ensemble_pipelines.keys()
)

if actual_ensemble_model_names != expected_ensemble_model_names:

    missing_models = (
        expected_ensemble_model_names
        - actual_ensemble_model_names
    )

    unexpected_models = (
        actual_ensemble_model_names
        - expected_ensemble_model_names
    )

    raise ValueError(
        "Daftar model pada ensemble_pipelines tidak sesuai.\n"
        f"Model belum tersedia: {sorted(missing_models)}\n"
        f"Model tidak diharapkan: {sorted(unexpected_models)}"
    )


# ============================================================
# 3. VALIDASI REFERENSI SCALE_POS_WEIGHT
# ============================================================

negative_subject_count = int(
    np.sum(
        ensemble_y == 0
    )
)

positive_subject_count = int(
    np.sum(
        ensemble_y == 1
    )
)

calculated_scale_pos_weight = float(
    negative_subject_count
    / positive_subject_count
)

if not np.isclose(
    calculated_scale_pos_weight,
    ensemble_scale_pos_weight_reference
):
    print(
        "PERINGATAN: Referensi scale_pos_weight sebelumnya "
        "tidak sama dengan distribusi kelas aktual."
    )

    print(
        "Nilai lama  :",
        ensemble_scale_pos_weight_reference
    )

    print(
        "Nilai aktual:",
        calculated_scale_pos_weight
    )

    ensemble_scale_pos_weight_reference = (
        calculated_scale_pos_weight
    )


# ============================================================
# 4. RANDOM FOREST — FULL GRID
# ============================================================

random_forest_full_grid = {
    "classifier__n_estimators": [
        100,
        300,
        500
    ],

    "classifier__criterion": [
        "gini",
        "entropy"
    ],

    "classifier__max_depth": [
        2,
        3,
        5,
        None
    ],

    "classifier__min_samples_split": [
        2,
        4,
        6
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        3
    ],

    "classifier__max_features": [
        "sqrt",
        "log2",
        None
    ],

    "classifier__bootstrap": [
        True
    ],

    "classifier__class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}


# ============================================================
# 5. EXTRA TREES — FULL GRID
# ============================================================

extra_trees_full_grid = {
    "classifier__n_estimators": [
        100,
        300,
        500
    ],

    "classifier__criterion": [
        "gini",
        "entropy"
    ],

    "classifier__max_depth": [
        2,
        3,
        5,
        None
    ],

    "classifier__min_samples_split": [
        2,
        4,
        6
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        3
    ],

    "classifier__max_features": [
        "sqrt",
        "log2",
        None
    ],

    "classifier__bootstrap": [
        False,
        True
    ],

    "classifier__class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}


# ============================================================
# 6. GRADIENT BOOSTING — FULL GRID
# ============================================================

# GradientBoostingClassifier tidak memiliki class_weight
# langsung. Regularisasi dilakukan melalui:
# - shallow trees;
# - learning rate;
# - subsampling;
# - minimum samples per leaf.

gradient_boosting_full_grid = {
    "classifier__n_estimators": [
        50,
        100,
        200
    ],

    "classifier__learning_rate": [
        0.01,
        0.05,
        0.1
    ],

    "classifier__max_depth": [
        1,
        2,
        3
    ],

    "classifier__min_samples_split": [
        2,
        4,
        6
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        3
    ],

    "classifier__subsample": [
        0.7,
        0.85,
        1.0
    ],

    "classifier__max_features": [
        None,
        "sqrt",
        "log2"
    ]
}


# ============================================================
# 7. ADABOOST — FULL GRID
# ============================================================

# AdaBoost tidak memiliki class_weight langsung.
# Hyperparameter utama:
# - jumlah weak learners;
# - learning rate.
#
# Base estimator tidak dituning pada eksperimen utama agar
# jumlah kombinasi tetap rasional untuk 30 subject.

adaboost_full_grid = {
    "classifier__n_estimators": [
        25,
        50,
        100,
        200
    ],

    "classifier__learning_rate": [
        0.01,
        0.05,
        0.1,
        0.5,
        1.0
    ]
}


# ============================================================
# 8. XGBOOST — FULL GRID
# ============================================================

# scale_pos_weight menguji:
# - 0.5  : rasio kelas negatif/positif aktual;
# - 0.75 : penyesuaian moderat;
# - 1.0  : tanpa pembobotan khusus.
#
# Regularisasi XGBoost:
# - max_depth
# - min_child_weight
# - gamma
# - reg_alpha
# - reg_lambda
# - subsample
# - colsample_bytree

xgboost_full_grid = {
    "classifier__n_estimators": [
        50,
        100,
        200
    ],

    "classifier__learning_rate": [
        0.01,
        0.05,
        0.1
    ],

    "classifier__max_depth": [
        1,
        2,
        3
    ],

    "classifier__min_child_weight": [
        1,
        2,
        4
    ],

    "classifier__gamma": [
        0.0,
        0.1,
        0.5
    ],

    "classifier__subsample": [
        0.7,
        0.85,
        1.0
    ],

    "classifier__colsample_bytree": [
        0.7,
        0.85,
        1.0
    ],

    "classifier__reg_alpha": [
        0.0,
        0.01,
        0.1
    ],

    "classifier__reg_lambda": [
        1.0,
        5.0,
        10.0
    ],

    "classifier__scale_pos_weight": [
        float(
            ensemble_scale_pos_weight_reference
        ),
        0.75,
        1.0
    ]
}


# ============================================================
# 9. MENYIMPAN FULL GRID
# ============================================================

ensemble_full_param_grids = {
    "Random Forest": (
        random_forest_full_grid
    ),

    "Extra Trees": (
        extra_trees_full_grid
    ),

    "Gradient Boosting": (
        gradient_boosting_full_grid
    ),

    "AdaBoost": (
        adaboost_full_grid
    ),

    "XGBoost": (
        xgboost_full_grid
    )
}


# ============================================================
# 10. RANDOM FOREST — COMPACT GRID
# ============================================================

random_forest_compact_grid = {
    "classifier__n_estimators": [
        100,
        300
    ],

    "classifier__criterion": [
        "gini",
        "entropy"
    ],

    "classifier__max_depth": [
        2,
        3,
        5,
        None
    ],

    "classifier__min_samples_leaf": [
        1,
        2
    ],

    "classifier__max_features": [
        "sqrt"
    ],

    "classifier__bootstrap": [
        True
    ],

    "classifier__class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}


# ============================================================
# 11. EXTRA TREES — COMPACT GRID
# ============================================================

extra_trees_compact_grid = {
    "classifier__n_estimators": [
        100,
        300
    ],

    "classifier__criterion": [
        "gini",
        "entropy"
    ],

    "classifier__max_depth": [
        2,
        3,
        5,
        None
    ],

    "classifier__min_samples_leaf": [
        1,
        2
    ],

    "classifier__max_features": [
        "sqrt"
    ],

    "classifier__bootstrap": [
        False,
        True
    ],

    "classifier__class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}


# ============================================================
# 12. GRADIENT BOOSTING — COMPACT GRID
# ============================================================

gradient_boosting_compact_grid = {
    "classifier__n_estimators": [
        50,
        100,
        200
    ],

    "classifier__learning_rate": [
        0.01,
        0.05,
        0.1
    ],

    "classifier__max_depth": [
        1,
        2
    ],

    "classifier__min_samples_leaf": [
        1,
        2
    ],

    "classifier__subsample": [
        0.8,
        1.0
    ],

    "classifier__max_features": [
        None,
        "sqrt"
    ]
}


# ============================================================
# 13. ADABOOST — COMPACT GRID
# ============================================================

adaboost_compact_grid = {
    "classifier__n_estimators": [
        25,
        50,
        100,
        200
    ],

    "classifier__learning_rate": [
        0.01,
        0.05,
        0.1,
        0.5,
        1.0
    ]
}


# ============================================================
# 14. XGBOOST — COMPACT GRID
# ============================================================

# Grid XGBoost sengaja dibatasi agar tidak menghasilkan
# ribuan kombinasi untuk dataset yang hanya memiliki
# 30 subject.

xgboost_compact_grid = {
    "classifier__n_estimators": [
        50,
        100,
        200
    ],

    "classifier__learning_rate": [
        0.01,
        0.05,
        0.1
    ],

    "classifier__max_depth": [
        1,
        2,
        3
    ],

    "classifier__min_child_weight": [
        1,
        2
    ],

    "classifier__gamma": [
        0.0,
        0.1
    ],

    "classifier__subsample": [
        0.8,
        1.0
    ],

    "classifier__colsample_bytree": [
        0.8,
        1.0
    ],

    "classifier__reg_alpha": [
        0.0,
        0.1
    ],

    "classifier__reg_lambda": [
        1.0,
        5.0
    ],

    "classifier__scale_pos_weight": [
        float(
            ensemble_scale_pos_weight_reference
        ),
        0.75,
        1.0
    ]
}


# ============================================================
# 15. MENYIMPAN COMPACT GRID
# ============================================================

ensemble_compact_param_grids = {
    "Random Forest": (
        random_forest_compact_grid
    ),

    "Extra Trees": (
        extra_trees_compact_grid
    ),

    "Gradient Boosting": (
        gradient_boosting_compact_grid
    ),

    "AdaBoost": (
        adaboost_compact_grid
    ),

    "XGBoost": (
        xgboost_compact_grid
    )
}


# ============================================================
# 16. VALIDASI KESESUAIAN NAMA MODEL
# ============================================================

pipeline_model_names = set(
    ensemble_pipelines.keys()
)

full_grid_model_names = set(
    ensemble_full_param_grids.keys()
)

compact_grid_model_names = set(
    ensemble_compact_param_grids.keys()
)

if pipeline_model_names != full_grid_model_names:
    raise ValueError(
        "Nama model pada ensemble_pipelines dan "
        "ensemble_full_param_grids tidak sama."
    )

if pipeline_model_names != compact_grid_model_names:
    raise ValueError(
        "Nama model pada ensemble_pipelines dan "
        "ensemble_compact_param_grids tidak sama."
    )


# ============================================================
# 17. FUNGSI VALIDASI PARAMETER GRID
# ============================================================

def validate_ensemble_parameter_grids(
    pipelines,
    parameter_grids,
    grid_name
):
    """
    Memastikan seluruh nama parameter tersedia pada pipeline.
    """

    validation_records = []

    for model_name, parameter_grid in (
        parameter_grids.items()
    ):

        model_pipeline = (
            pipelines[
                model_name
            ]
        )

        valid_parameter_names = set(
            model_pipeline
            .get_params()
            .keys()
        )

        invalid_parameters = [
            parameter_name
            for parameter_name
            in parameter_grid.keys()
            if parameter_name
            not in valid_parameter_names
        ]

        if invalid_parameters:
            raise ValueError(
                f"{grid_name} untuk {model_name} memiliki "
                "parameter tidak valid: "
                f"{invalid_parameters}"
            )

        empty_parameters = [
            parameter_name
            for parameter_name, parameter_values
            in parameter_grid.items()
            if len(
                parameter_values
            ) == 0
        ]

        if empty_parameters:
            raise ValueError(
                f"{grid_name} untuk {model_name} memiliki "
                "daftar kandidat kosong: "
                f"{empty_parameters}"
            )

        number_of_combinations = int(
            np.prod([
                len(
                    parameter_values
                )
                for parameter_values
                in parameter_grid.values()
            ])
        )

        validation_records.append({
            "grid_name": grid_name,
            "model": model_name,
            "number_of_parameters": int(
                len(
                    parameter_grid
                )
            ),
            "number_of_combinations": int(
                number_of_combinations
            ),
            "includes_class_weight": bool(
                "classifier__class_weight"
                in parameter_grid
            ),
            "includes_scale_pos_weight": bool(
                "classifier__scale_pos_weight"
                in parameter_grid
            ),
            "parameter_names": ", ".join(
                parameter_grid.keys()
            )
        })

    return pd.DataFrame(
        validation_records
    )


# ============================================================
# 18. VALIDASI FULL DAN COMPACT GRID
# ============================================================

ensemble_full_grid_summary_df = (
    validate_ensemble_parameter_grids(
        pipelines=ensemble_pipelines,
        parameter_grids=(
            ensemble_full_param_grids
        ),
        grid_name="full"
    )
)

ensemble_compact_grid_summary_df = (
    validate_ensemble_parameter_grids(
        pipelines=ensemble_pipelines,
        parameter_grids=(
            ensemble_compact_param_grids
        ),
        grid_name="compact"
    )
)


# ============================================================
# 19. VALIDASI CLASS-IMBALANCE PARAMETERS
# ============================================================

class_weight_ensemble_models = [
    "Random Forest",
    "Extra Trees"
]

for model_name in (
    class_weight_ensemble_models
):

    for grid_dictionary in [
        ensemble_full_param_grids,
        ensemble_compact_param_grids
    ]:

        if (
            "classifier__class_weight"
            not in grid_dictionary[
                model_name
            ]
        ):
            raise ValueError(
                f"{model_name} belum memiliki "
                "classifier__class_weight."
            )


for grid_dictionary in [
    ensemble_full_param_grids,
    ensemble_compact_param_grids
]:

    if (
        "classifier__scale_pos_weight"
        not in grid_dictionary[
            "XGBoost"
        ]
    ):
        raise ValueError(
            "XGBoost belum memiliki "
            "classifier__scale_pos_weight."
        )


for model_name in [
    "Gradient Boosting",
    "AdaBoost"
]:

    for grid_dictionary in [
        ensemble_full_param_grids,
        ensemble_compact_param_grids
    ]:

        if (
            "classifier__class_weight"
            in grid_dictionary[
                model_name
            ]
        ):
            raise ValueError(
                f"{model_name} tidak mendukung "
                "classifier__class_weight langsung."
            )


# ============================================================
# 20. ESTIMASI JUMLAH FIT
# ============================================================

for summary_df in [
    ensemble_full_grid_summary_df,
    ensemble_compact_grid_summary_df
]:

    summary_df[
        "estimated_inner_fits_per_outer_fold"
    ] = (
        summary_df[
            "number_of_combinations"
        ]
        * INNER_N_SPLITS
    )

    summary_df[
        "estimated_total_nested_fits"
    ] = (
        summary_df[
            "estimated_inner_fits_per_outer_fold"
        ]
        * OUTER_N_SPLITS
    )


TOTAL_FULL_ENSEMBLE_FITS = int(
    ensemble_full_grid_summary_df[
        "estimated_total_nested_fits"
    ].sum()
)

TOTAL_COMPACT_ENSEMBLE_FITS = int(
    ensemble_compact_grid_summary_df[
        "estimated_total_nested_fits"
    ].sum()
)


# ============================================================
# 21. MEMILIH MODE GRID
# ============================================================

# Untuk dataset 30 subject, gunakan compact sebagai
# eksperimen utama.
#
# Full grid dapat digunakan sebagai sensitivity analysis,
# tetapi berisiko memerlukan waktu yang sangat panjang dan
# meningkatkan risiko selection overfitting.

ENSEMBLE_GRID_MODE = "compact"

if ENSEMBLE_GRID_MODE == "compact":

    active_ensemble_param_grids = (
        ensemble_compact_param_grids
    )

    active_ensemble_grid_summary_df = (
        ensemble_compact_grid_summary_df
        .copy()
    )

elif ENSEMBLE_GRID_MODE == "full":

    active_ensemble_param_grids = (
        ensemble_full_param_grids
    )

    active_ensemble_grid_summary_df = (
        ensemble_full_grid_summary_df
        .copy()
    )

else:
    raise ValueError(
        "ENSEMBLE_GRID_MODE harus bernilai "
        "'compact' atau 'full'."
    )


TOTAL_ACTIVE_ENSEMBLE_FITS = int(
    active_ensemble_grid_summary_df[
        "estimated_total_nested_fits"
    ].sum()
)


# ============================================================
# 22. MENAMPILKAN RINGKASAN GRID
# ============================================================

print("=" * 76)
print("PURE ENSEMBLE HYPERPARAMETER GRIDS")
print("=" * 76)

print(
    "Models included              :",
    len(
        ensemble_pipelines
    )
)

print(
    "Models                       :",
    list(
        ensemble_pipelines.keys()
    )
)

print(
    "PCA included                 :",
    False
)

print(
    "StandardScaler included      :",
    False
)

print(
    "Grid mode                    :",
    ENSEMBLE_GRID_MODE
)

print(
    "Inner CV folds               :",
    INNER_N_SPLITS
)

print(
    "Outer CV folds               :",
    OUTER_N_SPLITS
)

print(
    "Class-weight models          :",
    class_weight_ensemble_models
)

print(
    "XGBoost scale_pos_weight     :",
    active_ensemble_param_grids[
        "XGBoost"
    ][
        "classifier__scale_pos_weight"
    ]
)

print(
    "Full-grid estimated fits     :",
    TOTAL_FULL_ENSEMBLE_FITS
)

print(
    "Compact-grid estimated fits  :",
    TOTAL_COMPACT_ENSEMBLE_FITS
)

print(
    "Active-grid estimated fits   :",
    TOTAL_ACTIVE_ENSEMBLE_FITS
)


print("\nFull grid summary:")

display(
    ensemble_full_grid_summary_df
)


print("\nActive grid summary:")

display(
    active_ensemble_grid_summary_df
)


# ============================================================
# 23. MENAMPILKAN DETAIL ACTIVE GRID
# ============================================================

for model_name, parameter_grid in (
    active_ensemble_param_grids.items()
):

    print("\n" + "=" * 76)
    print(model_name.upper())
    print("=" * 76)

    for parameter_name, parameter_values in (
        parameter_grid.items()
    ):

        print(
            f"{parameter_name}:"
        )

        print(
            parameter_values
        )


# ============================================================
# 24. MENYIMPAN RINGKASAN GRID
# ============================================================

ensemble_full_grid_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_full_hyperparameter_grid_summary.csv"
)

ensemble_full_grid_summary_df.to_csv(
    ensemble_full_grid_summary_path,
    index=False
)


ensemble_compact_grid_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_compact_hyperparameter_grid_summary.csv"
)

ensemble_compact_grid_summary_df.to_csv(
    ensemble_compact_grid_summary_path,
    index=False
)


ensemble_active_grid_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_active_hyperparameter_grid_summary.csv"
)

active_ensemble_grid_summary_df.to_csv(
    ensemble_active_grid_summary_path,
    index=False
)


# ============================================================
# 25. MENYIMPAN ACTIVE GRID KE JSON
# ============================================================

ensemble_active_grid_json_path = os.path.join(
    ENSEMBLE_OUTPUT_DIR,
    "ensemble_active_hyperparameter_grids.json"
)


serializable_active_ensemble_grids = {}

for model_name, parameter_grid in (
    active_ensemble_param_grids.items()
):

    serializable_active_ensemble_grids[
        model_name
    ] = {}

    for parameter_name, parameter_values in (
        parameter_grid.items()
    ):

        serializable_values = []

        for parameter_value in parameter_values:

            if isinstance(
                parameter_value,
                np.generic
            ):
                parameter_value = (
                    parameter_value.item()
                )

            serializable_values.append(
                parameter_value
            )

        serializable_active_ensemble_grids[
            model_name
        ][
            parameter_name
        ] = serializable_values


with open(
    ensemble_active_grid_json_path,
    "w",
    encoding="utf-8"
) as grid_file:

    json.dump(
        serializable_active_ensemble_grids,
        grid_file,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 26. MENYIMPAN CONFIGURATION SUMMARY
# ============================================================

ensemble_grid_configuration_summary = {
    "grid_mode": (
        ENSEMBLE_GRID_MODE
    ),

    "models": list(
        active_ensemble_param_grids.keys()
    ),

    "pca_used": False,

    "standard_scaler_used": False,

    "inner_cv_folds": int(
        INNER_N_SPLITS
    ),

    "outer_cv_folds": int(
        OUTER_N_SPLITS
    ),

    "reference_scale_pos_weight": float(
        ensemble_scale_pos_weight_reference
    ),

    "total_estimated_nested_fits": int(
        TOTAL_ACTIVE_ENSEMBLE_FITS
    )
}


ensemble_grid_configuration_summary_path = (
    os.path.join(
        ENSEMBLE_OUTPUT_DIR,
        "ensemble_grid_configuration_summary.json"
    )
)


with open(
    ensemble_grid_configuration_summary_path,
    "w",
    encoding="utf-8"
) as summary_file:

    json.dump(
        ensemble_grid_configuration_summary,
        summary_file,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 27. KONFIRMASI STEP 4
# ============================================================

print("\n" + "=" * 76)
print("STEP 4 COMPLETED")
print("=" * 76)

print(
    "Hyperparameter grid lima ensemble tanpa PCA "
    "berhasil disiapkan."
)

print(
    "\nActive grid mode:"
)

print(
    ENSEMBLE_GRID_MODE
)

print(
    "\nClass-imbalance optimization:"
)

print(
    "- Random Forest: class_weight"
)

print(
    "- Extra Trees: class_weight"
)

print(
    "- Gradient Boosting: regularization parameters"
)

print(
    "- AdaBoost: learning rate and number of estimators"
)

print(
    "- XGBoost: scale_pos_weight"
)

print(
    "\nObjek yang siap digunakan pada STEP 5:"
)

print(
    "- active_ensemble_param_grids"
)

print(
    "- active_ensemble_grid_summary_df"
)

print(
    "\nSTEP berikutnya:"
)

print(
    "STEP 5 — Nested Stratified Cross-Validation "
    "with inner GridSearchCV for five ensemble models."
)

In [ ]:
# ============================================================
# PRE-STEP 5. AUDIT STRATIFIED OUTER AND INNER FOLDS
# ============================================================

print("=" * 76)
print("CROSS-VALIDATION DISTRIBUTION AUDIT")
print("=" * 76)

outer_audit_records = []
inner_audit_records = []

for outer_fold, (
    outer_train_index,
    outer_test_index
) in enumerate(
    outer_cv.split(
        ensemble_X_array,
        ensemble_y_array
    ),
    start=1
):

    y_outer_train = ensemble_y_array[
        outer_train_index
    ]

    y_outer_test = ensemble_y_array[
        outer_test_index
    ]

    outer_audit_records.append({
        "outer_fold": outer_fold,
        "train_total": len(
            y_outer_train
        ),
        "train_LOW_NO": int(
            np.sum(
                y_outer_train == 0
            )
        ),
        "train_HIGH": int(
            np.sum(
                y_outer_train == 1
            )
        ),
        "test_total": len(
            y_outer_test
        ),
        "test_LOW_NO": int(
            np.sum(
                y_outer_test == 0
            )
        ),
        "test_HIGH": int(
            np.sum(
                y_outer_test == 1
            )
        )
    })

    inner_cv_audit = StratifiedKFold(
        n_splits=INNER_N_SPLITS,
        shuffle=True,
        random_state=(
            RANDOM_STATE
            + outer_fold
        )
    )

    for inner_fold, (
        inner_train_index,
        inner_validation_index
    ) in enumerate(
        inner_cv_audit.split(
            ensemble_X_array[
                outer_train_index
            ],
            y_outer_train
        ),
        start=1
    ):

        y_inner_train = y_outer_train[
            inner_train_index
        ]

        y_inner_validation = y_outer_train[
            inner_validation_index
        ]

        inner_audit_records.append({
            "outer_fold": outer_fold,
            "inner_fold": inner_fold,
            "inner_train_total": len(
                y_inner_train
            ),
            "inner_train_LOW_NO": int(
                np.sum(
                    y_inner_train == 0
                )
            ),
            "inner_train_HIGH": int(
                np.sum(
                    y_inner_train == 1
                )
            ),
            "inner_validation_total": len(
                y_inner_validation
            ),
            "inner_validation_LOW_NO": int(
                np.sum(
                    y_inner_validation == 0
                )
            ),
            "inner_validation_HIGH": int(
                np.sum(
                    y_inner_validation == 1
                )
            )
        })


outer_fold_distribution_df = pd.DataFrame(
    outer_audit_records
)

inner_fold_distribution_df = pd.DataFrame(
    inner_audit_records
)


print("\nOuter-fold distribution:")

display(
    outer_fold_distribution_df
)


print("\nInner-fold distribution:")

display(
    inner_fold_distribution_df
)


# ============================================================
# VALIDASI SETIAP FOLD MEMILIKI DUA KELAS
# ============================================================

invalid_outer_folds = (
    outer_fold_distribution_df[
        (
            outer_fold_distribution_df[
                "train_LOW_NO"
            ] == 0
        )
        |
        (
            outer_fold_distribution_df[
                "train_HIGH"
            ] == 0
        )
        |
        (
            outer_fold_distribution_df[
                "test_LOW_NO"
            ] == 0
        )
        |
        (
            outer_fold_distribution_df[
                "test_HIGH"
            ] == 0
        )
    ]
)

invalid_inner_folds = (
    inner_fold_distribution_df[
        (
            inner_fold_distribution_df[
                "inner_train_LOW_NO"
            ] == 0
        )
        |
        (
            inner_fold_distribution_df[
                "inner_train_HIGH"
            ] == 0
        )
        |
        (
            inner_fold_distribution_df[
                "inner_validation_LOW_NO"
            ] == 0
        )
        |
        (
            inner_fold_distribution_df[
                "inner_validation_HIGH"
            ] == 0
        )
    ]
)


if not invalid_outer_folds.empty:
    raise ValueError(
        "Ditemukan outer fold yang tidak memiliki "
        "kedua kelas."
    )

if not invalid_inner_folds.empty:
    raise ValueError(
        "Ditemukan inner fold yang tidak memiliki "
        "kedua kelas."
    )


print("\n" + "=" * 76)
print("CROSS-VALIDATION AUDIT PASSED")
print("=" * 76)

print(
    "Seluruh outer dan inner fold memiliki kelas "
    "LOW/NO dan HIGH."
)

In [ ]:
# ============================================================
# STEP 5. RUN NESTED CROSS-VALIDATION FOR PURE ENSEMBLE MODELS
# Outer Stratified 5-Fold + Inner 3-Fold GridSearchCV
# ============================================================

# Model yang dievaluasi:
# 1. Random Forest
# 2. Extra Trees
# 3. Gradient Boosting
# 4. AdaBoost
# 5. XGBoost
#
# Pipeline setiap model:
# Median Imputation
#       ↓
# Ensemble Classifier
#
# Tidak digunakan:
# - StandardScaler
# - PCA
#
# Pada setiap outer fold:
# 1. Outer training data digunakan untuk GridSearchCV.
# 2. Hyperparameter terbaik dipilih melalui inner CV.
# 3. Best estimator diuji pada outer test fold.
# 4. Setiap subject memperoleh tepat satu out-of-fold prediction.
#
# Metrik utama:
# - ROC-AUC
#
# Metrik tambahan:
# - Accuracy
# - Balanced Accuracy
# - Precision
# - Sensitivity / Recall
# - Specificity
# - F1-score
# - Confusion Matrix components


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP 1–4
# ============================================================

required_step_5_objects = [
    "ensemble_X",
    "ensemble_y",
    "ensemble_subjects",
    "ensemble_feature_columns",
    "ensemble_pipelines",
    "active_ensemble_param_grids",
    "ensemble_model_score_methods",
    "ensemble_model_abbreviations",
    "outer_cv",
    "INNER_N_SPLITS",
    "OUTER_N_SPLITS",
    "GRID_SCORING",
    "GRID_N_JOBS",
    "GRID_VERBOSE",
    "GRID_REFIT",
    "GRID_ERROR_SCORE",
    "RANDOM_STATE",
    "CLASS_LABEL_MAPPING",
    "ENSEMBLE_TABLE_DIR",
    "ENSEMBLE_LOG_DIR"
]

missing_step_5_objects = [
    object_name
    for object_name in required_step_5_objects
    if object_name not in globals()
]

if missing_step_5_objects:
    raise NameError(
        "Objek untuk STEP 5 belum lengkap: "
        f"{missing_step_5_objects}. "
        "Jalankan STEP 1–4 terlebih dahulu."
    )


# ============================================================
# 2. VALIDASI KESESUAIAN PIPELINE DAN PARAMETER GRID
# ============================================================

pipeline_model_names = set(
    ensemble_pipelines.keys()
)

active_grid_model_names = set(
    active_ensemble_param_grids.keys()
)

score_method_model_names = set(
    ensemble_model_score_methods.keys()
)

abbreviation_model_names = set(
    ensemble_model_abbreviations.keys()
)

if pipeline_model_names != active_grid_model_names:
    raise ValueError(
        "Nama model pada ensemble_pipelines dan "
        "active_ensemble_param_grids tidak sama."
    )

if pipeline_model_names != score_method_model_names:
    raise ValueError(
        "Nama model pada ensemble_pipelines dan "
        "ensemble_model_score_methods tidak sama."
    )

if pipeline_model_names != abbreviation_model_names:
    raise ValueError(
        "Nama model pada ensemble_pipelines dan "
        "ensemble_model_abbreviations tidak sama."
    )


# ============================================================
# 3. KONVERSI INPUT MENJADI ARRAY
# ============================================================

X_ensemble_nested = (
    ensemble_X
    .to_numpy(
        dtype=float
    )
)

y_ensemble_nested = np.asarray(
    ensemble_y,
    dtype=int
)

subjects_ensemble_nested = np.asarray(
    ensemble_subjects
)


if X_ensemble_nested.shape[0] != len(
    y_ensemble_nested
):
    raise ValueError(
        "Jumlah baris X tidak sama dengan jumlah target."
    )

if X_ensemble_nested.shape[0] != len(
    subjects_ensemble_nested
):
    raise ValueError(
        "Jumlah baris X tidak sama dengan jumlah subject."
    )

if X_ensemble_nested.shape[1] != len(
    ensemble_feature_columns
):
    raise ValueError(
        "Jumlah kolom X tidak sama dengan "
        "ensemble_feature_columns."
    )

if not np.array_equal(
    np.unique(
        y_ensemble_nested
    ),
    np.array([
        0,
        1
    ])
):
    raise ValueError(
        "Target harus memiliki kelas [0, 1]."
    )


# ============================================================
# 4. FUNGSI MENGAMBIL PROBABILITAS KELAS HIGH
# ============================================================

def extract_ensemble_positive_score(
    fitted_estimator,
    X_input
):
    """
    Mengambil probabilitas kelas HIGH = 1.

    Parameters
    ----------
    fitted_estimator : sklearn Pipeline
        Pipeline ensemble yang sudah dilatih.

    X_input : array-like
        Data input untuk prediksi.

    Returns
    -------
    numpy.ndarray
        Probabilitas kelas HIGH.
    """

    if not hasattr(
        fitted_estimator,
        "predict_proba"
    ):
        raise AttributeError(
            "Estimator tidak menyediakan predict_proba()."
        )

    classifier = (
        fitted_estimator
        .named_steps["classifier"]
    )

    classifier_classes = np.asarray(
        classifier.classes_
    )

    positive_class_indices = np.where(
        classifier_classes == 1
    )[0]

    if len(
        positive_class_indices
    ) != 1:
        raise ValueError(
            "Kelas positif HIGH=1 tidak ditemukan secara unik. "
            f"classes_={classifier_classes}"
        )

    positive_class_index = int(
        positive_class_indices[0]
    )

    probability_matrix = (
        fitted_estimator
        .predict_proba(
            X_input
        )
    )

    if probability_matrix.ndim != 2:
        raise ValueError(
            "Output predict_proba harus berupa matriks 2D."
        )

    if probability_matrix.shape[1] != len(
        classifier_classes
    ):
        raise ValueError(
            "Jumlah kolom probability tidak sesuai dengan "
            "jumlah kelas classifier."
        )

    positive_scores = np.asarray(
        probability_matrix[
            :,
            positive_class_index
        ],
        dtype=float
    )

    if not np.isfinite(
        positive_scores
    ).all():
        raise ValueError(
            "Probability score mengandung NaN atau infinite."
        )

    if (
        np.any(
            positive_scores < 0.0
        )
        or np.any(
            positive_scores > 1.0
        )
    ):
        raise ValueError(
            "Probability score berada di luar rentang [0, 1]."
        )

    return positive_scores


# ============================================================
# 5. FUNGSI MENGHITUNG METRIK BINARY CLASSIFICATION
# ============================================================

def calculate_ensemble_binary_metrics(
    y_true,
    y_pred,
    y_score
):
    """
    Menghitung metrik klasifikasi biner.

    Target:
    0 = LOW/NO
    1 = HIGH
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int
    )

    y_score = np.asarray(
        y_score,
        dtype=float
    )

    if not (
        len(y_true)
        == len(y_pred)
        == len(y_score)
    ):
        raise ValueError(
            "Panjang y_true, y_pred, dan y_score harus sama."
        )

    binary_confusion_matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1
        ]
    )

    tn, fp, fn, tp = (
        binary_confusion_matrix
        .ravel()
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    positive_predictive_value = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    negative_predictive_value = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    roc_auc = (
        roc_auc_score(
            y_true,
            y_score
        )
        if len(
            np.unique(
                y_true
            )
        ) == 2
        else np.nan
    )

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred
            )
        ),

        "precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            )
        ),

        "recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            )
        ),

        "sensitivity": float(
            sensitivity
        ),

        "specificity": float(
            specificity
        ),

        "positive_predictive_value": float(
            positive_predictive_value
        ),

        "negative_predictive_value": float(
            negative_predictive_value
        ),

        "f1_score": float(
            f1_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            )
        ),

        "roc_auc": float(
            roc_auc
        ),

        "true_negative": int(
            tn
        ),

        "false_positive": int(
            fp
        ),

        "false_negative": int(
            fn
        ),

        "true_positive": int(
            tp
        )
    }


# ============================================================
# 6. INISIALISASI PENYIMPAN HASIL
# ============================================================

ensemble_nested_cv_results = {}

ensemble_all_prediction_frames = []

ensemble_all_fold_metric_frames = []

ensemble_all_best_parameter_frames = []

ensemble_all_grid_result_frames = []

ensemble_model_summary_records = []

ensemble_execution_log_records = []


# ============================================================
# 7. MENAMPILKAN KONFIGURASI EVALUASI
# ============================================================

print("=" * 76)
print("PURE ENSEMBLE NESTED CROSS-VALIDATION")
print("=" * 76)

print(
    "Number of subjects          :",
    len(
        y_ensemble_nested
    )
)

print(
    "Number of features          :",
    X_ensemble_nested.shape[1]
)

print(
    "Number of models            :",
    len(
        ensemble_pipelines
    )
)

print(
    "Models                      :",
    list(
        ensemble_pipelines.keys()
    )
)

print(
    "Outer CV                    :",
    f"Stratified {OUTER_N_SPLITS}-Fold"
)

print(
    "Inner GridSearchCV          :",
    f"Stratified {INNER_N_SPLITS}-Fold"
)

print(
    "Grid scoring                :",
    GRID_SCORING
)

print(
    "PCA used                    :",
    False
)

print(
    "StandardScaler used         :",
    False
)

print(
    "Prediction score            :",
    "Probability of HIGH class"
)


# ============================================================
# 8. EVALUASI SETIAP MODEL
# ============================================================

for model_index, (
    model_name,
    base_pipeline
) in enumerate(
    ensemble_pipelines.items(),
    start=1
):

    print("\n" + "=" * 76)

    print(
        f"MODEL {model_index}/{len(ensemble_pipelines)}: "
        f"{model_name.upper()}"
    )

    print("=" * 76)

    parameter_grid = (
        active_ensemble_param_grids[
            model_name
        ]
    )

    model_abbreviation = (
        ensemble_model_abbreviations[
            model_name
        ]
    )

    model_start_time = time.time()

    model_prediction_records = []

    model_fold_metric_records = []

    model_best_parameter_records = []

    model_grid_result_frames = []

    model_y_true = []

    model_y_pred = []

    model_y_score = []

    model_test_subjects = []


    # ========================================================
    # 9. OUTER STRATIFIED CROSS-VALIDATION
    # ========================================================

    for outer_fold, (
        outer_train_indices,
        outer_test_indices
    ) in enumerate(
        outer_cv.split(
            X_ensemble_nested,
            y_ensemble_nested
        ),
        start=1
    ):

        fold_start_time = time.time()

        X_outer_train = (
            X_ensemble_nested[
                outer_train_indices
            ]
        )

        X_outer_test = (
            X_ensemble_nested[
                outer_test_indices
            ]
        )

        y_outer_train = (
            y_ensemble_nested[
                outer_train_indices
            ]
        )

        y_outer_test = (
            y_ensemble_nested[
                outer_test_indices
            ]
        )

        subjects_outer_test = (
            subjects_ensemble_nested[
                outer_test_indices
            ]
        )


        # ====================================================
        # 10. VALIDASI DISTRIBUSI KELAS OUTER FOLD
        # ====================================================

        train_class_counts = (
            pd.Series(
                y_outer_train
            )
            .value_counts()
            .sort_index()
        )

        test_class_counts = (
            pd.Series(
                y_outer_test
            )
            .value_counts()
            .sort_index()
        )

        if set(
            np.unique(
                y_outer_train
            )
        ) != {
            0,
            1
        }:
            raise ValueError(
                f"Outer training fold {outer_fold} "
                "tidak memiliki dua kelas."
            )

        if set(
            np.unique(
                y_outer_test
            )
        ) != {
            0,
            1
        }:
            raise ValueError(
                f"Outer test fold {outer_fold} "
                "tidak memiliki dua kelas."
            )


        # ====================================================
        # 11. INNER STRATIFIED CV KHUSUS OUTER FOLD
        # ====================================================

        inner_cv_fold = StratifiedKFold(
            n_splits=INNER_N_SPLITS,
            shuffle=True,
            random_state=(
                RANDOM_STATE
                + outer_fold
            )
        )


        # ====================================================
        # 12. GRIDSEARCHCV PADA OUTER TRAINING FOLD
        # ====================================================

        grid_search = GridSearchCV(
            estimator=clone(
                base_pipeline
            ),

            param_grid=(
                parameter_grid
            ),

            scoring=(
                GRID_SCORING
            ),

            cv=(
                inner_cv_fold
            ),

            n_jobs=(
                GRID_N_JOBS
            ),

            verbose=(
                GRID_VERBOSE
            ),

            refit=(
                GRID_REFIT
            ),

            return_train_score=True,

            error_score=(
                GRID_ERROR_SCORE
            )
        )


        grid_search.fit(
            X_outer_train,
            y_outer_train
        )


        # ====================================================
        # 13. BEST ESTIMATOR DAN HYPERPARAMETER
        # ====================================================

        best_estimator = (
            grid_search
            .best_estimator_
        )

        best_parameters = dict(
            grid_search
            .best_params_
        )

        best_inner_auc = float(
            grid_search
            .best_score_
        )

        best_classifier = (
            best_estimator
            .named_steps["classifier"]
        )

        best_classifier_classes = np.asarray(
            best_classifier.classes_
        )

        if not np.array_equal(
            best_classifier_classes,
            np.array([
                0,
                1
            ])
        ):
            raise ValueError(
                f"Urutan kelas {model_name}, fold "
                f"{outer_fold}, bukan [0, 1]: "
                f"{best_classifier_classes}"
            )


        # ====================================================
        # 14. PREDIKSI OUTER TEST FOLD
        # ====================================================

        outer_predicted_labels = (
            best_estimator
            .predict(
                X_outer_test
            )
            .astype(int)
        )

        outer_prediction_scores = (
            extract_ensemble_positive_score(
                fitted_estimator=(
                    best_estimator
                ),
                X_input=(
                    X_outer_test
                )
            )
        )


        if len(
            outer_predicted_labels
        ) != len(
            y_outer_test
        ):
            raise ValueError(
                f"Jumlah predicted label pada fold "
                f"{outer_fold} tidak sesuai."
            )

        if len(
            outer_prediction_scores
        ) != len(
            y_outer_test
        ):
            raise ValueError(
                f"Jumlah prediction score pada fold "
                f"{outer_fold} tidak sesuai."
            )


        # ====================================================
        # 15. METRIK OUTER FOLD
        # ====================================================

        fold_metrics = (
            calculate_ensemble_binary_metrics(
                y_true=(
                    y_outer_test
                ),

                y_pred=(
                    outer_predicted_labels
                ),

                y_score=(
                    outer_prediction_scores
                )
            )
        )


        # ====================================================
        # 16. TRAINING SCORE UNTUK AUDIT OVERFITTING
        # ====================================================

        outer_train_predicted_labels = (
            best_estimator
            .predict(
                X_outer_train
            )
            .astype(int)
        )

        outer_train_prediction_scores = (
            extract_ensemble_positive_score(
                fitted_estimator=(
                    best_estimator
                ),

                X_input=(
                    X_outer_train
                )
            )
        )

        outer_train_metrics = (
            calculate_ensemble_binary_metrics(
                y_true=(
                    y_outer_train
                ),

                y_pred=(
                    outer_train_predicted_labels
                ),

                y_score=(
                    outer_train_prediction_scores
                )
            )
        )

        train_test_auc_gap = float(
            outer_train_metrics[
                "roc_auc"
            ]
            - fold_metrics[
                "roc_auc"
            ]
        )


        # ====================================================
        # 17. MENYIMPAN PREDIKSI PER SUBJECT
        # ====================================================

        for local_index, subject_name in enumerate(
            subjects_outer_test
        ):

            actual_label = int(
                y_outer_test[
                    local_index
                ]
            )

            predicted_label = int(
                outer_predicted_labels[
                    local_index
                ]
            )

            prediction_score = float(
                outer_prediction_scores[
                    local_index
                ]
            )

            model_prediction_records.append({
                "model": (
                    model_name
                ),

                "model_abbreviation": (
                    model_abbreviation
                ),

                "outer_fold": int(
                    outer_fold
                ),

                "subject": str(
                    subject_name
                ),

                "actual_label": (
                    actual_label
                ),

                "actual_class": (
                    CLASS_LABEL_MAPPING[
                        actual_label
                    ]
                ),

                "predicted_label": (
                    predicted_label
                ),

                "predicted_class": (
                    CLASS_LABEL_MAPPING[
                        predicted_label
                    ]
                ),

                "probability_HIGH": (
                    prediction_score
                ),

                "score_method": (
                    "predict_proba"
                ),

                "correct_prediction": bool(
                    actual_label
                    == predicted_label
                )
            })


        # ====================================================
        # 18. MENYIMPAN METRIK PER OUTER FOLD
        # ====================================================

        fold_elapsed_seconds = float(
            time.time()
            - fold_start_time
        )

        model_fold_metric_records.append({
            "model": (
                model_name
            ),

            "model_abbreviation": (
                model_abbreviation
            ),

            "outer_fold": int(
                outer_fold
            ),

            "outer_train_size": int(
                len(
                    outer_train_indices
                )
            ),

            "outer_test_size": int(
                len(
                    outer_test_indices
                )
            ),

            "training_LOW_NO": int(
                train_class_counts.get(
                    0,
                    0
                )
            ),

            "training_HIGH": int(
                train_class_counts.get(
                    1,
                    0
                )
            ),

            "test_LOW_NO": int(
                test_class_counts.get(
                    0,
                    0
                )
            ),

            "test_HIGH": int(
                test_class_counts.get(
                    1,
                    0
                )
            ),

            "best_inner_auc": (
                best_inner_auc
            ),

            "training_accuracy": (
                outer_train_metrics[
                    "accuracy"
                ]
            ),

            "training_balanced_accuracy": (
                outer_train_metrics[
                    "balanced_accuracy"
                ]
            ),

            "training_roc_auc": (
                outer_train_metrics[
                    "roc_auc"
                ]
            ),

            "train_test_auc_gap": (
                train_test_auc_gap
            ),

            "elapsed_seconds": (
                fold_elapsed_seconds
            ),

            **fold_metrics
        })


        # ====================================================
        # 19. MENYIMPAN HYPERPARAMETER TERBAIK
        # ====================================================

        best_parameter_record = {
            "model": (
                model_name
            ),

            "model_abbreviation": (
                model_abbreviation
            ),

            "outer_fold": int(
                outer_fold
            ),

            "best_inner_auc": (
                best_inner_auc
            )
        }

        for parameter_name, parameter_value in (
            best_parameters.items()
        ):

            if isinstance(
                parameter_value,
                np.generic
            ):
                parameter_value = (
                    parameter_value.item()
                )

            best_parameter_record[
                parameter_name
            ] = parameter_value

        model_best_parameter_records.append(
            best_parameter_record
        )


        # ====================================================
        # 20. MENYIMPAN SELURUH HASIL GRID SEARCH
        # ====================================================

        fold_grid_result_df = pd.DataFrame(
            grid_search.cv_results_
        )

        grid_result_columns = [
            column_name
            for column_name in [
                "rank_test_score",
                "mean_test_score",
                "std_test_score",
                "mean_train_score",
                "std_train_score",
                "mean_fit_time",
                "std_fit_time",
                "mean_score_time",
                "std_score_time",
                "params"
            ]
            if column_name
            in fold_grid_result_df.columns
        ]

        fold_grid_result_df = (
            fold_grid_result_df[
                grid_result_columns
            ]
            .copy()
        )

        fold_grid_result_df.insert(
            0,
            "outer_fold",
            outer_fold
        )

        fold_grid_result_df.insert(
            0,
            "model",
            model_name
        )

        model_grid_result_frames.append(
            fold_grid_result_df
        )


        # ====================================================
        # 21. MENAMBAHKAN KE OUT-OF-FOLD ARRAY
        # ====================================================

        model_y_true.extend(
            y_outer_test.tolist()
        )

        model_y_pred.extend(
            outer_predicted_labels.tolist()
        )

        model_y_score.extend(
            outer_prediction_scores.tolist()
        )

        model_test_subjects.extend(
            subjects_outer_test.tolist()
        )


        # ====================================================
        # 22. LOG EKSEKUSI
        # ====================================================

        ensemble_execution_log_records.append({
            "model": (
                model_name
            ),

            "outer_fold": int(
                outer_fold
            ),

            "best_inner_auc": (
                best_inner_auc
            ),

            "outer_test_auc": (
                fold_metrics[
                    "roc_auc"
                ]
            ),

            "training_auc": (
                outer_train_metrics[
                    "roc_auc"
                ]
            ),

            "train_test_auc_gap": (
                train_test_auc_gap
            ),

            "elapsed_seconds": (
                fold_elapsed_seconds
            ),

            "best_parameters": json.dumps(
                best_parameters,
                default=str
            )
        })


        # ====================================================
        # 23. MENAMPILKAN HASIL OUTER FOLD
        # ====================================================

        print(
            f"\nOuter fold "
            f"{outer_fold}/{OUTER_N_SPLITS}"
        )

        print(
            "  Train/Test subjects :",
            f"{len(outer_train_indices)}/"
            f"{len(outer_test_indices)}"
        )

        print(
            "  Train distribution  :",
            f"LOW/NO={train_class_counts.get(0, 0)}, "
            f"HIGH={train_class_counts.get(1, 0)}"
        )

        print(
            "  Test distribution   :",
            f"LOW/NO={test_class_counts.get(0, 0)}, "
            f"HIGH={test_class_counts.get(1, 0)}"
        )

        print(
            "  Best inner ROC-AUC  :",
            f"{best_inner_auc:.4f}"
        )

        print(
            "  Training ROC-AUC    :",
            f"{outer_train_metrics['roc_auc']:.4f}"
        )

        print(
            "  Outer test ROC-AUC  :",
            f"{fold_metrics['roc_auc']:.4f}"
        )

        print(
            "  Outer accuracy      :",
            f"{fold_metrics['accuracy']:.4f}"
        )

        print(
            "  Balanced accuracy   :",
            f"{fold_metrics['balanced_accuracy']:.4f}"
        )

        print(
            "  Sensitivity         :",
            f"{fold_metrics['sensitivity']:.4f}"
        )

        print(
            "  Specificity         :",
            f"{fold_metrics['specificity']:.4f}"
        )

        print(
            "  Train-test AUC gap  :",
            f"{train_test_auc_gap:.4f}"
        )

        print(
            "  Best parameters     :",
            best_parameters
        )


    # ========================================================
    # 24. VALIDASI OUT-OF-FOLD PREDICTIONS
    # ========================================================

    model_y_true = np.asarray(
        model_y_true,
        dtype=int
    )

    model_y_pred = np.asarray(
        model_y_pred,
        dtype=int
    )

    model_y_score = np.asarray(
        model_y_score,
        dtype=float
    )

    model_test_subjects = np.asarray(
        model_test_subjects
    )

    if len(
        model_y_true
    ) != len(
        y_ensemble_nested
    ):
        raise ValueError(
            f"Jumlah out-of-fold predictions untuk "
            f"{model_name} tidak sama dengan jumlah subject."
        )

    if len(
        model_test_subjects
    ) != len(
        subjects_ensemble_nested
    ):
        raise ValueError(
            f"Jumlah subject prediction untuk "
            f"{model_name} tidak sesuai."
        )

    if len(
        np.unique(
            model_test_subjects
        )
    ) != len(
        subjects_ensemble_nested
    ):
        raise ValueError(
            f"Subject pada model {model_name} tidak memperoleh "
            "tepat satu out-of-fold prediction."
        )

    if set(
        model_test_subjects.tolist()
    ) != set(
        subjects_ensemble_nested.tolist()
    ):
        raise ValueError(
            f"Daftar subject out-of-fold {model_name} "
            "tidak sama dengan daftar subject asli."
        )


    # ========================================================
    # 25. METRIK OUT-OF-FOLD KESELURUHAN
    # ========================================================

    overall_metrics = (
        calculate_ensemble_binary_metrics(
            y_true=(
                model_y_true
            ),

            y_pred=(
                model_y_pred
            ),

            y_score=(
                model_y_score
            )
        )
    )

    model_elapsed_seconds = float(
        time.time()
        - model_start_time
    )

    model_fold_metrics_df = pd.DataFrame(
        model_fold_metric_records
    )

    mean_training_auc = float(
        model_fold_metrics_df[
            "training_roc_auc"
        ].mean()
    )

    mean_outer_fold_auc = float(
        model_fold_metrics_df[
            "roc_auc"
        ].mean()
    )

    std_outer_fold_auc = float(
        model_fold_metrics_df[
            "roc_auc"
        ].std(
            ddof=1
        )
    )

    mean_train_test_auc_gap = float(
        model_fold_metrics_df[
            "train_test_auc_gap"
        ].mean()
    )

    mean_best_inner_auc = float(
        model_fold_metrics_df[
            "best_inner_auc"
        ].mean()
    )

    std_best_inner_auc = float(
        model_fold_metrics_df[
            "best_inner_auc"
        ].std(
            ddof=1
        )
    )


    ensemble_model_summary_records.append({
        "model": (
            model_name
        ),

        "model_abbreviation": (
            model_abbreviation
        ),

        "number_of_subjects": int(
            len(
                model_y_true
            )
        ),

        "number_of_features": int(
            X_ensemble_nested.shape[1]
        ),

        "number_of_outer_folds": int(
            OUTER_N_SPLITS
        ),

        "mean_best_inner_auc": (
            mean_best_inner_auc
        ),

        "std_best_inner_auc": (
            std_best_inner_auc
        ),

        "mean_training_auc": (
            mean_training_auc
        ),

        "mean_outer_fold_auc": (
            mean_outer_fold_auc
        ),

        "std_outer_fold_auc": (
            std_outer_fold_auc
        ),

        "mean_train_test_auc_gap": (
            mean_train_test_auc_gap
        ),

        "total_elapsed_seconds": (
            model_elapsed_seconds
        ),

        **overall_metrics
    })


    # ========================================================
    # 26. MEMBENTUK DATAFRAME HASIL MODEL
    # ========================================================

    model_predictions_df = pd.DataFrame(
        model_prediction_records
    )

    model_best_parameters_df = pd.DataFrame(
        model_best_parameter_records
    )

    model_grid_search_results_df = pd.concat(
        model_grid_result_frames,
        ignore_index=True
    )


    # ========================================================
    # 27. MENYIMPAN HASIL MODEL KE DICTIONARY
    # ========================================================

    ensemble_nested_cv_results[
        model_name
    ] = {
        "predictions": (
            model_predictions_df
        ),

        "fold_metrics": (
            model_fold_metrics_df
        ),

        "best_parameters": (
            model_best_parameters_df
        ),

        "grid_search_results": (
            model_grid_search_results_df
        ),

        "y_true": (
            model_y_true
        ),

        "y_pred": (
            model_y_pred
        ),

        "y_score": (
            model_y_score
        ),

        "subjects": (
            model_test_subjects
        ),

        "overall_metrics": (
            overall_metrics
        )
    }


    # ========================================================
    # 28. MENAMBAHKAN KE HASIL GABUNGAN
    # ========================================================

    ensemble_all_prediction_frames.append(
        model_predictions_df
    )

    ensemble_all_fold_metric_frames.append(
        model_fold_metrics_df
    )

    ensemble_all_best_parameter_frames.append(
        model_best_parameters_df
    )

    ensemble_all_grid_result_frames.append(
        model_grid_search_results_df
    )


    # ========================================================
    # 29. MENAMPILKAN HASIL KESELURUHAN MODEL
    # ========================================================

    print("\n" + "-" * 76)

    print(
        f"OVERALL OUT-OF-FOLD RESULTS: "
        f"{model_name}"
    )

    print("-" * 76)

    print(
        "Accuracy               :",
        f"{overall_metrics['accuracy']:.4f}"
    )

    print(
        "Balanced accuracy      :",
        f"{overall_metrics['balanced_accuracy']:.4f}"
    )

    print(
        "Precision              :",
        f"{overall_metrics['precision']:.4f}"
    )

    print(
        "Sensitivity / Recall   :",
        f"{overall_metrics['sensitivity']:.4f}"
    )

    print(
        "Specificity            :",
        f"{overall_metrics['specificity']:.4f}"
    )

    print(
        "F1-score               :",
        f"{overall_metrics['f1_score']:.4f}"
    )

    print(
        "Out-of-fold ROC-AUC    :",
        f"{overall_metrics['roc_auc']:.4f}"
    )

    print(
        "Mean outer-fold AUC    :",
        f"{mean_outer_fold_auc:.4f} ± "
        f"{std_outer_fold_auc:.4f}"
    )

    print(
        "Mean training AUC      :",
        f"{mean_training_auc:.4f}"
    )

    print(
        "Mean train-test gap    :",
        f"{mean_train_test_auc_gap:.4f}"
    )

    print(
        "Elapsed time           :",
        f"{model_elapsed_seconds:.2f} seconds"
    )


# ============================================================
# 30. MENGGABUNGKAN SELURUH HASIL MODEL
# ============================================================

ensemble_nested_predictions_df = pd.concat(
    ensemble_all_prediction_frames,
    ignore_index=True
)

ensemble_nested_fold_metrics_df = pd.concat(
    ensemble_all_fold_metric_frames,
    ignore_index=True
)

ensemble_nested_best_parameters_df = pd.concat(
    ensemble_all_best_parameter_frames,
    ignore_index=True
)

ensemble_nested_grid_search_results_df = pd.concat(
    ensemble_all_grid_result_frames,
    ignore_index=True
)

ensemble_execution_log_df = pd.DataFrame(
    ensemble_execution_log_records
)


# ============================================================
# 31. MEMBENTUK RINGKASAN PERFORMA
# ============================================================

ensemble_nested_model_summary_df = (
    pd.DataFrame(
        ensemble_model_summary_records
    )
    .sort_values(
        by=[
            "roc_auc",
            "balanced_accuracy",
            "specificity",
            "f1_score"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

ensemble_nested_model_summary_df.insert(
    0,
    "performance_rank",
    np.arange(
        1,
        len(
            ensemble_nested_model_summary_df
        ) + 1
    )
)


# ============================================================
# 32. VALIDASI JUMLAH PREDIKSI
# ============================================================

expected_total_prediction_rows = (
    len(
        ensemble_pipelines
    )
    * len(
        ensemble_y
    )
)

if len(
    ensemble_nested_predictions_df
) != expected_total_prediction_rows:
    raise ValueError(
        "Jumlah seluruh prediction rows tidak sesuai. "
        f"Diharapkan {expected_total_prediction_rows}, "
        f"diperoleh {len(ensemble_nested_predictions_df)}."
    )


prediction_count_by_model = (
    ensemble_nested_predictions_df
    .groupby("model")
    .size()
)

if not (
    prediction_count_by_model
    == len(
        ensemble_y
    )
).all():
    raise ValueError(
        "Tidak semua model menghasilkan satu prediksi "
        "untuk setiap subject."
    )


# ============================================================
# 33. MENAMPILKAN RINGKASAN PERFORMA
# ============================================================

ensemble_summary_display_columns = [
    "performance_rank",
    "model",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1_score",
    "roc_auc",
    "mean_outer_fold_auc",
    "std_outer_fold_auc",
    "mean_training_auc",
    "mean_train_test_auc_gap"
]

print("\n" + "=" * 76)
print("PURE ENSEMBLE NESTED CV MODEL SUMMARY")
print("=" * 76)

display(
    ensemble_nested_model_summary_df[
        ensemble_summary_display_columns
    ]
)


# ============================================================
# 34. MENENTUKAN MODEL TERBAIK SEMENTARA
# ============================================================

best_ensemble_nested_row = (
    ensemble_nested_model_summary_df
    .iloc[0]
)

best_ensemble_model_name = str(
    best_ensemble_nested_row[
        "model"
    ]
)

best_ensemble_model_auc = float(
    best_ensemble_nested_row[
        "roc_auc"
    ]
)


print("\n" + "=" * 76)
print("BEST PURE ENSEMBLE MODEL")
print("=" * 76)

print(
    "Model              :",
    best_ensemble_model_name
)

print(
    "ROC-AUC            :",
    f"{best_ensemble_model_auc:.4f}"
)

print(
    "Accuracy           :",
    f"{best_ensemble_nested_row['accuracy']:.4f}"
)

print(
    "Balanced accuracy  :",
    f"{best_ensemble_nested_row['balanced_accuracy']:.4f}"
)

print(
    "Sensitivity        :",
    f"{best_ensemble_nested_row['sensitivity']:.4f}"
)

print(
    "Specificity        :",
    f"{best_ensemble_nested_row['specificity']:.4f}"
)

print(
    "F1-score           :",
    f"{best_ensemble_nested_row['f1_score']:.4f}"
)

print(
    "Training AUC       :",
    f"{best_ensemble_nested_row['mean_training_auc']:.4f}"
)

print(
    "Train-test AUC gap :",
    f"{best_ensemble_nested_row['mean_train_test_auc_gap']:.4f}"
)


# ============================================================
# 35. PERINGATAN OVERFITTING
# ============================================================

if (
    best_ensemble_nested_row[
        "mean_train_test_auc_gap"
    ]
    > 0.30
):

    print(
        "\nPERINGATAN: Model terbaik menunjukkan "
        "train-test AUC gap yang sangat besar."
    )

    print(
        "Hasil harus diinterpretasikan sebagai indikasi "
        "overfitting pada dataset kecil."
    )

elif (
    best_ensemble_nested_row[
        "mean_train_test_auc_gap"
    ]
    > 0.15
):

    print(
        "\nCatatan: Model terbaik menunjukkan "
        "train-test AUC gap moderat."
    )

else:

    print(
        "\nTrain-test AUC gap model terbaik relatif kecil."
    )


# ============================================================
# 36. MENYIMPAN SELURUH HASIL STEP 5
# ============================================================

ensemble_model_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_nested_cv_model_summary.csv"
)

ensemble_nested_model_summary_df.to_csv(
    ensemble_model_summary_path,
    index=False
)


ensemble_predictions_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_nested_cv_subject_predictions.csv"
)

ensemble_nested_predictions_df.to_csv(
    ensemble_predictions_path,
    index=False
)


ensemble_fold_metrics_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_nested_cv_fold_metrics.csv"
)

ensemble_nested_fold_metrics_df.to_csv(
    ensemble_fold_metrics_path,
    index=False
)


ensemble_best_parameters_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_nested_cv_best_parameters.csv"
)

ensemble_nested_best_parameters_df.to_csv(
    ensemble_best_parameters_path,
    index=False
)


ensemble_grid_results_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_nested_cv_grid_search_results.csv"
)

ensemble_nested_grid_search_results_df.to_csv(
    ensemble_grid_results_path,
    index=False
)


ensemble_execution_log_path = os.path.join(
    ENSEMBLE_LOG_DIR,
    "ensemble_nested_cv_execution_log.csv"
)

ensemble_execution_log_df.to_csv(
    ensemble_execution_log_path,
    index=False
)


# ============================================================
# 37. KONFIRMASI STEP 5
# ============================================================

print("\n" + "=" * 76)
print("STEP 5 COMPLETED")
print("=" * 76)

print(
    "Nested Stratified Cross-Validation dan "
    "inner GridSearchCV berhasil dijalankan untuk "
    "lima model ensemble."
)

print(
    "\nBest ensemble model:"
)

print(
    best_ensemble_model_name
)

print(
    "\nBest out-of-fold ROC-AUC:"
)

print(
    f"{best_ensemble_model_auc:.4f}"
)

print(
    "\nModel summary saved to:"
)

print(
    ensemble_model_summary_path
)

print(
    "\nSTEP berikutnya:"
)

print(
    "STEP 6 — Analyze hyperparameter stability, "
    "class-imbalance settings, and fold performance."
)

In [ ]:
# ============================================================
# STEP 6. ANALYZE ENSEMBLE HYPERPARAMETER STABILITY
# Best Parameters, Class-Imbalance Settings, and Fold Stability
# ============================================================

# Tujuan STEP 6:
# 1. Menganalisis hyperparameter terbaik pada setiap outer fold.
# 2. Mengidentifikasi konfigurasi yang paling sering dipilih.
# 3. Menganalisis penggunaan class_weight dan scale_pos_weight.
# 4. Mengukur kestabilan performa antar-fold.
# 5. Mengidentifikasi indikasi overfitting.
#
# Sumber hasil:
# - ensemble_nested_best_parameters_df
# - ensemble_nested_fold_metrics_df
# - ensemble_nested_model_summary_df


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP 5
# ============================================================

required_step_6_objects = [
    "ensemble_nested_best_parameters_df",
    "ensemble_nested_fold_metrics_df",
    "ensemble_nested_model_summary_df",
    "ensemble_nested_cv_results",
    "ensemble_pipelines",
    "ensemble_model_abbreviations",
    "best_ensemble_model_name",
    "ENSEMBLE_TABLE_DIR",
    "ENSEMBLE_FIGURE_DIR"
]

missing_step_6_objects = [
    object_name
    for object_name in required_step_6_objects
    if object_name not in globals()
]

if missing_step_6_objects:
    raise NameError(
        "Objek untuk STEP 6 belum lengkap: "
        f"{missing_step_6_objects}. "
        "Jalankan STEP 5 terlebih dahulu."
    )


# ============================================================
# 2. MEMBUAT SALINAN DATAFRAME HASIL
# ============================================================

ensemble_best_parameters_analysis_df = (
    ensemble_nested_best_parameters_df
    .copy()
)

ensemble_fold_metrics_analysis_df = (
    ensemble_nested_fold_metrics_df
    .copy()
)

ensemble_model_summary_analysis_df = (
    ensemble_nested_model_summary_df
    .copy()
)


# ============================================================
# 3. VALIDASI JUMLAH OUTER FOLD SETIAP MODEL
# ============================================================

fold_count_by_model = (
    ensemble_fold_metrics_analysis_df
    .groupby("model")[
        "outer_fold"
    ]
    .nunique()
)

invalid_fold_counts = (
    fold_count_by_model[
        fold_count_by_model
        != OUTER_N_SPLITS
    ]
)

if not invalid_fold_counts.empty:
    raise ValueError(
        "Jumlah outer fold tidak sesuai pada model berikut:\n"
        f"{invalid_fold_counts.to_dict()}"
    )


# ============================================================
# 4. IDENTIFIKASI KOLOM HYPERPARAMETER
# ============================================================

ensemble_parameter_columns = [
    column_name
    for column_name in (
        ensemble_best_parameters_analysis_df.columns
    )
    if column_name.startswith(
        "classifier__"
    )
]

if not ensemble_parameter_columns:
    raise ValueError(
        "Tidak ditemukan kolom hyperparameter classifier."
    )


print("=" * 76)
print("ENSEMBLE HYPERPARAMETER COLUMNS")
print("=" * 76)

print(
    "Number of hyperparameter columns:",
    len(
        ensemble_parameter_columns
    )
)

for parameter_name in ensemble_parameter_columns:
    print(
        "-",
        parameter_name
    )


# ============================================================
# 5. FREKUENSI HYPERPARAMETER TERPILIH
# ============================================================

ensemble_parameter_frequency_frames = []

for model_name in ensemble_pipelines.keys():

    model_parameter_df = (
        ensemble_best_parameters_analysis_df[
            ensemble_best_parameters_analysis_df[
                "model"
            ]
            == model_name
        ]
        .copy()
    )

    for parameter_name in (
        ensemble_parameter_columns
    ):

        if parameter_name not in (
            model_parameter_df.columns
        ):
            continue

        non_missing_parameter_values = (
            model_parameter_df[
                parameter_name
            ]
            .dropna()
        )

        if non_missing_parameter_values.empty:
            continue

        parameter_frequency_df = (
            non_missing_parameter_values
            .astype(str)
            .value_counts(
                dropna=False
            )
            .rename_axis(
                "parameter_value"
            )
            .reset_index(
                name="frequency"
            )
        )

        parameter_frequency_df.insert(
            0,
            "parameter",
            parameter_name
        )

        parameter_frequency_df.insert(
            0,
            "model",
            model_name
        )

        parameter_frequency_df[
            "selection_percentage"
        ] = (
            parameter_frequency_df[
                "frequency"
            ]
            / len(
                model_parameter_df
            )
            * 100
        )

        ensemble_parameter_frequency_frames.append(
            parameter_frequency_df
        )


ensemble_parameter_frequency_df = pd.concat(
    ensemble_parameter_frequency_frames,
    ignore_index=True
)


# ============================================================
# 6. HYPERPARAMETER PALING SERING DIPILIH
# ============================================================

ensemble_most_frequent_parameters_df = (
    ensemble_parameter_frequency_df
    .sort_values(
        by=[
            "model",
            "parameter",
            "frequency",
            "parameter_value"
        ],
        ascending=[
            True,
            True,
            False,
            True
        ]
    )
    .groupby(
        [
            "model",
            "parameter"
        ],
        as_index=False
    )
    .first()
)


print("\n" + "=" * 76)
print("MOST FREQUENT HYPERPARAMETERS")
print("=" * 76)

display(
    ensemble_most_frequent_parameters_df
)


# ============================================================
# 7. ANALISIS CLASS_WEIGHT RANDOM FOREST DAN EXTRA TREES
# ============================================================

class_weight_parameter = (
    "classifier__class_weight"
)

if (
    class_weight_parameter
    in ensemble_best_parameters_analysis_df.columns
):

    ensemble_class_weight_selection_df = (
        ensemble_best_parameters_analysis_df[
            ensemble_best_parameters_analysis_df[
                "model"
            ].isin([
                "Random Forest",
                "Extra Trees"
            ])
        ]
        .groupby([
            "model",
            class_weight_parameter
        ],
        dropna=False)
        .size()
        .reset_index(
            name="frequency"
        )
    )

    ensemble_class_weight_selection_df[
        "selection_percentage"
    ] = (
        ensemble_class_weight_selection_df
        .groupby("model")[
            "frequency"
        ]
        .transform(
            lambda values: (
                values
                / values.sum()
                * 100
            )
        )
    )

else:

    ensemble_class_weight_selection_df = (
        pd.DataFrame(
            columns=[
                "model",
                class_weight_parameter,
                "frequency",
                "selection_percentage"
            ]
        )
    )


print("\n" + "=" * 76)
print("CLASS-WEIGHT SELECTION")
print("=" * 76)

display(
    ensemble_class_weight_selection_df
)


# ============================================================
# 8. ANALISIS SCALE_POS_WEIGHT XGBOOST
# ============================================================

scale_pos_weight_parameter = (
    "classifier__scale_pos_weight"
)

if (
    scale_pos_weight_parameter
    in ensemble_best_parameters_analysis_df.columns
):

    ensemble_xgboost_weight_selection_df = (
        ensemble_best_parameters_analysis_df[
            ensemble_best_parameters_analysis_df[
                "model"
            ]
            == "XGBoost"
        ]
        .groupby(
            scale_pos_weight_parameter,
            dropna=False
        )
        .size()
        .reset_index(
            name="frequency"
        )
    )

    ensemble_xgboost_weight_selection_df[
        "selection_percentage"
    ] = (
        ensemble_xgboost_weight_selection_df[
            "frequency"
        ]
        / ensemble_xgboost_weight_selection_df[
            "frequency"
        ].sum()
        * 100
    )

else:

    ensemble_xgboost_weight_selection_df = (
        pd.DataFrame(
            columns=[
                scale_pos_weight_parameter,
                "frequency",
                "selection_percentage"
            ]
        )
    )


print("\n" + "=" * 76)
print("XGBOOST SCALE_POS_WEIGHT SELECTION")
print("=" * 76)

display(
    ensemble_xgboost_weight_selection_df
)


# ============================================================
# 9. RINGKASAN PERFORMA ANTAR-FOLD
# ============================================================

ensemble_fold_performance_summary_df = (
    ensemble_fold_metrics_analysis_df
    .groupby("model")
    .agg(
        mean_accuracy=(
            "accuracy",
            "mean"
        ),
        std_accuracy=(
            "accuracy",
            "std"
        ),
        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean"
        ),
        std_balanced_accuracy=(
            "balanced_accuracy",
            "std"
        ),
        mean_sensitivity=(
            "sensitivity",
            "mean"
        ),
        std_sensitivity=(
            "sensitivity",
            "std"
        ),
        mean_specificity=(
            "specificity",
            "mean"
        ),
        std_specificity=(
            "specificity",
            "std"
        ),
        mean_f1_score=(
            "f1_score",
            "mean"
        ),
        std_f1_score=(
            "f1_score",
            "std"
        ),
        mean_roc_auc=(
            "roc_auc",
            "mean"
        ),
        std_roc_auc=(
            "roc_auc",
            "std"
        ),
        min_roc_auc=(
            "roc_auc",
            "min"
        ),
        max_roc_auc=(
            "roc_auc",
            "max"
        ),
        mean_training_auc=(
            "training_roc_auc",
            "mean"
        ),
        std_training_auc=(
            "training_roc_auc",
            "std"
        ),
        mean_train_test_auc_gap=(
            "train_test_auc_gap",
            "mean"
        ),
        max_train_test_auc_gap=(
            "train_test_auc_gap",
            "max"
        ),
        mean_best_inner_auc=(
            "best_inner_auc",
            "mean"
        ),
        std_best_inner_auc=(
            "best_inner_auc",
            "std"
        )
    )
    .reset_index()
)


# ============================================================
# 10. COEFFICIENT OF VARIATION ROC-AUC
# ============================================================

ensemble_fold_performance_summary_df[
    "roc_auc_coefficient_of_variation"
] = np.where(
    ensemble_fold_performance_summary_df[
        "mean_roc_auc"
    ].abs() > 1e-12,

    ensemble_fold_performance_summary_df[
        "std_roc_auc"
    ]
    / ensemble_fold_performance_summary_df[
        "mean_roc_auc"
    ].abs(),

    np.nan
)


# ============================================================
# 11. KATEGORI STABILITAS MODEL
# ============================================================

def classify_ensemble_stability(
    roc_auc_std
):
    """
    Mengategorikan kestabilan berdasarkan SD outer-fold AUC.
    """

    if pd.isna(
        roc_auc_std
    ):
        return "UNDEFINED"

    if roc_auc_std <= 0.10:
        return "HIGH"

    if roc_auc_std <= 0.20:
        return "MODERATE"

    return "LOW"


ensemble_fold_performance_summary_df[
    "stability_category"
] = (
    ensemble_fold_performance_summary_df[
        "std_roc_auc"
    ]
    .apply(
        classify_ensemble_stability
    )
)


# ============================================================
# 12. KATEGORI OVERFITTING
# ============================================================

def classify_ensemble_overfitting(
    auc_gap
):
    """
    Mengategorikan indikasi overfitting berdasarkan
    mean training-test ROC-AUC gap.
    """

    if pd.isna(
        auc_gap
    ):
        return "UNDEFINED"

    if auc_gap <= 0.10:
        return "LOW"

    if auc_gap <= 0.20:
        return "MODERATE"

    if auc_gap <= 0.30:
        return "HIGH"

    return "VERY HIGH"


ensemble_fold_performance_summary_df[
    "overfitting_category"
] = (
    ensemble_fold_performance_summary_df[
        "mean_train_test_auc_gap"
    ]
    .apply(
        classify_ensemble_overfitting
    )
)


# ============================================================
# 13. MENGGABUNGKAN DENGAN HASIL OOF
# ============================================================

ensemble_stability_summary_df = (
    ensemble_model_summary_analysis_df[
        [
            "performance_rank",
            "model",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "sensitivity",
            "specificity",
            "f1_score",
            "roc_auc"
        ]
    ]
    .merge(
        ensemble_fold_performance_summary_df,
        on="model",
        how="left"
    )
    .sort_values(
        by="performance_rank"
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 76)
print("ENSEMBLE FOLD PERFORMANCE AND STABILITY")
print("=" * 76)

display(
    ensemble_stability_summary_df[
        [
            "performance_rank",
            "model",
            "roc_auc",
            "mean_roc_auc",
            "std_roc_auc",
            "min_roc_auc",
            "max_roc_auc",
            "mean_training_auc",
            "mean_train_test_auc_gap",
            "stability_category",
            "overfitting_category"
        ]
    ]
)


# ============================================================
# 14. HASIL PER OUTER FOLD
# ============================================================

ensemble_outer_fold_auc_pivot_df = (
    ensemble_fold_metrics_analysis_df
    .pivot(
        index="outer_fold",
        columns="model",
        values="roc_auc"
    )
    .sort_index()
)


ensemble_outer_fold_balanced_accuracy_pivot_df = (
    ensemble_fold_metrics_analysis_df
    .pivot(
        index="outer_fold",
        columns="model",
        values="balanced_accuracy"
    )
    .sort_index()
)


print("\n" + "=" * 76)
print("OUTER-FOLD ROC-AUC MATRIX")
print("=" * 76)

display(
    ensemble_outer_fold_auc_pivot_df
)


print("\n" + "=" * 76)
print("OUTER-FOLD BALANCED ACCURACY MATRIX")
print("=" * 76)

display(
    ensemble_outer_fold_balanced_accuracy_pivot_df
)


# ============================================================
# 15. ANALISIS KESTABILAN HYPERPARAMETER
# ============================================================

ensemble_parameter_stability_records = []

for model_name in ensemble_pipelines.keys():

    model_parameter_frequency_df = (
        ensemble_parameter_frequency_df[
            ensemble_parameter_frequency_df[
                "model"
            ]
            == model_name
        ]
    )

    available_parameters = (
        model_parameter_frequency_df[
            "parameter"
        ]
        .unique()
    )

    for parameter_name in available_parameters:

        parameter_selection_df = (
            model_parameter_frequency_df[
                model_parameter_frequency_df[
                    "parameter"
                ]
                == parameter_name
            ]
        )

        maximum_frequency = int(
            parameter_selection_df[
                "frequency"
            ].max()
        )

        number_of_unique_values = int(
            len(
                parameter_selection_df
            )
        )

        modal_selection_percentage = float(
            maximum_frequency
            / OUTER_N_SPLITS
            * 100
        )

        if modal_selection_percentage >= 80:
            stability_label = "HIGH"

        elif modal_selection_percentage >= 60:
            stability_label = "MODERATE"

        else:
            stability_label = "LOW"

        ensemble_parameter_stability_records.append({
            "model": model_name,
            "parameter": parameter_name,
            "number_of_unique_selected_values": (
                number_of_unique_values
            ),
            "modal_frequency": (
                maximum_frequency
            ),
            "modal_selection_percentage": (
                modal_selection_percentage
            ),
            "parameter_stability": (
                stability_label
            )
        })


ensemble_parameter_stability_df = pd.DataFrame(
    ensemble_parameter_stability_records
)


print("\n" + "=" * 76)
print("HYPERPARAMETER STABILITY")
print("=" * 76)

display(
    ensemble_parameter_stability_df
)


# ============================================================
# 16. PARAMETER TERPILIH MODEL TERBAIK
# ============================================================

best_ensemble_parameter_records_df = (
    ensemble_best_parameters_analysis_df[
        ensemble_best_parameters_analysis_df[
            "model"
        ]
        == best_ensemble_model_name
    ]
    .sort_values(
        by="outer_fold"
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 76)
print("BEST MODEL PARAMETERS ACROSS OUTER FOLDS")
print("=" * 76)

print(
    "Best model:",
    best_ensemble_model_name
)

display(
    best_ensemble_parameter_records_df
)


# ============================================================
# 17. MODEL DENGAN OVERFITTING TERKECIL
# ============================================================

lowest_overfitting_model_row = (
    ensemble_stability_summary_df
    .sort_values(
        by=[
            "mean_train_test_auc_gap",
            "std_roc_auc"
        ],
        ascending=[
            True,
            True
        ]
    )
    .iloc[0]
)


most_stable_model_row = (
    ensemble_stability_summary_df
    .sort_values(
        by=[
            "std_roc_auc",
            "mean_train_test_auc_gap"
        ],
        ascending=[
            True,
            True
        ]
    )
    .iloc[0]
)


print("\n" + "=" * 76)
print("STABILITY HIGHLIGHTS")
print("=" * 76)

print(
    "Best OOF ROC-AUC model       :",
    best_ensemble_model_name
)

print(
    "Most stable outer-fold model :",
    most_stable_model_row[
        "model"
    ]
)

print(
    "Lowest train-test gap model  :",
    lowest_overfitting_model_row[
        "model"
    ]
)


# ============================================================
# 18. VISUALISASI ROC-AUC PER OUTER FOLD
# ============================================================

plt.figure(
    figsize=(11, 6)
)

for model_name in (
    ensemble_outer_fold_auc_pivot_df.columns
):

    plt.plot(
        ensemble_outer_fold_auc_pivot_df.index,
        ensemble_outer_fold_auc_pivot_df[
            model_name
        ],
        marker="o",
        linewidth=2,
        label=(
            ensemble_model_abbreviations[
                model_name
            ]
        )
    )


plt.xlabel(
    "Outer Fold"
)

plt.ylabel(
    "ROC-AUC"
)

plt.title(
    "Pure Ensemble ROC-AUC Across Outer Folds"
)

plt.xticks(
    ensemble_outer_fold_auc_pivot_df.index
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    alpha=0.25
)

plt.legend(
    title="Model",
    loc="best"
)

plt.tight_layout()


ensemble_fold_auc_figure_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "ensemble_outer_fold_roc_auc_stability.png"
)

plt.savefig(
    ensemble_fold_auc_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 19. VISUALISASI TRAINING DAN TEST AUC
# ============================================================

training_test_plot_df = (
    ensemble_fold_performance_summary_df[
        [
            "model",
            "mean_training_auc",
            "mean_roc_auc"
        ]
    ]
    .set_index(
        "model"
    )
)

training_test_plot_df.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.xlabel(
    "Ensemble Model"
)

plt.ylabel(
    "Mean ROC-AUC"
)

plt.title(
    "Training versus Outer-Fold ROC-AUC"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.legend([
    "Training AUC",
    "Outer-Fold AUC"
])

plt.tight_layout()


ensemble_training_test_figure_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "ensemble_training_vs_outer_auc.png"
)

plt.savefig(
    ensemble_training_test_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 20. VISUALISASI TRAIN-TEST AUC GAP
# ============================================================

auc_gap_plot_df = (
    ensemble_fold_performance_summary_df[
        [
            "model",
            "mean_train_test_auc_gap"
        ]
    ]
    .sort_values(
        by="mean_train_test_auc_gap",
        ascending=True
    )
)


plt.figure(
    figsize=(10, 5.5)
)

plt.bar(
    auc_gap_plot_df[
        "model"
    ],
    auc_gap_plot_df[
        "mean_train_test_auc_gap"
    ]
)

plt.xlabel(
    "Ensemble Model"
)

plt.ylabel(
    "Mean Training–Test ROC-AUC Gap"
)

plt.title(
    "Overfitting Gap of Pure Ensemble Models"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()


ensemble_auc_gap_figure_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "ensemble_mean_train_test_auc_gap.png"
)

plt.savefig(
    ensemble_auc_gap_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 21. MENYIMPAN HASIL ANALISIS
# ============================================================

ensemble_parameter_frequency_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_hyperparameter_frequency.csv"
)

ensemble_parameter_frequency_df.to_csv(
    ensemble_parameter_frequency_path,
    index=False
)


ensemble_most_frequent_parameters_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_most_frequent_hyperparameters.csv"
)

ensemble_most_frequent_parameters_df.to_csv(
    ensemble_most_frequent_parameters_path,
    index=False
)


ensemble_parameter_stability_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_hyperparameter_stability.csv"
)

ensemble_parameter_stability_df.to_csv(
    ensemble_parameter_stability_path,
    index=False
)


ensemble_class_weight_selection_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_class_weight_selection.csv"
)

ensemble_class_weight_selection_df.to_csv(
    ensemble_class_weight_selection_path,
    index=False
)


ensemble_xgboost_weight_selection_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_xgboost_scale_pos_weight_selection.csv"
)

ensemble_xgboost_weight_selection_df.to_csv(
    ensemble_xgboost_weight_selection_path,
    index=False
)


ensemble_fold_performance_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_fold_performance_stability.csv"
)

ensemble_fold_performance_summary_df.to_csv(
    ensemble_fold_performance_summary_path,
    index=False
)


ensemble_stability_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_model_stability_summary.csv"
)

ensemble_stability_summary_df.to_csv(
    ensemble_stability_summary_path,
    index=False
)


ensemble_fold_auc_pivot_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_outer_fold_roc_auc_pivot.csv"
)

ensemble_outer_fold_auc_pivot_df.to_csv(
    ensemble_fold_auc_pivot_path
)


ensemble_fold_balanced_accuracy_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_outer_fold_balanced_accuracy_pivot.csv"
)

ensemble_outer_fold_balanced_accuracy_pivot_df.to_csv(
    ensemble_fold_balanced_accuracy_path
)


best_ensemble_parameters_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "best_ensemble_model_parameters_by_fold.csv"
)

best_ensemble_parameter_records_df.to_csv(
    best_ensemble_parameters_path,
    index=False
)


# ============================================================
# 22. KONFIRMASI STEP 6
# ============================================================

print("\n" + "=" * 76)
print("STEP 6 COMPLETED")
print("=" * 76)

print(
    "Analisis hyperparameter, class imbalance, "
    "stabilitas outer fold, dan overfitting berhasil dilakukan."
)

print(
    "\nBest OOF model:"
)

print(
    best_ensemble_model_name
)

print(
    "\nMost stable model:"
)

print(
    most_stable_model_row[
        "model"
    ]
)

print(
    "\nModel with lowest train-test AUC gap:"
)

print(
    lowest_overfitting_model_row[
        "model"
    ]
)

print(
    "\nSTEP berikutnya:"
)

print(
    "STEP 7 — Generate ROC curves, Precision–Recall curves, "
    "confusion matrices, and ensemble performance plots."
)

In [ ]:
# ============================================================
# STEP 7. VISUALIZE PURE ENSEMBLE MODEL PERFORMANCE
# ROC Curves, Precision–Recall Curves, Confusion Matrices,
# and Comparative Performance Plots
# ============================================================

# Tujuan STEP 7:
# 1. Membuat ROC curve seluruh model ensemble.
# 2. Membuat Precision–Recall curve.
# 3. Membuat confusion matrix setiap model.
# 4. Membuat confusion matrix khusus model terbaik.
# 5. Membandingkan accuracy, balanced accuracy, sensitivity,
#    specificity, F1-score, dan ROC-AUC.
# 6. Menyimpan seluruh gambar dan tabel visualisasi.
#
# Seluruh visualisasi menggunakan out-of-fold prediction
# dari outer nested cross-validation.
#
# Setiap subject diprediksi hanya ketika berada pada
# outer test fold. Oleh karena itu, confusion matrix dan
# ROC-AUC pada tahap ini bukan hasil training-set prediction.


# ============================================================
# 1. VALIDASI OBJEK HASIL STEP 5 DAN STEP 6
# ============================================================

required_step_7_objects = [
    "ensemble_nested_cv_results",
    "ensemble_nested_model_summary_df",
    "ensemble_nested_predictions_df",
    "ensemble_nested_fold_metrics_df",
    "ensemble_pipelines",
    "ensemble_model_abbreviations",
    "best_ensemble_model_name",
    "best_ensemble_model_auc",
    "CLASS_LABEL_MAPPING",
    "ensemble_y",
    "ENSEMBLE_FIGURE_DIR",
    "ENSEMBLE_TABLE_DIR"
]

missing_step_7_objects = [
    object_name
    for object_name in required_step_7_objects
    if object_name not in globals()
]

if missing_step_7_objects:
    raise NameError(
        "Objek untuk STEP 7 belum lengkap: "
        f"{missing_step_7_objects}. "
        "Jalankan STEP 5 dan STEP 6 terlebih dahulu."
    )


# ============================================================
# 2. IMPORT LIBRARY VISUALISASI DAN METRIK
# ============================================================

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)


# ============================================================
# 3. KONFIGURASI GAMBAR
# ============================================================

ENSEMBLE_FIGURE_DPI = 300

ENSEMBLE_CLASS_IDS = [
    0,
    1
]

ENSEMBLE_CLASS_NAMES = [
    CLASS_LABEL_MAPPING[0],
    CLASS_LABEL_MAPPING[1]
]

os.makedirs(
    ENSEMBLE_FIGURE_DIR,
    exist_ok=True
)

os.makedirs(
    ENSEMBLE_TABLE_DIR,
    exist_ok=True
)


# ============================================================
# 4. VALIDASI OUT-OF-FOLD RESULTS SETIAP MODEL
# ============================================================

for model_name in ensemble_pipelines.keys():

    if model_name not in ensemble_nested_cv_results:
        raise KeyError(
            f"Hasil nested CV untuk {model_name} "
            "tidak ditemukan."
        )

    model_result = (
        ensemble_nested_cv_results[
            model_name
        ]
    )

    required_result_keys = [
        "y_true",
        "y_pred",
        "y_score"
    ]

    missing_result_keys = [
        key_name
        for key_name in required_result_keys
        if key_name not in model_result
    ]

    if missing_result_keys:
        raise KeyError(
            f"Hasil {model_name} belum memiliki key: "
            f"{missing_result_keys}"
        )

    model_y_true = np.asarray(
        model_result["y_true"],
        dtype=int
    )

    model_y_pred = np.asarray(
        model_result["y_pred"],
        dtype=int
    )

    model_y_score = np.asarray(
        model_result["y_score"],
        dtype=float
    )

    if not (
        len(model_y_true)
        == len(model_y_pred)
        == len(model_y_score)
        == len(ensemble_y)
    ):
        raise ValueError(
            f"Jumlah hasil OOF {model_name} tidak sesuai "
            "dengan jumlah subject."
        )

    if not np.isfinite(
        model_y_score
    ).all():
        raise ValueError(
            f"Skor prediksi {model_name} mengandung "
            "NaN atau infinite."
        )

    if (
        np.any(model_y_score < 0.0)
        or np.any(model_y_score > 1.0)
    ):
        raise ValueError(
            f"Probabilitas {model_name} berada di luar "
            "rentang [0, 1]."
        )


# ============================================================
# 5. COMBINED ROC CURVES
# ============================================================

ensemble_roc_curve_records = []

plt.figure(
    figsize=(9, 7)
)

for model_name in ensemble_pipelines.keys():

    model_result = (
        ensemble_nested_cv_results[
            model_name
        ]
    )

    model_y_true = np.asarray(
        model_result["y_true"],
        dtype=int
    )

    model_y_score = np.asarray(
        model_result["y_score"],
        dtype=float
    )

    false_positive_rate, true_positive_rate, (
        roc_thresholds
    ) = roc_curve(
        model_y_true,
        model_y_score,
        pos_label=1
    )

    model_roc_auc = float(
        roc_auc_score(
            model_y_true,
            model_y_score
        )
    )

    plt.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        marker="o",
        markersize=3,
        label=(
            f"{ensemble_model_abbreviations[model_name]} "
            f"(AUC = {model_roc_auc:.3f})"
        )
    )

    for point_index in range(
        len(false_positive_rate)
    ):

        ensemble_roc_curve_records.append({
            "model": model_name,
            "model_abbreviation": (
                ensemble_model_abbreviations[
                    model_name
                ]
            ),
            "point_index": int(
                point_index
            ),
            "false_positive_rate": float(
                false_positive_rate[
                    point_index
                ]
            ),
            "true_positive_rate": float(
                true_positive_rate[
                    point_index
                ]
            ),
            "threshold": float(
                roc_thresholds[
                    point_index
                ]
            ),
            "roc_auc": model_roc_auc
        })


plt.plot(
    [
        0,
        1
    ],
    [
        0,
        1
    ],
    linestyle="--",
    linewidth=1.5,
    label="Chance Level"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curves of Pure Ensemble Models"
)

plt.xlim(
    0.0,
    1.0
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    alpha=0.25
)

plt.legend(
    loc="lower right"
)

plt.tight_layout()


ensemble_combined_roc_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "ensemble_combined_nested_cv_roc_curves.png"
)

plt.savefig(
    ensemble_combined_roc_path,
    dpi=ENSEMBLE_FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


ensemble_roc_curve_df = pd.DataFrame(
    ensemble_roc_curve_records
)


# ============================================================
# 6. COMBINED PRECISION–RECALL CURVES
# ============================================================

ensemble_pr_curve_records = []

positive_class_prevalence = float(
    np.mean(
        np.asarray(
            ensemble_y,
            dtype=int
        )
        == 1
    )
)

plt.figure(
    figsize=(9, 7)
)

for model_name in ensemble_pipelines.keys():

    model_result = (
        ensemble_nested_cv_results[
            model_name
        ]
    )

    model_y_true = np.asarray(
        model_result["y_true"],
        dtype=int
    )

    model_y_score = np.asarray(
        model_result["y_score"],
        dtype=float
    )

    precision_values, recall_values, (
        pr_thresholds
    ) = precision_recall_curve(
        model_y_true,
        model_y_score,
        pos_label=1
    )

    average_precision = float(
        average_precision_score(
            model_y_true,
            model_y_score
        )
    )

    plt.plot(
        recall_values,
        precision_values,
        linewidth=2,
        marker="o",
        markersize=3,
        label=(
            f"{ensemble_model_abbreviations[model_name]} "
            f"(AP = {average_precision:.3f})"
        )
    )

    for point_index in range(
        len(precision_values)
    ):

        if point_index < len(
            pr_thresholds
        ):
            threshold_value = float(
                pr_thresholds[
                    point_index
                ]
            )
        else:
            threshold_value = np.nan

        ensemble_pr_curve_records.append({
            "model": model_name,
            "model_abbreviation": (
                ensemble_model_abbreviations[
                    model_name
                ]
            ),
            "point_index": int(
                point_index
            ),
            "recall": float(
                recall_values[
                    point_index
                ]
            ),
            "precision": float(
                precision_values[
                    point_index
                ]
            ),
            "threshold": (
                threshold_value
            ),
            "average_precision": (
                average_precision
            )
        })


plt.axhline(
    y=positive_class_prevalence,
    linestyle="--",
    linewidth=1.5,
    label=(
        "HIGH-Class Prevalence "
        f"({positive_class_prevalence:.3f})"
    )
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Precision–Recall Curves of Pure Ensemble Models"
)

plt.xlim(
    0.0,
    1.0
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    alpha=0.25
)

plt.legend(
    loc="best"
)

plt.tight_layout()


ensemble_combined_pr_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "ensemble_combined_precision_recall_curves.png"
)

plt.savefig(
    ensemble_combined_pr_path,
    dpi=ENSEMBLE_FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


ensemble_pr_curve_df = pd.DataFrame(
    ensemble_pr_curve_records
)


# ============================================================
# 7. CONFUSION MATRIX SETIAP MODEL
# ============================================================

ensemble_confusion_matrix_records = []

ensemble_classification_report_frames = []

ensemble_confusion_figure_records = []


for model_name in ensemble_pipelines.keys():

    model_result = (
        ensemble_nested_cv_results[
            model_name
        ]
    )

    model_y_true = np.asarray(
        model_result["y_true"],
        dtype=int
    )

    model_y_pred = np.asarray(
        model_result["y_pred"],
        dtype=int
    )

    model_confusion_matrix = confusion_matrix(
        model_y_true,
        model_y_pred,
        labels=ENSEMBLE_CLASS_IDS
    )

    tn, fp, fn, tp = (
        model_confusion_matrix
        .ravel()
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    precision_high = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    negative_predictive_value = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    ensemble_confusion_matrix_records.append({
        "model": model_name,
        "model_abbreviation": (
            ensemble_model_abbreviations[
                model_name
            ]
        ),
        "true_negative": int(
            tn
        ),
        "false_positive": int(
            fp
        ),
        "false_negative": int(
            fn
        ),
        "true_positive": int(
            tp
        ),
        "sensitivity": float(
            sensitivity
        ),
        "specificity": float(
            specificity
        ),
        "precision_HIGH": float(
            precision_high
        ),
        "negative_predictive_value": float(
            negative_predictive_value
        ),
        "correct_predictions": int(
            tn + tp
        ),
        "incorrect_predictions": int(
            fp + fn
        ),
        "total_subjects": int(
            model_confusion_matrix.sum()
        )
    })


    # ========================================================
    # Classification report
    # ========================================================

    classification_report_dictionary = (
        classification_report(
            model_y_true,
            model_y_pred,
            labels=ENSEMBLE_CLASS_IDS,
            target_names=ENSEMBLE_CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )
    )

    model_classification_report_df = (
        pd.DataFrame(
            classification_report_dictionary
        )
        .transpose()
        .reset_index()
        .rename(
            columns={
                "index": "class_or_metric"
            }
        )
    )

    model_classification_report_df.insert(
        0,
        "model",
        model_name
    )

    ensemble_classification_report_frames.append(
        model_classification_report_df
    )


    # ========================================================
    # Plot confusion matrix
    # ========================================================

    figure, axis = plt.subplots(
        figsize=(6, 5)
    )

    confusion_display = ConfusionMatrixDisplay(
        confusion_matrix=(
            model_confusion_matrix
        ),
        display_labels=(
            ENSEMBLE_CLASS_NAMES
        )
    )

    confusion_display.plot(
        ax=axis,
        values_format="d",
        colorbar=False
    )

    axis.set_title(
        f"Nested CV Confusion Matrix—{model_name}"
    )

    axis.set_xlabel(
        "Predicted Fall-Risk Class"
    )

    axis.set_ylabel(
        "Actual Fall-Risk Class"
    )

    plt.tight_layout()


    safe_model_name = (
        model_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )

    model_confusion_figure_path = os.path.join(
        ENSEMBLE_FIGURE_DIR,
        (
            "ensemble_nested_cv_confusion_matrix_"
            f"{safe_model_name}.png"
        )
    )

    plt.savefig(
        model_confusion_figure_path,
        dpi=ENSEMBLE_FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()

    ensemble_confusion_figure_records.append({
        "model": model_name,
        "figure_path": (
            model_confusion_figure_path
        )
    })


ensemble_confusion_matrix_summary_df = pd.DataFrame(
    ensemble_confusion_matrix_records
)

ensemble_classification_reports_df = pd.concat(
    ensemble_classification_report_frames,
    ignore_index=True
)

ensemble_confusion_figure_paths_df = pd.DataFrame(
    ensemble_confusion_figure_records
)


# ============================================================
# 8. NORMALIZED CONFUSION MATRIX SETIAP MODEL
# ============================================================

for model_name in ensemble_pipelines.keys():

    model_result = (
        ensemble_nested_cv_results[
            model_name
        ]
    )

    model_y_true = np.asarray(
        model_result["y_true"],
        dtype=int
    )

    model_y_pred = np.asarray(
        model_result["y_pred"],
        dtype=int
    )

    normalized_confusion_matrix = confusion_matrix(
        model_y_true,
        model_y_pred,
        labels=ENSEMBLE_CLASS_IDS,
        normalize="true"
    )

    figure, axis = plt.subplots(
        figsize=(6, 5)
    )

    normalized_display = ConfusionMatrixDisplay(
        confusion_matrix=(
            normalized_confusion_matrix
        ),
        display_labels=(
            ENSEMBLE_CLASS_NAMES
        )
    )

    normalized_display.plot(
        ax=axis,
        values_format=".2f",
        colorbar=False
    )

    axis.set_title(
        (
            "Normalized Nested CV Confusion Matrix—"
            f"{model_name}"
        )
    )

    axis.set_xlabel(
        "Predicted Fall-Risk Class"
    )

    axis.set_ylabel(
        "Actual Fall-Risk Class"
    )

    plt.tight_layout()


    safe_model_name = (
        model_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            "_"
        )
    )

    normalized_confusion_path = os.path.join(
        ENSEMBLE_FIGURE_DIR,
        (
            "ensemble_normalized_confusion_matrix_"
            f"{safe_model_name}.png"
        )
    )

    plt.savefig(
        normalized_confusion_path,
        dpi=ENSEMBLE_FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# 9. CONFUSION MATRIX MODEL TERBAIK
# ============================================================

best_ensemble_result = (
    ensemble_nested_cv_results[
        best_ensemble_model_name
    ]
)

best_ensemble_y_true = np.asarray(
    best_ensemble_result[
        "y_true"
    ],
    dtype=int
)

best_ensemble_y_pred = np.asarray(
    best_ensemble_result[
        "y_pred"
    ],
    dtype=int
)

best_ensemble_y_score = np.asarray(
    best_ensemble_result[
        "y_score"
    ],
    dtype=float
)


best_ensemble_confusion_matrix = confusion_matrix(
    best_ensemble_y_true,
    best_ensemble_y_pred,
    labels=ENSEMBLE_CLASS_IDS
)

best_tn, best_fp, best_fn, best_tp = (
    best_ensemble_confusion_matrix
    .ravel()
)


best_ensemble_confusion_df = pd.DataFrame(
    best_ensemble_confusion_matrix,
    index=[
        "Actual LOW/NO",
        "Actual HIGH"
    ],
    columns=[
        "Predicted LOW/NO",
        "Predicted HIGH"
    ]
)


print("=" * 76)
print("BEST ENSEMBLE MODEL CONFUSION MATRIX")
print("=" * 76)

print(
    "Best model:",
    best_ensemble_model_name
)

display(
    best_ensemble_confusion_df
)


# ============================================================
# 10. PLOT CONFUSION MATRIX MODEL TERBAIK
# ============================================================

figure, axis = plt.subplots(
    figsize=(6.5, 5.5)
)

best_confusion_display = ConfusionMatrixDisplay(
    confusion_matrix=(
        best_ensemble_confusion_matrix
    ),
    display_labels=(
        ENSEMBLE_CLASS_NAMES
    )
)

best_confusion_display.plot(
    ax=axis,
    values_format="d",
    colorbar=False
)

axis.set_title(
    (
        "Best Pure Ensemble Model Confusion Matrix\n"
        f"{best_ensemble_model_name}"
    )
)

axis.set_xlabel(
    "Predicted Fall-Risk Class"
)

axis.set_ylabel(
    "Actual Fall-Risk Class"
)

plt.tight_layout()


best_ensemble_confusion_figure_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "best_ensemble_nested_cv_confusion_matrix.png"
)

plt.savefig(
    best_ensemble_confusion_figure_path,
    dpi=ENSEMBLE_FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 11. CLASSIFICATION REPORT MODEL TERBAIK
# ============================================================

best_ensemble_classification_report_dictionary = (
    classification_report(
        best_ensemble_y_true,
        best_ensemble_y_pred,
        labels=ENSEMBLE_CLASS_IDS,
        target_names=ENSEMBLE_CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )
)

best_ensemble_classification_report_df = (
    pd.DataFrame(
        best_ensemble_classification_report_dictionary
    )
    .transpose()
)


print("\n" + "=" * 76)
print("BEST ENSEMBLE MODEL CLASSIFICATION REPORT")
print("=" * 76)

display(
    best_ensemble_classification_report_df
)

print(
    classification_report(
        best_ensemble_y_true,
        best_ensemble_y_pred,
        labels=ENSEMBLE_CLASS_IDS,
        target_names=ENSEMBLE_CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 12. INTERPRETASI CONFUSION MATRIX MODEL TERBAIK
# ============================================================

best_ensemble_sensitivity = (
    best_tp / (
        best_tp + best_fn
    )
    if (
        best_tp + best_fn
    ) > 0
    else np.nan
)

best_ensemble_specificity = (
    best_tn / (
        best_tn + best_fp
    )
    if (
        best_tn + best_fp
    ) > 0
    else np.nan
)

best_ensemble_precision = (
    best_tp / (
        best_tp + best_fp
    )
    if (
        best_tp + best_fp
    ) > 0
    else np.nan
)

best_ensemble_accuracy_from_cm = (
    (
        best_tn + best_tp
    )
    / best_ensemble_confusion_matrix.sum()
)

best_ensemble_balanced_accuracy_from_cm = (
    (
        best_ensemble_sensitivity
        + best_ensemble_specificity
    )
    / 2.0
)


print("\n" + "=" * 76)
print("BEST ENSEMBLE CONFUSION MATRIX INTERPRETATION")
print("=" * 76)

print(
    "True Negative  :",
    best_tn,
    "LOW/NO subjects correctly predicted"
)

print(
    "False Positive :",
    best_fp,
    "LOW/NO subjects predicted as HIGH"
)

print(
    "False Negative :",
    best_fn,
    "HIGH subjects predicted as LOW/NO"
)

print(
    "True Positive  :",
    best_tp,
    "HIGH subjects correctly predicted"
)

print(
    "\nAccuracy          :",
    f"{best_ensemble_accuracy_from_cm:.4f}"
)

print(
    "Balanced accuracy :",
    f"{best_ensemble_balanced_accuracy_from_cm:.4f}"
)

print(
    "Sensitivity        :",
    f"{best_ensemble_sensitivity:.4f}"
)

print(
    "Specificity        :",
    f"{best_ensemble_specificity:.4f}"
)

print(
    "Precision          :",
    f"{best_ensemble_precision:.4f}"
)

print(
    "ROC-AUC            :",
    f"{best_ensemble_model_auc:.4f}"
)


# ============================================================
# 13. PERFORMANCE COMPARISON TABLE
# ============================================================

ensemble_performance_comparison_df = (
    ensemble_nested_model_summary_df[
        [
            "performance_rank",
            "model",
            "model_abbreviation",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "sensitivity",
            "specificity",
            "f1_score",
            "roc_auc",
            "mean_training_auc",
            "mean_train_test_auc_gap"
        ]
    ]
    .copy()
)


print("\n" + "=" * 76)
print("PURE ENSEMBLE PERFORMANCE COMPARISON")
print("=" * 76)

display(
    ensemble_performance_comparison_df
)


# ============================================================
# 14. BAR CHART SETIAP METRIK
# ============================================================

ensemble_performance_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1_score",
    "roc_auc"
]


for metric_name in ensemble_performance_metrics:

    metric_plot_df = (
        ensemble_nested_model_summary_df[
            [
                "model",
                metric_name
            ]
        ]
        .sort_values(
            by=metric_name,
            ascending=False
        )
        .copy()
    )

    plt.figure(
        figsize=(10, 5.5)
    )

    plt.bar(
        metric_plot_df[
            "model"
        ],
        metric_plot_df[
            metric_name
        ]
    )

    plt.xlabel(
        "Ensemble Model"
    )

    plt.ylabel(
        metric_name
        .replace(
            "_",
            " "
        )
        .title()
    )

    plt.title(
        (
            "Nested Cross-Validation "
            f"{metric_name.replace('_', ' ').title()}"
        )
    )

    plt.xticks(
        rotation=25,
        ha="right"
    )

    plt.ylim(
        0.0,
        1.05
    )

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.tight_layout()


    metric_figure_path = os.path.join(
        ENSEMBLE_FIGURE_DIR,
        (
            "ensemble_model_comparison_"
            f"{metric_name}.png"
        )
    )

    plt.savefig(
        metric_figure_path,
        dpi=ENSEMBLE_FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# 15. COMBINED PERFORMANCE METRICS
# ============================================================

selected_combined_metrics = [
    "accuracy",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "f1_score",
    "roc_auc"
]

combined_performance_df = (
    ensemble_nested_model_summary_df[
        [
            "model"
        ]
        + selected_combined_metrics
    ]
    .set_index(
        "model"
    )
)


combined_performance_df.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.xlabel(
    "Ensemble Model"
)

plt.ylabel(
    "Performance Score"
)

plt.title(
    "Performance Comparison of Pure Ensemble Models"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.legend(
    title="Metric",
    loc="best"
)

plt.tight_layout()


ensemble_combined_performance_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "ensemble_combined_performance_metrics.png"
)

plt.savefig(
    ensemble_combined_performance_path,
    dpi=ENSEMBLE_FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 16. OUTER-FOLD ROC-AUC VARIABILITY
# ============================================================

ensemble_fold_auc_pivot_df = (
    ensemble_nested_fold_metrics_df
    .pivot(
        index="outer_fold",
        columns="model",
        values="roc_auc"
    )
    .sort_index()
)


plt.figure(
    figsize=(11, 6)
)

for model_name in (
    ensemble_fold_auc_pivot_df.columns
):

    plt.plot(
        ensemble_fold_auc_pivot_df.index,
        ensemble_fold_auc_pivot_df[
            model_name
        ],
        marker="o",
        linewidth=2,
        label=(
            ensemble_model_abbreviations[
                model_name
            ]
        )
    )

plt.xlabel(
    "Outer Fold"
)

plt.ylabel(
    "ROC-AUC"
)

plt.title(
    "ROC-AUC Variability Across Outer Folds"
)

plt.xticks(
    ensemble_fold_auc_pivot_df.index
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    alpha=0.25
)

plt.legend(
    title="Model",
    loc="best"
)

plt.tight_layout()


ensemble_fold_auc_variability_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "ensemble_outer_fold_roc_auc_variability.png"
)

plt.savefig(
    ensemble_fold_auc_variability_path,
    dpi=ENSEMBLE_FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 17. OUTER-FOLD ROC-AUC BOXPLOT
# ============================================================

ensemble_boxplot_data = []

ensemble_boxplot_labels = []

for model_name in ensemble_pipelines.keys():

    model_fold_auc = (
        ensemble_nested_fold_metrics_df.loc[
            ensemble_nested_fold_metrics_df[
                "model"
            ]
            == model_name,
            "roc_auc"
        ]
        .dropna()
        .to_numpy()
    )

    ensemble_boxplot_data.append(
        model_fold_auc
    )

    ensemble_boxplot_labels.append(
        ensemble_model_abbreviations[
            model_name
        ]
    )


plt.figure(
    figsize=(10, 6)
)

plt.boxplot(
    ensemble_boxplot_data,
    tick_labels=(
        ensemble_boxplot_labels
    )
)

plt.xlabel(
    "Ensemble Model"
)

plt.ylabel(
    "Outer-Fold ROC-AUC"
)

plt.title(
    "Distribution of ROC-AUC Across Outer Folds"
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()


ensemble_fold_auc_boxplot_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "ensemble_outer_fold_roc_auc_boxplot.png"
)

plt.savefig(
    ensemble_fold_auc_boxplot_path,
    dpi=ENSEMBLE_FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 18. TRAINING AUC VERSUS OOF AUC
# ============================================================

ensemble_training_oof_comparison_df = (
    ensemble_nested_model_summary_df[
        [
            "model",
            "mean_training_auc",
            "roc_auc"
        ]
    ]
    .set_index(
        "model"
    )
)

ensemble_training_oof_comparison_df.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.xlabel(
    "Ensemble Model"
)

plt.ylabel(
    "ROC-AUC"
)

plt.title(
    "Mean Training ROC-AUC versus Out-of-Fold ROC-AUC"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.legend(
    [
        "Mean Training AUC",
        "Out-of-Fold AUC"
    ]
)

plt.tight_layout()


ensemble_training_oof_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "ensemble_training_vs_oof_roc_auc.png"
)

plt.savefig(
    ensemble_training_oof_path,
    dpi=ENSEMBLE_FIGURE_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 19. MENYIMPAN TABEL ROC DAN PR CURVE
# ============================================================

ensemble_roc_curve_table_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_nested_cv_roc_curve_points.csv"
)

ensemble_roc_curve_df.to_csv(
    ensemble_roc_curve_table_path,
    index=False
)


ensemble_pr_curve_table_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_precision_recall_curve_points.csv"
)

ensemble_pr_curve_df.to_csv(
    ensemble_pr_curve_table_path,
    index=False
)


# ============================================================
# 20. MENYIMPAN CONFUSION MATRIX RESULTS
# ============================================================

ensemble_confusion_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_confusion_matrix_summary.csv"
)

ensemble_confusion_matrix_summary_df.to_csv(
    ensemble_confusion_summary_path,
    index=False
)


ensemble_classification_reports_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_classification_reports_all_models.csv"
)

ensemble_classification_reports_df.to_csv(
    ensemble_classification_reports_path,
    index=False
)


ensemble_confusion_paths_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_confusion_matrix_figure_paths.csv"
)

ensemble_confusion_figure_paths_df.to_csv(
    ensemble_confusion_paths_path,
    index=False
)


best_ensemble_confusion_table_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "best_ensemble_confusion_matrix.csv"
)

best_ensemble_confusion_df.to_csv(
    best_ensemble_confusion_table_path
)


best_ensemble_classification_report_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "best_ensemble_classification_report.csv"
)

best_ensemble_classification_report_df.to_csv(
    best_ensemble_classification_report_path
)


# ============================================================
# 21. MENYIMPAN PERFORMANCE COMPARISON
# ============================================================

ensemble_performance_comparison_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_performance_comparison.csv"
)

ensemble_performance_comparison_df.to_csv(
    ensemble_performance_comparison_path,
    index=False
)


ensemble_fold_auc_pivot_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_outer_fold_roc_auc_visualization_data.csv"
)

ensemble_fold_auc_pivot_df.to_csv(
    ensemble_fold_auc_pivot_path
)


# ============================================================
# 22. RINGKASAN ARTEFAK VISUALISASI
# ============================================================

ensemble_visualization_summary_df = pd.DataFrame([
    {
        "visualization": (
            "Combined ROC curves"
        ),
        "path": (
            ensemble_combined_roc_path
        )
    },
    {
        "visualization": (
            "Combined Precision–Recall curves"
        ),
        "path": (
            ensemble_combined_pr_path
        )
    },
    {
        "visualization": (
            "Best model confusion matrix"
        ),
        "path": (
            best_ensemble_confusion_figure_path
        )
    },
    {
        "visualization": (
            "Combined performance metrics"
        ),
        "path": (
            ensemble_combined_performance_path
        )
    },
    {
        "visualization": (
            "Outer-fold ROC-AUC variability"
        ),
        "path": (
            ensemble_fold_auc_variability_path
        )
    },
    {
        "visualization": (
            "Outer-fold ROC-AUC boxplot"
        ),
        "path": (
            ensemble_fold_auc_boxplot_path
        )
    },
    {
        "visualization": (
            "Training versus OOF ROC-AUC"
        ),
        "path": (
            ensemble_training_oof_path
        )
    }
])


ensemble_visualization_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_visualization_output_summary.csv"
)

ensemble_visualization_summary_df.to_csv(
    ensemble_visualization_summary_path,
    index=False
)


# ============================================================
# 23. MENAMPILKAN RINGKASAN CONFUSION MATRIX
# ============================================================

print("\n" + "=" * 76)
print("ENSEMBLE CONFUSION MATRIX SUMMARY")
print("=" * 76)

display(
    ensemble_confusion_matrix_summary_df[
        [
            "model",
            "true_negative",
            "false_positive",
            "false_negative",
            "true_positive",
            "sensitivity",
            "specificity",
            "precision_HIGH",
            "negative_predictive_value"
        ]
    ]
)


# ============================================================
# 24. MENAMPILKAN RINGKASAN ARTEFAK
# ============================================================

print("\n" + "=" * 76)
print("ENSEMBLE VISUALIZATION OUTPUTS")
print("=" * 76)

display(
    ensemble_visualization_summary_df
)


# ============================================================
# 25. KONFIRMASI STEP 7
# ============================================================

print("\n" + "=" * 76)
print("STEP 7 COMPLETED")
print("=" * 76)

print(
    "Visualisasi performa lima model ensemble berhasil dibuat."
)

print(
    "\nBest ensemble model:"
)

print(
    best_ensemble_model_name
)

print(
    "\nBest out-of-fold ROC-AUC:"
)

print(
    f"{best_ensemble_model_auc:.4f}"
)

print(
    "\nBest-model confusion matrix:"
)

print(
    best_ensemble_confusion_figure_path
)

print(
    "\nCombined ROC curve:"
)

print(
    ensemble_combined_roc_path
)

print(
    "\nCombined Precision–Recall curve:"
)

print(
    ensemble_combined_pr_path
)

print(
    "\nSTEP berikutnya:"
)

print(
    "STEP 8 — Statistical uncertainty analysis, bootstrap "
    "confidence intervals, and model comparison."
)

In [ ]:
# ============================================================
# STEP 8. STATISTICAL UNCERTAINTY AND MODEL COMPARISON
# Bootstrap Confidence Intervals, Fold-Level Statistics,
# Friedman Test, and Pairwise Wilcoxon Tests
# ============================================================

# Tujuan STEP 8:
# 1. Menghitung mean ± standard deviation setiap metrik
#    pada outer folds.
# 2. Menghitung 95% confidence interval berbasis distribusi t.
# 3. Menghitung stratified bootstrap 95% confidence interval
#    dari out-of-fold predictions pada tingkat subject.
# 4. Membandingkan kelima model menggunakan Friedman test.
# 5. Melakukan pairwise Wilcoxon signed-rank test.
# 6. Menerapkan Holm correction untuk multiple comparisons.
#
# Catatan:
# - Outer fold hanya berjumlah 5.
# - Uji Friedman dan Wilcoxon bersifat eksploratif.
# - Bootstrap subject-level digunakan sebagai analisis
#   ketidakpastian utama.
# - Performa ilmiah tetap berasal dari out-of-fold predictions.


# ============================================================
# 1. IMPORT LIBRARY STATISTIK
# ============================================================

from itertools import combinations

from scipy.stats import (
    t,
    friedmanchisquare,
    wilcoxon
)


# ============================================================
# 2. VALIDASI OBJEK HASIL STEP 5–7
# ============================================================

required_step_8_objects = [
    "ensemble_nested_cv_results",
    "ensemble_nested_model_summary_df",
    "ensemble_nested_fold_metrics_df",
    "ensemble_nested_predictions_df",
    "ensemble_pipelines",
    "ensemble_model_abbreviations",
    "best_ensemble_model_name",
    "best_ensemble_model_auc",
    "OUTER_N_SPLITS",
    "RANDOM_STATE",
    "ENSEMBLE_TABLE_DIR",
    "ENSEMBLE_FIGURE_DIR"
]

missing_step_8_objects = [
    object_name
    for object_name in required_step_8_objects
    if object_name not in globals()
]

if missing_step_8_objects:
    raise NameError(
        "Objek untuk STEP 8 belum lengkap: "
        f"{missing_step_8_objects}. "
        "Jalankan STEP 5–7 terlebih dahulu."
    )


# ============================================================
# 3. KONFIGURASI ANALISIS
# ============================================================

ENSEMBLE_CONFIDENCE_LEVEL = 0.95

ENSEMBLE_ALPHA = (
    1.0
    - ENSEMBLE_CONFIDENCE_LEVEL
)

ENSEMBLE_BOOTSTRAP_ITERATIONS = 5000

ENSEMBLE_BOOTSTRAP_RANDOM_STATE = (
    RANDOM_STATE + 800
)

ENSEMBLE_PRIMARY_STATISTICAL_METRIC = (
    "roc_auc"
)

ENSEMBLE_STATISTICAL_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1_score",
    "roc_auc"
]


# ============================================================
# 4. FUNGSI CONFIDENCE INTERVAL BERBASIS t
# ============================================================

def calculate_ensemble_t_confidence_interval(
    values,
    confidence_level=0.95
):
    """
    Menghitung mean, standard deviation, standard error,
    dan confidence interval berbasis distribusi t.

    Parameters
    ----------
    values : array-like
        Nilai metrik dari outer folds.

    confidence_level : float
        Tingkat confidence interval.

    Returns
    -------
    dict
        Ringkasan statistik.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(
            values
        )
    ]

    sample_size = len(
        values
    )

    if sample_size == 0:

        return {
            "n_folds": 0,
            "mean": np.nan,
            "std": np.nan,
            "standard_error": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "minimum": np.nan,
            "maximum": np.nan
        }

    mean_value = float(
        np.mean(
            values
        )
    )

    minimum_value = float(
        np.min(
            values
        )
    )

    maximum_value = float(
        np.max(
            values
        )
    )

    if sample_size == 1:

        return {
            "n_folds": 1,
            "mean": mean_value,
            "std": np.nan,
            "standard_error": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "minimum": minimum_value,
            "maximum": maximum_value
        }

    standard_deviation = float(
        np.std(
            values,
            ddof=1
        )
    )

    standard_error = float(
        standard_deviation
        / np.sqrt(
            sample_size
        )
    )

    degrees_of_freedom = (
        sample_size - 1
    )

    critical_value = float(
        t.ppf(
            (
                1.0
                + confidence_level
            )
            / 2.0,
            degrees_of_freedom
        )
    )

    margin_of_error = float(
        critical_value
        * standard_error
    )

    confidence_interval_lower = float(
        mean_value
        - margin_of_error
    )

    confidence_interval_upper = float(
        mean_value
        + margin_of_error
    )

    confidence_interval_lower = max(
        0.0,
        confidence_interval_lower
    )

    confidence_interval_upper = min(
        1.0,
        confidence_interval_upper
    )

    return {
        "n_folds": int(
            sample_size
        ),
        "mean": mean_value,
        "std": standard_deviation,
        "standard_error": standard_error,
        "ci_lower": confidence_interval_lower,
        "ci_upper": confidence_interval_upper,
        "minimum": minimum_value,
        "maximum": maximum_value
    }


# ============================================================
# 5. OUTER-FOLD MEAN ± SD DAN 95% CI
# ============================================================

ensemble_fold_uncertainty_records = []

for model_name in ensemble_pipelines.keys():

    model_fold_df = (
        ensemble_nested_fold_metrics_df[
            ensemble_nested_fold_metrics_df[
                "model"
            ]
            == model_name
        ]
        .sort_values(
            by="outer_fold"
        )
        .copy()
    )

    for metric_name in (
        ENSEMBLE_STATISTICAL_METRICS
    ):

        metric_summary = (
            calculate_ensemble_t_confidence_interval(
                values=model_fold_df[
                    metric_name
                ].to_numpy(),
                confidence_level=(
                    ENSEMBLE_CONFIDENCE_LEVEL
                )
            )
        )

        ensemble_fold_uncertainty_records.append({
            "model": model_name,
            "model_abbreviation": (
                ensemble_model_abbreviations[
                    model_name
                ]
            ),
            "metric": metric_name,
            "confidence_level": (
                ENSEMBLE_CONFIDENCE_LEVEL
            ),
            **metric_summary
        })


ensemble_fold_uncertainty_df = pd.DataFrame(
    ensemble_fold_uncertainty_records
)


ensemble_fold_uncertainty_df[
    "mean_plus_minus_std"
] = (
    ensemble_fold_uncertainty_df.apply(
        lambda row: (
            f"{row['mean']:.4f} ± "
            f"{row['std']:.4f}"
        ),
        axis=1
    )
)


ensemble_fold_uncertainty_df[
    "confidence_interval_text"
] = (
    ensemble_fold_uncertainty_df.apply(
        lambda row: (
            f"[{row['ci_lower']:.4f}, "
            f"{row['ci_upper']:.4f}]"
        ),
        axis=1
    )
)


print("=" * 76)
print("ENSEMBLE OUTER-FOLD PERFORMANCE UNCERTAINTY")
print("=" * 76)

display(
    ensemble_fold_uncertainty_df[
        [
            "model",
            "metric",
            "n_folds",
            "mean_plus_minus_std",
            "standard_error",
            "confidence_interval_text",
            "minimum",
            "maximum"
        ]
    ]
)


# ============================================================
# 6. TABEL WIDE MEAN ± SD
# ============================================================

ensemble_mean_sd_performance_table = (
    ensemble_fold_uncertainty_df
    .pivot(
        index="model",
        columns="metric",
        values="mean_plus_minus_std"
    )
    .reset_index()
)

ensemble_mean_sd_performance_table.columns.name = None


print("\n" + "=" * 76)
print("ENSEMBLE MEAN ± STANDARD DEVIATION")
print("=" * 76)

display(
    ensemble_mean_sd_performance_table
)


# ============================================================
# 7. FUNGSI METRIK UNTUK BOOTSTRAP
# ============================================================

def calculate_ensemble_bootstrap_metrics(
    y_true,
    y_pred,
    y_score
):
    """
    Menghitung metrik utama untuk satu bootstrap sample.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int
    )

    y_score = np.asarray(
        y_score,
        dtype=float
    )

    bootstrap_confusion_matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1
        ]
    )

    tn, fp, fn, tp = (
        bootstrap_confusion_matrix
        .ravel()
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    if len(
        np.unique(
            y_true
        )
    ) == 2:

        roc_auc = float(
            roc_auc_score(
                y_true,
                y_score
            )
        )

    else:

        roc_auc = np.nan

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred
            )
        ),

        "precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            )
        ),

        "sensitivity": float(
            sensitivity
        ),

        "specificity": float(
            specificity
        ),

        "f1_score": float(
            f1_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            )
        ),

        "roc_auc": (
            roc_auc
        )
    }


# ============================================================
# 8. STRATIFIED SUBJECT-LEVEL BOOTSTRAP
# ============================================================

def stratified_ensemble_bootstrap(
    y_true,
    y_pred,
    y_score,
    n_iterations=5000,
    confidence_level=0.95,
    random_state=42
):
    """
    Menghitung confidence interval dengan stratified bootstrap.

    Sampling dilakukan secara terpisah untuk:
    - LOW/NO = 0
    - HIGH = 1

    Stratifikasi menjaga kedua kelas tersedia pada setiap
    bootstrap sample.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int
    )

    y_score = np.asarray(
        y_score,
        dtype=float
    )

    if not (
        len(y_true)
        == len(y_pred)
        == len(y_score)
    ):
        raise ValueError(
            "Panjang y_true, y_pred, dan y_score harus sama."
        )

    class_zero_indices = np.where(
        y_true == 0
    )[0]

    class_one_indices = np.where(
        y_true == 1
    )[0]

    if (
        len(class_zero_indices) == 0
        or len(class_one_indices) == 0
    ):
        raise ValueError(
            "Bootstrap membutuhkan kedua kelas."
        )

    random_generator = np.random.default_rng(
        random_state
    )

    bootstrap_metric_records = []

    for bootstrap_iteration in range(
        n_iterations
    ):

        sampled_class_zero_indices = (
            random_generator.choice(
                class_zero_indices,
                size=len(
                    class_zero_indices
                ),
                replace=True
            )
        )

        sampled_class_one_indices = (
            random_generator.choice(
                class_one_indices,
                size=len(
                    class_one_indices
                ),
                replace=True
            )
        )

        bootstrap_indices = np.concatenate(
            [
                sampled_class_zero_indices,
                sampled_class_one_indices
            ]
        )

        random_generator.shuffle(
            bootstrap_indices
        )

        bootstrap_metrics = (
            calculate_ensemble_bootstrap_metrics(
                y_true=y_true[
                    bootstrap_indices
                ],
                y_pred=y_pred[
                    bootstrap_indices
                ],
                y_score=y_score[
                    bootstrap_indices
                ]
            )
        )

        bootstrap_metric_records.append(
            bootstrap_metrics
        )


    bootstrap_distribution_df = pd.DataFrame(
        bootstrap_metric_records
    )


    lower_percentile = (
        (
            1.0
            - confidence_level
        )
        / 2.0
        * 100
    )

    upper_percentile = (
        (
            1.0
            + confidence_level
        )
        / 2.0
        * 100
    )


    observed_metrics = (
        calculate_ensemble_bootstrap_metrics(
            y_true=y_true,
            y_pred=y_pred,
            y_score=y_score
        )
    )


    bootstrap_summary_records = []

    for metric_name in (
        ENSEMBLE_STATISTICAL_METRICS
    ):

        valid_bootstrap_values = (
            bootstrap_distribution_df[
                metric_name
            ]
            .dropna()
            .to_numpy()
        )

        bootstrap_summary_records.append({
            "metric": metric_name,

            "observed_value": float(
                observed_metrics[
                    metric_name
                ]
            ),

            "bootstrap_mean": float(
                np.mean(
                    valid_bootstrap_values
                )
            ),

            "bootstrap_std": float(
                np.std(
                    valid_bootstrap_values,
                    ddof=1
                )
            ),

            "ci_lower": float(
                np.percentile(
                    valid_bootstrap_values,
                    lower_percentile
                )
            ),

            "ci_upper": float(
                np.percentile(
                    valid_bootstrap_values,
                    upper_percentile
                )
            ),

            "valid_bootstrap_samples": int(
                len(
                    valid_bootstrap_values
                )
            )
        })


    return (
        pd.DataFrame(
            bootstrap_summary_records
        ),
        bootstrap_distribution_df
    )


# ============================================================
# 9. BOOTSTRAP SELURUH MODEL
# ============================================================

ensemble_bootstrap_summary_frames = []

ensemble_bootstrap_distribution_frames = []


for model_index, model_name in enumerate(
    ensemble_pipelines.keys(),
    start=1
):

    model_result = (
        ensemble_nested_cv_results[
            model_name
        ]
    )

    (
        model_bootstrap_summary_df,
        model_bootstrap_distribution_df
    ) = stratified_ensemble_bootstrap(
        y_true=model_result[
            "y_true"
        ],
        y_pred=model_result[
            "y_pred"
        ],
        y_score=model_result[
            "y_score"
        ],
        n_iterations=(
            ENSEMBLE_BOOTSTRAP_ITERATIONS
        ),
        confidence_level=(
            ENSEMBLE_CONFIDENCE_LEVEL
        ),
        random_state=(
            ENSEMBLE_BOOTSTRAP_RANDOM_STATE
            + model_index
        )
    )


    model_bootstrap_summary_df.insert(
        0,
        "model_abbreviation",
        ensemble_model_abbreviations[
            model_name
        ]
    )

    model_bootstrap_summary_df.insert(
        0,
        "model",
        model_name
    )


    model_bootstrap_distribution_df.insert(
        0,
        "bootstrap_iteration",
        np.arange(
            1,
            len(
                model_bootstrap_distribution_df
            ) + 1
        )
    )

    model_bootstrap_distribution_df.insert(
        0,
        "model",
        model_name
    )


    ensemble_bootstrap_summary_frames.append(
        model_bootstrap_summary_df
    )

    ensemble_bootstrap_distribution_frames.append(
        model_bootstrap_distribution_df
    )


ensemble_bootstrap_confidence_interval_df = (
    pd.concat(
        ensemble_bootstrap_summary_frames,
        ignore_index=True
    )
)


ensemble_bootstrap_distribution_df = (
    pd.concat(
        ensemble_bootstrap_distribution_frames,
        ignore_index=True
    )
)


ensemble_bootstrap_confidence_interval_df[
    "confidence_interval_text"
] = (
    ensemble_bootstrap_confidence_interval_df.apply(
        lambda row: (
            f"[{row['ci_lower']:.4f}, "
            f"{row['ci_upper']:.4f}]"
        ),
        axis=1
    )
)


print("\n" + "=" * 76)
print("ENSEMBLE STRATIFIED BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 76)

display(
    ensemble_bootstrap_confidence_interval_df[
        [
            "model",
            "metric",
            "observed_value",
            "bootstrap_mean",
            "bootstrap_std",
            "confidence_interval_text",
            "valid_bootstrap_samples"
        ]
    ]
)


# ============================================================
# 10. BOOTSTRAP ROC-AUC SUMMARY
# ============================================================

ensemble_bootstrap_auc_summary_df = (
    ensemble_bootstrap_confidence_interval_df[
        ensemble_bootstrap_confidence_interval_df[
            "metric"
        ]
        == "roc_auc"
    ]
    .sort_values(
        by="observed_value",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 76)
print("ENSEMBLE BOOTSTRAP ROC-AUC SUMMARY")
print("=" * 76)

display(
    ensemble_bootstrap_auc_summary_df[
        [
            "model",
            "observed_value",
            "bootstrap_mean",
            "bootstrap_std",
            "ci_lower",
            "ci_upper",
            "confidence_interval_text"
        ]
    ]
)


# ============================================================
# 11. MATRIKS OUTER-FOLD ROC-AUC
# ============================================================

ensemble_primary_metric_pivot_df = (
    ensemble_nested_fold_metrics_df
    .pivot(
        index="outer_fold",
        columns="model",
        values=(
            ENSEMBLE_PRIMARY_STATISTICAL_METRIC
        )
    )
    .sort_index()
)


ensemble_primary_metric_pivot_df = (
    ensemble_primary_metric_pivot_df[
        list(
            ensemble_pipelines.keys()
        )
    ]
)


print("\n" + "=" * 76)
print("ENSEMBLE OUTER-FOLD ROC-AUC MATRIX")
print("=" * 76)

display(
    ensemble_primary_metric_pivot_df
)


# ============================================================
# 12. FRIEDMAN TEST
# ============================================================

ensemble_friedman_statistic = np.nan

ensemble_friedman_p_value = np.nan

ensemble_friedman_interpretation = (
    "Not available"
)


if (
    ensemble_primary_metric_pivot_df.shape[1]
    >= 3
    and not ensemble_primary_metric_pivot_df
    .isna()
    .any()
    .any()
):

    ensemble_friedman_result = (
        friedmanchisquare(
            *[
                ensemble_primary_metric_pivot_df[
                    model_name
                ].to_numpy()
                for model_name in (
                    ensemble_primary_metric_pivot_df.columns
                )
            ]
        )
    )

    ensemble_friedman_statistic = float(
        ensemble_friedman_result.statistic
    )

    ensemble_friedman_p_value = float(
        ensemble_friedman_result.pvalue
    )

    if (
        ensemble_friedman_p_value
        < ENSEMBLE_ALPHA
    ):

        ensemble_friedman_interpretation = (
            "Significant difference among ensemble models"
        )

    else:

        ensemble_friedman_interpretation = (
            "No statistically significant difference"
        )


ensemble_friedman_result_df = pd.DataFrame([
    {
        "test": "Friedman test",

        "metric": (
            ENSEMBLE_PRIMARY_STATISTICAL_METRIC
        ),

        "number_of_models": int(
            ensemble_primary_metric_pivot_df.shape[1]
        ),

        "number_of_outer_folds": int(
            ensemble_primary_metric_pivot_df.shape[0]
        ),

        "statistic": (
            ensemble_friedman_statistic
        ),

        "p_value": (
            ensemble_friedman_p_value
        ),

        "alpha": (
            ENSEMBLE_ALPHA
        ),

        "interpretation": (
            ensemble_friedman_interpretation
        )
    }
])


print("\n" + "=" * 76)
print("ENSEMBLE FRIEDMAN TEST")
print("=" * 76)

display(
    ensemble_friedman_result_df
)


# ============================================================
# 13. PAIRWISE WILCOXON SIGNED-RANK TEST
# ============================================================

ensemble_pairwise_wilcoxon_records = []

ensemble_model_pairs = list(
    combinations(
        ensemble_primary_metric_pivot_df.columns,
        2
    )
)


for model_a, model_b in (
    ensemble_model_pairs
):

    model_a_values = (
        ensemble_primary_metric_pivot_df[
            model_a
        ]
        .to_numpy(
            dtype=float
        )
    )

    model_b_values = (
        ensemble_primary_metric_pivot_df[
            model_b
        ]
        .to_numpy(
            dtype=float
        )
    )

    valid_pair_mask = (
        np.isfinite(
            model_a_values
        )
        & np.isfinite(
            model_b_values
        )
    )

    model_a_valid = (
        model_a_values[
            valid_pair_mask
        ]
    )

    model_b_valid = (
        model_b_values[
            valid_pair_mask
        ]
    )

    paired_differences = (
        model_a_valid
        - model_b_valid
    )


    wilcoxon_statistic = np.nan

    wilcoxon_p_value = np.nan

    test_note = ""


    if len(
        model_a_valid
    ) < 2:

        test_note = (
            "Insufficient paired observations"
        )

    elif np.allclose(
        paired_differences,
        0.0
    ):

        wilcoxon_statistic = 0.0

        wilcoxon_p_value = 1.0

        test_note = (
            "All paired differences are zero"
        )

    else:

        try:

            wilcoxon_result = wilcoxon(
                model_a_valid,
                model_b_valid,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto"
            )

            wilcoxon_statistic = float(
                wilcoxon_result.statistic
            )

            wilcoxon_p_value = float(
                wilcoxon_result.pvalue
            )

            test_note = "Completed"

        except ValueError as error:

            test_note = str(
                error
            )


    ensemble_pairwise_wilcoxon_records.append({
        "model_a": model_a,

        "model_b": model_b,

        "metric": (
            ENSEMBLE_PRIMARY_STATISTICAL_METRIC
        ),

        "number_of_pairs": int(
            len(
                model_a_valid
            )
        ),

        "mean_model_a": float(
            np.mean(
                model_a_valid
            )
        ),

        "mean_model_b": float(
            np.mean(
                model_b_valid
            )
        ),

        "mean_difference_a_minus_b": float(
            np.mean(
                paired_differences
            )
        ),

        "median_difference_a_minus_b": float(
            np.median(
                paired_differences
            )
        ),

        "wilcoxon_statistic": (
            wilcoxon_statistic
        ),

        "raw_p_value": (
            wilcoxon_p_value
        ),

        "test_note": (
            test_note
        )
    })


ensemble_pairwise_wilcoxon_df = pd.DataFrame(
    ensemble_pairwise_wilcoxon_records
)


# ============================================================
# 14. HOLM MULTIPLE-COMPARISON CORRECTION
# ============================================================

def ensemble_holm_adjust_p_values(
    p_values
):
    """
    Melakukan Holm step-down correction.
    """

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    adjusted_p_values = np.full(
        shape=len(
            p_values
        ),
        fill_value=np.nan,
        dtype=float
    )

    valid_indices = np.where(
        np.isfinite(
            p_values
        )
    )[0]

    if len(
        valid_indices
    ) == 0:

        return adjusted_p_values


    valid_p_values = (
        p_values[
            valid_indices
        ]
    )

    sorted_local_indices = np.argsort(
        valid_p_values
    )

    number_of_tests = len(
        valid_p_values
    )

    running_maximum = 0.0


    for rank_position, local_index in enumerate(
        sorted_local_indices
    ):

        multiplier = (
            number_of_tests
            - rank_position
        )

        adjusted_value = min(
            1.0,
            valid_p_values[
                local_index
            ]
            * multiplier
        )

        running_maximum = max(
            running_maximum,
            adjusted_value
        )

        original_index = (
            valid_indices[
                local_index
            ]
        )

        adjusted_p_values[
            original_index
        ] = running_maximum


    return adjusted_p_values


ensemble_pairwise_wilcoxon_df[
    "holm_adjusted_p_value"
] = (
    ensemble_holm_adjust_p_values(
        ensemble_pairwise_wilcoxon_df[
            "raw_p_value"
        ].to_numpy()
    )
)


ensemble_pairwise_wilcoxon_df[
    "significant_after_holm"
] = (
    ensemble_pairwise_wilcoxon_df[
        "holm_adjusted_p_value"
    ]
    < ENSEMBLE_ALPHA
)


ensemble_pairwise_wilcoxon_df[
    "interpretation"
] = np.where(
    ensemble_pairwise_wilcoxon_df[
        "significant_after_holm"
    ],
    "Statistically significant",
    "Not statistically significant"
)


print("\n" + "=" * 76)
print("PAIRWISE WILCOXON TESTS WITH HOLM CORRECTION")
print("=" * 76)

display(
    ensemble_pairwise_wilcoxon_df
)


# ============================================================
# 15. MODEL RANKING PER OUTER FOLD
# ============================================================

ensemble_fold_rank_df = (
    ensemble_primary_metric_pivot_df
    .rank(
        axis=1,
        method="average",
        ascending=False
    )
)


ensemble_mean_model_rank_df = (
    ensemble_fold_rank_df
    .mean(
        axis=0
    )
    .rename(
        "mean_rank"
    )
    .reset_index()
)


ensemble_mean_model_rank_df.columns = [
    "model",
    "mean_rank"
]


ensemble_mean_model_rank_df = (
    ensemble_mean_model_rank_df
    .sort_values(
        by="mean_rank",
        ascending=True
    )
    .reset_index(drop=True)
)


ensemble_mean_model_rank_df.insert(
    0,
    "rank_position",
    np.arange(
        1,
        len(
            ensemble_mean_model_rank_df
        ) + 1
    )
)


print("\n" + "=" * 76)
print("MEAN ENSEMBLE MODEL RANK ACROSS OUTER FOLDS")
print("=" * 76)

display(
    ensemble_mean_model_rank_df
)


# ============================================================
# 16. RINGKASAN FINAL DENGAN BOOTSTRAP CI
# ============================================================

ensemble_final_uncertainty_summary_df = (
    ensemble_nested_model_summary_df[
        [
            "performance_rank",
            "model",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "sensitivity",
            "specificity",
            "f1_score",
            "roc_auc",
            "mean_training_auc",
            "mean_train_test_auc_gap"
        ]
    ]
    .merge(
        ensemble_bootstrap_auc_summary_df[
            [
                "model",
                "ci_lower",
                "ci_upper",
                "bootstrap_std"
            ]
        ].rename(
            columns={
                "ci_lower": (
                    "roc_auc_bootstrap_ci_lower"
                ),
                "ci_upper": (
                    "roc_auc_bootstrap_ci_upper"
                ),
                "bootstrap_std": (
                    "roc_auc_bootstrap_std"
                )
            }
        ),
        on="model",
        how="left"
    )
    .merge(
        ensemble_mean_model_rank_df[
            [
                "model",
                "mean_rank"
            ]
        ],
        on="model",
        how="left"
    )
)


ensemble_final_uncertainty_summary_df[
    "roc_auc_95_ci"
] = (
    ensemble_final_uncertainty_summary_df.apply(
        lambda row: (
            f"[{row['roc_auc_bootstrap_ci_lower']:.4f}, "
            f"{row['roc_auc_bootstrap_ci_upper']:.4f}]"
        ),
        axis=1
    )
)


print("\n" + "=" * 76)
print("FINAL ENSEMBLE PERFORMANCE WITH UNCERTAINTY")
print("=" * 76)

display(
    ensemble_final_uncertainty_summary_df[
        [
            "performance_rank",
            "model",
            "accuracy",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "f1_score",
            "roc_auc",
            "roc_auc_95_ci",
            "mean_rank",
            "mean_train_test_auc_gap"
        ]
    ]
)


# ============================================================
# 17. VISUALISASI BOOTSTRAP ROC-AUC CI
# ============================================================

ensemble_auc_ci_plot_df = (
    ensemble_bootstrap_auc_summary_df
    .sort_values(
        by="observed_value",
        ascending=False
    )
    .copy()
)


ensemble_auc_ci_plot_df[
    "lower_error"
] = (
    ensemble_auc_ci_plot_df[
        "observed_value"
    ]
    - ensemble_auc_ci_plot_df[
        "ci_lower"
    ]
)


ensemble_auc_ci_plot_df[
    "upper_error"
] = (
    ensemble_auc_ci_plot_df[
        "ci_upper"
    ]
    - ensemble_auc_ci_plot_df[
        "observed_value"
    ]
)


plt.figure(
    figsize=(10, 6)
)

plt.errorbar(
    x=ensemble_auc_ci_plot_df[
        "model"
    ],
    y=ensemble_auc_ci_plot_df[
        "observed_value"
    ],
    yerr=[
        ensemble_auc_ci_plot_df[
            "lower_error"
        ],
        ensemble_auc_ci_plot_df[
            "upper_error"
        ]
    ],
    fmt="o",
    capsize=5
)

plt.xlabel(
    "Ensemble Model"
)

plt.ylabel(
    "Out-of-Fold ROC-AUC"
)

plt.title(
    "Bootstrap 95% Confidence Intervals of ROC-AUC"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.ylim(
    0.0,
    1.05
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()


ensemble_auc_ci_figure_path = os.path.join(
    ENSEMBLE_FIGURE_DIR,
    "ensemble_bootstrap_roc_auc_confidence_intervals.png"
)

plt.savefig(
    ensemble_auc_ci_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 18. MENYIMPAN HASIL STATISTIK
# ============================================================

ensemble_fold_uncertainty_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_outer_fold_performance_uncertainty.csv"
)

ensemble_fold_uncertainty_df.to_csv(
    ensemble_fold_uncertainty_path,
    index=False
)


ensemble_mean_sd_table_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_outer_fold_mean_std_table.csv"
)

ensemble_mean_sd_performance_table.to_csv(
    ensemble_mean_sd_table_path,
    index=False
)


ensemble_bootstrap_ci_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_bootstrap_confidence_intervals.csv"
)

ensemble_bootstrap_confidence_interval_df.to_csv(
    ensemble_bootstrap_ci_path,
    index=False
)


ensemble_bootstrap_distribution_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_bootstrap_metric_distributions.csv"
)

ensemble_bootstrap_distribution_df.to_csv(
    ensemble_bootstrap_distribution_path,
    index=False
)


ensemble_friedman_result_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_friedman_test_results.csv"
)

ensemble_friedman_result_df.to_csv(
    ensemble_friedman_result_path,
    index=False
)


ensemble_pairwise_wilcoxon_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_pairwise_wilcoxon_holm_results.csv"
)

ensemble_pairwise_wilcoxon_df.to_csv(
    ensemble_pairwise_wilcoxon_path,
    index=False
)


ensemble_mean_model_rank_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_mean_model_rank_outer_folds.csv"
)

ensemble_mean_model_rank_df.to_csv(
    ensemble_mean_model_rank_path,
    index=False
)


ensemble_final_uncertainty_summary_path = os.path.join(
    ENSEMBLE_TABLE_DIR,
    "ensemble_final_performance_with_uncertainty.csv"
)

ensemble_final_uncertainty_summary_df.to_csv(
    ensemble_final_uncertainty_summary_path,
    index=False
)


# ============================================================
# 19. RINGKASAN MODEL TERBAIK
# ============================================================

best_ensemble_uncertainty_row = (
    ensemble_final_uncertainty_summary_df[
        ensemble_final_uncertainty_summary_df[
            "model"
        ]
        == best_ensemble_model_name
    ]
    .iloc[0]
)


print("\n" + "=" * 76)
print("BEST ENSEMBLE MODEL WITH UNCERTAINTY")
print("=" * 76)

print(
    "Model              :",
    best_ensemble_model_name
)

print(
    "ROC-AUC            :",
    f"{best_ensemble_uncertainty_row['roc_auc']:.4f}"
)

print(
    "Bootstrap 95% CI   :",
    best_ensemble_uncertainty_row[
        "roc_auc_95_ci"
    ]
)

print(
    "Balanced accuracy  :",
    f"{best_ensemble_uncertainty_row['balanced_accuracy']:.4f}"
)

print(
    "Sensitivity        :",
    f"{best_ensemble_uncertainty_row['sensitivity']:.4f}"
)

print(
    "Specificity        :",
    f"{best_ensemble_uncertainty_row['specificity']:.4f}"
)

print(
    "F1-score           :",
    f"{best_ensemble_uncertainty_row['f1_score']:.4f}"
)

print(
    "Mean model rank    :",
    f"{best_ensemble_uncertainty_row['mean_rank']:.4f}"
)

print(
    "Train-test AUC gap :",
    f"{best_ensemble_uncertainty_row['mean_train_test_auc_gap']:.4f}"
)


# ============================================================
# 20. CATATAN INTERPRETASI
# ============================================================

print("\n" + "=" * 76)
print("STATISTICAL INTERPRETATION NOTES")
print("=" * 76)

print(
    "1. Outer-fold analysis hanya menggunakan "
    f"{OUTER_N_SPLITS} fold."
)

print(
    "2. Bootstrap menggunakan "
    f"{ENSEMBLE_BOOTSTRAP_ITERATIONS} stratified "
    "subject-level resamples."
)

print(
    "3. Friedman dan Wilcoxon tests bersifat eksploratif "
    "karena jumlah outer folds kecil."
)

print(
    "4. Training sets antar-fold saling tumpang tindih, "
    "sehingga fold bukan observasi independen sempurna."
)

print(
    "5. Model terbaik tidak boleh dipilih hanya berdasarkan "
    "p-value."
)

print(
    "6. Pertimbangkan ROC-AUC, bootstrap CI, balanced "
    "accuracy, sensitivity, specificity, F1-score, "
    "dan train-test gap secara bersama-sama."
)


# ============================================================
# 21. KONFIRMASI STEP 8
# ============================================================

print("\n" + "=" * 76)
print("STEP 8 COMPLETED")
print("=" * 76)

print(
    "Analisis uncertainty dan perbandingan statistik "
    "kelima model ensemble berhasil dilakukan."
)

print(
    "\nBest ensemble model:"
)

print(
    best_ensemble_model_name
)

print(
    "\nBest ensemble ROC-AUC:"
)

print(
    f"{best_ensemble_model_auc:.4f}"
)

print(
    "\nFinal uncertainty summary saved to:"
)

print(
    ensemble_final_uncertainty_summary_path
)

print(
    "\nSTEP berikutnya:"
)

print(
    "STEP 9 — Select and train the final ensemble model "
    "using full-data GridSearchCV for deployment."
)

MLP